# 4\.6\.1 Compare Forecasting Model Families

The previous forecasting notebooks answered four separate questions:

- What is the strongest **classical time-series** forecasting system?
- What is the strongest **global panel-tree** forecasting system?
- What is the strongest **neural sequence** forecasting system?
- What is the strongest **Transformer** forecasting system?

This notebook asks the next question:

> **Which finalized family provides the strongest deployable forecasting system for each Target × Horizon?**

The comparison remains downstream of the family-specific tournaments. We do not retune SARIMAX models, reopen LightGBM development, revisit the neural architecture tournament, or alter the frozen Transformer design.

The decisive model-selection evidence comes from **ordinary rolling validation**. For each Target × Horizon, the notebook constructs one benchmark-supported physical validation population. A family uses its advanced forecast wherever that forecast is available and falls back to **Previous week · same Daypart** wherever it is not.

This makes forecast coverage part of the comparison rather than allowing unavailable predictions to disappear from the denominator.

The notebook also retains the exact rows where all four advanced models are available. That exact-common surface answers the narrower diagnostic question of which advanced model is most accurate **conditional on all four being able to forecast**; it is not the primary selector.

The main evidence therefore includes:

- **Coverage-adjusted system MAE** — the primary selector within each Target × Horizon.
- **Advanced-model coverage** — how often the family can use its advanced forecast rather than fallback.
- **Exact-common MAE** — conditional advanced-model accuracy on the shared four-family subset.
- **System Relative MAE (%)** — error size relative to observed mobility.
- **System benchmark skill** — improvement over Previous week · same Daypart.
- **Paired uncertainty and stability evidence** — whether the apparent system-level advantage is convincing enough to support selection.

Only ordinary-validation evidence participates in model selection. January–March 2026 final-holdout performance is reserved for the later generalization test.

In [1]:
!pip install lightgbm==4.7.0
!pip install statsmodels==0.15.0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Notebook Setup

All notebook-wide imports, paths, forecasting constants, execution controls, family handoff definitions, statistical settings, and reusable helpers are defined here before Section 4.6.1.1 begins.

The setup follows two principles.

First, completed upstream work is treated as an **authoritative contract**. This notebook reads the finalized registries, row-level ordinary-validation predictions, QA files, and manifests exported by 4.3.1–4.5.2 rather than reconstructing their family-specific tournaments.

Second, the cross-family row-level handoffs contain millions of forecasts. Large joins therefore remain in **DuckDB and Parquet**, while Pandas receives only compact audit tables, scorecards, weekly bootstrap summaries, and stability diagnostics. This keeps the notebook suitable for the available Deepnote memory while preserving exact forecast-row comparisons.

The first setup block imports only the analytical tools needed for cross-family evaluation. No model-training libraries are required yet because this part of 4.6.1 consumes completed validation forecasts rather than fitting new models.

In [2]:
# ---------------------------------------------------------------------
# 4.6.1.0A — Imports
# ---------------------------------------------------------------------

import os
import uuid
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
from plotly.subplots import make_subplots

from project_branding import (
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_SEQUENTIAL_HEATMAP_SCALE,
    BRAND_COLORS,
    apply_branding,
)


# Reader-facing tables stay compact without reducing the precision retained
# in analytical objects or written artifacts.
pd.options.display.float_format = "{:,.3f}".format

The next block freezes the Chapter 4 forecasting universe and the selection rules **before** any cross-family performance result is inspected.

Two execution controls deserve special attention:

- `RUN_COMMON_SUPPORT_BUILD` governs the potentially heavier four-way row intersection.
- `RUN_PAIRWISE_BOOTSTRAP` governs the repeated time-block uncertainty analysis.

Both are restartable. Existing valid intermediates are recovered automatically; missing expensive artifacts are never silently rebuilt unless the corresponding `RUN_*` control is enabled.

In [3]:
# ---------------------------------------------------------------------
# 4.6.1.0B — Paths, scope, execution controls, and selection constants
# ---------------------------------------------------------------------

PROJECT_ROOT = Path(".")
PIPELINE_DIR = PROJECT_ROOT / "pipeline_data"

INPUT_412_FINAL_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INPUT_424_FINAL_DIR = PIPELINE_DIR / "4.2.4.final_tables"

INPUT_431_FINAL_DIR = PIPELINE_DIR / "4.3.1.final_tables"
INPUT_441_FINAL_DIR = PIPELINE_DIR / "4.4.1.final_tables"
INPUT_451_FINAL_DIR = PIPELINE_DIR / "4.5.1.final_tables"
INPUT_452_FINAL_DIR = PIPELINE_DIR / "4.5.2.final_tables"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.6.1.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.6.1.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Frozen upstream contracts
# ---------------------------------------------------------------------

TRACK_CONTRACT_PATH = (
    INPUT_412_FINAL_DIR / "forecasting_track_contract_registry.parquet"
)
ADVANCED_BENCHMARK_REGISTRY_PATH = (
    INPUT_424_FINAL_DIR / "advanced_model_benchmark_registry.parquet"
)

POLICY_START_DATE = pd.Timestamp("2025-01-05")
FINAL_HOLDOUT_START_DATE = pd.Timestamp("2026-01-05")

ORDINARY_TRACKS = [
    "post_cp_primary",
    "full_history_regime_aware",
]

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
}

FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "subway_ridership": "Subway ridership",
}

TARGET_SHORT_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway",
}

FORECAST_HORIZONS = [1, 2, 5]
EXPECTED_JOB_COUNT = len(FORECAST_TARGETS) * len(FORECAST_HORIZONS)

EXPECTED_BENCHMARK_LABEL = "Previous week · same Daypart"

# Selection evaluates the deployable forecasting system: use the advanced
# prediction when it exists and the frozen benchmark when it does not.
SELECTION_PREDICTION_FIELD = "system_prediction"
SELECTION_METRIC = "coverage_adjusted_system_mae"


# ---------------------------------------------------------------------
# Statistical selection contract
# ---------------------------------------------------------------------
# A difference smaller than 1% of the frozen benchmark's MAE is treated as
# practically small enough to require stronger evidence before declaring that
# one finalist is meaningfully superior.

PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION = 0.01
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_REPEATS = 5_000
BOOTSTRAP_RANDOM_SEED = 696

OUTPUT_RULE_TOLERANCE = 1e-8

# The exact-common artifact remains useful for conditional-accuracy diagnosis.
# The primary selection surface has a separate contract because unavailable
# advanced forecasts remain in the denominator and receive benchmark fallback.
COMMON_VALIDATION_CONTRACT_ID = "four_family_common_validation_v2"
SELECTION_VALIDATION_CONTRACT_ID = (
    "four_family_coverage_adjusted_validation_v1"
)

CROSS_ARTIFACT_ATOL = 5e-6
CROSS_ARTIFACT_RTOL = 1e-7


# ---------------------------------------------------------------------
# Restartable execution controls
# ---------------------------------------------------------------------

RUN_COMMON_SUPPORT_BUILD = False
RESET_COMMON_SUPPORT = False

RUN_SELECTION_SURFACE_BUILD = False
RESET_SELECTION_SURFACE = False

RUN_PAIRWISE_BOOTSTRAP = False
RESET_PAIRWISE_BOOTSTRAP = False

WRITE_INTERMEDIATES = True


# ---------------------------------------------------------------------
# 4.6.1 intermediate outputs used before the first Decision Point
# ---------------------------------------------------------------------

# Secondary diagnostic: rows where all four advanced models are available.
COMMON_VALIDATION_PATH = (
    INTERMEDIATE_DIR / "cross_family_common_validation.parquet"
)
COMMON_VALIDATION_AUDIT_PATH = (
    INTERMEDIATE_DIR / "cross_family_common_validation_recovery_audit.parquet"
)

VALIDATION_SUPPORT_SUMMARY_PATH = (
    INTERMEDIATE_DIR / "cross_family_validation_support_summary.parquet"
)

# Primary selection surface: one benchmark-supported population, with model
# coverage and benchmark fallback represented explicitly for every family.
SELECTION_VALIDATION_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_validation.parquet"
)
SELECTION_VALIDATION_AUDIT_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_validation_recovery_audit.parquet"
)

SELECTION_COVERAGE_SUMMARY_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_support_summary.parquet"
)
VALIDATION_SCORECARD_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_scorecard.parquet"
)
WEEKLY_ERROR_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_weekly_error_summary.parquet"
)
MONTHLY_STABILITY_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_monthly_stability.parquet"
)
ZONE_STABILITY_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_zone_stability.parquet"
)
PAIRWISE_BOOTSTRAP_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_pairwise_bootstrap.parquet"
)
SELECTION_SUPPORT_PATH = (
    INTERMEDIATE_DIR / "cross_family_coverage_adjusted_selection_support.parquet"
)

Each forecasting branch exposes a different family-specific registry, but the downstream role is the same: **one frozen 15-job representative system plus one row-level ordinary-validation forecast handoff**.

The definitions below record those authoritative entry points. They also document the small schema differences that 4.6.1 must normalize—for example, the neural handoff names its observed target `target_observed_value`, whereas the other finalized row-level handoffs expose the same quantity as `actual`.

In [4]:
# ---------------------------------------------------------------------
# 4.6.1.0C — Authoritative family handoff contracts
# ---------------------------------------------------------------------

FAMILY_ORDER = ["classical", "tree", "neural", "transformer"]

FAMILY_LABELS = {
    "classical": "Classical",
    "tree": "Tree",
    "neural": "Neural",
    "transformer": "Transformer",
}

FAMILY_REPRESENTATIVES = {
    "classical": "SARIMAX system",
    "tree": "LightGBM",
    "neural": "Lag-window MLP system",
    "transformer": "PatchTST-inspired Transformer",
}

FAMILY_COLORS = {
    family: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
    for index, family in enumerate(FAMILY_ORDER)
}


# Registry and prediction artifacts do not always use identical physical
# column names. Keep each source contract explicit, then normalize them only
# after the file itself has been validated.
FAMILY_HANDOFFS = {
    "classical": {
        "registry": (
            INPUT_431_FINAL_DIR
            / "classical_final_winner_registry.parquet"
        ),
        "predictions": (
            INPUT_431_FINAL_DIR
            / "classical_selected_validation_predictions.parquet"
        ),
        "qa": (
            INPUT_431_FINAL_DIR
            / "final_classical_model_selection_qa.parquet"
        ),
        "manifest": (
            INPUT_431_FINAL_DIR
            / "classical_model_selection_manifest.parquet"
        ),
        "registry_track_column": "modeling_track",
        "prediction_track_column": "modeling_track",
        "actual_column": "actual",
        "configuration_columns": [
            "candidate_label",
            "representation",
        ],
    },
    "tree": {
        "registry": (
            INPUT_441_FINAL_DIR
            / "lgbm_selected_model_registry.parquet"
        ),
        "predictions": (
            INPUT_441_FINAL_DIR
            / "lgbm_selected_row_level_validation_predictions.parquet"
        ),
        "qa": (
            INPUT_441_FINAL_DIR
            / "lgbm_selected_qa.parquet"
        ),
        "manifest": (
            INPUT_441_FINAL_DIR
            / "lgbm_selected_manifest.parquet"
        ),
        "registry_track_column": "modeling_track",
        "prediction_track_column": "modeling_track",
        "actual_column": "actual",
        "configuration_columns": [
            "production_config_id",
            "target_representation",
        ],
    },
    "neural": {
        "registry": (
            INPUT_451_FINAL_DIR
            / "neural_final_model_registry.parquet"
        ),
        "predictions": (
            INPUT_451_FINAL_DIR
            / "neural_selected_validation_predictions.parquet"
        ),
        "qa": (
            INPUT_451_FINAL_DIR
            / "neural_closeout_qa.parquet"
        ),
        "manifest": (
            INPUT_451_FINAL_DIR
            / "neural_handoff_manifest.parquet"
        ),
        "registry_track_column": "modeling_track",
        "prediction_track_column": "modeling_track",
        "actual_column": "target_observed_value",
        "configuration_columns": [
            "profile_id",
            "model_definition_id",
        ],
    },
    "transformer": {
        "registry": (
            INPUT_452_FINAL_DIR
            / "transformer_final_model_registry.parquet"
        ),
        "predictions": (
            INPUT_452_FINAL_DIR
            / "transformer_selected_validation_predictions.parquet"
        ),
        "qa": (
            INPUT_452_FINAL_DIR
            / "transformer_closeout_qa.parquet"
        ),
        "manifest": (
            INPUT_452_FINAL_DIR
            / "transformer_handoff_manifest.parquet"
        ),

        # The Transformer registry preserves its internal `track` field,
        # while the downstream row-level prediction handoff standardizes the
        # same concept to `modeling_track`. Treat those as separate contracts.
        "registry_track_column": "track",
        "prediction_track_column": "modeling_track",
        "actual_column": "actual",
        "configuration_columns": [
            "representation_id",
            "profile_id",
            "model_definition_id",
        ],
    },
}


COMMON_FORECAST_KEY_COLUMNS = [
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

COMMON_PREDICTION_COLUMNS = [
    *COMMON_FORECAST_KEY_COLUMNS,
    "target_date",
    "modeling_track",
    "actual",
    "benchmark_prediction",
    "prediction_raw",
    "prediction_final",
]

The general helpers below handle compact schema checks, atomic Parquet writes, QA interpretation, and registry normalization. They deliberately fail loudly when an upstream contract changes: a downstream comparison should never silently reinterpret a renamed field or incomplete finalist package.

In [5]:
# ---------------------------------------------------------------------
# 4.6.1.0D — General contract and persistence helpers
# ---------------------------------------------------------------------

def require_columns(frame, required_columns, label):
    """Fail clearly when a required upstream table no longer matches its contract."""
    missing = sorted(set(required_columns) - set(frame.columns))

    if missing:
        raise KeyError(
            f"{label} is missing required columns: {missing}"
        )


def parquet_columns(path):
    """Return a Parquet file's physical column names without loading its rows."""
    return pq.ParquetFile(path).schema_arrow.names


def parquet_row_count(path):
    """Return a Parquet row count from metadata without reading the full table."""
    return int(pq.ParquetFile(path).metadata.num_rows)


def atomic_write_parquet(frame, path):
    """Publish one completed Parquet artifact without exposing a partial write."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        f".{path.stem}.{uuid.uuid4().hex[:8]}.tmp.parquet"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )
    os.replace(temporary_path, path)



def write_artifact_recovery_audit(path, audit_path, contract_id):
    """
    Record the small physical fingerprint used by ordinary checkpoint recovery.

    Full row-level semantics are validated when the large artifact is created.
    Clean reruns only need to prove that the exact validated file is still here.
    """
    path = Path(path)
    audit_path = Path(audit_path)

    if not path.exists():
        raise FileNotFoundError(f"Cannot audit missing artifact: {path}")

    file_stat = path.stat()

    audit_df = pd.DataFrame([{
        "contract_id": contract_id,
        "artifact_path": str(path),
        "rows": parquet_row_count(path),
        "schema_signature": "||".join(parquet_columns(path)),
        "file_size_bytes": int(file_stat.st_size),
        "modified_ns": int(file_stat.st_mtime_ns),
        "status": "success",
    }])

    atomic_write_parquet(audit_df, audit_path)


def artifact_recovery_audit_matches(path, audit_path, contract_id):
    """
    Check one validated artifact through metadata rather than rescanning its rows.

    A changed file intentionally fails this hot-path check. The calling cell can
    then perform one full semantic validation before trusting it again.
    """
    path = Path(path)
    audit_path = Path(audit_path)

    if not path.exists() or not audit_path.exists():
        return False

    try:
        audit_df = pd.read_parquet(audit_path)
        require_columns(
            audit_df,
            [
                "contract_id",
                "artifact_path",
                "rows",
                "schema_signature",
                "file_size_bytes",
                "modified_ns",
                "status",
            ],
            f"{path.name} recovery audit",
        )

        if len(audit_df) != 1:
            return False

        audit = audit_df.iloc[0]
        file_stat = path.stat()

        return bool(
            str(audit["contract_id"]) == str(contract_id)
            and str(audit["artifact_path"]) == str(path)
            and str(audit["status"]) == "success"
            and int(audit["rows"]) == parquet_row_count(path)
            and str(audit["schema_signature"]) == "||".join(parquet_columns(path))
            and int(audit["file_size_bytes"]) == int(file_stat.st_size)
            and int(audit["modified_ns"]) == int(file_stat.st_mtime_ns)
        )

    except Exception:
        return False

def qa_file_passes(path):
    """
    Interpret the final QA formats already used by the four source notebooks.

    Classical QA stores PASS/FAIL text in `status`; the other three finalized
    handoffs expose a Boolean `passed` field.
    """
    qa_df = pd.read_parquet(path)

    if "passed" in qa_df.columns:
        passed = qa_df["passed"].fillna(False).astype(bool).all()
        return bool(passed), qa_df

    if "status" in qa_df.columns:
        passed = qa_df["status"].eq("PASS").all()
        return bool(passed), qa_df

    raise KeyError(
        f"{path.name}: expected either a `passed` or `status` QA field."
    )


def expected_target_horizon_pairs():
    """Return the frozen 15 Target × Horizon jobs."""
    return {
        (metric, int(horizon))
        for metric in FORECAST_TARGETS
        for horizon in FORECAST_HORIZONS
    }


def normalize_finalist_registry(family, frame):
    """
    Normalize one family's finalist registry to the shared tournament vocabulary.

    Registry schemas remain authoritative upstream. This helper changes only
    the small set of names needed by 4.6.1 after validating the source fields.
    """
    spec = FAMILY_HANDOFFS[family]
    registry_track_column = spec["registry_track_column"]

    required_columns = [
        "metric",
        "horizon",
        registry_track_column,
        *spec["configuration_columns"],
    ]

    require_columns(
        frame,
        required_columns,
        f"{FAMILY_LABELS[family]} finalist registry",
    )

    normalized = frame.copy().rename(
        columns={registry_track_column: "modeling_track"}
    )

    normalized["family"] = family
    normalized["Family"] = FAMILY_LABELS[family]
    normalized["Representative"] = FAMILY_REPRESENTATIVES[family]
    normalized["horizon"] = normalized["horizon"].astype(int)

    normalized["configuration_id"] = (
        normalized[spec["configuration_columns"]]
        .astype(str)
        .agg(" · ".join, axis=1)
    )

    return normalized


def artifact_materialization_table():
    """Summarize which finalist handoff artifacts currently exist."""
    rows = []

    for family in FAMILY_ORDER:
        spec = FAMILY_HANDOFFS[family]

        for role in ["registry", "predictions", "qa", "manifest"]:
            path = Path(spec[role])

            rows.append({
                "Family": FAMILY_LABELS[family],
                "Artifact role": role.title(),
                "File": path.name,
                "Materialized": path.exists(),
                "Rows": (
                    parquet_row_count(path)
                    if path.exists()
                    else np.nan
                ),
            })

    return pd.DataFrame(rows)


def require_complete_family_handoffs():
    """Stop empirical comparison until all four canonical finalist packages exist."""
    materialization_df = artifact_materialization_table()

    missing_df = materialization_df.loc[
        ~materialization_df["Materialized"]
    ]

    if not missing_df.empty:
        missing_text = "\n".join(
            f"- {row.Family}: {row.File}"
            for row in missing_df.itertuples(index=False)
        )

        raise FileNotFoundError(
            "4.6.1 cannot execute the empirical family tournament until all "
            "canonical finalist handoffs are complete.\n"
            f"Missing artifacts:\n{missing_text}"
        )

    return materialization_df

The row-level helpers normalize the four prediction handoffs into one common vocabulary without loading them into notebook memory.

A forecast row is identified by its target, horizon, Forecast Sequence, Taxi Zone, forecast origin, and future target position. **Modeling track is intentionally not part of the cross-family join key.** Different finalists were allowed to select different frozen training-history tracks; the fair comparison asks whether they forecast the same future outcome, not whether they were trained from identical histories.

In [6]:
# ---------------------------------------------------------------------
# 4.6.1.0E — DuckDB forecast-row and common-support helpers
# ---------------------------------------------------------------------

def sql_path(path):
    """Escape one local filesystem path for use inside DuckDB SQL."""
    return str(Path(path)).replace("'", "''")


def create_family_prediction_views(connection):
    """
    Normalize all family prediction handoffs while preserving archival rows.

    Availability is explicit when upstream provides it and otherwise inferred
    from finite physical values. Unavailable rows remain visible for audit;
    the exact-common and coverage-adjusted builders decide later how those
    rows enter their respective comparison surfaces.
    """
    for family in FAMILY_ORDER:
        spec = FAMILY_HANDOFFS[family]
        path = Path(spec["predictions"])
        source_columns = set(parquet_columns(path))
        prediction_track_column = spec["prediction_track_column"]
        actual_column = spec["actual_column"]

        required_source_columns = {
            *COMMON_FORECAST_KEY_COLUMNS,
            "target_date",
            prediction_track_column,
            actual_column,
            "benchmark_prediction",
            "prediction_raw",
            "prediction_final",
        }
        missing = sorted(required_source_columns - source_columns)
        if missing:
            raise KeyError(
                f"{FAMILY_LABELS[family]} prediction handoff is missing "
                f"required fields: {missing}"
            )

        if "model_prediction_available" in source_columns:
            model_available_sql = (
                "CAST(model_prediction_available AS BOOLEAN)"
            )
        else:
            model_available_sql = """
                (
                    COALESCE(isfinite(CAST(prediction_raw AS DOUBLE)), FALSE)
                    AND COALESCE(isfinite(CAST(prediction_final AS DOUBLE)), FALSE)
                )
            """

        if "benchmark_prediction_available" in source_columns:
            benchmark_available_sql = (
                "CAST(benchmark_prediction_available AS BOOLEAN)"
            )
        else:
            benchmark_available_sql = """
                COALESCE(
                    isfinite(CAST(benchmark_prediction AS DOUBLE)),
                    FALSE
                )
            """

        path_sql = sql_path(path)
        connection.execute(f"""
            CREATE OR REPLACE VIEW pred_{family} AS
            SELECT
                CAST(metric AS VARCHAR) AS metric,
                CAST(horizon AS INTEGER) AS horizon,
                CAST(forecast_sequence_id AS VARCHAR)
                    AS forecast_sequence_id,
                CAST(taxi_zone_id AS BIGINT) AS taxi_zone_id,
                CAST(origin_observation_sequence_id AS BIGINT)
                    AS origin_observation_sequence_id,
                CAST(target_observation_sequence_id AS BIGINT)
                    AS target_observation_sequence_id,
                CAST(target_date AS DATE) AS target_date,
                CAST({prediction_track_column} AS VARCHAR)
                    AS modeling_track,
                CAST({actual_column} AS DOUBLE) AS actual,
                CAST(benchmark_prediction AS DOUBLE)
                    AS benchmark_prediction,
                CAST(prediction_raw AS DOUBLE) AS prediction_raw,
                CAST(prediction_final AS DOUBLE) AS prediction_final,
                {model_available_sql} AS model_prediction_available,
                {benchmark_available_sql}
                    AS benchmark_prediction_available
            FROM read_parquet('{path_sql}')
        """)


def audit_prediction_view(connection, family):
    """
    Audit archival rows separately from rows scoreable by the tournament.

    A family may pass with less than complete scoreable coverage, provided all
    15 jobs have scoreable evidence and the availability contracts are honest.
    """
    view = f"pred_{family}"
    key_sql = ", ".join(COMMON_FORECAST_KEY_COLUMNS)

    summary = connection.execute(f"""
        SELECT
            COUNT(*) AS forecast_rows,
            MIN(target_date) AS minimum_target_date,
            MAX(target_date) AS maximum_target_date,
            SUM(
                CASE WHEN target_date >= DATE '{FINAL_HOLDOUT_START_DATE.date()}'
                     THEN 1 ELSE 0 END
            ) AS final_holdout_rows,
            SUM(CASE WHEN COALESCE(isfinite(actual), FALSE)
                     THEN 1 ELSE 0 END) AS actual_available_rows,
            SUM(CASE WHEN COALESCE(model_prediction_available, FALSE)
                     THEN 1 ELSE 0 END) AS model_available_rows,
            SUM(CASE WHEN COALESCE(benchmark_prediction_available, FALSE)
                     THEN 1 ELSE 0 END) AS benchmark_available_rows,
            SUM(
                CASE
                    WHEN COALESCE(isfinite(actual), FALSE)
                     AND COALESCE(model_prediction_available, FALSE)
                     AND COALESCE(benchmark_prediction_available, FALSE)
                     AND COALESCE(isfinite(prediction_raw), FALSE)
                     AND COALESCE(isfinite(prediction_final), FALSE)
                     AND COALESCE(isfinite(benchmark_prediction), FALSE)
                    THEN 1 ELSE 0
                END
            ) AS scoreable_rows,
            SUM(
                CASE
                    WHEN COALESCE(model_prediction_available, FALSE) <>
                         (
                             COALESCE(isfinite(prediction_raw), FALSE)
                             AND COALESCE(isfinite(prediction_final), FALSE)
                         )
                    THEN 1 ELSE 0
                END
            ) AS model_availability_violations,
            SUM(
                CASE
                    WHEN COALESCE(benchmark_prediction_available, FALSE) <>
                         COALESCE(isfinite(benchmark_prediction), FALSE)
                    THEN 1 ELSE 0
                END
            ) AS benchmark_availability_violations,
            SUM(
                CASE
                    WHEN COALESCE(model_prediction_available, FALSE)
                     AND COALESCE(isfinite(prediction_raw), FALSE)
                     AND COALESCE(isfinite(prediction_final), FALSE)
                     AND ABS(
                         prediction_final - GREATEST(0.0, prediction_raw)
                     ) > {OUTPUT_RULE_TOLERANCE}
                    THEN 1 ELSE 0
                END
            ) AS output_rule_violations
        FROM {view}
    """).df().iloc[0]

    job_count = int(connection.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT metric, horizon FROM {view}
        )
    """).fetchone()[0])

    scoreable_filter = """
        COALESCE(isfinite(actual), FALSE)
        AND COALESCE(model_prediction_available, FALSE)
        AND COALESCE(benchmark_prediction_available, FALSE)
        AND COALESCE(isfinite(prediction_raw), FALSE)
        AND COALESCE(isfinite(prediction_final), FALSE)
        AND COALESCE(isfinite(benchmark_prediction), FALSE)
    """

    scoreable_job_count = int(connection.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT metric, horizon
            FROM {view}
            WHERE {scoreable_filter}
        )
    """).fetchone()[0])

    duplicate_rows = int(connection.execute(f"""
        SELECT COALESCE(SUM(row_count - 1), 0)
        FROM (
            SELECT {key_sql}, COUNT(*) AS row_count
            FROM {view}
            GROUP BY {key_sql}
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0])

    forecast_rows = int(summary["forecast_rows"])
    scoreable_rows = int(summary["scoreable_rows"])

    return {
        "family": family,
        "Family": FAMILY_LABELS[family],
        "forecast_rows": forecast_rows,
        "scoreable_rows": scoreable_rows,
        "scoreable_coverage_pct": (
            100.0 * scoreable_rows / forecast_rows
            if forecast_rows else np.nan
        ),
        "jobs": job_count,
        "scoreable_jobs": scoreable_job_count,
        "minimum_target_date": pd.Timestamp(summary["minimum_target_date"]),
        "maximum_target_date": pd.Timestamp(summary["maximum_target_date"]),
        "duplicate_rows": duplicate_rows,
        "final_holdout_rows": int(summary["final_holdout_rows"]),
        "actual_available_rows": int(summary["actual_available_rows"]),
        "model_available_rows": int(summary["model_available_rows"]),
        "benchmark_available_rows": int(
            summary["benchmark_available_rows"]
        ),
        "model_availability_violations": int(
            summary["model_availability_violations"]
        ),
        "benchmark_availability_violations": int(
            summary["benchmark_availability_violations"]
        ),
        "output_rule_violations": int(summary["output_rule_violations"]),
    }


def common_support_artifact_is_valid(path):
    """Return True only for a complete artifact from this contract version."""
    path = Path(path)
    if not path.exists():
        return False

    required_columns = {
        *COMMON_FORECAST_KEY_COLUMNS,
        "target_date",
        "actual",
        "benchmark_prediction",
        "common_validation_contract_id",
        "actual_alignment_delta",
        "actual_alignment_scale",
        "actual_alignment_tolerance",
        "actual_alignment_ok",
        "benchmark_alignment_delta",
        "benchmark_alignment_scale",
        "benchmark_alignment_tolerance",
        "benchmark_alignment_ok",
        "date_alignment_ok",
        *[
            f"{family}_prediction_raw"
            for family in FAMILY_ORDER
        ],
        *[
            f"{family}_prediction_final"
            for family in FAMILY_ORDER
        ],
        *[
            f"{family}_modeling_track"
            for family in FAMILY_ORDER
        ],
    }

    if not required_columns.issubset(set(parquet_columns(path))):
        return False
    if parquet_row_count(path) == 0:
        return False

    path_sql = sql_path(path)
    key_sql = ", ".join(COMMON_FORECAST_KEY_COLUMNS)
    prediction_checks = " AND ".join(
        f"""
        COALESCE(isfinite({family}_prediction_raw), FALSE)
        AND COALESCE(isfinite({family}_prediction_final), FALSE)
        AND ABS(
            {family}_prediction_final
            - GREATEST(0.0, {family}_prediction_raw)
        ) <= {OUTPUT_RULE_TOLERANCE}
        """
        for family in FAMILY_ORDER
    )

    with duckdb.connect() as connection:
        jobs = connection.execute(f"""
            SELECT metric, horizon
            FROM read_parquet('{path_sql}')
            GROUP BY metric, horizon
        """).fetchall()

        duplicate_rows = int(connection.execute(f"""
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT {key_sql}, COUNT(*) AS row_count
                FROM read_parquet('{path_sql}')
                GROUP BY {key_sql}
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0])

        invalid_rows = int(connection.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{path_sql}')
            WHERE COALESCE(common_validation_contract_id, '') <>
                      '{COMMON_VALIDATION_CONTRACT_ID}'
               OR target_date >= DATE '{FINAL_HOLDOUT_START_DATE.date()}'
               OR NOT COALESCE(date_alignment_ok, FALSE)
               OR NOT COALESCE(actual_alignment_ok, FALSE)
               OR NOT COALESCE(benchmark_alignment_ok, FALSE)
               OR NOT COALESCE(isfinite(actual), FALSE)
               OR NOT COALESCE(isfinite(benchmark_prediction), FALSE)
               OR NOT ({prediction_checks})
        """).fetchone()[0])

    return (
        set((metric, int(horizon)) for metric, horizon in jobs)
        == expected_target_horizon_pairs()
        and duplicate_rows == 0
        and invalid_rows == 0
    )


def build_common_validation_artifact(output_path):
    """
    Materialize the exact intersection of rows all four finalists can score.

    Source-specific actual and benchmark copies are checked with a symmetric
    float-aware tolerance before Classical values become the canonical copies.
    """
    output_path = Path(output_path)
    temporary_path = output_path.with_name(
        f".{output_path.stem}.{uuid.uuid4().hex[:8]}.tmp.parquet"
    )
    key_sql = ", ".join(COMMON_FORECAST_KEY_COLUMNS)
    output_sql = sql_path(temporary_path)

    scoreable_filter = """
        COALESCE(model_prediction_available, FALSE)
        AND COALESCE(benchmark_prediction_available, FALSE)
        AND COALESCE(isfinite(actual), FALSE)
        AND COALESCE(isfinite(benchmark_prediction), FALSE)
        AND COALESCE(isfinite(prediction_raw), FALSE)
        AND COALESCE(isfinite(prediction_final), FALSE)
    """

    with duckdb.connect() as connection:
        create_family_prediction_views(connection)
        connection.execute(f"""
            COPY (
                WITH
                classical_scored AS (
                    SELECT * FROM pred_classical
                    WHERE {scoreable_filter}
                ),
                tree_scored AS (
                    SELECT * FROM pred_tree
                    WHERE {scoreable_filter}
                ),
                neural_scored AS (
                    SELECT * FROM pred_neural
                    WHERE {scoreable_filter}
                ),
                transformer_scored AS (
                    SELECT * FROM pred_transformer
                    WHERE {scoreable_filter}
                ),
                joined AS (
                    SELECT
                        c.* EXCLUDE (
                            target_date,
                            modeling_track,
                            actual,
                            benchmark_prediction,
                            prediction_raw,
                            prediction_final,
                            model_prediction_available,
                            benchmark_prediction_available
                        ),
                        c.target_date AS classical_target_date,
                        t.target_date AS tree_target_date,
                        n.target_date AS neural_target_date,
                        x.target_date AS transformer_target_date,
                        c.actual AS classical_actual,
                        t.actual AS tree_actual,
                        n.actual AS neural_actual,
                        x.actual AS transformer_actual,
                        c.benchmark_prediction AS classical_benchmark,
                        t.benchmark_prediction AS tree_benchmark,
                        n.benchmark_prediction AS neural_benchmark,
                        x.benchmark_prediction AS transformer_benchmark,
                        c.prediction_raw AS classical_prediction_raw,
                        c.prediction_final AS classical_prediction_final,
                        t.prediction_raw AS tree_prediction_raw,
                        t.prediction_final AS tree_prediction_final,
                        n.prediction_raw AS neural_prediction_raw,
                        n.prediction_final AS neural_prediction_final,
                        x.prediction_raw AS transformer_prediction_raw,
                        x.prediction_final AS transformer_prediction_final,
                        c.modeling_track AS classical_modeling_track,
                        t.modeling_track AS tree_modeling_track,
                        n.modeling_track AS neural_modeling_track,
                        x.modeling_track AS transformer_modeling_track
                    FROM classical_scored AS c
                    INNER JOIN tree_scored AS t USING ({key_sql})
                    INNER JOIN neural_scored AS n USING ({key_sql})
                    INNER JOIN transformer_scored AS x USING ({key_sql})
                )
                SELECT
                    {key_sql},
                    classical_target_date AS target_date,
                    classical_actual AS actual,
                    classical_benchmark AS benchmark_prediction,
                    GREATEST(
                        ABS(classical_actual - tree_actual),
                        ABS(classical_actual - neural_actual),
                        ABS(classical_actual - transformer_actual)
                    ) AS actual_alignment_delta,
                    GREATEST(
                        ABS(classical_actual),
                        ABS(tree_actual),
                        ABS(neural_actual),
                        ABS(transformer_actual)
                    ) AS actual_alignment_scale,
                    (
                        {CROSS_ARTIFACT_ATOL}
                        + {CROSS_ARTIFACT_RTOL}
                        * GREATEST(
                            ABS(classical_actual),
                            ABS(tree_actual),
                            ABS(neural_actual),
                            ABS(transformer_actual)
                        )
                    ) AS actual_alignment_tolerance,
                    (
                        GREATEST(
                            ABS(classical_actual - tree_actual),
                            ABS(classical_actual - neural_actual),
                            ABS(classical_actual - transformer_actual)
                        )
                        <= (
                            {CROSS_ARTIFACT_ATOL}
                            + {CROSS_ARTIFACT_RTOL}
                            * GREATEST(
                                ABS(classical_actual),
                                ABS(tree_actual),
                                ABS(neural_actual),
                                ABS(transformer_actual)
                            )
                        )
                    ) AS actual_alignment_ok,
                    (
                        classical_target_date = tree_target_date
                        AND classical_target_date = neural_target_date
                        AND classical_target_date = transformer_target_date
                    ) AS date_alignment_ok,
                    GREATEST(
                        ABS(classical_benchmark - tree_benchmark),
                        ABS(classical_benchmark - neural_benchmark),
                        ABS(classical_benchmark - transformer_benchmark)
                    ) AS benchmark_alignment_delta,
                    GREATEST(
                        ABS(classical_benchmark),
                        ABS(tree_benchmark),
                        ABS(neural_benchmark),
                        ABS(transformer_benchmark)
                    ) AS benchmark_alignment_scale,
                    (
                        {CROSS_ARTIFACT_ATOL}
                        + {CROSS_ARTIFACT_RTOL}
                        * GREATEST(
                            ABS(classical_benchmark),
                            ABS(tree_benchmark),
                            ABS(neural_benchmark),
                            ABS(transformer_benchmark)
                        )
                    ) AS benchmark_alignment_tolerance,
                    (
                        GREATEST(
                            ABS(classical_benchmark - tree_benchmark),
                            ABS(classical_benchmark - neural_benchmark),
                            ABS(classical_benchmark - transformer_benchmark)
                        )
                        <= (
                            {CROSS_ARTIFACT_ATOL}
                            + {CROSS_ARTIFACT_RTOL}
                            * GREATEST(
                                ABS(classical_benchmark),
                                ABS(tree_benchmark),
                                ABS(neural_benchmark),
                                ABS(transformer_benchmark)
                            )
                        )
                    ) AS benchmark_alignment_ok,
                    classical_prediction_raw,
                    classical_prediction_final,
                    tree_prediction_raw,
                    tree_prediction_final,
                    neural_prediction_raw,
                    neural_prediction_final,
                    transformer_prediction_raw,
                    transformer_prediction_final,
                    classical_modeling_track,
                    tree_modeling_track,
                    neural_modeling_track,
                    transformer_modeling_track,
                    '{COMMON_VALIDATION_CONTRACT_ID}'
                        AS common_validation_contract_id
                FROM joined
            )
            TO '{output_sql}'
            (FORMAT PARQUET, COMPRESSION ZSTD);
        """)

    os.replace(temporary_path, output_path)
    path_sql = sql_path(output_path)
    key_sql = ", ".join(COMMON_FORECAST_KEY_COLUMNS)

    with duckdb.connect() as connection:
        qa = connection.execute(f"""
            SELECT
                COUNT(*) AS common_rows,
                COUNT(DISTINCT metric || '|' || CAST(horizon AS VARCHAR))
                    AS jobs,
                SUM(
                    CASE WHEN target_date >=
                        DATE '{FINAL_HOLDOUT_START_DATE.date()}'
                         THEN 1 ELSE 0 END
                ) AS final_holdout_rows,
                SUM(CASE WHEN NOT date_alignment_ok THEN 1 ELSE 0 END)
                    AS date_alignment_failures,
                SUM(CASE WHEN NOT COALESCE(actual_alignment_ok, FALSE) THEN 1 ELSE 0 END)
                    AS actual_alignment_failures,
                SUM(CASE WHEN NOT COALESCE(benchmark_alignment_ok, FALSE) THEN 1 ELSE 0 END)
                    AS benchmark_alignment_failures,
                SUM(CASE WHEN COALESCE(common_validation_contract_id, '') <>
                    '{COMMON_VALIDATION_CONTRACT_ID}'
                    THEN 1 ELSE 0 END) AS contract_failures,
                SUM(CASE WHEN NOT COALESCE(isfinite(actual), FALSE)
                    THEN 1 ELSE 0 END) AS actual_nonfinite,
                SUM(CASE WHEN NOT COALESCE(isfinite(benchmark_prediction), FALSE)
                    THEN 1 ELSE 0 END) AS benchmark_nonfinite,
                SUM(CASE WHEN NOT (
                    COALESCE(isfinite(classical_prediction_raw), FALSE)
                    AND COALESCE(isfinite(classical_prediction_final), FALSE)
                    AND COALESCE(isfinite(tree_prediction_raw), FALSE)
                    AND COALESCE(isfinite(tree_prediction_final), FALSE)
                    AND COALESCE(isfinite(neural_prediction_raw), FALSE)
                    AND COALESCE(isfinite(neural_prediction_final), FALSE)
                    AND COALESCE(isfinite(transformer_prediction_raw), FALSE)
                    AND COALESCE(isfinite(transformer_prediction_final), FALSE)
                ) THEN 1 ELSE 0 END) AS prediction_nonfinite
            FROM read_parquet('{path_sql}')
        """).df().iloc[0]

        duplicate_rows = int(connection.execute(f"""
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT {key_sql}, COUNT(*) AS row_count
                FROM read_parquet('{path_sql}')
                GROUP BY {key_sql}
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0])

    failures = []
    if int(qa["common_rows"]) == 0:
        failures.append("nonzero rows")
    if int(qa["jobs"]) != EXPECTED_JOB_COUNT:
        failures.append("exactly 15 Target × Horizon jobs")
    if duplicate_rows != 0:
        failures.append("zero duplicate common forecast keys")
    if int(qa["final_holdout_rows"]) != 0:
        failures.append("zero final-holdout rows")
    if int(qa["date_alignment_failures"]) != 0:
        failures.append("exact target-date alignment")
    if int(qa["actual_alignment_failures"]) != 0:
        failures.append("float-aware actual equivalence")
    if int(qa["benchmark_alignment_failures"]) != 0:
        failures.append("float-aware benchmark equivalence")
    if int(qa["contract_failures"]) != 0:
        failures.append("common validation contract ID")
    if int(qa["actual_nonfinite"]) != 0:
        failures.append("finite canonical actual values")
    if int(qa["benchmark_nonfinite"]) != 0:
        failures.append("finite canonical benchmark values")
    if int(qa["prediction_nonfinite"]) != 0:
        failures.append("finite family prediction values")

    if failures:
        output_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Common-validation artifact failed QA: "
            + ", ".join(failures)
        )

    if not common_support_artifact_is_valid(output_path):
        output_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Common-validation artifact failed its persisted recovery contract."
        )

    return output_path

# ---------------------------------------------------------------------
# Coverage-adjusted validation selection surface
# ---------------------------------------------------------------------

def selection_surface_artifact_is_valid(path):
    """Return True only for a complete coverage-adjusted selection artifact."""
    path = Path(path)
    if not path.exists():
        return False

    required_columns = {
        *COMMON_FORECAST_KEY_COLUMNS,
        "target_date",
        "actual",
        "benchmark_prediction",
        "source_family_count",
        "date_alignment_ok",
        "actual_alignment_tolerance",
        "actual_alignment_ok",
        "benchmark_alignment_tolerance",
        "benchmark_alignment_ok",
        "selection_validation_contract_id",
    }

    for family in FAMILY_ORDER:
        required_columns.update({
            f"{family}_modeling_track",
            f"{family}_row_present",
            f"{family}_source_target_date",
            f"{family}_source_actual",
            f"{family}_source_benchmark_prediction",
            f"{family}_model_prediction_raw",
            f"{family}_model_prediction_final",
            f"{family}_model_prediction_available",
            f"{family}_used_benchmark_fallback",
            f"{family}_system_prediction",
        })

    if not required_columns.issubset(set(parquet_columns(path))):
        return False
    if parquet_row_count(path) == 0:
        return False

    path_sql = sql_path(path)
    key_sql = ", ".join(COMMON_FORECAST_KEY_COLUMNS)

    family_checks = []

    for family in FAMILY_ORDER:
        family_checks.extend([
            f"NOT COALESCE(isfinite({family}_system_prediction), FALSE)",
            (
                f"{family}_used_benchmark_fallback <> "
                f"(NOT {family}_model_prediction_available)"
            ),
            (
                f"({family}_model_prediction_available AND "
                f"ABS({family}_system_prediction - "
                f"{family}_model_prediction_final) > "
                f"{OUTPUT_RULE_TOLERANCE})"
            ),
            (
                f"({family}_used_benchmark_fallback AND "
                f"ABS({family}_system_prediction - benchmark_prediction) > "
                f"{OUTPUT_RULE_TOLERANCE})"
            ),
            (
                f"({family}_row_present AND "
                f"{family}_source_target_date IS NOT NULL AND "
                f"{family}_source_target_date <> target_date)"
            ),
            (
                f"({family}_row_present AND "
                f"COALESCE(isfinite({family}_source_actual), FALSE) AND "
                f"ABS({family}_source_actual - actual) > "
                f"actual_alignment_tolerance)"
            ),
            (
                f"({family}_row_present AND "
                f"COALESCE(isfinite({family}_source_benchmark_prediction), FALSE) "
                f"AND ABS({family}_source_benchmark_prediction - "
                f"benchmark_prediction) > benchmark_alignment_tolerance)"
            ),
        ])

    family_check_sql = " OR ".join(family_checks)

    with duckdb.connect() as connection:
        jobs = connection.execute(f"""
            SELECT metric, horizon
            FROM read_parquet('{path_sql}')
            GROUP BY metric, horizon
        """).fetchall()

        duplicate_rows = int(connection.execute(f"""
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT {key_sql}, COUNT(*) AS row_count
                FROM read_parquet('{path_sql}')
                GROUP BY {key_sql}
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0])

        invalid_rows = int(connection.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{path_sql}')
            WHERE COALESCE(selection_validation_contract_id, '') <>
                      '{SELECTION_VALIDATION_CONTRACT_ID}'
               OR target_date >= DATE '{FINAL_HOLDOUT_START_DATE.date()}'
               OR NOT COALESCE(date_alignment_ok, FALSE)
               OR NOT COALESCE(actual_alignment_ok, FALSE)
               OR NOT COALESCE(benchmark_alignment_ok, FALSE)
               OR NOT COALESCE(isfinite(actual), FALSE)
               OR NOT COALESCE(isfinite(benchmark_prediction), FALSE)
               OR ({family_check_sql})
        """).fetchone()[0])

    return (
        set((metric, int(horizon)) for metric, horizon in jobs)
        == expected_target_horizon_pairs()
        and duplicate_rows == 0
        and invalid_rows == 0
    )


def build_coverage_adjusted_validation_artifact(output_path):
    """
    Build one benchmark-supported validation population for all four families.

    A physical forecast row enters the population when at least one finalized
    family handoff contains a finite actual and frozen benchmark value. Each
    family is then LEFT JOINED onto that common population. When its advanced
    model prediction is unavailable, the frozen weekly benchmark becomes the
    system prediction for that family on that row.

    This makes missing advanced forecasts visible as coverage rather than
    silently deleting those observations from model selection.
    """
    output_path = Path(output_path)
    temporary_path = output_path.with_name(
        f".{output_path.stem}.{uuid.uuid4().hex[:8]}.tmp.parquet"
    )

    key_sql = ", ".join(COMMON_FORECAST_KEY_COLUMNS)
    output_sql = sql_path(temporary_path)

    benchmark_supported_filter = f"""
        target_date < DATE '{FINAL_HOLDOUT_START_DATE.date()}'
        AND COALESCE(isfinite(actual), FALSE)
        AND COALESCE(benchmark_prediction_available, FALSE)
        AND COALESCE(isfinite(benchmark_prediction), FALSE)
    """

    eligible_union_sql = "\nUNION ALL\n".join(
        f"""
        SELECT
            {key_sql},
            target_date,
            actual,
            benchmark_prediction,
            '{family}' AS source_family
        FROM pred_{family}
        WHERE {benchmark_supported_filter}
        """
        for family in FAMILY_ORDER
    )

    family_joins = []
    family_projections = []

    for family in FAMILY_ORDER:
        alias = f"p_{family}"

        # Availability must agree with both the upstream flag and the physical
        # prediction values. A row that fails either condition uses fallback.
        model_available_sql = f"""
            (
                COALESCE({alias}.model_prediction_available, FALSE)
                AND COALESCE(isfinite({alias}.prediction_raw), FALSE)
                AND COALESCE(isfinite({alias}.prediction_final), FALSE)
            )
        """

        family_joins.append(
            f"LEFT JOIN pred_{family} AS {alias} USING ({key_sql})"
        )

        family_projections.extend([
            f"{alias}.modeling_track AS {family}_modeling_track",
            (
                f"({alias}.forecast_sequence_id IS NOT NULL) "
                f"AS {family}_row_present"
            ),
            f"{alias}.target_date AS {family}_source_target_date",
            f"{alias}.actual AS {family}_source_actual",
            (
                f"{alias}.benchmark_prediction "
                f"AS {family}_source_benchmark_prediction"
            ),
            (
                f"{alias}.prediction_raw "
                f"AS {family}_model_prediction_raw"
            ),
            (
                f"{alias}.prediction_final "
                f"AS {family}_model_prediction_final"
            ),
            (
                f"{model_available_sql} "
                f"AS {family}_model_prediction_available"
            ),
            (
                f"NOT {model_available_sql} "
                f"AS {family}_used_benchmark_fallback"
            ),
            (
                "CASE "
                f"WHEN {model_available_sql} "
                f"THEN {alias}.prediction_final "
                "ELSE canonical.benchmark_prediction "
                f"END AS {family}_system_prediction"
            ),
        ])

    family_join_sql = "\n".join(family_joins)
    family_projection_sql = ",\n                    ".join(family_projections)

    with duckdb.connect() as connection:
        create_family_prediction_views(connection)

        connection.execute(f"""
            COPY (
                WITH eligible_union AS (
                    {eligible_union_sql}
                ),
                canonical AS (
                    SELECT
                        {key_sql},
                        MIN(target_date) AS target_date,
                        MIN(actual) AS actual,
                        MIN(benchmark_prediction) AS benchmark_prediction,
                        COUNT(DISTINCT source_family) AS source_family_count,

                        COUNT(DISTINCT target_date) = 1
                            AS date_alignment_ok,

                        MAX(actual) - MIN(actual)
                            AS actual_alignment_delta,
                        MAX(ABS(actual))
                            AS actual_alignment_scale,
                        (
                            {CROSS_ARTIFACT_ATOL}
                            + {CROSS_ARTIFACT_RTOL} * MAX(ABS(actual))
                        ) AS actual_alignment_tolerance,
                        (
                            MAX(actual) - MIN(actual)
                            <= (
                                {CROSS_ARTIFACT_ATOL}
                                + {CROSS_ARTIFACT_RTOL} * MAX(ABS(actual))
                            )
                        ) AS actual_alignment_ok,

                        MAX(benchmark_prediction) - MIN(benchmark_prediction)
                            AS benchmark_alignment_delta,
                        MAX(ABS(benchmark_prediction))
                            AS benchmark_alignment_scale,
                        (
                            {CROSS_ARTIFACT_ATOL}
                            + {CROSS_ARTIFACT_RTOL}
                            * MAX(ABS(benchmark_prediction))
                        ) AS benchmark_alignment_tolerance,
                        (
                            MAX(benchmark_prediction)
                            - MIN(benchmark_prediction)
                            <= (
                                {CROSS_ARTIFACT_ATOL}
                                + {CROSS_ARTIFACT_RTOL}
                                * MAX(ABS(benchmark_prediction))
                            )
                        ) AS benchmark_alignment_ok

                    FROM eligible_union
                    GROUP BY {key_sql}
                )

                SELECT
                    canonical.*,
                    {family_projection_sql},
                    '{SELECTION_VALIDATION_CONTRACT_ID}'
                        AS selection_validation_contract_id
                FROM canonical
                {family_join_sql}
            )
            TO '{output_sql}'
            (FORMAT PARQUET, COMPRESSION ZSTD);
        """)

    os.replace(temporary_path, output_path)

    if not selection_surface_artifact_is_valid(output_path):
        output_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Coverage-adjusted validation artifact failed its persisted "
            "selection-surface contract."
        )

    return output_path

The primary tournament evaluates each finalist as a **complete forecasting system**, not only on the rows where its advanced model happened to produce a prediction.

For every Target × Horizon, the notebook therefore builds one benchmark-supported validation population. On each row:

- use the family's advanced `prediction_final` when it is available; otherwise
- use the frozen **Previous week · same Daypart** prediction as the fallback.

This produces a **system prediction** with the same denominator for all four families.

The metric helper retains the project's established forecasting measures:

- **System MAE** — the average absolute miss of the complete forecasting system; this is the primary selector.
- **Model coverage** — the share of canonical validation rows served by the advanced model rather than fallback.
- **Model-only MAE** — accuracy on the rows where that family's advanced model is available; diagnostic only.
- **Exact-common MAE** — advanced-model accuracy on the rows where all four families can forecast; diagnostic only.
- **System Relative MAE (%)** — total absolute system error divided by total absolute observed mobility.
- **RMSE** — gives extra weight to unusually large misses.
- **System bias** — shows whether the complete forecasting system systematically runs high or low.
- **sMAPE and R²** — supporting diagnostics rather than winner selectors.
- **Benchmark skill** — expresses how much system MAE improves on the frozen weekly benchmark.

System Relative MAE is:

$$
100 \times
\frac{\sum |system\_prediction - actual|}
     {\sum |actual|}
$$

It is **not** MAPE and does not average row-level percentage errors.

Raw native-unit MAE is compared only within the same Target × Horizon. Taxi trips, vehicle speed, and Subway ridership are never averaged together as though they shared the same units.

In [7]:
# ---------------------------------------------------------------------
# 4.6.1.0F — Coverage-adjusted scoring and stability helpers
# ---------------------------------------------------------------------

def build_coverage_adjusted_support_summary(selection_path):
    """Summarize advanced-model coverage and benchmark fallback by job."""
    path_sql = sql_path(selection_path)
    family_queries = []

    for family in FAMILY_ORDER:
        family_queries.append(f"""
            SELECT
                metric,
                horizon,
                '{family}' AS family,
                COUNT(*) AS canonical_rows,
                SUM(
                    CASE WHEN {family}_model_prediction_available
                         THEN 1 ELSE 0 END
                ) AS model_available_rows,
                SUM(
                    CASE WHEN {family}_used_benchmark_fallback
                         THEN 1 ELSE 0 END
                ) AS fallback_rows
            FROM read_parquet('{path_sql}')
            GROUP BY metric, horizon
        """)

    with duckdb.connect() as connection:
        support_df = connection.execute(
            "\nUNION ALL\n".join(family_queries)
        ).df()

    support_df["model_coverage_pct"] = (
        100.0
        * support_df["model_available_rows"]
        / support_df["canonical_rows"]
    )
    support_df["fallback_pct"] = (
        100.0 * support_df["fallback_rows"] / support_df["canonical_rows"]
    )

    if not (
        support_df["model_available_rows"] + support_df["fallback_rows"]
    ).eq(support_df["canonical_rows"]).all():
        raise RuntimeError(
            "Advanced-model coverage plus benchmark fallback does not equal "
            "the canonical validation population for at least one family/job."
        )

    return (
        support_df
        .sort_values(["metric", "horizon", "family"])
        .reset_index(drop=True)
    )


def build_validation_scorecard(selection_path, exact_common_path):
    """
    Score each finalist as a complete forecasting system.

    The primary MAE uses the advanced model when available and the frozen
    weekly benchmark otherwise. This keeps one denominator for all families
    while making forecast coverage part of model selection.

    Exact-common MAE is retained only as a secondary diagnostic: it answers
    how accurate the advanced models are on rows where all four can forecast.
    """
    selection_sql = sql_path(selection_path)
    common_sql = sql_path(exact_common_path)

    # -----------------------------------------------------------------
    # 1) Compute quantities shared by every family exactly once
    # -----------------------------------------------------------------
    with duckdb.connect() as connection:
        shared_job_df = connection.execute(f"""
            SELECT
                metric,
                horizon,
                COUNT(*) AS evaluation_rows,
                SUM(ABS(actual)) AS observed_abs_sum,
                VAR_POP(actual) AS actual_var_pop,
                SUM(
                    ABS(benchmark_prediction - actual)
                ) AS benchmark_absolute_error_sum,
                AVG(
                    ABS(benchmark_prediction - actual)
                ) AS benchmark_mae
            FROM read_parquet('{selection_sql}')
            GROUP BY metric, horizon
        """).df()

    if len(shared_job_df) != EXPECTED_JOB_COUNT:
        raise RuntimeError(
            "Expected exactly 15 Target × Horizon selection summaries; "
            f"found {len(shared_job_df)}."
        )

    if (shared_job_df["evaluation_rows"] <= 0).any():
        raise RuntimeError(
            "At least one Target × Horizon selection population contains no rows."
        )

    if (shared_job_df["observed_abs_sum"] <= 0).any():
        raise RuntimeError(
            "At least one forecasting job has no positive observed-activity "
            "denominator for Relative MAE."
        )

    # -----------------------------------------------------------------
    # 2) Compute family-dependent system error and model coverage
    # -----------------------------------------------------------------
    family_queries = []

    for family in FAMILY_ORDER:
        system_prediction = f"{family}_system_prediction"
        model_prediction = f"{family}_model_prediction_final"
        model_available = f"{family}_model_prediction_available"
        fallback = f"{family}_used_benchmark_fallback"

        family_queries.append(f"""
            SELECT
                metric,
                horizon,
                '{family}' AS family,
                COUNT(*) AS family_evaluation_rows,

                SUM(
                    CASE WHEN {model_available} THEN 1 ELSE 0 END
                ) AS model_available_rows,

                SUM(
                    CASE WHEN {fallback} THEN 1 ELSE 0 END
                ) AS fallback_rows,

                SUM(
                    ABS({system_prediction} - actual)
                ) AS absolute_error_sum,

                SUM(
                    POWER({system_prediction} - actual, 2)
                ) AS squared_error_sum,

                AVG(
                    ABS({system_prediction} - actual)
                ) AS mae,

                SQRT(
                    AVG(POWER({system_prediction} - actual, 2))
                ) AS rmse,

                AVG(
                    {system_prediction} - actual
                ) AS bias,

                AVG(
                    CASE
                        WHEN ABS({system_prediction}) + ABS(actual) > 0
                        THEN (
                            2.0 * ABS({system_prediction} - actual)
                            / (ABS({system_prediction}) + ABS(actual))
                        )
                        ELSE NULL
                    END
                ) * 100.0 AS smape_pct,

                AVG(
                    CASE
                        WHEN {model_available}
                        THEN ABS({model_prediction} - actual)
                        ELSE NULL
                    END
                ) AS model_only_mae

            FROM read_parquet('{selection_sql}')
            GROUP BY metric, horizon
        """)

    with duckdb.connect() as connection:
        family_score_df = connection.execute(
            "\nUNION ALL\n".join(family_queries)
        ).df()

    # -----------------------------------------------------------------
    # 3) Retain exact-common MAE as a conditional-accuracy diagnostic
    # -----------------------------------------------------------------
    exact_common_queries = []

    for family in FAMILY_ORDER:
        prediction = f"{family}_prediction_final"

        exact_common_queries.append(f"""
            SELECT
                metric,
                horizon,
                '{family}' AS family,
                COUNT(*) AS exact_common_rows,
                AVG(ABS({prediction} - actual)) AS exact_common_mae
            FROM read_parquet('{common_sql}')
            GROUP BY metric, horizon
        """)

    with duckdb.connect() as connection:
        exact_common_df = connection.execute(
            "\nUNION ALL\n".join(exact_common_queries)
        ).df()

    # -----------------------------------------------------------------
    # 4) Attach the one canonical denominator and diagnostic evidence
    # -----------------------------------------------------------------
    scorecard_df = (
        family_score_df
        .merge(
            shared_job_df,
            on=["metric", "horizon"],
            how="left",
            validate="many_to_one",
        )
        .merge(
            exact_common_df,
            on=["metric", "horizon", "family"],
            how="left",
            validate="one_to_one",
        )
    )

    expected_rows = len(FAMILY_ORDER) * EXPECTED_JOB_COUNT

    if len(scorecard_df) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} Family × Target × Horizon scorecard "
            f"rows; found {len(scorecard_df)}."
        )

    if not scorecard_df["family_evaluation_rows"].eq(
        scorecard_df["evaluation_rows"]
    ).all():
        raise RuntimeError(
            "At least one family does not use the full canonical selection "
            "population."
        )

    if not (
        scorecard_df["model_available_rows"] + scorecard_df["fallback_rows"]
    ).eq(scorecard_df["evaluation_rows"]).all():
        raise RuntimeError(
            "Advanced-model rows plus fallback rows do not equal the full "
            "selection population."
        )

    # -----------------------------------------------------------------
    # 5) Derive coverage, normalized error, and benchmark-relative skill
    # -----------------------------------------------------------------
    scorecard_df["model_coverage_pct"] = (
        100.0
        * scorecard_df["model_available_rows"]
        / scorecard_df["evaluation_rows"]
    )

    scorecard_df["fallback_pct"] = (
        100.0 * scorecard_df["fallback_rows"] / scorecard_df["evaluation_rows"]
    )

    scorecard_df["nmae"] = (
        scorecard_df["absolute_error_sum"]
        / scorecard_df["observed_abs_sum"]
    )
    scorecard_df["relative_mae_pct"] = 100.0 * scorecard_df["nmae"]

    scorecard_df["benchmark_nmae"] = (
        scorecard_df["benchmark_absolute_error_sum"]
        / scorecard_df["observed_abs_sum"]
    )
    scorecard_df["benchmark_relative_mae_pct"] = (
        100.0 * scorecard_df["benchmark_nmae"]
    )

    scorecard_df["benchmark_skill_pct"] = (
        100.0
        * (
            scorecard_df["benchmark_mae"] - scorecard_df["mae"]
        )
        / scorecard_df["benchmark_mae"]
    )

    scorecard_df["abs_bias"] = scorecard_df["bias"].abs()

    model_mse = (
        scorecard_df["squared_error_sum"]
        / scorecard_df["evaluation_rows"]
    )

    scorecard_df["r2"] = np.where(
        scorecard_df["actual_var_pop"] > 0,
        1.0 - model_mse / scorecard_df["actual_var_pop"],
        np.nan,
    )

    # -----------------------------------------------------------------
    # 6) Rank complete forecasting systems, not only available model rows
    # -----------------------------------------------------------------
    scorecard_df["mae_rank"] = (
        scorecard_df.groupby(
            ["metric", "horizon"],
            observed=True,
        )["mae"]
        .rank(method="first", ascending=True)
        .astype(int)
    )

    scorecard_df["mae_comparison_rank"] = (
        scorecard_df.groupby(
            ["metric", "horizon"],
            observed=True,
        )["mae"]
        .rank(method="min", ascending=True)
    )

    scorecard_df["relative_mae_rank"] = (
        scorecard_df.groupby(
            ["metric", "horizon"],
            observed=True,
        )["relative_mae_pct"]
        .rank(method="min", ascending=True)
    )

    scorecard_df["benchmark_skill_rank"] = (
        scorecard_df.groupby(
            ["metric", "horizon"],
            observed=True,
        )["benchmark_skill_pct"]
        .rank(method="min", ascending=False)
    )

    return (
        scorecard_df
        .drop(columns="family_evaluation_rows")
        .sort_values(["metric", "horizon", "mae_rank"])
        .reset_index(drop=True)
    )


def build_temporal_and_geographic_stability(selection_path):
    """Create month- and Taxi-Zone-level system-error evidence."""
    path_sql = sql_path(selection_path)

    monthly_queries = []
    zone_queries = []

    for family in FAMILY_ORDER:
        prediction = f"{family}_system_prediction"

        monthly_queries.append(f"""
            SELECT
                metric,
                horizon,
                DATE_TRUNC('month', target_date)::DATE AS month_start,
                '{family}' AS family,
                COUNT(*) AS evaluation_rows,
                AVG(ABS({prediction} - actual)) AS mae,
                AVG({prediction} - actual) AS bias,
                AVG(ABS(benchmark_prediction - actual)) AS benchmark_mae
            FROM read_parquet('{path_sql}')
            GROUP BY metric, horizon, month_start
        """)

        zone_queries.append(f"""
            SELECT
                metric,
                horizon,
                taxi_zone_id,
                '{family}' AS family,
                COUNT(*) AS evaluation_rows,
                AVG(ABS({prediction} - actual)) AS mae,
                AVG({prediction} - actual) AS bias
            FROM read_parquet('{path_sql}')
            GROUP BY metric, horizon, taxi_zone_id
        """)

    with duckdb.connect() as connection:
        monthly_df = connection.execute(
            "\nUNION ALL\n".join(monthly_queries)
        ).df()

        zone_df = connection.execute(
            "\nUNION ALL\n".join(zone_queries)
        ).df()

    monthly_df["skill_vs_benchmark_pct"] = (
        100.0
        * (monthly_df["benchmark_mae"] - monthly_df["mae"])
        / monthly_df["benchmark_mae"]
    )
    monthly_df["beats_benchmark"] = (
        monthly_df["mae"] < monthly_df["benchmark_mae"]
    )

    monthly_df["family_rank"] = (
        monthly_df.groupby(
            ["metric", "horizon", "month_start"],
            observed=True,
        )["mae"]
        .rank(method="min", ascending=True)
    )
    monthly_df["is_month_best"] = monthly_df["family_rank"].eq(1)

    zone_df["family_rank"] = (
        zone_df.groupby(
            ["metric", "horizon", "taxi_zone_id"],
            observed=True,
        )["mae"]
        .rank(method="min", ascending=True)
    )
    zone_df["is_zone_best"] = zone_df["family_rank"].eq(1)

    return monthly_df, zone_df


def summarize_stability(monthly_df, zone_df):
    """Condense system-stability evidence to one row per Family × Job."""
    monthly_summary_df = (
        monthly_df.groupby(["metric", "horizon", "family"], observed=True)
        .agg(
            validation_months=("month_start", "nunique"),
            months_best=("is_month_best", "sum"),
            months_beating_benchmark=("beats_benchmark", "sum"),
            monthly_mae_std=("mae", "std"),
            worst_monthly_skill_pct=("skill_vs_benchmark_pct", "min"),
        )
        .reset_index()
    )

    monthly_summary_df["months_best_share_pct"] = (
        100.0
        * monthly_summary_df["months_best"]
        / monthly_summary_df["validation_months"]
    )

    zone_summary_df = (
        zone_df.groupby(["metric", "horizon", "family"], observed=True)
        .agg(
            evaluated_zones=("taxi_zone_id", "nunique"),
            zones_best=("is_zone_best", "sum"),
        )
        .reset_index()
    )

    zone_summary_df["zones_best_share_pct"] = (
        100.0
        * zone_summary_df["zones_best"]
        / zone_summary_df["evaluated_zones"]
    )

    return monthly_summary_df.merge(
        zone_summary_df,
        on=["metric", "horizon", "family"],
        how="outer",
        validate="one_to_one",
    )

The final statistical helper implements the paired **calendar-week block bootstrap** on the same coverage-adjusted system errors used by the primary selector.

Why use whole weeks instead of individual forecast rows?

Forecast errors from the same period are related. A snowstorm, holiday, service disruption, or citywide travel pattern can affect many Taxi Zones at once, while neighboring days can also resemble one another. Pretending that millions of individual rows are completely independent would make tiny model differences look far more certain than they really are.

The procedure therefore keeps each calendar week's canonical validation rows together. It repeatedly resamples whole weeks and recalculates the **system-MAE** difference between two finalists.

For point-estimate leader \(A\) and challenger \(B\):

$$
\Delta MAE = System\ MAE_B - System\ MAE_A
$$

Positive values favor \(A\).

A difference is a **clear practical win** only when the lower edge of the 95% bootstrap interval exceeds 1% of the frozen benchmark MAE. A 95% interval entirely inside the ±1% practical-equivalence band is treated as practical equivalence. Everything between those cases remains uncertain enough to require secondary evidence.

In [8]:
# ---------------------------------------------------------------------
# 4.6.1.0G — Paired calendar-week block-bootstrap helpers
# ---------------------------------------------------------------------

def build_weekly_error_summary(selection_path):
    """
    Reduce the coverage-adjusted validation surface to weekly error totals.

    Each family is scored on the same canonical rows. The system prediction
    uses the advanced model when available and the frozen weekly benchmark
    when that model cannot produce a forecast.
    """
    path_sql = sql_path(selection_path)
    family_queries = []

    for family in FAMILY_ORDER:
        prediction = f"{family}_system_prediction"

        family_queries.append(f"""
            SELECT
                metric,
                horizon,
                DATE_TRUNC('week', target_date)::DATE AS week_start,
                '{family}' AS family,
                COUNT(*) AS evaluation_rows,
                SUM(ABS({prediction} - actual)) AS absolute_error_sum
            FROM read_parquet('{path_sql}')
            GROUP BY metric, horizon, week_start
        """)

    with duckdb.connect() as connection:
        return connection.execute(
            "\nUNION ALL\n".join(family_queries)
        ).df()


def bootstrap_pairwise_mae_advantage(
    weekly_df,
    metric,
    horizon,
    reference_family,
    challenger_family,
    benchmark_mae,
    repeats,
    seed,
):
    """
    Bootstrap the paired MAE advantage of one point-estimate winner.

    Positive advantage means the challenger has larger MAE and therefore the
    reference family performs better.
    """
    job_df = weekly_df.loc[
        weekly_df["metric"].eq(metric)
        & weekly_df["horizon"].astype(int).eq(int(horizon))
        & weekly_df["family"].isin([reference_family, challenger_family])
    ].copy()

    error_wide = job_df.pivot(
        index="week_start",
        columns="family",
        values="absolute_error_sum",
    )
    rows_wide = job_df.pivot(
        index="week_start",
        columns="family",
        values="evaluation_rows",
    )

    required_families = {reference_family, challenger_family}

    if set(error_wide.columns) != required_families:
        raise RuntimeError(
            f"{metric} · h={horizon}: incomplete weekly family pairing."
        )

    if not rows_wide[reference_family].equals(rows_wide[challenger_family]):
        raise RuntimeError(
            f"{metric} · h={horizon}: paired families do not share "
            "the same weekly row counts."
        )

    error_wide = error_wide.sort_index()
    rows = rows_wide.loc[error_wide.index, reference_family].to_numpy(dtype=float)

    weekly_difference_sum = (
        error_wide[challenger_family].to_numpy(dtype=float)
        - error_wide[reference_family].to_numpy(dtype=float)
    )

    point_advantage = weekly_difference_sum.sum() / rows.sum()

    rng = np.random.default_rng(seed)
    n_blocks = len(error_wide)

    sample_indices = rng.integers(
        low=0,
        high=n_blocks,
        size=(repeats, n_blocks),
    )

    sampled_difference = weekly_difference_sum[sample_indices].sum(axis=1)
    sampled_rows = rows[sample_indices].sum(axis=1)
    bootstrap_advantage = sampled_difference / sampled_rows

    alpha = 1.0 - BOOTSTRAP_CONFIDENCE_LEVEL
    ci_low, ci_high = np.quantile(
        bootstrap_advantage,
        [alpha / 2.0, 1.0 - alpha / 2.0],
    )

    practical_margin = (
        PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION * benchmark_mae
    )

    if ci_low > practical_margin:
        comparison_status = "clear_win"
    elif ci_low >= -practical_margin and ci_high <= practical_margin:
        comparison_status = "practical_equivalence"
    else:
        comparison_status = "uncertain_advantage"

    return {
        "metric": metric,
        "horizon": int(horizon),
        "reference_family": reference_family,
        "challenger_family": challenger_family,
        "calendar_week_blocks": n_blocks,
        "point_advantage_mae": float(point_advantage),
        "ci_low_mae": float(ci_low),
        "ci_high_mae": float(ci_high),
        "benchmark_mae": float(benchmark_mae),
        "practical_margin_mae": float(practical_margin),
        "point_advantage_benchmark_pct":
            100.0 * point_advantage / benchmark_mae,
        "ci_low_benchmark_pct":
            100.0 * ci_low / benchmark_mae,
        "ci_high_benchmark_pct":
            100.0 * ci_high / benchmark_mae,
        "practical_margin_benchmark_pct":
            100.0 * practical_margin / benchmark_mae,
        "comparison_status": comparison_status,
    }


def build_pairwise_bootstrap_table(scorecard_df, weekly_df):
    """Compare each point-estimate MAE leader with all three challengers."""
    records = []
    pair_number = 0

    point_winners_df = scorecard_df.loc[
        scorecard_df["mae_rank"].eq(1)
    ].copy()

    for winner in point_winners_df.itertuples(index=False):
        challengers = [
            family
            for family in FAMILY_ORDER
            if family != winner.family
        ]

        for challenger in challengers:
            pair_number += 1

            records.append(
                bootstrap_pairwise_mae_advantage(
                    weekly_df=weekly_df,
                    metric=winner.metric,
                    horizon=int(winner.horizon),
                    reference_family=winner.family,
                    challenger_family=challenger,
                    benchmark_mae=float(winner.benchmark_mae),
                    repeats=BOOTSTRAP_REPEATS,
                    seed=BOOTSTRAP_RANDOM_SEED + pair_number,
                )
            )

    return pd.DataFrame(records)

The last setup helpers create the reader-facing visuals used before model selection.

The charts are intentionally complementary:

1. **Finalist track mix** — what each family is actually entering.
2. **Relative-MAE small multiples** — how large each family's error is relative to the observed mobility amount across all five targets and three horizons.
3. **Benchmark-relative skill heatmap** — how much each finalist improves on Previous week · same Daypart.
4. **Point-estimate leader board** — which family has the lowest validation MAE before uncertainty is considered.
5. **Bootstrap interval view** — whether the closest competition is clearly separated, practically tied, or still uncertain.

Relative MAE uses the project's **sequential heatmap palette** because lower values are simply better; there is no meaningful zero-centered midpoint. Benchmark skill remains diverging because zero represents exact parity with the frozen baseline.

The champion map itself is not frozen here; that happens only at the Decision Point.

In [9]:
# ---------------------------------------------------------------------
# 4.6.1.0H — Reader-facing visualization helpers
# ---------------------------------------------------------------------

def plot_finalist_track_mix(registry_df):
    """Show how many of each family's 15 jobs use each frozen history track."""
    track_mix_df = (
        registry_df.groupby(
            ["Family", "modeling_track"],
            observed=True,
            as_index=False,
        )
        .size()
        .rename(columns={"size": "Jobs"})
    )

    track_mix_df["Track"] = (
        track_mix_df["modeling_track"].map(TRACK_LABELS)
    )

    fig = px.bar(
        track_mix_df,
        x="Family",
        y="Jobs",
        color="Track",
        barmode="stack",
        text="Jobs",
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        title="Four finalized forecasting systems enter the tournament",
        labels={"Family": "", "Jobs": "Target × Horizon jobs"},
    )

    apply_branding(fig)

    fig.update_layout(
        yaxis=dict(range=[0, EXPECTED_JOB_COUNT + 1]),
        legend_title_text="Frozen training-history track",
    )
    fig.update_traces(textposition="inside")

    return fig


def plot_relative_mae_heatmaps(scorecard_df):
    """
    Compare all 60 coverage-adjusted Relative-MAE cells on one color scale.

    A star marks the lowest system Relative MAE for that Target × Horizon.
    Hover text preserves model coverage and exact-common MAE so readers can
    distinguish deployable-system performance from conditional model accuracy.
    """
    display_df = scorecard_df.copy()

    display_df["Target"] = display_df["metric"].map(TARGET_SHORT_LABELS)
    display_df["Family"] = display_df["family"].map(FAMILY_LABELS)

    job_minimum = display_df.groupby(
        ["metric", "horizon"],
        observed=True,
    )["relative_mae_pct"].transform("min")

    display_df["is_job_winner"] = np.isclose(
        display_df["relative_mae_pct"],
        job_minimum,
        rtol=0,
        atol=1e-12,
    )

    target_order = [
        TARGET_SHORT_LABELS[metric]
        for metric in FORECAST_TARGETS
    ]
    horizon_order = FORECAST_HORIZONS

    zmin = 0.0
    zmax = float(display_df["relative_mae_pct"].max())

    fig = make_subplots(
        rows=2,
        cols=2,
        shared_yaxes="all",
        subplot_titles=[
            FAMILY_LABELS[family]
            for family in FAMILY_ORDER
        ],
        horizontal_spacing=0.08,
        vertical_spacing=0.14,
    )

    panel_positions = {
        "classical": (1, 1),
        "tree": (1, 2),
        "neural": (2, 1),
        "transformer": (2, 2),
    }

    for family in FAMILY_ORDER:
        row, col = panel_positions[family]

        family_df = display_df.loc[
            display_df["family"].eq(family)
        ].copy()

        family_df["cell_text"] = family_df.apply(
            lambda record: (
                f"{record['relative_mae_pct']:.1f} ★"
                if record["is_job_winner"]
                else f"{record['relative_mae_pct']:.1f}"
            ),
            axis=1,
        )

        relative_matrix = (
            family_df.pivot(
                index="Target",
                columns="horizon",
                values="relative_mae_pct",
            )
            .reindex(index=target_order, columns=horizon_order)
        )

        text_matrix = (
            family_df.pivot(
                index="Target",
                columns="horizon",
                values="cell_text",
            )
            .reindex(index=target_order, columns=horizon_order)
        )

        system_mae_matrix = (
            family_df.pivot(
                index="Target",
                columns="horizon",
                values="mae",
            )
            .reindex(index=target_order, columns=horizon_order)
        )

        skill_matrix = (
            family_df.pivot(
                index="Target",
                columns="horizon",
                values="benchmark_skill_pct",
            )
            .reindex(index=target_order, columns=horizon_order)
        )

        coverage_matrix = (
            family_df.pivot(
                index="Target",
                columns="horizon",
                values="model_coverage_pct",
            )
            .reindex(index=target_order, columns=horizon_order)
        )

        exact_common_matrix = (
            family_df.pivot(
                index="Target",
                columns="horizon",
                values="exact_common_mae",
            )
            .reindex(index=target_order, columns=horizon_order)
        )

        customdata = np.dstack([
            system_mae_matrix.to_numpy(),
            skill_matrix.to_numpy(),
            coverage_matrix.to_numpy(),
            exact_common_matrix.to_numpy(),
        ])

        fig.add_trace(
            go.Heatmap(
                z=relative_matrix.to_numpy(),
                x=["h=1", "h=2", "h=5"],
                y=target_order,
                text=text_matrix.to_numpy(),
                texttemplate="%{text}",
                customdata=customdata,
                coloraxis="coloraxis",
                hovertemplate=(
                    "<b>%{y} · %{x}</b><br>"
                    "System Relative MAE: %{z:.3f}%<br>"
                    "System MAE: %{customdata[0]:.3f}<br>"
                    "Skill vs weekly benchmark: %{customdata[1]:.3f}%<br>"
                    "Advanced-model coverage: %{customdata[2]:.3f}%<br>"
                    "Exact-common model MAE: %{customdata[3]:.3f}"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

    # Every panel shows the same five mobility targets, so use one synchronized
    # categorical scale. Repeat labels only on the left to keep the grid compact.
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=target_order,
        autorange="reversed",
    )

    fig.update_yaxes(showticklabels=True, row=1, col=1)
    fig.update_yaxes(showticklabels=False, row=1, col=2)
    fig.update_yaxes(showticklabels=True, row=2, col=1)
    fig.update_yaxes(showticklabels=False, row=2, col=2)

    fig.update_layout(
        title=(
            "Coverage-adjusted forecast error across families, targets, and horizons"
        ),
        coloraxis=dict(
            colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
            cmin=zmin,
            cmax=zmax,
            colorbar=dict(title="System Relative<br>MAE (%)"),
        ),
        width = 940,
        height=740,
    )

    apply_branding(fig)

    fig.add_annotation(
        text="★ = lowest coverage-adjusted Relative MAE for that Target × Horizon",
        xref="paper",
        yref="paper",
        x=0,
        y=-0.10,
        showarrow=False,
        xanchor="left",
    )

    return fig


def plot_validation_skill_heatmap(scorecard_df):
    """Show coverage-adjusted benchmark skill for all 60 finalist systems."""
    display_df = scorecard_df.copy()

    display_df["job_label"] = (
        display_df["metric"].map(TARGET_SHORT_LABELS)
        + " · h="
        + display_df["horizon"].astype(int).astype(str)
    )
    display_df["Family"] = display_df["family"].map(FAMILY_LABELS)

    job_order = [
        f"{TARGET_SHORT_LABELS[metric]} · h={horizon}"
        for metric in FORECAST_TARGETS
        for horizon in FORECAST_HORIZONS
    ]

    matrix_df = (
        display_df.pivot(
            index="Family",
            columns="job_label",
            values="benchmark_skill_pct",
        )
        .reindex(
            index=[FAMILY_LABELS[family] for family in FAMILY_ORDER],
            columns=job_order,
        )
    )

    fig = go.Figure(
        data=go.Heatmap(
            z=matrix_df.to_numpy(),
            x=matrix_df.columns,
            y=matrix_df.index,
            colorscale=BRAND_DIVERGING_SEQUENCE[::-1],
            zmid=0,
            colorbar=dict(title="System skill vs<br>benchmark (%)"),
            hovertemplate=(
                "Family: %{y}<br>"
                "Job: %{x}<br>"
                "System-MAE improvement vs benchmark: %{z:.3f}%"
                "<extra></extra>"
            ),
        )
    )

    apply_branding(fig)

    fig.update_layout(
        title="Coverage-adjusted validation skill relative to the frozen benchmark",
        xaxis_title="Target × Horizon",
        yaxis_title="",
        height=430,
    )
    fig.update_xaxes(tickangle=-45)

    return fig


def build_point_winner_summary(scorecard_df):
    """Summarize the first- and second-lowest coverage-adjusted system MAE."""
    ranked_df = scorecard_df.sort_values(
        ["metric", "horizon", "mae"]
    ).copy()

    winner_df = (
        ranked_df.groupby(["metric", "horizon"], observed=True)
        .nth(0)
        .reset_index()
        .rename(columns={
            "family": "point_winner_family",
            "mae": "point_winner_mae",
            "model_coverage_pct": "point_winner_coverage_pct",
            "exact_common_mae": "point_winner_exact_common_mae",
        })
    )

    runner_up_df = (
        ranked_df.groupby(["metric", "horizon"], observed=True)
        .nth(1)
        .reset_index()
        .rename(columns={
            "family": "runner_up_family",
            "mae": "runner_up_mae",
            "model_coverage_pct": "runner_up_coverage_pct",
            "exact_common_mae": "runner_up_exact_common_mae",
        })
    )

    summary_df = winner_df[
        [
            "metric",
            "horizon",
            "point_winner_family",
            "point_winner_mae",
            "point_winner_coverage_pct",
            "point_winner_exact_common_mae",
            "benchmark_mae",
        ]
    ].merge(
        runner_up_df[
            [
                "metric",
                "horizon",
                "runner_up_family",
                "runner_up_mae",
                "runner_up_coverage_pct",
                "runner_up_exact_common_mae",
            ]
        ],
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )

    summary_df["mae_margin"] = (
        summary_df["runner_up_mae"] - summary_df["point_winner_mae"]
    )
    summary_df["margin_vs_benchmark_pct"] = (
        100.0 * summary_df["mae_margin"] / summary_df["benchmark_mae"]
    )

    return summary_df


def plot_point_winner_board(point_winner_df):
    """Show the provisional coverage-adjusted leader for every forecasting job."""
    fig = go.Figure()

    for family in FAMILY_ORDER:
        family_df = point_winner_df.loc[
            point_winner_df["point_winner_family"].eq(family)
        ].copy()

        if family_df.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=family_df["horizon"],
                y=family_df["metric"].map(TARGET_LABELS),
                mode="markers",
                name=FAMILY_LABELS[family],
                marker=dict(
                    size=24,
                    color=FAMILY_COLORS[family],
                    line=dict(width=1),
                ),
                customdata=np.column_stack([
                    family_df["point_winner_mae"],
                    family_df["point_winner_coverage_pct"],
                    family_df["runner_up_family"].map(FAMILY_LABELS),
                    family_df["runner_up_mae"],
                    family_df["runner_up_coverage_pct"],
                    family_df["margin_vs_benchmark_pct"],
                ]),
                hovertemplate=(
                    "<b>%{y} · h=%{x}</b><br>"
                    f"System-MAE leader: {FAMILY_LABELS[family]}<br>"
                    "Leader system MAE: %{customdata[0]:.3f}<br>"
                    "Leader model coverage: %{customdata[1]:.3f}%<br>"
                    "Runner-up: %{customdata[2]}<br>"
                    "Runner-up system MAE: %{customdata[3]:.3f}<br>"
                    "Runner-up model coverage: %{customdata[4]:.3f}%<br>"
                    "Lead as % of benchmark MAE: %{customdata[5]:.3f}%"
                    "<extra></extra>"
                ),
            )
        )

    apply_branding(fig)

    fig.update_layout(
        title="Lowest coverage-adjusted system MAE before uncertainty testing",
        xaxis=dict(
            title="Forecast horizon",
            tickmode="array",
            tickvals=FORECAST_HORIZONS,
            ticktext=["h=1", "h=2", "h=5"],
        ),
        yaxis_title="",
        legend_title_text="Point-estimate leader",
        height=500,
    )

    return fig


def plot_bootstrap_forest(pairwise_df):
    """Show the closest challenger's system-MAE uncertainty interval."""
    closest_df = (
        pairwise_df.sort_values(
            ["metric", "horizon", "point_advantage_mae"]
        )
        .groupby(["metric", "horizon"], observed=True)
        .first()
        .reset_index()
    )

    closest_df["job_label"] = (
        closest_df["metric"].map(TARGET_SHORT_LABELS)
        + " · h="
        + closest_df["horizon"].astype(int).astype(str)
    )

    status_labels = {
        "clear_win": "Clear win",
        "practical_equivalence": "Practical equivalence",
        "uncertain_advantage": "Uncertain advantage",
    }

    closest_df["Status"] = (
        closest_df["comparison_status"].map(status_labels)
    )

    status_colors = {
        "Clear win": BRAND_COLOR_SEQUENCE[0],
        "Practical equivalence": BRAND_COLOR_SEQUENCE[1],
        "Uncertain advantage": BRAND_COLOR_SEQUENCE[2],
    }

    fig = go.Figure()

    for status in status_labels.values():
        status_df = closest_df.loc[closest_df["Status"].eq(status)]

        if status_df.empty:
            continue

        x = status_df["point_advantage_benchmark_pct"]
        lower = x - status_df["ci_low_benchmark_pct"]
        upper = status_df["ci_high_benchmark_pct"] - x

        fig.add_trace(
            go.Scatter(
                x=x,
                y=status_df["job_label"],
                mode="markers",
                name=status,
                marker=dict(size=10, color=status_colors[status]),
                error_x=dict(
                    type="data",
                    symmetric=False,
                    array=upper,
                    arrayminus=lower,
                    thickness=1.5,
                ),
                customdata=np.column_stack([
                    status_df["reference_family"].map(FAMILY_LABELS),
                    status_df["challenger_family"].map(FAMILY_LABELS),
                    status_df["ci_low_benchmark_pct"],
                    status_df["ci_high_benchmark_pct"],
                ]),
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "System-MAE leader: %{customdata[0]}<br>"
                    "Closest challenger: %{customdata[1]}<br>"
                    "System-MAE advantage: %{x:.3f}% of benchmark MAE<br>"
                    "95% interval: %{customdata[2]:.3f}% to "
                    "%{customdata[3]:.3f}%"
                    "<extra></extra>"
                ),
            )
        )

    fig.add_vrect(
        x0=-100 * PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION,
        x1=100 * PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION,
        opacity=0.10,
        line_width=0,
        annotation_text="Practical-equivalence band",
        annotation_position="top left",
    )
    fig.add_vline(x=0, line_dash="dot")
    fig.add_vline(
        x=100 * PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION,
        line_dash="dash",
    )

    apply_branding(fig)

    fig.update_layout(
        title=(
            "Does the coverage-adjusted validation leader "
            "have a meaningful advantage?"
        ),
        xaxis_title=(
            "Leader system-MAE advantage as % of frozen benchmark MAE"
        ),
        yaxis_title="",
        height=650,
        legend_title_text="Bootstrap interpretation",
    )

    return fig

## 4.6.1.1 Frame the Cross-Family Tournament

Before comparing performance, freeze the rules of the competition.

That separation matters because a winner rule should not change after we learn which family benefits from it. The notebook therefore establishes the forecasting jobs, validation population, treatment of unavailable forecasts, primary metric, uncertainty procedure, practical-equivalence rule, secondary evidence, and final-holdout firewall first.

### 4.6.1.1.1 Define the Comparison Question and Frozen Scope

The tournament contains four finalized forecasting systems:

1. **Classical** — the selected SARIMAX system from 4.3.1.
2. **Tree** — the selected LightGBM system from 4.4.1.
3. **Neural** — the target-specific Lag-window MLP system from 4.5.1.
4. **Transformer** — the target-specific PatchTST-inspired system from 4.5.2.

Each system competes across:

$$
5\ targets \times 3\ horizons = 15\ Target \times Horizon\ jobs
$$

A family may legitimately use a different frozen training-history track or configuration. That history remains model provenance rather than part of the physical forecast key.

The central comparison is:

> **Across the same benchmark-supported validation population, which finalized family produces the lowest forecasting-system error after unavailable advanced forecasts fall back to the frozen benchmark?**

The exact four-family model intersection remains visible as a secondary diagnostic, but it no longer determines the selection denominator.

Final-holdout performance does not participate in this tournament.

The compact contract below makes the tournament dimensions explicit and provides a quick guard against accidental drift from the forecasting problem established upstream.

In [10]:
# ---------------------------------------------------------------------
# 4.6.1.1.1A — Display and validate the frozen tournament scope
# ---------------------------------------------------------------------

tournament_scope_df = pd.DataFrame([
    {
        "Contract": "Advanced families",
        "Frozen value": "Classical · Tree · Neural · Transformer",
    },
    {
        "Contract": "Forecast targets",
        "Frozen value": f"{len(FORECAST_TARGETS)} targets",
    },
    {
        "Contract": "Horizons",
        "Frozen value": "h=1 · h=2 · h=5",
    },
    {
        "Contract": "Selection jobs",
        "Frozen value": f"{EXPECTED_JOB_COUNT} Target × Horizon jobs",
    },
    {
        "Contract": "Primary selector",
        "Frozen value": (
            "Coverage-adjusted system MAE on one benchmark-supported "
            "validation population"
        ),
    },
    {
        "Contract": "Unavailable advanced forecast",
        "Frozen value": "Use Previous week · same Daypart as fallback",
    },
    {
        "Contract": "Conditional-accuracy diagnostic",
        "Frozen value": "Advanced-model MAE on exact four-family common rows",
    },
    {
        "Contract": "Cross-target error lens",
        "Frozen value": "Coverage-adjusted Relative MAE (%)",
    },
    {
        "Contract": "Baseline-value lens",
        "Frozen value": "System-MAE skill vs Previous week · same Daypart",
    },
    {
        "Contract": "Production output rule",
        "Frozen value": "prediction_final = max(0, prediction_raw)",
    },
    {
        "Contract": "Frozen benchmark",
        "Frozen value": EXPECTED_BENCHMARK_LABEL,
    },
    {
        "Contract": "Final holdout role",
        "Frozen value": "Generalization test only; never a selector",
    },
])

assert len(FAMILY_ORDER) == 4
assert EXPECTED_JOB_COUNT == 15
assert set(FORECAST_HORIZONS) == {1, 2, 5}
assert FINAL_HOLDOUT_START_DATE == pd.Timestamp("2026-01-05")

display(tournament_scope_df)

,Contract,Frozen value
0,Advanced families,Classical · Tree · Neural · Transformer
1,Forecast targets,5 targets
2,Horizons,h=1 · h=2 · h=5
3,Selection jobs,15 Target × Horizon jobs
4,Primary selector,Coverage-adjusted system MAE on one benchmark-...
5,Unavailable advanced forecast,Use Previous week · same Daypart as fallback
6,Conditional-accuracy diagnostic,Advanced-model MAE on exact four-family common...
7,Cross-target error lens,Coverage-adjusted Relative MAE (%)
8,Baseline-value lens,System-MAE skill vs Previous week · same Daypart
9,Production output rule,"prediction_final = max(0, prediction_raw)"


**Findings.** The cross-family tournament contains four finalized families and 15 Target × Horizon jobs. Each family is evaluated as a **complete forecasting system** on one benchmark-supported validation population: advanced model where available, frozen weekly benchmark where unavailable.

Exact-common advanced-model accuracy remains a secondary diagnostic, while final-holdout performance is excluded from model selection.

### 4.6.1.1.2 Freeze the Winner-Selection Rule

A simple model-only leaderboard is not sufficient when finalists have different forecast coverage.

The selection hierarchy is therefore:

1. **Define one canonical population:** every legal ordinary-validation row with an observed outcome and frozen weekly benchmark.
2. **Construct each family's system forecast:** use its advanced prediction when available and the benchmark otherwise.
3. **Primary accuracy:** lower coverage-adjusted system MAE within each Target × Horizon.
4. **Conditional diagnostic:** retain exact-common advanced-model MAE to show how the models compare where all four can forecast.
5. **Interpret error scale:** use system Relative MAE across different mobility targets.
6. **Interpret value over a simple forecast:** use coverage-adjusted benchmark skill.
7. **Paired uncertainty:** compare the system-MAE leader with each challenger using whole calendar weeks as bootstrap blocks.
8. **Practical importance:** require a clear advantage beyond **1% of the frozen benchmark MAE**.
9. **Near-tie evidence:** inspect absolute system bias first, followed by temporal and geographic breadth.
10. **Operational complexity:** use only if predictive evidence remains effectively indistinguishable.

System MAE, system Relative MAE, and benchmark skill remain interpretation-equivalent within a job because every family uses the same canonical rows, actual values, and benchmark values.

The final holdout never breaks a validation tie.

The machine-readable method contract below records those rules before the tournament executes. This makes the later champion decision auditable rather than dependent on prose alone.

In [11]:
# ---------------------------------------------------------------------
# 4.6.1.1.2A — Freeze the cross-family selection-method contract
# ---------------------------------------------------------------------

selection_method_contract_df = pd.DataFrame([
    {
        "Step": 1,
        "Evidence": "Benchmark-supported validation population",
        "Decision role": (
            "Defines one physical validation denominator for all four families."
        ),
    },
    {
        "Step": 2,
        "Evidence": "Advanced-model coverage + frozen fallback",
        "Decision role": (
            "Uses the advanced prediction when available and the weekly "
            "benchmark otherwise, so unavailable forecasts do not disappear."
        ),
    },
    {
        "Step": 3,
        "Evidence": "Coverage-adjusted system MAE",
        "Decision role": (
            "Primary model-selection metric within each Target × Horizon."
        ),
    },
    {
        "Step": 4,
        "Evidence": "Exact-common advanced-model MAE",
        "Decision role": (
            "Secondary diagnostic of conditional model accuracy when all four "
            "advanced families can forecast the same row."
        ),
    },
    {
        "Step": 5,
        "Evidence": "Coverage-adjusted Relative MAE (%)",
        "Decision role": (
            "Places system-error size on a comparable scale across trips, "
            "riders, and speed without creating a separate vote."
        ),
    },
    {
        "Step": 6,
        "Evidence": "Coverage-adjusted benchmark skill",
        "Decision role": (
            "Shows how much the complete forecasting system improves on "
            "Previous week · same Daypart."
        ),
    },
    {
        "Step": 7,
        "Evidence": "Paired calendar-week block bootstrap",
        "Decision role": (
            "Tests whether the system-MAE advantage persists across whole "
            "periods rather than isolated forecast rows."
        ),
    },
    {
        "Step": 8,
        "Evidence": "1% of frozen benchmark MAE",
        "Decision role": (
            "Defines the practical-equivalence margin for each job."
        ),
    },
    {
        "Step": 9,
        "Evidence": "Absolute system bias",
        "Decision role": (
            "First secondary consideration when system MAE does not establish "
            "a clear practical winner."
        ),
    },
    {
        "Step": 10,
        "Evidence": "Monthly and Taxi Zone breadth",
        "Decision role": (
            "Shows whether near-tie system performance is broad or concentrated."
        ),
    },
    {
        "Step": 11,
        "Evidence": "Operational simplicity",
        "Decision role": (
            "Final tie-break consideration only when predictive evidence "
            "remains effectively indistinguishable."
        ),
    },
])

display(selection_method_contract_df)

,Step,Evidence,Decision role
0,1,Benchmark-supported validation population,Defines one physical validation denominator fo...
1,2,Advanced-model coverage + frozen fallback,Uses the advanced prediction when available an...
2,3,Coverage-adjusted system MAE,Primary model-selection metric within each Tar...
3,4,Exact-common advanced-model MAE,Secondary diagnostic of conditional model accu...
4,5,Coverage-adjusted Relative MAE (%),Places system-error size on a comparable scale...
5,6,Coverage-adjusted benchmark skill,Shows how much the complete forecasting system...
6,7,Paired calendar-week block bootstrap,Tests whether the system-MAE advantage persist...
7,8,1% of frozen benchmark MAE,Defines the practical-equivalence margin for e...
8,9,Absolute system bias,First secondary consideration when system MAE ...
9,10,Monthly and Taxi Zone breadth,Shows whether near-tie system performance is b...


**Findings.** Coverage-adjusted system MAE determines the point-estimate ordering, while exact-common MAE describes conditional model accuracy on rows where all four advanced models are available.

A numerical system-MAE lead becomes a clear winner only when it survives paired calendar-week uncertainty and exceeds the predeclared practical-equivalence margin.

### Section Summary

Four finalized family representatives compete across 15 Target × Horizon jobs on one benchmark-supported ordinary-validation population.

Coverage-adjusted **system MAE** is the primary selector. Unavailable advanced forecasts remain in the population and receive the frozen Previous week · same Daypart fallback. Exact-common advanced-model MAE provides a secondary view of conditional model accuracy.

System Relative MAE, benchmark skill, paired calendar-week uncertainty, bias, and breadth provide interpretation and tie-resolution evidence. Final-holdout performance remains outside model selection.

## 4.6.1.2 Validate the Four Finalists and Their Handoffs

The tournament can proceed only if each source notebook contributes one complete and clearly defined finalist system.

This section does not judge forecasting accuracy. It answers a more basic set of questions first:

- Are the four canonical handoffs physically available?
- Does each registry contain exactly the expected 15 Target × Horizon jobs?
- Did every upstream closeout QA pass?
- Do the row-level forecast files preserve the required forecast keys and scoring fields?
- Are raw forecasts preserved alongside the common nonnegative final-output rule?
- Is the observed target available at row level so Relative MAE can be recomputed independently?
- Are all rows still ordinary validation observations occurring before the final holdout?

A missing or malformed handoff is a source-contract problem, not a reason for 4.6.1 to manufacture a substitute entrant or copy summary metrics from an upstream notebook.

### 4.6.1.2.1 Audit the Four Tournament Entrants

Before comparing model families, verify that each finalist provides the artifacts required for a fair tournament. This audit confirms that each family has a finalized model definition, row-level ordinary-validation predictions, and the metadata needed to align forecasts by Target × Horizon.

In [12]:

# ---------------------------------------------------------------------
# 4.6.1.2.1A — Audit finalist-handoff materialization
# ---------------------------------------------------------------------

family_materialization_df = artifact_materialization_table()

display(
    family_materialization_df.assign(
        Status=np.where(
            family_materialization_df["Materialized"],
            "READY",
            "WAITING",
        )
    )[
        ["Family", "Artifact role", "File", "Status", "Rows"]
    ]
)

ready_artifacts = int(family_materialization_df["Materialized"].sum())
total_artifacts = len(family_materialization_df)

print(
    f"Finalist handoff materialization: "
    f"{ready_artifacts}/{total_artifacts} required artifacts ready.",
    flush=True,
)

,Family,Artifact role,File,Status,Rows
0,Classical,Registry,classical_final_winner_registry.parquet,READY,15
1,Classical,Predictions,classical_selected_validation_predictions.parquet,READY,3938000
2,Classical,Qa,final_classical_model_selection_qa.parquet,READY,20
3,Classical,Manifest,classical_model_selection_manifest.parquet,READY,8
4,Tree,Registry,lgbm_selected_model_registry.parquet,READY,15
5,Tree,Predictions,lgbm_selected_row_level_validation_predictions...,READY,4311531
6,Tree,Qa,lgbm_selected_qa.parquet,READY,9
7,Tree,Manifest,lgbm_selected_manifest.parquet,READY,9
8,Neural,Registry,neural_final_model_registry.parquet,READY,15
9,Neural,Predictions,neural_selected_validation_predictions.parquet,READY,4535124


Finalist handoff materialization: 16/16 required artifacts ready.


**Findings.** All **16 required finalist handoff artifacts are materialized and ready**: one registry, row-level prediction file, QA artifact, and manifest for each of the four model families. The prediction handoffs range from roughly 3.9 million to 4.6 million rows, confirming that all four completed family packages are available for the cross-family tournament. No family is waiting on an upstream artifact.

The next cell is the empirical tournament gate.

If 4.5.2 is still running, it should stop here with a clear missing-artifact message. Once all four completed packages exist, the same cell validates the upstream QA files and loads only the small finalist registries.

In [13]:
# ---------------------------------------------------------------------
# 4.6.1.2.1B — Require completed upstream handoffs and load finalist registries
# ---------------------------------------------------------------------

family_materialization_df = require_complete_family_handoffs()

normalized_registry_parts = []
qa_records = []

for family in FAMILY_ORDER:
    spec = FAMILY_HANDOFFS[family]

    qa_passed, qa_df = qa_file_passes(spec["qa"])

    qa_records.append({
        "family": family,
        "Family": FAMILY_LABELS[family],
        "QA checks": len(qa_df),
        "QA passed": qa_passed,
    })

    if not qa_passed:
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} final upstream QA did not pass."
        )

    registry_df = pd.read_parquet(spec["registry"])
    normalized_registry_parts.append(
        normalize_finalist_registry(family, registry_df)
    )

finalist_registry_df = pd.concat(
    normalized_registry_parts,
    ignore_index=True,
)

upstream_qa_summary_df = pd.DataFrame(qa_records)

With the packages loaded, the next audit asks whether each family really contributes one complete 15-job finalist system. Configuration counts are descriptive: some branches legitimately vary their frozen specification by target or horizon, while others use one common definition across all jobs.

In [14]:
# ---------------------------------------------------------------------
# 4.6.1.2.1C — Validate and summarize the four finalist systems
# ---------------------------------------------------------------------

expected_pairs = expected_target_horizon_pairs()

finalist_audit_records = []

for family in FAMILY_ORDER:
    family_df = finalist_registry_df.loc[
        finalist_registry_df["family"].eq(family)
    ].copy()

    observed_pairs = set(
        family_df[["metric", "horizon"]]
        .itertuples(index=False, name=None)
    )

    if observed_pairs != expected_pairs:
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} finalist registry does not contain "
            "the exact 15 Target × Horizon jobs."
        )

    if not family_df["modeling_track"].isin(ORDINARY_TRACKS).all():
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} contains a non-ordinary modeling track."
        )

    track_counts = (
        family_df.groupby("modeling_track", observed=True)
        .size()
        .to_dict()
    )

    track_text = " · ".join(
        f"{TRACK_LABELS[track]}: {count}"
        for track, count in track_counts.items()
    )

    finalist_audit_records.append({
        "Family": FAMILY_LABELS[family],
        "Advancing representative": FAMILY_REPRESENTATIVES[family],
        "Jobs": len(family_df),
        "Frozen configurations": family_df["configuration_id"].nunique(),
        "Track mix": track_text,
        "QA": "PASS",
        "Authoritative manifest": FAMILY_HANDOFFS[family]["manifest"].name,
    })

finalist_audit_df = pd.DataFrame(finalist_audit_records)

display(finalist_audit_df)

fig = plot_finalist_track_mix(finalist_registry_df)
fig.show()

,Family,Advancing representative,Jobs,Frozen configurations,Track mix,QA,Authoritative manifest
0,Classical,SARIMAX system,15,4,Full-History Regime-Aware: 6 · Post-CP Primary: 9,PASS,classical_model_selection_manifest.parquet
1,Tree,LightGBM,15,1,Full-History Regime-Aware: 15,PASS,lgbm_selected_manifest.parquet
2,Neural,Lag-window MLP system,15,3,Post-CP Primary: 15,PASS,neural_handoff_manifest.parquet
3,Transformer,PatchTST-inspired Transformer,15,2,Full-History Regime-Aware: 6 · Post-CP Primary: 9,PASS,transformer_handoff_manifest.parquet


**Findings.** All four upstream family representatives pass the finalist-roster audit with exactly **15 Target × Horizon jobs** apiece. Classical enters as a coordinated SARIMAX system with **4 frozen configurations** across **6 Full-History** and **9 Post-CP** jobs. Tree enters as one Full-History LightGBM definition across all 15 jobs. Neural enters as a target-specific Lag-window MLP system with **3 frozen configurations**, all on the Post-CP track. Transformer enters with **2 frozen configurations** across **6 Full-History** and **9 Post-CP** jobs.

Those training-history differences are intentional frozen provenance, not a comparison mismatch. The tournament will compare the families on the same future validation observations while preserving each finalist's original training-history choice.

### 4.6.1.2.2 Validate the Cross-Family Prediction Contract

A complete 15-row registry is not enough. The actual model tournament happens on row-level forecasts, so those files must also agree on the basic meaning of a forecast observation.

The adapter preserves archival rows even when a selected model is unavailable. It validates explicit availability semantics, identifies the scoreable subset, and confirms that every finalist contributes scoreable evidence for all 15 jobs. Fair comparison occurs later on the exact intersection of rows that all four finalists can score.

The output rule check matters because the tournament compares the final forecast each system would actually deliver. `prediction_raw` remains preserved for later diagnostics, but it is not substituted for the production forecast during family selection.

This audit works from Parquet metadata and DuckDB summaries rather than loading the large row-level forecast tables into Pandas.

In [15]:
# ---------------------------------------------------------------------
# 4.6.1.2.2A — Audit the four row-level validation handoffs
# ---------------------------------------------------------------------
# Source handoffs are allowed to preserve unavailable validation rows. What
# matters here is that availability is internally honest, every finalist still
# provides scoreable evidence for all 15 jobs, and no final-holdout information
# has entered the ordinary-validation tournament.

with duckdb.connect() as connection:
    create_family_prediction_views(connection)

    prediction_audit_df = pd.DataFrame([
        audit_prediction_view(connection, family)
        for family in FAMILY_ORDER
    ])


# ---------------------------------------------------------------------
# Evaluate genuine contract requirements
# ---------------------------------------------------------------------
audit_checks_df = pd.DataFrame({
    "15 jobs": (
        prediction_audit_df["jobs"].eq(EXPECTED_JOB_COUNT)
    ),
    "15 scoreable jobs": (
        prediction_audit_df["scoreable_jobs"].eq(EXPECTED_JOB_COUNT)
    ),
    "No duplicate rows": (
        prediction_audit_df["duplicate_rows"].eq(0)
    ),
    "No final-holdout rows": (
        prediction_audit_df["final_holdout_rows"].eq(0)
    ),
    "Model availability consistent": (
        prediction_audit_df["model_availability_violations"].eq(0)
    ),
    "Benchmark availability consistent": (
        prediction_audit_df["benchmark_availability_violations"].eq(0)
    ),
    "Final-output rule valid": (
        prediction_audit_df["output_rule_violations"].eq(0)
    ),
})

prediction_contract_pass = audit_checks_df.all(axis=1)

prediction_audit_df["Status"] = np.where(
    prediction_contract_pass,
    "PASS",
    "FAIL",
)


# ---------------------------------------------------------------------
# Reader-facing coverage audit
# ---------------------------------------------------------------------
prediction_audit_display_df = (
    prediction_audit_df[
        [
            "Status",
            "Family",
            "forecast_rows",
            "scoreable_rows",
            "scoreable_coverage_pct",
            "jobs",
            "scoreable_jobs",
            "minimum_target_date",
            "maximum_target_date",
            "duplicate_rows",
            "final_holdout_rows",
            "model_availability_violations",
            "benchmark_availability_violations",
            "output_rule_violations",
        ]
    ]
    .rename(
        columns={
            "forecast_rows": "Forecast rows",
            "scoreable_rows": "Scoreable rows",
            "scoreable_coverage_pct": "Scoreable coverage (%)",
            "jobs": "Jobs",
            "scoreable_jobs": "Scoreable jobs",
            "minimum_target_date": "Minimum target date",
            "maximum_target_date": "Maximum target date",
            "duplicate_rows": "Duplicate rows",
            "final_holdout_rows": "Final-holdout rows",
            "model_availability_violations": "Model availability violations",
            "benchmark_availability_violations": (
                "Benchmark availability violations"
            ),
            "output_rule_violations": "Output-rule violations",
        }
    )
)

prediction_audit_display_df[
    "Scoreable coverage (%)"
] = prediction_audit_display_df[
    "Scoreable coverage (%)"
].round(3)

display(prediction_audit_display_df)


# ---------------------------------------------------------------------
# Fail closed, but explain the exact failed condition
# ---------------------------------------------------------------------
if not prediction_contract_pass.all():
    failure_messages = []

    for row_index in prediction_audit_df.index[
        ~prediction_contract_pass
    ]:
        family = prediction_audit_df.loc[row_index, "Family"]

        failed_checks = [
            check_name
            for check_name in audit_checks_df.columns
            if not bool(audit_checks_df.loc[row_index, check_name])
        ]

        failure_messages.append(
            f"{family}: " + ", ".join(failed_checks)
        )

    raise RuntimeError(
        "Cross-family prediction contract failed:\n- "
        + "\n- ".join(failure_messages)
    )

print(
    "PASS — all four finalist prediction handoffs satisfy the "
    "cross-family validation contract.\n"
    "Coverage differences remain visible for audit; model comparison will "
    "use the exact four-family scoreable intersection.",
    flush=True,
)

,Status,Family,Forecast rows,Scoreable rows,Scoreable coverage (%),Jobs,Scoreable jobs,Minimum target date,Maximum target date,Duplicate rows,Final-holdout rows,Model availability violations,Benchmark availability violations,Output-rule violations
0,PASS,Classical,3938000,3115957,79.125,15,15,2025-04-05,2026-01-04,0,0,0,0,0
1,PASS,Tree,4311531,4311531,100.000,15,15,2025-04-05,2026-01-04,0,0,0,0,0
2,PASS,Neural,4535124,4535124,100.000,15,15,2025-04-05,2026-01-04,0,0,0,0,0
3,PASS,Transformer,4551891,4551891,100.000,15,15,2025-04-05,2026-01-04,0,0,0,0,0


PASS — all four finalist prediction handoffs satisfy the cross-family validation contract.
Coverage differences remain visible for audit; model comparison will use the exact four-family scoreable intersection.


**Findings.** All four row-level finalist handoffs pass the validation contract. Each family covers all **15 Target × Horizon jobs**, spans **April 5, 2025 through January 4, 2026**, contains no duplicate forecast rows, contains no final-holdout observations, and has no availability or final-output-rule violations.

Coverage does differ across families. Tree, Neural, and Transformer provide scoreable advanced-model predictions for **100.0%** of their archived validation rows, while Classical provides scoreable predictions for **79.125%**.

Forecast coverage therefore remains part of the tournament evidence. The selection surface keeps one benchmark-supported population and uses the frozen weekly benchmark wherever a finalist's advanced model is unavailable.

The final contract check compares the upstream benchmark registry with the source handoffs. The two ordinary modeling tracks are allowed to differ in training history, but the cross-family tournament requires the same frozen Post-CP forecasting benchmark for every Target × Horizon.

In [16]:
# ---------------------------------------------------------------------
# 4.6.1.2.2B — Confirm the frozen benchmark and final-holdout boundary
# ---------------------------------------------------------------------

track_contract_df = pd.read_parquet(TRACK_CONTRACT_PATH)
benchmark_registry_df = pd.read_parquet(ADVANCED_BENCHMARK_REGISTRY_PATH)


# ---------------------------------------------------------------------
# 1) Validate the inherited forecasting contracts
# ---------------------------------------------------------------------

require_columns(
    track_contract_df,
    [
        "modeling_track",
        "final_holdout_target_start_date",
        "final_holdout_target_end_date",
    ],
    "4.1.2 forecasting track contract",
)

require_columns(
    benchmark_registry_df,
    [
        "modeling_track",
        "metric",
        "horizon",
        "baseline_id",
        "baseline_label",
        "benchmark_label",
    ],
    "4.2.4 advanced-model benchmark registry",
)


# ---------------------------------------------------------------------
# 2) Recover the ordinary-track benchmark contract
# ---------------------------------------------------------------------
# In the 4.2.4 registry, `benchmark_label` names the benchmark experiment
# (for example, "Post-CP forecasting benchmark"). The actual baseline used
# by downstream models is identified by `baseline_id` and `baseline_label`.
# Validate those source fields directly rather than comparing the experiment
# label with the reader-facing baseline name.

ordinary_benchmark_df = benchmark_registry_df.loc[
    benchmark_registry_df["modeling_track"].isin(ORDINARY_TRACKS)
].copy()

expected_ordinary_registry_rows = (
    len(ORDINARY_TRACKS) * EXPECTED_JOB_COUNT
)

if len(ordinary_benchmark_df) != expected_ordinary_registry_rows:
    raise RuntimeError(
        "Expected "
        f"{expected_ordinary_registry_rows} ordinary-track benchmark rows "
        f"({len(ORDINARY_TRACKS)} tracks × {EXPECTED_JOB_COUNT} jobs); "
        f"found {len(ordinary_benchmark_df)}."
    )

if ordinary_benchmark_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any():
    raise RuntimeError(
        "The 4.2.4 ordinary benchmark registry is not unique by "
        "Modeling Track × Target × Horizon."
    )

if not ordinary_benchmark_df["baseline_id"].eq(
    "previous_week_same_daypart"
).all():
    unexpected_baselines = (
        ordinary_benchmark_df.loc[
            ~ordinary_benchmark_df["baseline_id"].eq(
                "previous_week_same_daypart"
            ),
            [
                "modeling_track",
                "metric",
                "horizon",
                "baseline_id",
                "baseline_label",
            ],
        ]
        .drop_duplicates()
        .sort_values(["modeling_track", "metric", "horizon"])
    )

    display(unexpected_baselines)

    raise RuntimeError(
        "At least one ordinary forecasting job no longer uses the frozen "
        "`previous_week_same_daypart` baseline."
    )

if not ordinary_benchmark_df["baseline_label"].eq(
    EXPECTED_BENCHMARK_LABEL
).all():
    unexpected_labels = (
        ordinary_benchmark_df.loc[
            ~ordinary_benchmark_df["baseline_label"].eq(
                EXPECTED_BENCHMARK_LABEL
            ),
            [
                "modeling_track",
                "metric",
                "horizon",
                "baseline_id",
                "baseline_label",
            ],
        ]
        .drop_duplicates()
        .sort_values(["modeling_track", "metric", "horizon"])
    )

    display(unexpected_labels)

    raise RuntimeError(
        "The frozen ordinary baseline ID is correct, but at least one "
        "reader-facing baseline label no longer matches "
        f"`{EXPECTED_BENCHMARK_LABEL}`."
    )


# ---------------------------------------------------------------------
# 3) Confirm both ordinary tracks resolve to one 15-job benchmark surface
# ---------------------------------------------------------------------

benchmark_pairs_df = (
    ordinary_benchmark_df[
        [
            "metric",
            "horizon",
            "baseline_id",
            "baseline_label",
        ]
    ]
    .drop_duplicates()
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(benchmark_pairs_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The two ordinary modeling tracks do not resolve to one common "
        "15-job benchmark contract."
    )

if set(
    zip(
        benchmark_pairs_df["metric"],
        benchmark_pairs_df["horizon"].astype(int),
    )
) != expected_target_horizon_pairs():
    raise RuntimeError(
        "The frozen ordinary benchmark does not cover the expected "
        "15 Target × Horizon jobs exactly."
    )


# ---------------------------------------------------------------------
# 4) Confirm the untouched final-holdout boundary
# ---------------------------------------------------------------------

ordinary_track_contract_df = track_contract_df.loc[
    track_contract_df["modeling_track"].isin(ORDINARY_TRACKS)
].copy()

observed_holdout_starts = pd.to_datetime(
    ordinary_track_contract_df["final_holdout_target_start_date"]
).dropna().unique()

if len(observed_holdout_starts) != 1:
    raise RuntimeError(
        "The ordinary modeling tracks do not share one final-holdout "
        "start boundary."
    )

observed_holdout_start = pd.Timestamp(
    observed_holdout_starts[0]
)

if observed_holdout_start != FINAL_HOLDOUT_START_DATE:
    raise RuntimeError(
        "The final-holdout boundary no longer matches the frozen "
        "January 5, 2026 contract."
    )


# ---------------------------------------------------------------------
# 5) Curate the handoff-contract QA summary
# ---------------------------------------------------------------------

handoff_contract_qa_df = pd.DataFrame([
    {
        "Check": "Four upstream QA packages passed",
        "Status": "PASS",
        "Evidence": f"{len(upstream_qa_summary_df)}/4 families",
    },
    {
        "Check": "Exact finalist roster",
        "Status": "PASS",
        "Evidence": "4 families × 15 Target × Horizon jobs",
    },
    {
        "Check": "Row-level forecast contract",
        "Status": "PASS",
        "Evidence": (
            "Unique validation rows, explicit availability semantics, "
            "and scoreable support for all 15 jobs"
        ),
    },
    {
        "Check": "Frozen benchmark",
        "Status": "PASS",
        "Evidence": EXPECTED_BENCHMARK_LABEL,
    },
    {
        "Check": "Final holdout remains sealed",
        "Status": "PASS",
        "Evidence": f"Begins {FINAL_HOLDOUT_START_DATE.date()}",
    },
])

display(handoff_contract_qa_df)

,Check,Status,Evidence
0,Four upstream QA packages passed,PASS,4/4 families
1,Exact finalist roster,PASS,4 families × 15 Target × Horizon jobs
2,Row-level forecast contract,PASS,"Unique validation rows, explicit availability ..."
3,Frozen benchmark,PASS,Previous week · same Daypart
4,Final holdout remains sealed,PASS,Begins 2026-01-05


**Findings.** Every entry gate for the cross-family tournament passes. All **4 upstream QA packages** are green, all four families contribute the exact **15-job** finalist roster, and the row-level handoffs preserve the identifiers, observed outcomes, raw forecasts, nonnegative final forecasts, availability semantics, and frozen benchmark predictions needed for independent downstream scoring.

The inherited benchmark is consistently **Previous week · same Daypart**, and the final holdout remains sealed beginning **January 5, 2026**. No holdout rows have entered model-family selection.

### Section Summary

All four finalized family packages are eligible for the tournament. Their registries define one frozen 15-job representative system per family, their upstream QA packages pass, and their row-level ordinary-validation handoffs provide the fields needed to reconstruct the cross-family evidence directly from source predictions.

The families do not need identical training-history tracks or identical advanced-model coverage. Training history remains model provenance; coverage remains visible in the tournament.

The next section therefore builds two complementary surfaces: an exact-common subset for conditional model-to-model diagnostics and one benchmark-supported population for the actual champion-selection decision.

## 4.6.1.3 Construct the Validation Comparison Surfaces

Different model families do not necessarily produce advanced forecasts for exactly the same rows.

Using only the four-way model intersection solves one problem—comparing predictions on identical observations—but creates another: a lower-coverage family can avoid being evaluated on rows where it cannot forecast.

This notebook therefore keeps **two distinct surfaces**:

1. **Exact-common diagnostic surface** — rows where all four advanced models produce valid forecasts. This answers: *when everybody can forecast, whose advanced model is most accurate?*
2. **Coverage-adjusted selection surface** — every benchmark-supported validation row, with each family's advanced prediction used when available and the frozen benchmark used otherwise. This answers: *which complete forecasting system performs best across the population we need to forecast?*

The second surface determines the champion. The first remains a useful diagnostic.

Building the four-way intersection is one of the heavier non-modeling operations in this notebook, so it is restartable.

If a valid common-support artifact already exists, it is recovered automatically. Otherwise, `RUN_COMMON_SUPPORT_BUILD` must be set to `True` in the global setup before this cell will perform the join.

In [17]:
# ---------------------------------------------------------------------
# 4.6.1.3.1A — Audit exact-common overlap and build the selection population
# ---------------------------------------------------------------------

if RESET_COMMON_SUPPORT:
    COMMON_VALIDATION_PATH.unlink(missing_ok=True)
    COMMON_VALIDATION_AUDIT_PATH.unlink(missing_ok=True)


# Ordinary reruns take the cheap path: verify that the exact file previously
# validated at row level has not changed.
common_support_ready = artifact_recovery_audit_matches(
    COMMON_VALIDATION_PATH,
    COMMON_VALIDATION_AUDIT_PATH,
    COMMON_VALIDATION_CONTRACT_ID,
)


# Existing artifacts from before the recovery-marker contract need one
# full semantic validation. Once it passes, later reruns stay on the hot path.
if not common_support_ready and COMMON_VALIDATION_PATH.exists():
    print(
        "Existing common-validation artifact has no current recovery marker; "
        "running one-time semantic validation...",
        flush=True,
    )

    if common_support_artifact_is_valid(COMMON_VALIDATION_PATH):
        write_artifact_recovery_audit(
            COMMON_VALIDATION_PATH,
            COMMON_VALIDATION_AUDIT_PATH,
            COMMON_VALIDATION_CONTRACT_ID,
        )
        common_support_ready = True


if common_support_ready:
    print(
        "Recovered existing four-family common-validation artifact: "
        f"{parquet_row_count(COMMON_VALIDATION_PATH):,} rows.",
        flush=True,
    )

elif RUN_COMMON_SUPPORT_BUILD:
    print(
        "Building exact common validation support across four finalist families...",
        flush=True,
    )

    COMMON_VALIDATION_AUDIT_PATH.unlink(missing_ok=True)
    build_common_validation_artifact(COMMON_VALIDATION_PATH)

    # build_common_validation_artifact() already performs the full semantic
    # validation. Only after that succeeds do we publish the recovery marker.
    write_artifact_recovery_audit(
        COMMON_VALIDATION_PATH,
        COMMON_VALIDATION_AUDIT_PATH,
        COMMON_VALIDATION_CONTRACT_ID,
    )

    print(
        "Common-validation artifact complete: "
        f"{parquet_row_count(COMMON_VALIDATION_PATH):,} rows.",
        flush=True,
    )

else:
    raise RuntimeError(
        "The exact common-validation artifact is unavailable or failed its "
        "recovery contract. Set RUN_COMMON_SUPPORT_BUILD = True in 4.6.1.0B "
        "to rebuild it."
    )


Recovered existing four-family common-validation artifact: 3,115,957 rows.


**Findings.** The four-family exact-common diagnostic contains **3,115,957 forecast rows**. Every row represents an observation for which all four advanced models are simultaneously available.

This surface measures conditional advanced-model accuracy; it does not define the primary selection population.

The overlap audit measures how restrictive the four-family advanced-model intersection is.

Large retention differences indicate that exact-common accuracy describes a narrower forecasting population than the complete-system comparison.

In [18]:
# ---------------------------------------------------------------------
# 4.6.1.3.1B — Audit exact common-support retention
# ---------------------------------------------------------------------

scoreable_filter = """
    COALESCE(isfinite(actual), FALSE)
    AND COALESCE(model_prediction_available, FALSE)
    AND COALESCE(benchmark_prediction_available, FALSE)
    AND COALESCE(isfinite(prediction_raw), FALSE)
    AND COALESCE(isfinite(prediction_final), FALSE)
    AND COALESCE(isfinite(benchmark_prediction), FALSE)
"""

with duckdb.connect() as connection:
    create_family_prediction_views(connection)

    common_job_df = connection.execute(f"""
        SELECT
            metric,
            horizon,
            COUNT(*) AS common_rows,
            COUNT(DISTINCT DATE_TRUNC('week', target_date))
                AS distinct_calendar_weeks
        FROM read_parquet('{sql_path(COMMON_VALIDATION_PATH)}')
        GROUP BY metric, horizon
    """).df()

    duplicate_common_keys = int(connection.execute(f"""
        SELECT COALESCE(SUM(row_count - 1), 0)
        FROM (
            SELECT
                {", ".join(COMMON_FORECAST_KEY_COLUMNS)},
                COUNT(*) AS row_count
            FROM read_parquet('{sql_path(COMMON_VALIDATION_PATH)}')
            GROUP BY {", ".join(COMMON_FORECAST_KEY_COLUMNS)}
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0])

    common_holdout_rows = int(connection.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{sql_path(COMMON_VALIDATION_PATH)}')
        WHERE target_date >= DATE '{FINAL_HOLDOUT_START_DATE.date()}'
    """).fetchone()[0])

    support_records = []
    for family in FAMILY_ORDER:
        family_scoreable_df = connection.execute(f"""
            SELECT metric, horizon, COUNT(*) AS scoreable_rows
            FROM pred_{family}
            WHERE {scoreable_filter}
            GROUP BY metric, horizon
        """).df()

        family_support_df = common_job_df.merge(
            family_scoreable_df,
            on=["metric", "horizon"],
            how="left",
            validate="one_to_one",
        )
        family_support_df["family"] = family
        family_support_df["Family"] = FAMILY_LABELS[family]
        family_support_df["common_rows_to_scoreable_pct"] = (
            100.0
            * family_support_df["common_rows"]
            / family_support_df["scoreable_rows"]
        )
        support_records.append(family_support_df)

validation_support_summary_df = (
    pd.concat(support_records, ignore_index=True)
    .sort_values(["metric", "horizon", "family"])
    .reset_index(drop=True)
)

if len(common_job_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Exact common support does not contain all 15 Target × Horizon jobs."
    )

if validation_support_summary_df[
    ["scoreable_rows", "common_rows", "distinct_calendar_weeks"]
].isna().any().any():
    raise RuntimeError(
        "At least one family or Target × Horizon is missing scoreable "
        "or common calendar support."
    )

if (validation_support_summary_df["common_rows"] <= 0).any():
    raise RuntimeError(
        "At least one Target × Horizon has zero four-family common rows."
    )

if (validation_support_summary_df["distinct_calendar_weeks"] <= 0).any():
    raise RuntimeError(
        "At least one Target × Horizon has no represented calendar weeks."
    )

if duplicate_common_keys != 0:
    raise RuntimeError("Exact common support contains duplicate forecast keys.")

if common_holdout_rows != 0:
    raise RuntimeError("Exact common support contains final-holdout rows.")

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        validation_support_summary_df,
        VALIDATION_SUPPORT_SUMMARY_PATH,
    )

support_display_df = (
    validation_support_summary_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Family",
            "scoreable_rows",
            "common_rows",
            "common_rows_to_scoreable_pct",
            "distinct_calendar_weeks",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "scoreable_rows": "Scoreable rows",
        "common_rows": "Four-family common rows",
        "common_rows_to_scoreable_pct": (
            "Common / scoreable (%)"
        ),
        "distinct_calendar_weeks": "Calendar weeks",
    })
)

overall_common_support_df = pd.DataFrame([
    {
        "Family": FAMILY_LABELS[family],
        "Scoreable rows": int(
            validation_support_summary_df.loc[
                validation_support_summary_df["family"].eq(family),
                "scoreable_rows",
            ].sum()
        ),
        "Four-family common rows": int(
            common_job_df["common_rows"].sum()
        ),
        "Common / scoreable (%)": round(
            100.0
            * common_job_df["common_rows"].sum()
            / validation_support_summary_df.loc[
                validation_support_summary_df["family"].eq(family),
                "scoreable_rows",
            ].sum(),
            3,
        ),
        "Calendar weeks": int(
            common_job_df["distinct_calendar_weeks"].sum()
        ),
    }
    for family in FAMILY_ORDER
])

display(
    support_display_df.sort_values(
        ["Target", "Horizon", "Family"]
    ).round(3)
)
display(overall_common_support_df.round(3))

,Target,Horizon,Family,Scoreable rows,Four-family common rows,Common / scoreable (%),Calendar weeks
0,FHVHV speed,1,Classical,332748,332748,100.000,40
1,FHVHV speed,1,Neural,351204,332748,94.745,40
2,FHVHV speed,1,Transformer,352319,332748,94.445,40
3,FHVHV speed,1,Tree,352319,332748,94.445,40
4,FHVHV speed,2,Classical,332748,332748,100.000,40
5,FHVHV speed,2,Neural,351204,332748,94.745,40
6,FHVHV speed,2,Transformer,352319,332748,94.445,40
7,FHVHV speed,2,Tree,352319,332748,94.445,40
8,FHVHV speed,5,Classical,336845,336845,100.000,40
9,FHVHV speed,5,Neural,351204,336845,95.911,40


,Family,Scoreable rows,Four-family common rows,Common / scoreable (%),Calendar weeks
0,Classical,3115957,3115957,100.000,600
1,Tree,4311531,3115957,72.270,600
2,Neural,4535124,3115957,68.707,600
3,Transformer,4551891,3115957,68.454,600


**Findings.** The exact four-family subset contains **3,115,957 validation rows** across all **15 Target × Horizon jobs**, with **40 calendar weeks represented in every job**.

The subset retains **100.0% of Classical's scoreable advanced-model rows**, compared with **72.3% of Tree**, **68.7% of Neural**, and **68.5% of Transformer** rows overall.

That asymmetry is analytically important. The four-way intersection is useful for conditional model-to-model accuracy, but it is largely bounded by the narrowest advanced-model support. It therefore cannot be the sole population used to select the deployable forecasting system.

The common-row intersection establishes that all four families refer to the same forecast keys, but matching identifiers alone is not enough. Before scoring the finalists, we also need to confirm that those matched rows carry the **same forecasting meaning** across handoffs.

This check verifies three things on the exact common surface:

- **Target dates match exactly** across all four families.
- Independently stored copies of the **observed outcome** and **frozen benchmark prediction** agree within the very small float-aware tolerance used for harmless storage-precision differences.
- Each family's **modeling-track identity remains preserved as provenance**, without requiring the tracks themselves to be identical.

Passing this check means the tournament can safely collapse the repeated outcome and benchmark fields into one canonical comparison surface while keeping each family's predictions and training-history provenance intact.

In [19]:
# ---------------------------------------------------------------------
# 4.6.1.3.1C — Validate semantic alignment on the exact common rows
# ---------------------------------------------------------------------

common_path_sql = sql_path(COMMON_VALIDATION_PATH)

with duckdb.connect() as connection:
    common_alignment_df = connection.execute(f"""
        SELECT
            COUNT(*) AS common_rows,
            COUNT(DISTINCT metric || '|' || CAST(horizon AS VARCHAR))
                AS jobs,
            MIN(target_date) AS minimum_target_date,
            MAX(target_date) AS maximum_target_date,
            MAX(actual_alignment_delta) AS maximum_actual_delta,
            MAX(actual_alignment_tolerance) AS maximum_actual_tolerance,
            MAX(benchmark_alignment_delta)
                AS maximum_benchmark_delta,
            MAX(benchmark_alignment_tolerance)
                AS maximum_benchmark_tolerance,
            SUM(CASE WHEN NOT date_alignment_ok THEN 1 ELSE 0 END)
                AS date_alignment_failures,
            SUM(CASE WHEN NOT COALESCE(actual_alignment_ok, FALSE) THEN 1 ELSE 0 END)
                AS actual_alignment_failures,
            SUM(
                CASE WHEN NOT COALESCE(benchmark_alignment_ok, FALSE) THEN 1 ELSE 0 END
            ) AS benchmark_alignment_failures
        FROM read_parquet('{common_path_sql}')
    """).df()

alignment_row = common_alignment_df.iloc[0]

if int(alignment_row["jobs"]) != EXPECTED_JOB_COUNT:
    raise RuntimeError("Common validation support lost at least one job.")

if (
    int(alignment_row["date_alignment_failures"]) != 0
    or int(alignment_row["actual_alignment_failures"]) != 0
    or int(alignment_row["benchmark_alignment_failures"]) != 0
):
    raise RuntimeError(
        "The common-support artifact contains date or float-aware "
        "semantic disagreement across source handoffs."
    )

display(common_alignment_df.round(3))

,common_rows,jobs,minimum_target_date,maximum_target_date,maximum_actual_delta,maximum_actual_tolerance,maximum_benchmark_delta,maximum_benchmark_tolerance,date_alignment_failures,actual_alignment_failures,benchmark_alignment_failures
0,3115957,15,2025-04-05,2026-01-04,0.000,0.009,0.000,0.009,0.000,0.000,0.000


**Findings.** The exact-common diagnostic contains **3,115,957 rows** across all **15 Target × Horizon jobs**, covering **April 5, 2025 through January 4, 2026**. There are zero target-date, actual-value, or benchmark-value alignment failures.

The diagnostic subset is therefore semantically valid. The next step expands beyond that intersection to the full benchmark-supported validation population while preserving the same physical forecast keys, actual outcomes, and frozen benchmark semantics.

<img src="image-20260913-145530.png" width="" align="" />

**4.6.1.3.1D — Build the coverage-adjusted selection population.**

The decisive selection surface begins with every ordinary-validation forecast row for which the observed outcome and frozen weekly benchmark are available.

Each finalist is then left-joined onto that same population:

- if its advanced model produced a valid `prediction_final`, that value becomes the family's system forecast;
- if the advanced prediction is unavailable, **Previous week · same Daypart** becomes the fallback forecast.

No unavailable model row disappears from selection. A family with narrower advanced-model coverage can still compete, but it receives no advanced-model improvement credit on rows where it must fall back to the benchmark.

In [20]:
# ---------------------------------------------------------------------
# 4.6.1.3.1D — Build or recover the coverage-adjusted selection surface
# ---------------------------------------------------------------------

if RESET_SELECTION_SURFACE:
    SELECTION_VALIDATION_PATH.unlink(missing_ok=True)
    SELECTION_VALIDATION_AUDIT_PATH.unlink(missing_ok=True)


# The normal recovery path checks only the fingerprint created after the last
# successful full semantic validation.
selection_surface_ready = artifact_recovery_audit_matches(
    SELECTION_VALIDATION_PATH,
    SELECTION_VALIDATION_AUDIT_PATH,
    SELECTION_VALIDATION_CONTRACT_ID,
)


# Migrate an already-valid artifact into the cheap recovery contract once.
if not selection_surface_ready and SELECTION_VALIDATION_PATH.exists():
    print(
        "Existing coverage-adjusted surface has no current recovery marker; "
        "running one-time semantic validation...",
        flush=True,
    )

    if selection_surface_artifact_is_valid(SELECTION_VALIDATION_PATH):
        write_artifact_recovery_audit(
            SELECTION_VALIDATION_PATH,
            SELECTION_VALIDATION_AUDIT_PATH,
            SELECTION_VALIDATION_CONTRACT_ID,
        )
        selection_surface_ready = True


if selection_surface_ready:
    print(
        "Recovered existing coverage-adjusted validation surface: "
        f"{parquet_row_count(SELECTION_VALIDATION_PATH):,} rows.",
        flush=True,
    )

elif RUN_SELECTION_SURFACE_BUILD:
    print(
        "Building one benchmark-supported validation population across "
        "all four finalist families...",
        flush=True,
    )

    SELECTION_VALIDATION_AUDIT_PATH.unlink(missing_ok=True)

    build_coverage_adjusted_validation_artifact(
        SELECTION_VALIDATION_PATH
    )

    # The builder has already proved row-level semantics at creation time.
    write_artifact_recovery_audit(
        SELECTION_VALIDATION_PATH,
        SELECTION_VALIDATION_AUDIT_PATH,
        SELECTION_VALIDATION_CONTRACT_ID,
    )

    print(
        "Coverage-adjusted validation surface complete: "
        f"{parquet_row_count(SELECTION_VALIDATION_PATH):,} rows.",
        flush=True,
    )

else:
    raise RuntimeError(
        "The coverage-adjusted validation surface is unavailable or failed "
        "its recovery contract. Set RUN_SELECTION_SURFACE_BUILD = True in "
        "4.6.1.0B to rebuild it."
    )


Recovered existing coverage-adjusted validation surface: 4,558,683 rows.


**Findings.** The coverage-adjusted selection surface contains **4,558,683 ordinary-validation forecast rows**, while the exact four-family advanced-model intersection contains **3,115,957 rows**. The broader selection population is therefore about **46.3% larger**.

Every retained row has a finite observed outcome, a valid frozen weekly benchmark, aligned forecast identity, and an explicit system prediction for each family. This provides one common benchmark-supported population for the deployable-system comparison.

**4.6.1.3.1E — Audit advanced-model coverage and benchmark fallback.**

Before scoring accuracy, verify exactly how much of the common validation population each family serves with its advanced model.

For every Family × Target × Horizon, this audit reports:

- canonical benchmark-supported rows;
- rows served by the advanced model;
- rows receiving benchmark fallback;
- advanced-model coverage; and
- the size of the exact-common diagnostic subset.

The identity check is simple but important:

$$
Advanced\ rows + Fallback\ rows = Canonical\ rows
$$

for every family and forecasting job.

In [21]:
# ---------------------------------------------------------------------
# 4.6.1.3.1E — Audit model coverage and benchmark fallback
# ---------------------------------------------------------------------

selection_coverage_df = build_coverage_adjusted_support_summary(
    SELECTION_VALIDATION_PATH
)

selection_sql = sql_path(SELECTION_VALIDATION_PATH)
common_sql = sql_path(COMMON_VALIDATION_PATH)

with duckdb.connect() as connection:
    canonical_job_df = connection.execute(f"""
        SELECT
            metric,
            horizon,
            COUNT(*) AS canonical_rows,
            COUNT(DISTINCT DATE_TRUNC('week', target_date))
                AS calendar_weeks
        FROM read_parquet('{selection_sql}')
        GROUP BY metric, horizon
    """).df()

    exact_common_job_df = connection.execute(f"""
        SELECT
            metric,
            horizon,
            COUNT(*) AS exact_common_rows
        FROM read_parquet('{common_sql}')
        GROUP BY metric, horizon
    """).df()

selection_coverage_df = (
    selection_coverage_df
    .merge(
        exact_common_job_df,
        on=["metric", "horizon"],
        how="left",
        validate="many_to_one",
    )
)

selection_coverage_df["exact_common_share_pct"] = (
    100.0
    * selection_coverage_df["exact_common_rows"]
    / selection_coverage_df["canonical_rows"]
)

if len(canonical_job_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Coverage-adjusted validation surface does not contain all 15 jobs."
    )

if len(selection_coverage_df) != len(FAMILY_ORDER) * EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Coverage audit does not contain all 60 Family × Target × Horizon rows."
    )

canonical_span_qa_df = (
    selection_coverage_df.groupby(
        ["metric", "horizon"],
        observed=True,
        as_index=False,
    )
    .agg(
        minimum_rows=("canonical_rows", "min"),
        maximum_rows=("canonical_rows", "max"),
    )
)

if not canonical_span_qa_df["minimum_rows"].eq(
    canonical_span_qa_df["maximum_rows"]
).all():
    raise RuntimeError(
        "Families do not share the same canonical validation denominator."
    )

if not selection_coverage_df["model_coverage_pct"].between(0, 100).all():
    raise RuntimeError(
        "At least one family has an invalid advanced-model coverage percentage."
    )

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        selection_coverage_df,
        SELECTION_COVERAGE_SUMMARY_PATH,
    )

coverage_display_df = (
    selection_coverage_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        Family=lambda frame: frame["family"].map(FAMILY_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Family",
            "canonical_rows",
            "model_available_rows",
            "fallback_rows",
            "model_coverage_pct",
            "exact_common_rows",
            "exact_common_share_pct",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "canonical_rows": "Canonical rows",
        "model_available_rows": "Advanced-model rows",
        "fallback_rows": "Benchmark-fallback rows",
        "model_coverage_pct": "Model coverage (%)",
        "exact_common_rows": "Four-family common rows",
        "exact_common_share_pct": "Exact-common share (%)",
    })
)

overall_coverage_df = (
    selection_coverage_df.groupby("family", observed=True, as_index=False)
    .agg(
        canonical_rows=("canonical_rows", "sum"),
        model_available_rows=("model_available_rows", "sum"),
        fallback_rows=("fallback_rows", "sum"),
    )
)

overall_coverage_df["Model coverage (%)"] = (
    100.0
    * overall_coverage_df["model_available_rows"]
    / overall_coverage_df["canonical_rows"]
)
overall_coverage_df["Family"] = (
    overall_coverage_df["family"].map(FAMILY_LABELS)
)

display(
    coverage_display_df.sort_values(
        ["Target", "Horizon", "Family"]
    ).round(3)
)

display(
    overall_coverage_df[
        [
            "Family",
            "canonical_rows",
            "model_available_rows",
            "fallback_rows",
            "Model coverage (%)",
        ]
    ]
    .rename(columns={
        "canonical_rows": "Canonical rows",
        "model_available_rows": "Advanced-model rows",
        "fallback_rows": "Benchmark-fallback rows",
    })
    .round(3)
)


,Target,Horizon,Family,Canonical rows,Advanced-model rows,Benchmark-fallback rows,Model coverage (%),Four-family common rows,Exact-common share (%)
0,FHVHV speed,1,Classical,352319,"332,748.000","19,571.000",94.445,332748,94.445
1,FHVHV speed,1,Neural,352319,"351,204.000","1,115.000",99.684,332748,94.445
2,FHVHV speed,1,Transformer,352319,"352,319.000",0.000,100.000,332748,94.445
3,FHVHV speed,1,Tree,352319,"352,319.000",0.000,100.000,332748,94.445
4,FHVHV speed,2,Classical,352319,"332,748.000","19,571.000",94.445,332748,94.445
5,FHVHV speed,2,Neural,352319,"351,204.000","1,115.000",99.684,332748,94.445
6,FHVHV speed,2,Transformer,352319,"352,319.000",0.000,100.000,332748,94.445
7,FHVHV speed,2,Tree,352319,"352,319.000",0.000,100.000,332748,94.445
8,FHVHV speed,5,Classical,352319,"336,845.000","15,474.000",95.608,336845,95.608
9,FHVHV speed,5,Neural,352319,"351,204.000","1,115.000",99.684,336845,95.608


,Family,Canonical rows,Advanced-model rows,Benchmark-fallback rows,Model coverage (%)
0,Classical,4558683,"3,115,957.000","1,442,726.000",68.352
1,Neural,4558683,"4,535,124.000","23,559.000",99.483
2,Transformer,4558683,"4,551,891.000","6,792.000",99.851
3,Tree,4558683,"4,311,531.000","247,152.000",94.578


**Findings.** Advanced-model coverage differs substantially across the four finalists. Transformer covers **99.851%** of the canonical validation population and Neural **99.483%**, while Tree covers **94.578%** and Classical **68.352%**. Rows without an advanced-model forecast remain in the evaluation through the frozen benchmark fallback.

Coverage differences are concentrated in particular jobs. Classical coverage falls to about **32.8% for FHVHV trips h=2/h=5**, **37.3% for Taxi speed**, and **59.8% for FHVHV trips h=1**. Tree is complete for most jobs but covers about **67.1% of Taxi-speed rows**. Neural and Transformer remain near-complete across almost every job.

The exact-common surface contains **3,115,957 rows**, the same number of rows served by Classical's advanced model. This confirms that exact-common accuracy describes a substantially narrower population than the coverage-adjusted system comparison.

### 4.6.1.3.2 Recompute and Interpret Coverage-Adjusted Accuracy

With the selection population established, the tournament can compare the four finalists as complete forecasting systems.

The evidence is presented in this order:

1. **Coverage and native-unit system accuracy** — how often can the advanced model forecast, and how far off is the complete system?
2. **System Relative MAE (%)** — how large is accumulated system error relative to observed mobility?
3. **System benchmark skill** — how much does the complete system improve on Previous week · same Daypart?
4. **Exact-common MAE** — how do the advanced models compare on the narrower subset where all four are available?
5. **Provisional system leadership** — which family has the lowest coverage-adjusted MAE before uncertainty testing?

Every primary metric uses the same benchmark-supported physical rows within a Target × Horizon.

**4.6.1.3.2A — Recompute the coverage-adjusted validation scorecard.**

The scorecard retains all 60 Family × Target × Horizon combinations. For each one it reports the common validation denominator, advanced-model coverage, benchmark-fallback count, system-level accuracy, and exact-common model accuracy.

The exact-common number is retained for interpretation; **system MAE is the selector**.

In [22]:
# ---------------------------------------------------------------------
# 4.6.1.3.2A — Recompute coverage-adjusted validation performance
# ---------------------------------------------------------------------

validation_scorecard_df = build_validation_scorecard(
    SELECTION_VALIDATION_PATH,
    COMMON_VALIDATION_PATH,
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        validation_scorecard_df,
        VALIDATION_SCORECARD_PATH,
    )

expected_scorecard_rows = len(FAMILY_ORDER) * EXPECTED_JOB_COUNT

if len(validation_scorecard_df) != expected_scorecard_rows:
    raise RuntimeError(
        f"Expected {expected_scorecard_rows} Family × Target × Horizon "
        f"scorecard rows; found {len(validation_scorecard_df)}."
    )

required_metric_columns = [
    "mae",
    "rmse",
    "nmae",
    "relative_mae_pct",
    "bias",
    "benchmark_mae",
    "benchmark_relative_mae_pct",
    "benchmark_skill_pct",
    "model_coverage_pct",
    "fallback_pct",
    "exact_common_mae",
]

if validation_scorecard_df[required_metric_columns].isna().any().any():
    raise RuntimeError(
        "Coverage-adjusted validation scorecard contains missing core metrics."
    )

display(
    validation_scorecard_df[
        [
            "metric",
            "horizon",
            "family",
            "evaluation_rows",
            "model_available_rows",
            "fallback_rows",
            "model_coverage_pct",
            "mae",
            "model_only_mae",
            "exact_common_mae",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "bias",
        ]
    ]
    .rename(columns={
        "metric": "Target",
        "horizon": "Horizon",
        "family": "Family",
        "evaluation_rows": "Canonical rows",
        "model_available_rows": "Advanced-model rows",
        "fallback_rows": "Benchmark-fallback rows",
        "model_coverage_pct": "Model coverage (%)",
        "mae": "System MAE",
        "model_only_mae": "Model-only MAE",
        "exact_common_mae": "Exact-common MAE",
        "relative_mae_pct": "System Relative MAE (%)",
        "benchmark_skill_pct": "System benchmark skill (%)",
        "bias": "System bias",
    })
    .round(3)
)

,Target,Horizon,Family,Canonical rows,Advanced-model rows,Benchmark-fallback rows,Model coverage (%),System MAE,Model-only MAE,Exact-common MAE,System Relative MAE (%),System benchmark skill (%),System bias
0,fhvhv_avg_trip_speed,1,transformer,352319,"352,319.000",0.000,100.000,0.921,0.921,0.815,5.460,30.726,-0.043
1,fhvhv_avg_trip_speed,1,tree,352319,"352,319.000",0.000,100.000,0.938,0.938,0.832,5.563,29.419,0.180
2,fhvhv_avg_trip_speed,1,neural,352319,"351,204.000","1,115.000",99.684,1.014,1.001,0.905,6.013,23.702,0.020
3,fhvhv_avg_trip_speed,1,classical,352319,"332,748.000","19,571.000",94.445,1.134,0.983,0.983,6.727,14.641,0.081
4,fhvhv_avg_trip_speed,2,transformer,352319,"352,319.000",0.000,100.000,0.973,0.973,0.869,5.771,26.773,0.007
5,fhvhv_avg_trip_speed,2,tree,352319,"352,319.000",0.000,100.000,0.982,0.982,0.878,5.824,26.107,0.238
6,fhvhv_avg_trip_speed,2,neural,352319,"351,204.000","1,115.000",99.684,1.076,1.063,0.969,6.381,19.033,-0.043
7,fhvhv_avg_trip_speed,2,classical,352319,"332,748.000","19,571.000",94.445,1.194,1.046,1.046,7.084,10.113,0.107
8,fhvhv_avg_trip_speed,5,tree,352319,"352,319.000",0.000,100.000,0.978,0.978,0.893,5.803,26.365,0.178
9,fhvhv_avg_trip_speed,5,transformer,352319,"352,319.000",0.000,100.000,0.990,0.990,0.905,5.872,25.492,-0.018


**Findings.** Coverage has a substantial effect when an advanced model is available only intermittently. Classical's FHVHV-trips h=2 MAE is **33.257 on advanced-model rows** but **69.255 for the complete forecasting system**, with only **32.830%** advanced-model coverage. Taxi-speed h=1 similarly has **1.502 model-only MAE** and **2.898 system MAE** at **37.297%** coverage.

Families with near-complete coverage show much smaller differences between model-only and system performance. The coverage-adjusted score therefore measures the operational forecasting system directly: advanced prediction where available, frozen weekly benchmark otherwise.

**4.6.1.3.2B — Examine native-unit system accuracy.**

Coverage-adjusted **system MAE** remains the primary selector because it answers the deployment question directly:

> **Across the full benchmark-supported validation population, how far off is this forecasting system on average?**

When the advanced model is available, its error contributes normally. When it is unavailable, the frozen weekly benchmark contributes the error for that row.

The units remain target-specific, so system MAE is compared only within the same Target × Horizon and is never averaged across trips, riders, and speed.

In [23]:
# ---------------------------------------------------------------------
# 4.6.1.3.2B — Compare coverage-adjusted system MAE within each job
# ---------------------------------------------------------------------

system_mae_table_df = (
    validation_scorecard_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        Family=lambda frame: frame["family"].map(FAMILY_LABELS),
    )
    .pivot(
        index=["Target", "horizon"],
        columns="Family",
        values="mae",
    )
    .reindex(columns=[FAMILY_LABELS[family] for family in FAMILY_ORDER])
    .reset_index()
    .rename(columns={"horizon": "Horizon"})
)

target_order_lookup = {
    TARGET_SHORT_LABELS[metric]: index
    for index, metric in enumerate(FORECAST_TARGETS)
}

system_mae_table_df["_target_order"] = (
    system_mae_table_df["Target"].map(target_order_lookup)
)

system_mae_table_df = (
    system_mae_table_df
    .sort_values(["_target_order", "Horizon"])
    .drop(columns="_target_order")
    .reset_index(drop=True)
)

display(system_mae_table_df.round(3))

Family,Target,Horizon,Classical,Tree,Neural,Transformer
0,Taxi trips,1,19.804,14.077,16.747,16.331
1,Taxi trips,2,19.886,15.078,18.981,19.101
2,Taxi trips,5,19.958,14.151,19.798,17.794
3,Taxi speed,1,2.898,2.599,2.178,2.168
4,Taxi speed,2,2.914,2.613,2.195,2.196
5,Taxi speed,5,2.917,2.581,2.220,2.227
6,FHVHV trips,1,66.296,47.632,62.251,58.265
7,FHVHV trips,2,69.255,52.439,62.527,61.866
8,FHVHV trips,5,69.401,49.978,66.862,62.447
9,FHVHV speed,1,1.134,0.938,1.014,0.921


**Findings.** Tree has the lowest system MAE for all three Taxi-trip horizons. For Taxi speed, Transformer leads at h=1 while Neural leads at h=2 and h=5. Tree leads all three FHVHV-trip horizons, Transformer leads FHVHV speed at h=1 and h=2, and Tree leads FHVHV speed at h=5. For Subway ridership, Tree leads h=1 and h=5 while Transformer leads h=2.

Coverage-adjusted scoring therefore produces a specialized leadership pattern rather than one universally dominant family. Tree is strongest across the trip-count targets, Transformer is strongest in several short-horizon speed and Subway forecasts, and Neural is strongest for Taxi speed at the longer two horizons.

**4.6.1.3.2C — Compare system Relative MAE across families, targets, and horizons.**

Trips, riders, and speed use different numerical scales, so their native-unit system MAEs cannot be compared directly across targets.

System Relative MAE expresses total absolute **system** forecast error as a percentage of total observed mobility:

$$
Relative\ MAE\ (\%) =
100 \times
\frac{\sum |system\_prediction - actual|}
     {\sum |actual|}
$$

This is not MAPE.

Because every family uses the same canonical rows and the same observed-value denominator within a Target × Horizon, system Relative MAE preserves the same family ordering as system MAE. Advanced-model coverage and exact-common MAE remain visible in the chart hover so the reader can understand why a family's deployable-system result may differ from its conditional model result.

In [24]:
# ---------------------------------------------------------------------
# 4.6.1.3.2C — Compare coverage-adjusted Relative MAE on one scale
# ---------------------------------------------------------------------

relative_denominator_qa_df = (
    validation_scorecard_df.groupby(
        ["metric", "horizon"],
        observed=True,
        as_index=False,
    )
    .agg(
        minimum_observed_abs_sum=("observed_abs_sum", "min"),
        maximum_observed_abs_sum=("observed_abs_sum", "max"),
    )
)

relative_denominator_qa_df["denominator_spread"] = (
    relative_denominator_qa_df["maximum_observed_abs_sum"]
    - relative_denominator_qa_df["minimum_observed_abs_sum"]
)

if relative_denominator_qa_df["denominator_spread"].abs().max() > 0.0:
    raise RuntimeError(
        "Relative-MAE denominators differ across families within at least "
        "one Target × Horizon. The coverage-adjusted selection surface is "
        "internally inconsistent."
    )

fig = plot_relative_mae_heatmaps(validation_scorecard_df)
fig.show()

**Findings.** Forecast difficulty remains strongly target-dependent. **FHVHV speed is the easiest target**, with system Relative MAE of roughly **5.5–7.2%** across families and horizons. FHVHV trips and Subway occupy the middle range, while **Taxi trips and Taxi speed are generally harder**, reaching roughly **19–21%** for the weakest systems.

Horizon effects are smaller and less consistent than target effects. Some family-target combinations worsen modestly with longer horizons, while others remain nearly flat or improve slightly. The dominant distinction is therefore **what is being forecast**, not simply how far ahead the forecast extends.

**4.6.1.3.2D — Compare the complete forecasting systems with the frozen weekly benchmark.**

Coverage-adjusted benchmark skill asks:

> **Across the full validation population, how much does this family's complete forecasting system reduce MAE compared with always using Previous week · same Daypart?**

Rows served by benchmark fallback contribute zero advanced-model improvement by construction. They do not disappear and they are not given an arbitrary numerical penalty.

Positive skill means the complete system improves on the frozen benchmark; zero means it ties the benchmark; negative skill means the combined advanced-model-plus-fallback system is worse.

In [25]:
# ---------------------------------------------------------------------
# 4.6.1.3.2D — Visualize improvement over the frozen weekly benchmark
# ---------------------------------------------------------------------

fig = plot_validation_skill_heatmap(validation_scorecard_df)
fig.show()

**Findings.** Tree and Transformer beat the frozen weekly benchmark on **all 15 Target × Horizon jobs**. Neural and Classical each beat it on **12 of 15**, with their negative-skill cases concentrated in Subway ridership.

Because unavailable advanced forecasts use the benchmark itself, those rows contribute zero advanced-model improvement rather than disappearing from evaluation. Positive system skill therefore represents improvement delivered across the complete validation population.

**4.6.1.3.2E — Validate metric consistency and build the provisional system leaderboard.**

System MAE, system Relative MAE, and system benchmark skill answer different reader questions, but they share the same canonical rows within each Target × Horizon.

Therefore:

- system MAE divides total system error by the same row count;
- system Relative MAE divides that error by the same observed-activity denominator; and
- benchmark skill compares system MAE with the same frozen benchmark MAE.

They must produce the same family ordering within a job. Any disagreement is a scoring-contract error.

Only after that invariant passes do we build the provisional **coverage-adjusted system-MAE** leaderboard.

In [26]:
# ---------------------------------------------------------------------
# 4.6.1.3.2E — Verify ranking invariants and build point leaders
# ---------------------------------------------------------------------

benchmark_consistency_df = (
    validation_scorecard_df.groupby(
        ["metric", "horizon"],
        observed=True,
        as_index=False,
    )
    .agg(
        minimum_benchmark_mae=("benchmark_mae", "min"),
        maximum_benchmark_mae=("benchmark_mae", "max"),
    )
)

benchmark_consistency_df["benchmark_mae_spread"] = (
    benchmark_consistency_df["maximum_benchmark_mae"]
    - benchmark_consistency_df["minimum_benchmark_mae"]
)

if benchmark_consistency_df["benchmark_mae_spread"].abs().max() > 0.0:
    raise RuntimeError(
        "Benchmark MAE differs across families within at least one job. "
        "The coverage-adjusted selection surface is internally inconsistent."
    )

rank_consistency_df = validation_scorecard_df[
    [
        "metric",
        "horizon",
        "family",
        "mae_comparison_rank",
        "relative_mae_rank",
        "benchmark_skill_rank",
    ]
].copy()

rank_consistency_df["mae_matches_relative_mae"] = (
    rank_consistency_df["mae_comparison_rank"]
    .eq(rank_consistency_df["relative_mae_rank"])
)

rank_consistency_df["mae_matches_benchmark_skill"] = (
    rank_consistency_df["mae_comparison_rank"]
    .eq(rank_consistency_df["benchmark_skill_rank"])
)

ranking_inconsistency_df = rank_consistency_df.loc[
    ~rank_consistency_df["mae_matches_relative_mae"]
    | ~rank_consistency_df["mae_matches_benchmark_skill"]
].copy()

if not ranking_inconsistency_df.empty:
    raise RuntimeError(
        "System MAE, system Relative MAE, and system benchmark-skill rankings "
        "disagree. With one shared denominator, this indicates a scoring bug."
    )

metric_rank_qa_df = pd.DataFrame([
    {
        "Check": "Canonical validation rows identical within every job",
        "Status": "PASS",
        "Violations": 0,
    },
    {
        "Check": "Relative-MAE denominator identical within every job",
        "Status": "PASS",
        "Violations": 0,
    },
    {
        "Check": "Benchmark MAE identical within every job",
        "Status": "PASS",
        "Violations": 0,
    },
    {
        "Check": "System MAE and Relative-MAE family rankings agree",
        "Status": "PASS",
        "Violations": 0,
    },
    {
        "Check": "System MAE and benchmark-skill family rankings agree",
        "Status": "PASS",
        "Violations": 0,
    },
])

display(metric_rank_qa_df)

point_winner_df = build_point_winner_summary(
    validation_scorecard_df
)

point_winner_display_df = (
    point_winner_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        **{
            "Point leader": lambda frame:
                frame["point_winner_family"].map(FAMILY_LABELS),
            "Runner-up": lambda frame:
                frame["runner_up_family"].map(FAMILY_LABELS),
        },
    )[
        [
            "Target",
            "horizon",
            "Point leader",
            "point_winner_mae",
            "point_winner_coverage_pct",
            "Runner-up",
            "runner_up_mae",
            "runner_up_coverage_pct",
            "margin_vs_benchmark_pct",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "point_winner_mae": "Leader system MAE",
        "point_winner_coverage_pct": "Leader model coverage (%)",
        "runner_up_mae": "Runner-up system MAE",
        "runner_up_coverage_pct": "Runner-up model coverage (%)",
        "margin_vs_benchmark_pct": "Lead as % of benchmark MAE",
    })
)

display(point_winner_display_df.round(3))

fig = plot_point_winner_board(point_winner_df)
fig.show()

,Check,Status,Violations
0,Canonical validation rows identical within eve...,PASS,0
1,Relative-MAE denominator identical within ever...,PASS,0
2,Benchmark MAE identical within every job,PASS,0
3,System MAE and Relative-MAE family rankings agree,PASS,0
4,System MAE and benchmark-skill family rankings...,PASS,0


,Target,Horizon,Point leader,Leader system MAE,Leader model coverage (%),Runner-up,Runner-up system MAE,Runner-up model coverage (%),Lead as % of benchmark MAE
0,FHVHV speed,1,Transformer,0.921,100.000,Tree,0.938,100.000,1.307
1,FHVHV speed,2,Transformer,0.973,100.000,Tree,0.982,100.000,0.666
2,FHVHV speed,5,Tree,0.978,100.000,Transformer,0.990,100.000,0.873
3,FHVHV trips,1,Tree,47.632,100.000,Transformer,58.265,99.680,15.315
4,FHVHV trips,2,Tree,52.439,100.000,Transformer,61.866,99.680,13.578
5,FHVHV trips,5,Tree,49.978,100.000,Transformer,62.447,99.680,17.961
6,Subway,1,Tree,416.479,100.000,Transformer,447.209,100.000,5.779
7,Subway,2,Transformer,475.990,100.000,Tree,478.138,100.000,0.404
8,Subway,5,Tree,441.537,100.000,Transformer,477.380,100.000,6.740
9,Taxi speed,1,Transformer,2.168,100.000,Neural,2.178,100.000,0.331


**Findings.** All five scoring invariants pass: every family uses the same canonical row count, observed-value denominator, and benchmark MAE within each job, and the system-MAE, Relative-MAE, and benchmark-skill rankings agree exactly.

The point-estimate leaderboard contains **9 Tree leads, 4 Transformer leads, 2 Neural leads, and no Classical leads**. Several races are extremely close: Neural's Taxi-speed h=2 lead is only **0.029% of benchmark MAE**, Taxi-speed h=5 only **0.240%**, and Transformer's Taxi-speed h=1 lead only **0.331%**. Tree's trip-count advantages are substantially larger, reaching **19.337% of benchmark MAE** for Taxi trips h=2.

These point estimates establish the provisional leaders; paired uncertainty determines whether the smaller advantages are strong enough to support selection.

### 4.6.1.3.3 Characterize the Tournament Landscape

Before statistical uncertainty is introduced, one final descriptive view summarizes how concentrated the point-estimate race is.

This section combines the perspectives established above without collapsing incompatible targets into one error average:

- Which families lead the most Target × Horizon jobs?
- Are those leads large or narrow?
- Do speed and demand targets appear to favor different families?
- Does Relative MAE deteriorate as the horizon moves from h=1 toward h=5?
- Is one architecture broadly dominant, or does the evidence support a coordinated multi-family forecasting system?

The aim is to understand the **shape of the tournament**, not to declare champions before uncertainty testing.

The summary below counts point-estimate MAE leads and separately preserves each Family × Target horizon profile. It does **not** average Relative MAE across the five targets.

In [27]:
# ---------------------------------------------------------------------
# 4.6.1.3.3A — Summarize system-MAE leadership and horizon profiles
# ---------------------------------------------------------------------

point_win_mix_df = (
    point_winner_df.groupby(
        "point_winner_family",
        observed=True,
        as_index=False,
    )
    .agg(
        point_estimate_wins=("horizon", "size"),
        median_margin_vs_benchmark_pct=(
            "margin_vs_benchmark_pct",
            "median",
        ),
        minimum_margin_vs_benchmark_pct=(
            "margin_vs_benchmark_pct",
            "min",
        ),
        maximum_margin_vs_benchmark_pct=(
            "margin_vs_benchmark_pct",
            "max",
        ),
    )
)

point_win_mix_df["Family"] = (
    point_win_mix_df["point_winner_family"].map(FAMILY_LABELS)
)

display(
    point_win_mix_df[
        [
            "Family",
            "point_estimate_wins",
            "median_margin_vs_benchmark_pct",
            "minimum_margin_vs_benchmark_pct",
            "maximum_margin_vs_benchmark_pct",
        ]
    ]
    .rename(columns={
        "point_estimate_wins": "Lowest system-MAE jobs",
        "median_margin_vs_benchmark_pct": "Median lead (% benchmark MAE)",
        "minimum_margin_vs_benchmark_pct": "Smallest lead (% benchmark MAE)",
        "maximum_margin_vs_benchmark_pct": "Largest lead (% benchmark MAE)",
    })
    .round(3)
)

relative_horizon_profile_df = (
    validation_scorecard_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        Family=lambda frame: frame["family"].map(FAMILY_LABELS),
    )
    .pivot(
        index=["Family", "Target"],
        columns="horizon",
        values="relative_mae_pct",
    )
    .reindex(columns=FORECAST_HORIZONS)
    .reset_index()
    .rename(columns={
        1: "h=1 System Relative MAE (%)",
        2: "h=2 System Relative MAE (%)",
        5: "h=5 System Relative MAE (%)",
    })
)

relative_horizon_profile_df["h=5 minus h=1 (pp)"] = (
    relative_horizon_profile_df["h=5 System Relative MAE (%)"]
    - relative_horizon_profile_df["h=1 System Relative MAE (%)"]
)

display(relative_horizon_profile_df.round(3))

,Family,Lowest system-MAE jobs,Median lead (% benchmark MAE),Smallest lead (% benchmark MAE),Largest lead (% benchmark MAE)
0,Neural,2,0.135,0.029,0.240
1,Transformer,4,0.535,0.331,1.307
2,Tree,9,13.578,0.873,19.337


horizon,Family,Target,h=1 System Relative MAE (%),h=2 System Relative MAE (%),h=5 System Relative MAE (%),h=5 minus h=1 (pp)
0,Classical,FHVHV speed,6.727,7.084,7.150,0.423
1,Classical,FHVHV trips,12.806,13.377,13.405,0.600
2,Classical,Subway,11.631,12.914,14.073,2.442
3,Classical,Taxi speed,20.974,21.087,21.108,0.134
4,Classical,Taxi trips,19.005,19.083,19.152,0.147
5,Neural,FHVHV speed,6.013,6.381,6.412,0.399
6,Neural,FHVHV trips,12.024,12.078,12.915,0.891
7,Neural,Subway,11.612,12.403,12.051,0.439
8,Neural,Taxi speed,15.759,15.884,16.066,0.307
9,Neural,Taxi trips,16.071,18.214,18.999,2.928


**Findings.** Tree leads **9 of 15** Target × Horizon jobs by system MAE, Transformer leads **4**, Neural leads **2**, and Classical leads none.

The family strengths are specialized. Tree is strongest across the trip-count jobs and several longer-horizon or Subway forecasts. Transformer is strongest in short-horizon speed forecasting and Subway h=2, while Neural leads Taxi speed at h=2 and h=5.

Horizon behavior is also family- and target-specific rather than uniformly worsening with distance. Tree's Taxi-speed Relative MAE, for example, is slightly lower at h=5 than h=1, while Neural's Taxi-trip error increases by almost three percentage points. Target structure and model family therefore matter more than horizon alone.

### Section Summary

The tournament evaluates all four finalists on **4,558,683 benchmark-supported validation rows**. Transformer and Neural provide advanced forecasts for nearly the entire population, Tree has broad coverage with a notable Taxi-speed gap, and Classical covers about two-thirds of the population overall.

Using advanced predictions when available and the frozen weekly benchmark otherwise, the system-MAE leaderboard is **Tree 9, Transformer 4, Neural 2, Classical 0**.

The leadership pattern is strongly target-specific. Tree dominates the trip-count jobs, Transformer is strongest in several speed and Subway forecasts, and Neural leads the longer Taxi-speed horizons. The next section tests whether these system-level advantages are stable and practically meaningful enough to support champion selection.

## 4.6.1.4 Compare Finalists and Select the Champion System

The previous section establishes the coverage-adjusted system-MAE leader for each Target × Horizon. This section asks whether that advantage is convincing enough to freeze a champion.

The paired uncertainty analysis uses the same canonical validation population and the same system forecasts used by the primary selector. Whole calendar weeks are resampled together so shared temporal conditions remain clustered rather than pretending that millions of zone-level rows are independent.

A point-estimate system-MAE leader becomes a clear winner only when its advantage survives the weekly block bootstrap and exceeds the predeclared practical-equivalence margin.

<img src="image-20260913-144042.png" width="" align="" />

### 4.6.1.4.1 Quantify Pairwise Uncertainty and Practical Equivalence

The bootstrap operates on weekly **system-error totals**.

For each Target × Horizon:

1. keep the coverage-adjusted system-MAE leader fixed;
2. compare it separately with each challenger;
3. keep every canonical validation row from the same calendar week together;
4. repeatedly resample those weeks with replacement;
5. recompute the paired system-MAE advantage 5,000 times; and
6. use the middle 95% as the uncertainty interval.

The question is not whether two architectures are mathematically identical. It is whether one **deployable forecasting system** has a stable, practically meaningful validation advantage.

The weekly aggregation is compact and reusable, so it is written once before the bootstrap begins.

In [28]:
# ---------------------------------------------------------------------
# 4.6.1.4.1A — Build compact calendar-week system-error blocks
# ---------------------------------------------------------------------

# The bootstrap must use the same coverage-adjusted population as the
# primary selector so unavailable advanced forecasts remain represented
# through the frozen benchmark fallback.
weekly_error_df = build_weekly_error_summary(
    SELECTION_VALIDATION_PATH
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        weekly_error_df,
        WEEKLY_ERROR_PATH,
    )

weekly_block_audit_df = (
    weekly_error_df.groupby(
        ["metric", "horizon", "family"],
        observed=True,
        as_index=False,
    )
    .agg(
        calendar_week_blocks=("week_start", "nunique"),
        evaluation_rows=("evaluation_rows", "sum"),
    )
)

display(
    weekly_block_audit_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        Family=lambda frame: frame["family"].map(FAMILY_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Family",
            "calendar_week_blocks",
            "evaluation_rows",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "calendar_week_blocks": "Calendar-week blocks",
        "evaluation_rows": "Canonical rows",
    })
)

,Target,Horizon,Family,Calendar-week blocks,Canonical rows
0,FHVHV speed,1,Classical,40,352319
1,FHVHV speed,1,Neural,40,352319
2,FHVHV speed,1,Transformer,40,352319
3,FHVHV speed,1,Tree,40,352319
4,FHVHV speed,2,Classical,40,352319
5,FHVHV speed,2,Neural,40,352319
6,FHVHV speed,2,Transformer,40,352319
7,FHVHV speed,2,Tree,40,352319
8,FHVHV speed,5,Classical,40,352319
9,FHVHV speed,5,Neural,40,352319


**Findings.** All **60 Family × Target × Horizon weekly summaries** contain exactly **40 calendar-week blocks**. Within every Target × Horizon, all four families also contain the same canonical row count.

The bootstrap is therefore fully paired: each family is evaluated over the same weeks and physical validation observations, so differences reflect forecasting performance rather than unequal evaluation support.

The bootstrap itself is restartable. If the completed pairwise table already exists, it is reused. Otherwise the notebook requires the global `RUN_PAIRWISE_BOOTSTRAP` control to be enabled explicitly.

In [29]:
# ---------------------------------------------------------------------
# 4.6.1.4.1B — Run or recover the coverage-adjusted weekly bootstrap
# ---------------------------------------------------------------------

# Initialize the analytical object explicitly so downstream blocks can consume
# bootstrap evidence only after it satisfies the current execution contract.
pairwise_bootstrap_df = None

if RESET_PAIRWISE_BOOTSTRAP and PAIRWISE_BOOTSTRAP_PATH.exists():
    PAIRWISE_BOOTSTRAP_PATH.unlink()

if PAIRWISE_BOOTSTRAP_PATH.exists():
    pairwise_bootstrap_df = pd.read_parquet(
        PAIRWISE_BOOTSTRAP_PATH
    )

    print(
        "Recovered existing coverage-adjusted pairwise bootstrap: "
        f"{len(pairwise_bootstrap_df):,} comparisons.",
        flush=True,
    )

elif RUN_PAIRWISE_BOOTSTRAP:
    pairwise_bootstrap_df = build_pairwise_bootstrap_table(
        validation_scorecard_df,
        weekly_error_df,
    )

    if WRITE_INTERMEDIATES:
        atomic_write_parquet(
            pairwise_bootstrap_df,
            PAIRWISE_BOOTSTRAP_PATH,
        )

    print(
        "Coverage-adjusted paired bootstrap complete: "
        f"{len(pairwise_bootstrap_df):,} comparisons × "
        f"{BOOTSTRAP_REPEATS:,} weekly resamples.",
        flush=True,
    )

else:
    raise RuntimeError(
        "Coverage-adjusted pairwise bootstrap evidence is not available. "
        "Set RUN_PAIRWISE_BOOTSTRAP = True in 4.6.1.0B, rerun 0B, "
        "and then rerun this block."
    )


# ---------------------------------------------------------------------
# Validate bootstrap alignment with the system-MAE leaderboard
# ---------------------------------------------------------------------
# Every reference family must match the point-estimate leader for the same
# Target × Horizon before the bootstrap can support champion selection.

expected_comparisons = EXPECTED_JOB_COUNT * (len(FAMILY_ORDER) - 1)

if len(pairwise_bootstrap_df) != expected_comparisons:
    raise RuntimeError(
        f"Expected {expected_comparisons} leader-versus-challenger "
        f"comparisons; found {len(pairwise_bootstrap_df)}."
    )

pair_key = [
    "metric",
    "horizon",
    "reference_family",
    "challenger_family",
]

if pairwise_bootstrap_df.duplicated(pair_key).any():
    raise RuntimeError(
        "Pairwise bootstrap contains duplicate leader-versus-challenger rows."
    )

allowed_statuses = {
    "clear_win",
    "practical_equivalence",
    "uncertain_advantage",
}

unexpected_statuses = (
    set(pairwise_bootstrap_df["comparison_status"])
    - allowed_statuses
)

if unexpected_statuses:
    raise RuntimeError(
        "Pairwise bootstrap contains unexpected comparison statuses: "
        f"{sorted(unexpected_statuses)}."
    )

expected_leader_df = (
    point_winner_df[
        ["metric", "horizon", "point_winner_family"]
    ]
    .rename(
        columns={
            "point_winner_family": "expected_reference_family",
        }
    )
)

observed_leader_df = (
    pairwise_bootstrap_df[
        ["metric", "horizon", "reference_family"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "reference_family": "observed_reference_family",
        }
    )
)

leader_audit_df = expected_leader_df.merge(
    observed_leader_df,
    on=["metric", "horizon"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

leader_mismatch_df = leader_audit_df.loc[
    ~leader_audit_df["_merge"].eq("both")
    | ~leader_audit_df["expected_reference_family"].eq(
        leader_audit_df["observed_reference_family"]
    )
]

if not leader_mismatch_df.empty:
    raise RuntimeError(
        "Pairwise bootstrap reference families do not match the "
        "coverage-adjusted system-MAE leaders."
    )

job_bootstrap_audit_df = (
    pairwise_bootstrap_df.groupby(
        ["metric", "horizon", "reference_family"],
        observed=True,
        as_index=False,
    )
    .agg(
        challengers=("challenger_family", "nunique"),
        minimum_week_blocks=("calendar_week_blocks", "min"),
        maximum_week_blocks=("calendar_week_blocks", "max"),
    )
)

if not job_bootstrap_audit_df["challengers"].eq(
    len(FAMILY_ORDER) - 1
).all():
    raise RuntimeError(
        "At least one point leader was not compared with all three challengers."
    )

if not (
    job_bootstrap_audit_df["minimum_week_blocks"].eq(40)
    & job_bootstrap_audit_df["maximum_week_blocks"].eq(40)
).all():
    raise RuntimeError(
        "At least one bootstrap comparison no longer uses the expected "
        "40 calendar-week blocks."
    )

print(
    "PASS — bootstrap evidence matches all 15 coverage-adjusted "
    "system-MAE leaders.",
    flush=True,
)

Recovered existing coverage-adjusted pairwise bootstrap: 45 comparisons.
PASS — bootstrap evidence matches all 15 coverage-adjusted system-MAE leaders.


**Findings.** The paired calendar-week bootstrap contains the expected **45 leader-versus-challenger comparisons**: three challengers for each of the 15 system-MAE leaders. Every comparison uses **40 validation-week blocks** and **5,000 resamples**.

All reference families match the corresponding coverage-adjusted point-estimate leaders, so the uncertainty analysis is aligned with the primary selection metric.

For readability, the chart shows the **closest challenger** to each coverage-adjusted system-MAE leader.

The horizontal axis expresses the leader's system-MAE advantage as a percentage of the frozen benchmark MAE:

- `0%` means the two complete forecasting systems have the same MAE.
- Positive values favor the point-estimate system leader.
- The shaded ±1% area is the practical-equivalence region.
- An interval entirely to the right of `+1%` is a clear practical win.
- An interval entirely inside the shaded region indicates practical equivalence.
- Anything overlapping those boundaries remains uncertain enough to require secondary evidence.

The uncertainty test therefore evaluates the same complete forecasting systems used by the primary leaderboard.

In [30]:
# ---------------------------------------------------------------------
# 4.6.1.4.1C — Visualize uncertainty against the equivalence margin
# ---------------------------------------------------------------------

# Require a complete bootstrap object because this visualization interprets
# statistical evidence rather than constructing it.
if (
    pairwise_bootstrap_df is None
    or not isinstance(pairwise_bootstrap_df, pd.DataFrame)
    or pairwise_bootstrap_df.empty
):
    raise RuntimeError(
        "Current coverage-adjusted bootstrap evidence is not loaded. "
        "Run 4.6.1.4.1B successfully before this block."
    )

bootstrap_status_summary_df = (
    pairwise_bootstrap_df.groupby(
        "comparison_status",
        observed=True,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Pairwise comparisons"})
)

display(
    bootstrap_status_summary_df.assign(
        Status=lambda frame: frame["comparison_status"].map({
            "clear_win": "Clear win",
            "practical_equivalence": "Practical equivalence",
            "uncertain_advantage": "Uncertain advantage",
        })
    )[["Status", "Pairwise comparisons"]]
)

fig = plot_bootstrap_forest(pairwise_bootstrap_df)
fig.show()

,Status,Pairwise comparisons
0,Clear win,38
1,Practical equivalence,3
2,Uncertain advantage,4


**Findings.** Of the **45 leader-versus-challenger comparisons**, **38 are clear practical wins**, **3 fall inside the practical-equivalence region**, and **4 remain uncertain**.

Eight of the 15 Target × Horizon leaders clearly defeat all three challengers. Seven jobs retain at least one plausible alternative: all three FHVHV-speed horizons, Subway h=2, and all three Taxi-speed horizons.

The remaining uncertainty is therefore concentrated in speed forecasting plus one Subway job rather than being spread across the full tournament.

### 4.6.1.4.2 Resolve Near-Ties With Decision-Relevant Stability

The uncertainty analysis deliberately does not force a champion when the **coverage-adjusted system-MAE** evidence remains too close.

For those jobs, the notebook examines:

1. **System bias:** does one complete forecasting system consistently run too high or too low?
2. **Temporal breadth:** which system performs best across more validation months?
3. **Geographic breadth:** which system performs best across more Taxi Zones?

Advanced-model coverage and exact-common MAE remain visible alongside those tie-break measures for interpretation. Coverage is **not penalized a second time**: its effect is already incorporated into system MAE because unavailable advanced forecasts use the frozen benchmark fallback.

These are targeted selection diagnostics, not the deeper failure analysis reserved for 4.6.2.

The next block computes monthly and Taxi Zone performance from the **same coverage-adjusted validation population** used by the primary tournament. This keeps the secondary evidence aligned with the system-MAE comparison rather than quietly changing the rows being evaluated.

In [31]:
# ---------------------------------------------------------------------
# 4.6.1.4.2A — Build temporal and geographic system-stability evidence
# ---------------------------------------------------------------------

monthly_stability_df, zone_stability_df = (
    build_temporal_and_geographic_stability(
        SELECTION_VALIDATION_PATH
    )
)

stability_summary_df = summarize_stability(
    monthly_stability_df,
    zone_stability_df,
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        monthly_stability_df,
        MONTHLY_STABILITY_PATH,
    )
    atomic_write_parquet(
        zone_stability_df,
        ZONE_STABILITY_PATH,
    )

stability_summary_df = stability_summary_df.merge(
    validation_scorecard_df[
        [
            "metric",
            "horizon",
            "family",
            "mae",
            "bias",
            "abs_bias",
            "benchmark_skill_pct",
            "model_coverage_pct",
            "exact_common_mae",
        ]
    ],
    on=["metric", "horizon", "family"],
    how="left",
    validate="one_to_one",
)

A job is considered **clearly resolved by primary evidence** only if its coverage-adjusted system-MAE leader has a clear practical win over **all three** challengers.

Otherwise, every challenger that was not clearly beaten remains in the contender set. The notebook exposes system bias and breadth evidence for those surviving finalists rather than turning a tiny numerical difference into an automatic champion.

In [32]:
# ---------------------------------------------------------------------
# 4.6.1.4.2B — Identify clearly resolved jobs and near-tie contender sets
# ---------------------------------------------------------------------

pairwise_resolution_df = (
    pairwise_bootstrap_df.groupby(
        ["metric", "horizon", "reference_family"],
        observed=True,
        as_index=False,
    )
    .agg(
        challengers_tested=("challenger_family", "nunique"),
        clear_challengers_beaten=(
            "comparison_status",
            lambda values: int((values == "clear_win").sum()),
        ),
    )
)

pairwise_resolution_df["clear_primary_winner"] = (
    pairwise_resolution_df["challengers_tested"].eq(len(FAMILY_ORDER) - 1)
    & pairwise_resolution_df["clear_challengers_beaten"].eq(
        len(FAMILY_ORDER) - 1
    )
)

contender_records = []

for winner in point_winner_df.itertuples(index=False):
    job_pairs_df = pairwise_bootstrap_df.loc[
        pairwise_bootstrap_df["metric"].eq(winner.metric)
        & pairwise_bootstrap_df["horizon"].astype(int).eq(int(winner.horizon))
        & pairwise_bootstrap_df["reference_family"].eq(
            winner.point_winner_family
        )
    ]

    nonclear_challengers = job_pairs_df.loc[
        ~job_pairs_df["comparison_status"].eq("clear_win"),
        "challenger_family",
    ].tolist()

    contenders = [
        winner.point_winner_family,
        *nonclear_challengers,
    ]

    for family in contenders:
        contender_records.append({
            "metric": winner.metric,
            "horizon": int(winner.horizon),
            "point_winner_family": winner.point_winner_family,
            "family": family,
            "is_point_winner": family == winner.point_winner_family,
            "contender_count": len(contenders),
        })

contender_df = pd.DataFrame(contender_records)

tie_break_evidence_df = contender_df.merge(
    stability_summary_df,
    on=["metric", "horizon", "family"],
    how="left",
    validate="one_to_one",
)

tie_break_evidence_df["Family"] = (
    tie_break_evidence_df["family"].map(FAMILY_LABELS)
)
tie_break_evidence_df["Target"] = (
    tie_break_evidence_df["metric"].map(TARGET_SHORT_LABELS)
)

review_jobs_df = (
    contender_df.groupby(
        ["metric", "horizon", "point_winner_family"],
        observed=True,
        as_index=False,
    )
    .agg(
        contender_count=("family", "nunique"),
        contenders=(
            "family",
            lambda values: " · ".join(
                FAMILY_LABELS[family]
                for family in FAMILY_ORDER
                if family in set(values)
            ),
        ),
    )
)

review_jobs_df["selection_status"] = np.where(
    review_jobs_df["contender_count"].eq(1),
    "Clear primary winner",
    "Review near-tie evidence",
)

display(
    review_jobs_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        **{
            "Point leader": lambda frame:
                frame["point_winner_family"].map(FAMILY_LABELS),
        },
    )[
        [
            "Target",
            "horizon",
            "Point leader",
            "selection_status",
            "contenders",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "selection_status": "Selection status",
        "contenders": "Remaining contenders",
    })
)

,Target,Horizon,Point leader,Selection status,Remaining contenders
0,FHVHV speed,1,Transformer,Review near-tie evidence,Tree · Transformer
1,FHVHV speed,2,Transformer,Review near-tie evidence,Tree · Transformer
2,FHVHV speed,5,Tree,Review near-tie evidence,Tree · Transformer
3,FHVHV trips,1,Tree,Clear primary winner,Tree
4,FHVHV trips,2,Tree,Clear primary winner,Tree
5,FHVHV trips,5,Tree,Clear primary winner,Tree
6,Subway,1,Tree,Clear primary winner,Tree
7,Subway,2,Transformer,Review near-tie evidence,Tree · Transformer
8,Subway,5,Tree,Clear primary winner,Tree
9,Taxi speed,1,Transformer,Review near-tie evidence,Neural · Transformer


**Findings.** Primary system-MAE and bootstrap evidence fully resolves **8 of the 15 forecasting jobs**, while **7 require secondary evidence**.

All eight clear primary winners are **Tree**: FHVHV trips at h=1, h=2, and h=5; Subway at h=1 and h=5; and Taxi trips at h=1, h=2, and h=5.

The unresolved jobs divide into two clean groups. **Tree and Transformer** remain contenders for all three FHVHV-speed horizons and Subway h=2. **Neural and Transformer** remain contenders for all three Taxi-speed horizons.

Only jobs requiring secondary review need the detailed tie-break table.

For those jobs, the table keeps the evidence in the predeclared order:

1. coverage-adjusted system MAE;
2. absolute system bias;
3. monthly breadth; and
4. Taxi Zone breadth.

Advanced-model coverage and exact-common MAE are shown alongside the tie-break evidence to explain **why** the system-level ranking may differ from conditional model accuracy. They do not create additional independent votes.

In [33]:
# ---------------------------------------------------------------------
# 4.6.1.4.2C — Display near-tie bias, coverage, and breadth evidence
# ---------------------------------------------------------------------

near_tie_job_keys = set(
    review_jobs_df.loc[
        review_jobs_df["contender_count"].gt(1),
        ["metric", "horizon"],
    ].itertuples(index=False, name=None)
)

near_tie_evidence_df = tie_break_evidence_df.loc[
    [
        (metric, int(horizon)) in near_tie_job_keys
        for metric, horizon in zip(
            tie_break_evidence_df["metric"],
            tie_break_evidence_df["horizon"],
        )
    ]
].copy()

if near_tie_evidence_df.empty:
    print(
        "No Target × Horizon job requires secondary tie-break evidence: "
        "every system-MAE leader clearly beat all three challengers.",
        flush=True,
    )

else:
    near_tie_display_df = (
        near_tie_evidence_df[
            [
                "Target",
                "horizon",
                "Family",
                "is_point_winner",
                "mae",
                "model_coverage_pct",
                "exact_common_mae",
                "abs_bias",
                "months_best_share_pct",
                "months_beating_benchmark",
                "validation_months",
                "zones_best_share_pct",
                "worst_monthly_skill_pct",
            ]
        ]
        .rename(columns={
            "horizon": "Horizon",
            "is_point_winner": "System-MAE leader?",
            "mae": "System MAE",
            "model_coverage_pct": "Model coverage (%)",
            "exact_common_mae": "Exact-common MAE",
            "abs_bias": "Absolute system bias",
            "months_best_share_pct": "Months best (%)",
            "months_beating_benchmark": "Months beating benchmark",
            "validation_months": "Validation months",
            "zones_best_share_pct": "Zones best (%)",
            "worst_monthly_skill_pct": "Worst monthly benchmark skill (%)",
        })
        .sort_values(
            ["Target", "Horizon", "Absolute system bias", "Family"]
        )
    )

    display(near_tie_display_df.round(3))

,Target,Horizon,Family,System-MAE leader?,System MAE,Model coverage (%),Exact-common MAE,Absolute system bias,Months best (%),Months beating benchmark,Validation months,Zones best (%),Worst monthly benchmark skill (%)
0,FHVHV speed,1,Transformer,True,0.921,100.000,0.815,0.043,70.000,10,10,70.039,22.008
1,FHVHV speed,1,Tree,False,0.938,100.000,0.832,0.180,30.000,10,10,29.961,18.538
2,FHVHV speed,2,Transformer,True,0.973,100.000,0.869,0.007,60.000,10,10,56.420,17.907
3,FHVHV speed,2,Tree,False,0.982,100.000,0.878,0.238,40.000,10,10,43.580,13.624
5,FHVHV speed,5,Transformer,False,0.990,100.000,0.905,0.018,20.000,10,10,31.907,16.880
4,FHVHV speed,5,Tree,True,0.978,100.000,0.893,0.178,80.000,10,10,68.093,11.186
10,Subway,2,Transformer,True,475.990,100.000,486.628,2.276,50.000,6,10,28.758,-28.353
11,Subway,2,Tree,False,478.138,100.000,491.977,45.090,40.000,5,10,61.438,-47.615
14,Taxi speed,1,Neural,False,2.178,100.000,1.272,0.014,50.000,10,10,24.339,23.306
13,Taxi speed,1,Transformer,True,2.168,100.000,1.265,0.104,50.000,10,10,51.852,19.503


**Findings.** All finalists surviving into the seven near-tie reviews have **100% advanced-model coverage**, so the secondary decisions are driven by predictive behavior rather than forecast availability.

For **FHVHV speed**, Transformer has lower absolute system bias at every horizon: **0.043 vs 0.180** at h=1, **0.007 vs 0.238** at h=2, and **0.018 vs 0.178** at h=5. Transformer also leads monthly and geographic breadth at h=1 and h=2. At h=5, Tree has the lower system MAE and broader monthly/geographic wins, but Transformer retains the substantially smaller absolute bias.

For **Subway h=2**, Transformer has both the lower system MAE (**475.990 vs 478.138**) and much lower absolute bias (**2.276 vs 45.090**). Tree wins across more Taxi Zones, but the first secondary criterion strongly favors Transformer.

For **Taxi speed**, Neural has the lower absolute system bias at all three horizons: **0.014 vs 0.104** at h=1, **0.009 vs 0.022** at h=2, and **0.023 vs 0.118** at h=5. Monthly breadth is tied at all three horizons, while geographic breadth is mixed.

Under the predeclared hierarchy, absolute system bias therefore resolves all seven near ties without requiring a further operational-complexity tie-break.

The final pre-decision table combines the primary and secondary evidence into one compact 15-job review surface.

Jobs labeled **Clear primary winner** have already satisfied the coverage-adjusted system-MAE, uncertainty, and practical-significance requirements against all three challengers.

Jobs labeled **Review near-tie evidence** remain genuine decisions. Their point-estimate system leader is preserved for context, but no champion is frozen until the predeclared secondary hierarchy is applied.

In [34]:
# ---------------------------------------------------------------------
# 4.6.1.4.2D — Assemble the 15-job selection-review surface
# ---------------------------------------------------------------------

selection_support_df = (
    point_winner_df.merge(
        review_jobs_df[
            [
                "metric",
                "horizon",
                "contender_count",
                "contenders",
                "selection_status",
            ]
        ],
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        pairwise_resolution_df[
            [
                "metric",
                "horizon",
                "reference_family",
                "clear_challengers_beaten",
                "clear_primary_winner",
            ]
        ],
        left_on=["metric", "horizon", "point_winner_family"],
        right_on=["metric", "horizon", "reference_family"],
        how="left",
        validate="one_to_one",
    )
)

selection_support_df["provisional_champion"] = np.where(
    selection_support_df["clear_primary_winner"],
    selection_support_df["point_winner_family"],
    pd.NA,
)

if WRITE_INTERMEDIATES:
    atomic_write_parquet(
        selection_support_df,
        SELECTION_SUPPORT_PATH,
    )

selection_review_display_df = (
    selection_support_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
        **{
            "Point leader": lambda frame:
                frame["point_winner_family"].map(FAMILY_LABELS),
            "Provisional champion": lambda frame:
                frame["provisional_champion"].map(FAMILY_LABELS),
        },
    )[
        [
            "Target",
            "horizon",
            "Point leader",
            "selection_status",
            "contenders",
            "clear_challengers_beaten",
            "Provisional champion",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "selection_status": "Selection status",
        "contenders": "Remaining contenders",
        "clear_challengers_beaten": "Clear challenger wins",
    })
)

display(selection_review_display_df)

,Target,Horizon,Point leader,Selection status,Remaining contenders,Clear challenger wins,Provisional champion
0,FHVHV speed,1,Transformer,Review near-tie evidence,Tree · Transformer,2,NaN
1,FHVHV speed,2,Transformer,Review near-tie evidence,Tree · Transformer,2,NaN
2,FHVHV speed,5,Tree,Review near-tie evidence,Tree · Transformer,2,NaN
3,FHVHV trips,1,Tree,Clear primary winner,Tree,3,Tree
4,FHVHV trips,2,Tree,Clear primary winner,Tree,3,Tree
5,FHVHV trips,5,Tree,Clear primary winner,Tree,3,Tree
6,Subway,1,Tree,Clear primary winner,Tree,3,Tree
7,Subway,2,Transformer,Review near-tie evidence,Tree · Transformer,2,NaN
8,Subway,5,Tree,Clear primary winner,Tree,3,Tree
9,Taxi speed,1,Transformer,Review near-tie evidence,Neural · Transformer,2,NaN


**Findings.** The 15-job review surface contains **8 clear primary winners** and **7 near-tie decisions**.

Tree is already the clear champion for all three FHVHV-trip horizons, both resolved Subway horizons (h=1 and h=5), and all three Taxi-trip horizons. The remaining four Tree-versus-Transformer decisions are the three FHVHV-speed horizons and Subway h=2. The remaining three Taxi-speed decisions are Neural-versus-Transformer comparisons.

No Classical finalist remains in a champion contender set.

> Decision — Freeze the 15\-Job Champion System
The ordinary\-validation tournament selects Tree for 8 jobs, Transformer for 4, and Neural for 3\. Classical does not advance to the final holdout\.

Tree advances for all three Taxi\-trip horizons, all three FHVHV\-trip horizons, and Subway at h=1 and h=5\. These eight jobs are clear primary wins: Tree's coverage\-adjusted system\-MAE advantage survives the paired weekly bootstrap and exceeds the practical\-equivalence threshold against every challenger\.

The remaining seven jobs require the predeclared near\-tie rule\. Absolute system bias, the first secondary criterion, resolves every case\. Transformer advances for FHVHV speed at h=1, h=2, and h=5 and for Subway at h=2\. Neural advances for Taxi speed at h=1, h=2, and h=5\.

This means two point\-estimate leaders are intentionally overturned by the tie\-break hierarchy: Transformer replaces Tree for FHVHV speed h=5, while Neural replaces Transformer for Taxi speed h=1\. In both cases, the primary evidence leaves the finalists statistically close enough to require secondary evidence, and the selected family has the lower absolute system bias\.

No job requires temporal breadth, geographic breadth, or operational simplicity to break the tie\. The final champion system is therefore frozen entirely from ordinary\-validation predictive evidence, with the final holdout reserved for generalization testing\.

### 4.6.1.4.3 Materialize the Frozen Champion Registry

The champion Decision now needs a machine-readable execution contract.

For each Target × Horizon, the registry records the selected family, modeling track, upstream configuration, selection basis, and evidence used to resolve the job.

The registry also verifies the frozen decision rule:

- every clear primary winner must remain the system-MAE leader;
- every near-tie champion must belong to the surviving contender set; and
- each near tie must be uniquely resolved by the lowest absolute system bias.

Final-holdout performance remains outside model selection.

In [35]:
# ---------------------------------------------------------------------
# 4.6.1.4.3A — Materialize the frozen 15-job champion registry
# ---------------------------------------------------------------------

CHAMPION_SELECTION_CONTRACT_ID = (
    "coverage_adjusted_champion_selection_v1"
)
CHAMPION_REGISTRY_PATH = (
    FINAL_DIR / "forecasting_champion_registry.parquet"
)

# The final mapping follows the predeclared selection hierarchy:
# clear system-MAE wins first, then absolute system bias for near ties.
CHAMPION_FAMILY_BY_JOB = {
    ("taxi_trip_count", 1): "tree",
    ("taxi_trip_count", 2): "tree",
    ("taxi_trip_count", 5): "tree",

    ("taxi_avg_trip_speed", 1): "neural",
    ("taxi_avg_trip_speed", 2): "neural",
    ("taxi_avg_trip_speed", 5): "neural",

    ("fhvhv_trip_count", 1): "tree",
    ("fhvhv_trip_count", 2): "tree",
    ("fhvhv_trip_count", 5): "tree",

    ("fhvhv_avg_trip_speed", 1): "transformer",
    ("fhvhv_avg_trip_speed", 2): "transformer",
    ("fhvhv_avg_trip_speed", 5): "transformer",

    ("subway_ridership", 1): "tree",
    ("subway_ridership", 2): "transformer",
    ("subway_ridership", 5): "tree",
}

if set(CHAMPION_FAMILY_BY_JOB) != expected_target_horizon_pairs():
    raise RuntimeError(
        "The frozen champion mapping does not cover the exact "
        "15 Target × Horizon jobs."
    )

selection_lookup = selection_support_df.set_index(
    ["metric", "horizon"],
    verify_integrity=True,
)

champion_records = []

for metric in FORECAST_TARGETS:
    for horizon in FORECAST_HORIZONS:
        job_key = (metric, int(horizon))
        selected_family = CHAMPION_FAMILY_BY_JOB[job_key]
        support = selection_lookup.loc[job_key]

        point_family = str(support["point_winner_family"])
        clear_primary = bool(support["clear_primary_winner"])
        contender_count = int(support["contender_count"])

        if clear_primary:
            # A clear winner has already passed both the paired uncertainty
            # test and the practical-equivalence threshold.
            if selected_family != point_family:
                raise RuntimeError(
                    f"{metric} · h={horizon}: frozen champion "
                    f"{selected_family} does not match clear primary "
                    f"winner {point_family}."
                )

            selection_basis = "clear_primary_win"
            decision_evidence = (
                "System-MAE leader clearly beat all three challengers "
                "under the weekly block bootstrap and practical margin."
            )

        else:
            job_evidence_df = near_tie_evidence_df.loc[
                near_tie_evidence_df["metric"].eq(metric)
                & near_tie_evidence_df["horizon"].astype(int).eq(int(horizon))
            ].copy()

            if job_evidence_df.empty:
                raise RuntimeError(
                    f"{metric} · h={horizon}: near-tie evidence is missing."
                )

            contender_families = set(
                job_evidence_df["family"].astype(str)
            )

            if selected_family not in contender_families:
                raise RuntimeError(
                    f"{metric} · h={horizon}: frozen champion "
                    f"{selected_family} is not in the surviving "
                    "near-tie contender set."
                )

            abs_bias = pd.to_numeric(
                job_evidence_df["abs_bias"],
                errors="raise",
            ).to_numpy(dtype=float)

            minimum_abs_bias = float(np.min(abs_bias))
            bias_winner_mask = np.isclose(
                abs_bias,
                minimum_abs_bias,
                rtol=0.0,
                atol=1e-12,
            )

            bias_winners = (
                job_evidence_df.loc[bias_winner_mask, "family"]
                .astype(str)
                .tolist()
            )

            if len(bias_winners) != 1:
                raise RuntimeError(
                    f"{metric} · h={horizon}: absolute system bias "
                    "does not uniquely resolve the near tie."
                )

            if selected_family != bias_winners[0]:
                raise RuntimeError(
                    f"{metric} · h={horizon}: frozen champion "
                    f"{selected_family} does not match lowest-absolute-bias "
                    f"contender {bias_winners[0]}."
                )

            selection_basis = "near_tie_absolute_bias"
            decision_evidence = (
                "Primary system-MAE evidence did not establish a clear "
                "practical win; the unique lowest-absolute-bias contender "
                "wins under the frozen secondary hierarchy."
            )

        champion_records.append({
            "metric": metric,
            "horizon": int(horizon),
            "champion_family": selected_family,
            "point_winner_family": point_family,
            "clear_primary_winner": clear_primary,
            "contender_count": contender_count,
            "selection_basis": selection_basis,
            "decision_evidence": decision_evidence,
            "selection_contract_id": CHAMPION_SELECTION_CONTRACT_ID,
            "champion_frozen": True,
            "holdout_used_for_selection": False,
            "selection_prediction_field": SELECTION_PREDICTION_FIELD,
            "selection_metric": SELECTION_METRIC,
        })

champion_registry_df = pd.DataFrame(champion_records)


# ---------------------------------------------------------------------
# Attach the exact upstream configuration owned by each champion
# ---------------------------------------------------------------------

selected_finalist_config_df = (
    finalist_registry_df[
        [
            "family",
            "metric",
            "horizon",
            "Family",
            "Representative",
            "modeling_track",
            "configuration_id",
        ]
    ]
    .rename(columns={"family": "champion_family"})
    .copy()
)

champion_registry_df = champion_registry_df.merge(
    selected_finalist_config_df,
    on=["champion_family", "metric", "horizon"],
    how="left",
    validate="one_to_one",
)

required_champion_fields = [
    "Family",
    "Representative",
    "modeling_track",
    "configuration_id",
]

if len(champion_registry_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_JOB_COUNT} frozen champion rows; "
        f"found {len(champion_registry_df)}."
    )

if champion_registry_df[required_champion_fields].isna().any().any():
    raise RuntimeError(
        "At least one frozen champion is missing its authoritative "
        "upstream configuration or modeling track."
    )

if not champion_registry_df["modeling_track"].isin(ORDINARY_TRACKS).all():
    raise RuntimeError(
        "At least one frozen champion uses a non-ordinary modeling track."
    )

if champion_registry_df.duplicated(["metric", "horizon"]).any():
    raise RuntimeError(
        "The frozen champion registry is not unique by Target × Horizon."
    )


# ---------------------------------------------------------------------
# Validate the frozen selection composition
# ---------------------------------------------------------------------

selection_basis_counts = (
    champion_registry_df["selection_basis"]
    .value_counts()
    .to_dict()
)

if selection_basis_counts != {
    "clear_primary_win": 8,
    "near_tie_absolute_bias": 7,
}:
    raise RuntimeError(
        "The frozen registry does not reproduce the expected "
        "8 clear-primary and 7 absolute-bias selections."
    )

expected_champion_counts = {
    "tree": 8,
    "transformer": 4,
    "neural": 3,
}

observed_champion_counts = (
    champion_registry_df["champion_family"]
    .value_counts()
    .to_dict()
)

if observed_champion_counts != expected_champion_counts:
    raise RuntimeError(
        "The frozen champion-family composition is inconsistent with "
        f"the Decision: {observed_champion_counts}"
    )


# ---------------------------------------------------------------------
# Preserve the notebook's target/horizon reading order
# ---------------------------------------------------------------------

target_order = {
    metric: index
    for index, metric in enumerate(FORECAST_TARGETS)
}
horizon_order = {
    horizon: index
    for index, horizon in enumerate(FORECAST_HORIZONS)
}

champion_registry_df = (
    champion_registry_df.assign(
        _target_order=champion_registry_df["metric"].map(target_order),
        _horizon_order=champion_registry_df["horizon"].map(horizon_order),
    )
    .sort_values(["_target_order", "_horizon_order"])
    .drop(columns=["_target_order", "_horizon_order"])
    .reset_index(drop=True)
)

atomic_write_parquet(
    champion_registry_df,
    CHAMPION_REGISTRY_PATH,
)

champion_display_df = (
    champion_registry_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
    )
    .pivot(
        index="Target",
        columns="horizon",
        values="Family",
    )
    .reindex([
        TARGET_SHORT_LABELS[metric]
        for metric in FORECAST_TARGETS
    ])
    .rename(columns={
        1: "h=1",
        2: "h=2",
        5: "h=5",
    })
    .reset_index()
)

display(champion_display_df)

print(
    "PASS — frozen champion registry written before holdout evaluation.\n"
    f"  Jobs: {len(champion_registry_df):,}\n"
    "  Tree: 8 · Transformer: 4 · Neural: 3 · Classical: 0\n"
    f"  Path: {CHAMPION_REGISTRY_PATH}",
    flush=True,
)

horizon,Target,h=1,h=2,h=5
0,Taxi trips,Tree,Tree,Tree
1,Taxi speed,Neural,Neural,Neural
2,FHVHV trips,Tree,Tree,Tree
3,FHVHV speed,Transformer,Transformer,Transformer
4,Subway,Tree,Transformer,Tree


PASS — frozen champion registry written before holdout evaluation.
  Jobs: 15
  Tree: 8 · Transformer: 4 · Neural: 3 · Classical: 0
  Path: pipeline_data/4.6.1.final_tables/forecasting_champion_registry.parquet


**Findings.** The frozen champion registry contains all **15 Target × Horizon jobs** and reproduces the selection hierarchy exactly: **8 clear primary wins** and **7 near ties resolved by absolute system bias**.

The final system contains **8 Tree champions, 4 Transformer champions, and 3 Neural champions**. Classical does not own a final-holdout job.

Tree owns all Taxi-trip and FHVHV-trip horizons plus Subway h=1 and h=5. Transformer owns all FHVHV-speed horizons plus Subway h=2. Neural owns all three Taxi-speed horizons.

Every registry row preserves the selected modeling track and upstream configuration and records `holdout_used_for_selection = False`.

### Section Summary

The cross-family tournament produces a compact three-family champion system.

**Tree owns 8 jobs**, including every Taxi-trip and FHVHV-trip forecast. **Transformer owns 4**, covering all FHVHV-speed horizons and Subway h=2. **Neural owns 3**, covering every Taxi-speed horizon.

Eight jobs are clear primary wins; seven are resolved by the predeclared absolute-bias criterion. The resulting 15-job registry fixes each champion's family, modeling track, and upstream configuration before final-holdout evaluation begins.

## 4.6.1.5 Prepare the Unseen Final Holdout

Model selection is complete. The January–March 2026 holdout now tests whether the frozen champion system generalizes to future mobility.

Validation answered:

> **Which finalized forecasting system should own each Target × Horizon?**

The holdout asks:

> **How well do those frozen choices perform on unseen future outcomes?**

Holdout performance therefore evaluates the selected system; it does not revise the champion mapping.

Before fitting or scoring any champion, this section audits the legal evaluation population and binds each active family to the training lifecycle that produced its validation evidence.

Three families require final-holdout execution:

- Tree — 8 champion jobs;
- Neural — 3 champion jobs; and
- Transformer — 4 champion jobs.

Each family keeps its own frozen preprocessing, retraining, and checkpoint semantics while sharing the same final forecast-output contract.

### 4.6.1.5.1 Validate the Frozen Holdout Evaluation Surface

**Audit the unseen rows before asking any champion to predict.**

This check reads only the frozen forecasting matrices and simple-benchmark files. No Tree, Neural, or Transformer model is fitted.

For every selected Target × Horizon, the audit verifies:

- nonzero eligible and labeled final-holdout rows;
- support from the frozen `Previous week · same Daypart` benchmark;
- unique physical forecast rows;
- dates within the inherited final-holdout window; and
- agreement between the observed outcome stored in the forecasting matrix and the benchmark artifact.

Small storage-precision differences use the same float-aware equivalence rule applied throughout the cross-family comparison. Forecast accuracy is not evaluated here.

In [36]:
# ---------------------------------------------------------------------
# 4.6.1.5.1A — Audit the frozen champion holdout surface
# ---------------------------------------------------------------------

INPUT_423_FINAL_DIR = PIPELINE_DIR / "4.2.3.final_tables"
INPUT_424_INTERMEDIATE_DIR = (
    PIPELINE_DIR / "4.2.4.intermediate_tables"
)

FORECASTING_MATRIX_PATHS = {
    "post_cp_primary": (
        INPUT_423_FINAL_DIR
        / "forecasting_matrix__post_cp_primary.parquet"
    ),
    "full_history_regime_aware": (
        INPUT_423_FINAL_DIR
        / "forecasting_matrix__full_history_regime_aware.parquet"
    ),
}

BASELINE_PREDICTION_PATHS = {
    "post_cp_primary": (
        INPUT_424_INTERMEDIATE_DIR
        / "baseline_predictions__post_cp_primary.parquet"
    ),
    "full_history_regime_aware": (
        INPUT_424_INTERMEDIATE_DIR
        / "baseline_predictions__full_history_regime_aware.parquet"
    ),
}


def require_holdout_source_schema(
    path,
    required_columns,
    label,
):
    """Validate a large Parquet source from metadata without loading its rows."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {path}"
        )

    missing = sorted(
        set(required_columns) - set(parquet_columns(path))
    )

    if missing:
        raise KeyError(
            f"{label} is missing required columns: {missing}"
        )


# Read the durable registry rather than trusting the in-memory Decision state.
# This keeps the holdout stage downstream of a physically frozen selection.
frozen_champion_registry_df = pd.read_parquet(
    CHAMPION_REGISTRY_PATH
)

required_champion_columns = [
    "metric",
    "horizon",
    "champion_family",
    "Family",
    "Representative",
    "modeling_track",
    "configuration_id",
    "selection_basis",
    "selection_contract_id",
    "champion_frozen",
    "holdout_used_for_selection",
]

require_columns(
    frozen_champion_registry_df,
    required_champion_columns,
    "4.6.1 frozen champion registry",
)

if len(frozen_champion_registry_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The frozen champion registry no longer contains exactly 15 jobs."
    )

if not frozen_champion_registry_df["selection_contract_id"].eq(
    CHAMPION_SELECTION_CONTRACT_ID
).all():
    raise RuntimeError(
        "The champion registry does not match the current "
        "selection contract."
    )

if not frozen_champion_registry_df[
    "champion_frozen"
].fillna(False).all():
    raise RuntimeError(
        "At least one champion is not marked as frozen."
    )

if frozen_champion_registry_df[
    "holdout_used_for_selection"
].fillna(True).any():
    raise RuntimeError(
        "The champion registry indicates that holdout evidence "
        "influenced selection."
    )


# ---------------------------------------------------------------------
# 1) Recover the exact inherited holdout and benchmark contracts
# ---------------------------------------------------------------------

ordinary_holdout_end_values = pd.to_datetime(
    track_contract_df.loc[
        track_contract_df["modeling_track"].isin(ORDINARY_TRACKS),
        "final_holdout_target_end_date",
    ],
    errors="raise",
).dropna().dt.normalize().unique()

if len(ordinary_holdout_end_values) != 1:
    raise RuntimeError(
        "The ordinary modeling tracks do not share one "
        "final-holdout end date."
    )

FINAL_HOLDOUT_END_DATE = pd.Timestamp(
    ordinary_holdout_end_values[0]
)

baseline_ids = (
    ordinary_benchmark_df["baseline_id"]
    .dropna()
    .astype(str)
    .unique()
)

if len(baseline_ids) != 1:
    raise RuntimeError(
        "The ordinary forecasting jobs no longer share one "
        "frozen baseline ID."
    )

HOLDOUT_BASELINE_ID = str(baseline_ids[0])
HOLDOUT_BASELINE_COLUMN = (
    f"baseline__{HOLDOUT_BASELINE_ID}"
)


# ---------------------------------------------------------------------
# 2) Validate the large source schemas once before any row-level audit
# ---------------------------------------------------------------------

matrix_required_columns = [
    "modeling_track",
    "stage",
    "forecast_sequence_id",
    "taxi_zone_id",
    "metric",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "target_date",
    "target_observed_value",
    "target_label_available",
    "native_track_eligible",
]

# The 4.2.4 baseline files are track-specific. Their exact physical identity
# is carried by Forecast Sequence + origin/target positions + horizon + date.
# Do not invent taxi_zone_id or modeling_track as required baseline columns.
baseline_required_columns = [
    "stage",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "metric",
    "target_date",
    "target_observed_value",
    "target_label_available",
    HOLDOUT_BASELINE_COLUMN,
]

for track in ORDINARY_TRACKS:
    require_holdout_source_schema(
        FORECASTING_MATRIX_PATHS[track],
        matrix_required_columns,
        f"{TRACK_LABELS[track]} forecasting matrix",
    )
    require_holdout_source_schema(
        BASELINE_PREDICTION_PATHS[track],
        baseline_required_columns,
        f"{TRACK_LABELS[track]} baseline predictions",
    )


# ---------------------------------------------------------------------
# 3) Audit the legal holdout rows for every already-frozen champion
# ---------------------------------------------------------------------

holdout_audit_records = []

for champion in frozen_champion_registry_df.itertuples(index=False):
    track = str(champion.modeling_track)
    metric = str(champion.metric)
    horizon = int(champion.horizon)

    matrix_path_sql = sql_path(
        FORECASTING_MATRIX_PATHS[track]
    )
    baseline_path_sql = sql_path(
        BASELINE_PREDICTION_PATHS[track]
    )

    track_sql = track.replace("'", "''")
    metric_sql = metric.replace("'", "''")

    # DuckDB scans only one Track × Target × Horizon at a time. The aligned
    # rows are used only for QA here; no finalist is fitted or scored yet.
    with duckdb.connect() as connection:
        audit_row = connection.execute(f"""
            WITH
            matrix_rows AS (
                SELECT
                    CAST(forecast_sequence_id AS VARCHAR)
                        AS forecast_sequence_id,
                    CAST(origin_observation_sequence_id AS BIGINT)
                        AS origin_observation_sequence_id,
                    CAST(target_observation_sequence_id AS BIGINT)
                        AS target_observation_sequence_id,
                    CAST(horizon AS INTEGER) AS horizon,
                    CAST(metric AS VARCHAR) AS metric,
                    CAST(target_date AS DATE) AS target_date,
                    CAST(target_observed_value AS DOUBLE)
                        AS matrix_actual,
                    CAST(target_label_available AS BOOLEAN)
                        AS target_label_available
                FROM read_parquet('{matrix_path_sql}')
                WHERE CAST(modeling_track AS VARCHAR) = '{track_sql}'
                  AND CAST(stage AS VARCHAR) = 'final_holdout'
                  AND CAST(metric AS VARCHAR) = '{metric_sql}'
                  AND CAST(horizon AS INTEGER) = {horizon}
                  AND COALESCE(
                        CAST(native_track_eligible AS BOOLEAN),
                        FALSE
                      )
            ),
            matrix_labeled AS (
                SELECT *
                FROM matrix_rows
                WHERE COALESCE(target_label_available, FALSE)
                  AND COALESCE(isfinite(matrix_actual), FALSE)
            ),
            baseline_labeled AS (
                SELECT
                    CAST(forecast_sequence_id AS VARCHAR)
                        AS forecast_sequence_id,
                    CAST(origin_observation_sequence_id AS BIGINT)
                        AS origin_observation_sequence_id,
                    CAST(target_observation_sequence_id AS BIGINT)
                        AS target_observation_sequence_id,
                    CAST(horizon AS INTEGER) AS horizon,
                    CAST(metric AS VARCHAR) AS metric,
                    CAST(target_date AS DATE) AS target_date,
                    CAST(target_observed_value AS DOUBLE)
                        AS baseline_actual,
                    CAST({HOLDOUT_BASELINE_COLUMN} AS DOUBLE)
                        AS benchmark_prediction
                FROM read_parquet('{baseline_path_sql}')
                WHERE CAST(stage AS VARCHAR) = 'final_holdout'
                  AND CAST(metric AS VARCHAR) = '{metric_sql}'
                  AND CAST(horizon AS INTEGER) = {horizon}
                  AND COALESCE(
                        CAST(target_label_available AS BOOLEAN),
                        FALSE
                      )
                  AND COALESCE(
                        isfinite(
                            CAST(target_observed_value AS DOUBLE)
                        ),
                        FALSE
                      )
                  AND COALESCE(
                        isfinite(
                            CAST(
                                {HOLDOUT_BASELINE_COLUMN}
                                AS DOUBLE
                            )
                        ),
                        FALSE
                      )
            ),
            aligned AS (
                SELECT
                    m.*,
                    b.baseline_actual,
                    b.benchmark_prediction,
                    ABS(
                        m.matrix_actual - b.baseline_actual
                    ) AS actual_alignment_delta,
                    (
                        {CROSS_ARTIFACT_ATOL}
                        + {CROSS_ARTIFACT_RTOL}
                        * GREATEST(
                            ABS(m.matrix_actual),
                            ABS(b.baseline_actual)
                        )
                    ) AS actual_alignment_tolerance
                FROM matrix_labeled AS m
                INNER JOIN baseline_labeled AS b USING (
                    forecast_sequence_id,
                    origin_observation_sequence_id,
                    target_observation_sequence_id,
                    horizon,
                    metric,
                    target_date
                )
            ),
            matrix_duplicates AS (
                SELECT
                    COALESCE(
                        SUM(row_count - 1),
                        0
                    ) AS duplicate_rows
                FROM (
                    SELECT
                        forecast_sequence_id,
                        origin_observation_sequence_id,
                        target_observation_sequence_id,
                        horizon,
                        metric,
                        target_date,
                        COUNT(*) AS row_count
                    FROM matrix_labeled
                    GROUP BY ALL
                    HAVING COUNT(*) > 1
                )
            ),
            baseline_duplicates AS (
                SELECT
                    COALESCE(
                        SUM(row_count - 1),
                        0
                    ) AS duplicate_rows
                FROM (
                    SELECT
                        forecast_sequence_id,
                        origin_observation_sequence_id,
                        target_observation_sequence_id,
                        horizon,
                        metric,
                        target_date,
                        COUNT(*) AS row_count
                    FROM baseline_labeled
                    GROUP BY ALL
                    HAVING COUNT(*) > 1
                )
            )
            SELECT
                (
                    SELECT COUNT(*)
                    FROM matrix_rows
                ) AS eligible_rows,
                (
                    SELECT COUNT(*)
                    FROM matrix_labeled
                ) AS labeled_rows,
                (
                    SELECT COUNT(
                        DISTINCT forecast_sequence_id
                    )
                    FROM matrix_labeled
                ) AS forecast_sequences,
                (
                    SELECT MIN(target_date)
                    FROM matrix_labeled
                ) AS first_target_date,
                (
                    SELECT MAX(target_date)
                    FROM matrix_labeled
                ) AS last_target_date,
                COUNT(*) AS benchmark_scoreable_rows,
                COALESCE(
                    SUM(
                        CASE
                            WHEN actual_alignment_delta >
                                 actual_alignment_tolerance
                            THEN 1
                            ELSE 0
                        END
                    ),
                    0
                ) AS actual_alignment_violations,
                MAX(
                    actual_alignment_delta
                ) AS maximum_actual_delta,
                (
                    SELECT duplicate_rows
                    FROM matrix_duplicates
                ) AS matrix_duplicate_rows,
                (
                    SELECT duplicate_rows
                    FROM baseline_duplicates
                ) AS baseline_duplicate_rows
            FROM aligned
        """).df().iloc[0]

    eligible_rows = int(
        audit_row["eligible_rows"]
    )
    labeled_rows = int(
        audit_row["labeled_rows"]
    )
    benchmark_scoreable_rows = int(
        audit_row["benchmark_scoreable_rows"]
    )

    holdout_audit_records.append({
        "metric": metric,
        "horizon": horizon,
        "champion_family": champion.champion_family,
        "Family": champion.Family,
        "modeling_track": track,
        "eligible_rows": eligible_rows,
        "labeled_rows": labeled_rows,
        "forecast_sequences": int(
            audit_row["forecast_sequences"]
        ),
        "benchmark_scoreable_rows": benchmark_scoreable_rows,
        "benchmark_support_pct": (
            100.0 * benchmark_scoreable_rows / labeled_rows
            if labeled_rows
            else np.nan
        ),
        "first_target_date": pd.Timestamp(
            audit_row["first_target_date"]
        ),
        "last_target_date": pd.Timestamp(
            audit_row["last_target_date"]
        ),
        "actual_alignment_violations": int(
            audit_row["actual_alignment_violations"]
        ),
        "maximum_actual_delta": float(
            audit_row["maximum_actual_delta"]
            if pd.notna(audit_row["maximum_actual_delta"])
            else np.nan
        ),
        "matrix_duplicate_rows": int(
            audit_row["matrix_duplicate_rows"]
        ),
        "baseline_duplicate_rows": int(
            audit_row["baseline_duplicate_rows"]
        ),
    })

holdout_surface_audit_df = pd.DataFrame(
    holdout_audit_records
)


# ---------------------------------------------------------------------
# 4) Fail before fitting if the unseen evaluation surface is malformed
# ---------------------------------------------------------------------

observed_holdout_pairs = set(
    holdout_surface_audit_df[
        ["metric", "horizon"]
    ].itertuples(
        index=False,
        name=None,
    )
)

if observed_holdout_pairs != expected_target_horizon_pairs():
    raise RuntimeError(
        "The holdout surface does not cover the exact "
        "15 Target × Horizon jobs."
    )

if (
    (holdout_surface_audit_df["eligible_rows"] <= 0).any()
    or (holdout_surface_audit_df["labeled_rows"] <= 0).any()
    or (
        holdout_surface_audit_df["benchmark_scoreable_rows"]
        <= 0
    ).any()
):
    raise RuntimeError(
        "At least one frozen champion has zero legal, labeled, "
        "or benchmark-scoreable final-holdout rows."
    )

if (
    holdout_surface_audit_df["matrix_duplicate_rows"].ne(0).any()
    or holdout_surface_audit_df[
        "baseline_duplicate_rows"
    ].ne(0).any()
):
    raise RuntimeError(
        "The holdout matrix or frozen benchmark contains "
        "duplicate physical forecast rows."
    )

if holdout_surface_audit_df[
    "actual_alignment_violations"
].ne(0).any():
    raise RuntimeError(
        "The forecasting matrix and frozen benchmark disagree "
        "on at least one observed holdout outcome beyond the "
        "cross-artifact tolerance."
    )

if (
    holdout_surface_audit_df["first_target_date"]
    .lt(FINAL_HOLDOUT_START_DATE)
    .any()
    or holdout_surface_audit_df["last_target_date"]
    .gt(FINAL_HOLDOUT_END_DATE)
    .any()
):
    raise RuntimeError(
        "At least one holdout job extends outside the frozen "
        "evaluation window."
    )

if pd.Timestamp(
    holdout_surface_audit_df["first_target_date"].min()
) != FINAL_HOLDOUT_START_DATE:
    raise RuntimeError(
        "The audited holdout does not begin on the frozen "
        "January 5, 2026 boundary."
    )

if pd.Timestamp(
    holdout_surface_audit_df["last_target_date"].max()
) != FINAL_HOLDOUT_END_DATE:
    raise RuntimeError(
        "The audited holdout does not reach the frozen end boundary."
    )


# ---------------------------------------------------------------------
# 5) Curate the reader-facing support table
# ---------------------------------------------------------------------

holdout_surface_display_df = (
    holdout_surface_audit_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_SHORT_LABELS),
        Track=lambda frame:
            frame["modeling_track"].map(TRACK_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Family",
            "Track",
            "eligible_rows",
            "labeled_rows",
            "benchmark_scoreable_rows",
            "benchmark_support_pct",
            "first_target_date",
            "last_target_date",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "Family": "Champion",
        "eligible_rows": "Eligible rows",
        "labeled_rows": "Labeled rows",
        "benchmark_scoreable_rows": "Benchmark-scoreable rows",
        "benchmark_support_pct": "Benchmark support (%)",
        "first_target_date": "First target date",
        "last_target_date": "Last target date",
    })
)

display(
    holdout_surface_display_df.round(3)
)

,Target,Horizon,Champion,Track,Eligible rows,Labeled rows,Benchmark-scoreable rows,Benchmark support (%),First target date,Last target date
0,Taxi trips,1,Tree,Full-History Regime-Aware,110940,110693,110499,99.825,2026-01-05,2026-03-31
1,Taxi trips,2,Tree,Full-History Regime-Aware,110940,110693,110499,99.825,2026-01-05,2026-03-31
2,Taxi trips,5,Tree,Full-History Regime-Aware,110940,110693,110499,99.825,2026-01-05,2026-03-31
3,Taxi speed,1,Neural,Post-CP Primary,81270,79652,78325,98.334,2026-01-05,2026-03-31
4,Taxi speed,2,Neural,Post-CP Primary,81270,79652,78325,98.334,2026-01-05,2026-03-31
5,Taxi speed,5,Neural,Post-CP Primary,81270,79652,78325,98.334,2026-01-05,2026-03-31
6,FHVHV trips,1,Tree,Full-History Regime-Aware,110940,110693,110499,99.825,2026-01-05,2026-03-31
7,FHVHV trips,2,Tree,Full-History Regime-Aware,110940,110693,110499,99.825,2026-01-05,2026-03-31
8,FHVHV trips,5,Tree,Full-History Regime-Aware,110940,110693,110499,99.825,2026-01-05,2026-03-31
9,FHVHV speed,1,Transformer,Full-History Regime-Aware,110510,110231,110012,99.801,2026-01-05,2026-03-31


**Findings.** All **15 frozen champion jobs** have nonzero legal and labeled final-holdout support.

Taxi trips, Taxi speed, FHVHV trips, and FHVHV speed span **January 5 through March 31, 2026**. Subway ridership spans **January 5 through March 30**, reflecting its inherited observation calendar.

The frozen weekly benchmark covers nearly the entire labeled holdout population. Support ranges from **98.334% for Taxi speed** to **99.947% for Subway**, with Taxi trips and FHVHV trips at **99.825%** and FHVHV speed at **99.801%**.

These benchmark-supported rows define the primary holdout evaluation population, preserving the same complete-system semantics used during model selection.

With the holdout population verified, the next block freezes the **15-job execution roster**.

The roster joins each champion's family, modeling track, upstream configuration, and selection basis to its audited final-holdout support. It also preserves the shared output rule:

`prediction_final = max(0, prediction_raw)`

The roster defines **what must be executed**. The family-specific contracts in the next subsection define **how each selected system must be reproduced**.

In [37]:
# ---------------------------------------------------------------------
# 4.6.1.5.1B — Freeze the 15-job holdout execution roster
# ---------------------------------------------------------------------

HOLDOUT_EXECUTION_CONTRACT_ID = (
    "frozen_champion_holdout_execution_v2"
)
HOLDOUT_EXECUTION_MANIFEST_PATH = (
    INTERMEDIATE_DIR
    / "frozen_holdout_execution_manifest.parquet"
)

holdout_support_fields = holdout_surface_audit_df[
    [
        "metric",
        "horizon",
        "champion_family",
        "eligible_rows",
        "labeled_rows",
        "forecast_sequences",
        "benchmark_scoreable_rows",
        "benchmark_support_pct",
        "first_target_date",
        "last_target_date",
    ]
].copy()

holdout_execution_manifest_df = (
    frozen_champion_registry_df.merge(
        holdout_support_fields,
        on=["metric", "horizon", "champion_family"],
        how="left",
        validate="one_to_one",
    )
)

holdout_execution_manifest_df["holdout_execution_contract_id"] = (
    HOLDOUT_EXECUTION_CONTRACT_ID
)
holdout_execution_manifest_df["selection_frozen_before_holdout"] = True
holdout_execution_manifest_df["prediction_output_rule"] = (
    "prediction_final = max(0, prediction_raw)"
)
holdout_execution_manifest_df["runner_status"] = (
    "pending_source_runner_binding"
)

required_manifest_fields = [
    "metric",
    "horizon",
    "champion_family",
    "Family",
    "Representative",
    "modeling_track",
    "configuration_id",
    "selection_basis",
    "selection_contract_id",
    "holdout_execution_contract_id",
    "eligible_rows",
    "labeled_rows",
    "benchmark_scoreable_rows",
    "benchmark_support_pct",
    "first_target_date",
    "last_target_date",
]

if len(holdout_execution_manifest_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The frozen holdout execution roster does not contain "
        "exactly 15 jobs."
    )

if holdout_execution_manifest_df[
    required_manifest_fields
].isna().any().any():
    raise RuntimeError(
        "At least one frozen holdout job is missing selection "
        "or support metadata."
    )

observed_jobs = set(
    holdout_execution_manifest_df[
        ["metric", "horizon"]
    ].itertuples(index=False, name=None)
)

if observed_jobs != expected_target_horizon_pairs():
    raise RuntimeError(
        "The frozen holdout execution roster does not match "
        "the expected 15 Target × Horizon jobs."
    )

if not holdout_execution_manifest_df[
    "selection_frozen_before_holdout"
].all():
    raise RuntimeError(
        "The holdout execution roster does not preserve "
        "the selection freeze."
    )

if not holdout_execution_manifest_df[
    "selection_contract_id"
].eq(CHAMPION_SELECTION_CONTRACT_ID).all():
    raise RuntimeError(
        "The holdout execution roster does not match the "
        "champion-selection contract."
    )

expected_champion_counts = {
    "tree": 8,
    "transformer": 4,
    "neural": 3,
}

observed_champion_counts = (
    holdout_execution_manifest_df["champion_family"]
    .value_counts()
    .to_dict()
)

if observed_champion_counts != expected_champion_counts:
    raise RuntimeError(
        "The holdout roster does not reproduce the frozen "
        f"champion composition: {observed_champion_counts}"
    )

atomic_write_parquet(
    holdout_execution_manifest_df,
    HOLDOUT_EXECUTION_MANIFEST_PATH,
)

holdout_roster_display_df = (
    holdout_execution_manifest_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_SHORT_LABELS),
        Track=lambda frame:
            frame["modeling_track"].map(TRACK_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Family",
            "Representative",
            "Track",
            "selection_basis",
            "benchmark_support_pct",
            "runner_status",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "Family": "Champion",
        "selection_basis": "Selection basis",
        "benchmark_support_pct": "Benchmark support (%)",
        "runner_status": "Runner status",
    })
)

display(holdout_roster_display_df.round(3))

print(
    "PASS — final-holdout execution roster frozen.\n"
    f"  Jobs: {len(holdout_execution_manifest_df):,}\n"
    "  Tree: 8 · Transformer: 4 · Neural: 3\n"
    f"  Path: {HOLDOUT_EXECUTION_MANIFEST_PATH}",
    flush=True,
)

,Target,Horizon,Champion,Representative,Track,Selection basis,Benchmark support (%),Runner status
0,Taxi trips,1,Tree,LightGBM,Full-History Regime-Aware,clear_primary_win,99.825,pending_source_runner_binding
1,Taxi trips,2,Tree,LightGBM,Full-History Regime-Aware,clear_primary_win,99.825,pending_source_runner_binding
2,Taxi trips,5,Tree,LightGBM,Full-History Regime-Aware,clear_primary_win,99.825,pending_source_runner_binding
3,Taxi speed,1,Neural,Lag-window MLP system,Post-CP Primary,near_tie_absolute_bias,98.334,pending_source_runner_binding
4,Taxi speed,2,Neural,Lag-window MLP system,Post-CP Primary,near_tie_absolute_bias,98.334,pending_source_runner_binding
5,Taxi speed,5,Neural,Lag-window MLP system,Post-CP Primary,near_tie_absolute_bias,98.334,pending_source_runner_binding
6,FHVHV trips,1,Tree,LightGBM,Full-History Regime-Aware,clear_primary_win,99.825,pending_source_runner_binding
7,FHVHV trips,2,Tree,LightGBM,Full-History Regime-Aware,clear_primary_win,99.825,pending_source_runner_binding
8,FHVHV trips,5,Tree,LightGBM,Full-History Regime-Aware,clear_primary_win,99.825,pending_source_runner_binding
9,FHVHV speed,1,Transformer,PatchTST-inspired Transformer,Full-History Regime-Aware,near_tie_absolute_bias,99.801,pending_source_runner_binding


PASS — final-holdout execution roster frozen.
  Jobs: 15
  Tree: 8 · Transformer: 4 · Neural: 3
  Path: pipeline_data/4.6.1.intermediate_tables/frozen_holdout_execution_manifest.parquet


**Findings.** The final-holdout execution roster contains exactly **15 frozen jobs**: **8 Tree, 4 Transformer, and 3 Neural**.

Every job preserves its champion family, upstream representative, modeling track, configuration, selection basis, benchmark support, and final-output rule.

All jobs remain `pending_source_runner_binding`; no advanced-model final-holdout prediction has been generated.

### 4.6.1.5.2 Bind Frozen Champions to Their Source Execution Contracts

The champion registry determines **which system owns each forecasting job**. The execution contract determines **how that selected system must be reproduced**.

The three active families retain distinct training lifecycles:

**Tree** uses one fresh global-panel LightGBM fit per Target × Horizon.

**Neural** retrains monthly using expanding legal history. Each monthly refresh first selects a training duration from historical inner validation, then discards that selector and fits a fresh final model.

**Transformer** follows the same selector-then-final monthly lifecycle while rebuilding its frozen PatchTST-inspired sequence and context representation from legal history.

These execution rules reproduce the selected family systems without introducing new modeling choices.

**4.6.1.5.2A — Freeze the active family-specific holdout runner contracts.**

A fair final test reproduces each selected family under the training rules that produced its validation evidence.

This block records the execution unit, refit cadence, legal-history rule, preprocessing behavior, training-duration policy, output semantics, and checkpoint grain for the three active champion families.

In [38]:
# ---------------------------------------------------------------------
# 4.6.1.5.2A — Freeze active family-specific holdout runner contracts
# ---------------------------------------------------------------------

HOLDOUT_EXECUTION_MANIFEST_PATH = (
    INTERMEDIATE_DIR / "frozen_holdout_execution_manifest.parquet"
)
HOLDOUT_RUNNER_CONTRACT_PATH = (
    INTERMEDIATE_DIR / "holdout_runner_contract_registry.parquet"
)
HOLDOUT_RUNNER_BINDING_PATH = (
    INTERMEDIATE_DIR / "frozen_holdout_runner_bindings.parquet"
)
HOLDOUT_SOURCE_ARTIFACT_AUDIT_PATH = (
    INTERMEDIATE_DIR / "holdout_source_artifact_audit.parquet"
)
HOLDOUT_EXECUTION_PLAN_PATH = (
    INTERMEDIATE_DIR / "holdout_execution_plan_summary.parquet"
)

HOLDOUT_RUNNER_CONTRACT_ID = "active_champion_holdout_runners_v1"
ACTIVE_CHAMPION_FAMILIES = ["tree", "neural", "transformer"]


def write_small_contract(frame, path):
    """Persist one compact contract without exposing a partial file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        f".{path.stem}.tmp{path.suffix}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )
    temporary_path.replace(path)


# The three active families intentionally retain different training lifecycles.
runner_records = [
    {
        "champion_family": "tree",
        "runner_family_label": "Tree",
        "runner_contract_id": "lgbm_holdout_runtime_v1",
        "execution_unit": "Target × Horizon global panel model",
        "refit_cadence": "One fresh fit at final-holdout start",
        "training_history_rule": (
            "All legally observable training rows before the final holdout begins"
        ),
        "training_duration_policy": "Frozen LightGBM model definition",
        "selector_window_days": np.nan,
        "selector_max_epochs": np.nan,
        "selector_patience": np.nan,
        "selector_min_delta": np.nan,
        "selector_lifecycle": "Not applicable",
        "preprocessing_refit_rule": (
            "Use the frozen ordered predictor roster and categorical treatment"
        ),
        "state_carryover_rule": "Fresh fit; no cross-month model refresh",
        "output_availability_rule": (
            "Eligible rows require finite predictions from the fitted model"
        ),
        "output_rule": (
            "Preserve prediction_raw; finite prediction_final = "
            "max(0, prediction_raw)"
        ),
        "checkpoint_grain": "Target × Horizon job",
        "source_contract_summary": (
            "lgbm_selected_model_registry + lgbm_predictor_registry + "
            "lgbm_forecast_output_contract"
        ),
        "runner_binding_status":
            "source_contract_verified_pending_implementation",
    },
    {
        "champion_family": "neural",
        "runner_family_label": "Neural",
        "runner_contract_id": "mlp_holdout_runtime_v1",
        "execution_unit": "Target × Horizon × holdout month",
        "refit_cadence": "Monthly expanding-history retraining",
        "training_history_rule": (
            "Only outcomes legally observable before each monthly origin"
        ),
        "training_duration_policy": "Historical inner-validation selected epoch",
        "selector_window_days": 28,
        "selector_max_epochs": 24,
        "selector_patience": 4,
        "selector_min_delta": 1e-4,
        "selector_lifecycle": (
            "Select duration on historical inner validation; discard selector; "
            "train a fresh final model"
        ),
        "preprocessing_refit_rule": (
            "Refit preprocessing on all legally available pre-origin history"
        ),
        "state_carryover_rule": (
            "Fresh final model each month; no weight carryover"
        ),
        "output_availability_rule": (
            "Preserve source eligibility and finite-prediction requirements"
        ),
        "output_rule": (
            "Preserve prediction_raw; finite prediction_final = "
            "max(0, prediction_raw)"
        ),
        "checkpoint_grain": "Target × Horizon × holdout month",
        "source_contract_summary": (
            "neural_final_model_registry + neural_model_definition_contract + "
            "neural_input_registry + neural_sequence_preprocessing_contract + "
            "neural_training_policy_evidence"
        ),
        "runner_binding_status":
            "source_contract_verified_pending_implementation",
    },
    {
        "champion_family": "transformer",
        "runner_family_label": "Transformer",
        "runner_contract_id": "transformer_holdout_runtime_v1",
        "execution_unit": "Target × Horizon × holdout month",
        "refit_cadence": "Monthly expanding-history retraining",
        "training_history_rule": (
            "Only outcomes legally observable before each monthly origin"
        ),
        "training_duration_policy": "Historical inner-validation selected epoch",
        "selector_window_days": 28,
        "selector_max_epochs": 30,
        "selector_patience": 5,
        "selector_min_delta": 1e-4,
        "selector_lifecycle": (
            "Select duration on historical inner validation; discard selector; "
            "train a fresh final model"
        ),
        "preprocessing_refit_rule": (
            "Rebuild the frozen PatchTST-inspired sequence/context "
            "representation from all legal pre-origin history"
        ),
        "state_carryover_rule": (
            "Fresh final model each month; no weight carryover"
        ),
        "output_availability_rule": (
            "Preserve source eligibility and finite-prediction requirements"
        ),
        "output_rule": (
            "Preserve prediction_raw; finite prediction_final = "
            "max(0, prediction_raw)"
        ),
        "checkpoint_grain": "Target × Horizon × holdout month",
        "source_contract_summary": (
            "transformer_final_model_registry + "
            "transformer_model_definition_contract + "
            "transformer_context_contract + transformer_input_registry + "
            "transformer_production_schedule + "
            "transformer_training_policy_evidence"
        ),
        "runner_binding_status":
            "source_contract_verified_pending_implementation",
    },
]

holdout_runner_contract_df = pd.DataFrame(runner_records)
holdout_runner_contract_df["holdout_runner_contract_id"] = (
    HOLDOUT_RUNNER_CONTRACT_ID
)

observed_families = set(
    holdout_runner_contract_df["champion_family"]
)
expected_families = set(ACTIVE_CHAMPION_FAMILIES)

if observed_families != expected_families:
    raise AssertionError(
        "Holdout runner registry must contain exactly the active "
        "champion families."
    )

if holdout_runner_contract_df["champion_family"].duplicated().any():
    raise AssertionError(
        "Each active family must have exactly one holdout runner contract."
    )

if holdout_runner_contract_df["runner_contract_id"].duplicated().any():
    raise AssertionError(
        "Holdout runner contract IDs must be unique."
    )

if holdout_runner_contract_df["output_rule"].nunique(dropna=False) != 1:
    raise AssertionError(
        "All active families must preserve the shared raw/final "
        "forecast-output rule."
    )

write_small_contract(
    holdout_runner_contract_df,
    HOLDOUT_RUNNER_CONTRACT_PATH,
)

runner_display_df = (
    holdout_runner_contract_df[
        [
            "runner_family_label",
            "execution_unit",
            "refit_cadence",
            "training_duration_policy",
            "selector_lifecycle",
            "checkpoint_grain",
        ]
    ]
    .rename(columns={
        "runner_family_label": "Family",
        "execution_unit": "Execution unit",
        "refit_cadence": "Refit cadence",
        "training_duration_policy": "Training duration",
        "selector_lifecycle": "Selector lifecycle",
        "checkpoint_grain": "Checkpoint grain",
    })
)

display(runner_display_df)

print(
    f"Runner contracts frozen: {len(holdout_runner_contract_df)} families\n"
    f"Output: {HOLDOUT_RUNNER_CONTRACT_PATH}"
)

,Family,Execution unit,Refit cadence,Training duration,Selector lifecycle,Checkpoint grain
0,Tree,Target × Horizon global panel model,One fresh fit at final-holdout start,Frozen LightGBM model definition,Not applicable,Target × Horizon job
1,Neural,Target × Horizon × holdout month,Monthly expanding-history retraining,Historical inner-validation selected epoch,Select duration on historical inner validation...,Target × Horizon × holdout month
2,Transformer,Target × Horizon × holdout month,Monthly expanding-history retraining,Historical inner-validation selected epoch,Select duration on historical inner validation...,Target × Horizon × holdout month


Runner contracts frozen: 3 families
Output: pipeline_data/4.6.1.intermediate_tables/holdout_runner_contract_registry.parquet


**Findings.** The final-holdout system requires **three source-specific runner contracts**.

Tree uses one fresh LightGBM global-panel fit per champion job. Neural and Transformer both use monthly expanding-history retraining, with a historical training-duration selector followed by a fresh final model at each monthly refresh.

All three families preserve the same raw/final forecast-output rule while retaining their own preprocessing and training lifecycles.

**4.6.1.5.2B — Bind all 15 champions to their runner contracts.**

The family-level contracts describe how Tree, Neural, and Transformer operate. This block attaches the appropriate contract to each frozen Target × Horizon champion.

The binding is mechanical: it may add execution instructions, but it cannot change the champion family, modeling track, or upstream configuration.

In [39]:
# ---------------------------------------------------------------------
# 4.6.1.5.2B — Bind all 15 champions to source-specific runner contracts
# ---------------------------------------------------------------------

if not HOLDOUT_EXECUTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "The frozen holdout execution manifest is missing. "
        "Run 4.6.1.5.1B first."
    )

holdout_execution_manifest_df = pd.read_parquet(
    HOLDOUT_EXECUTION_MANIFEST_PATH
)

required_binding_columns = {
    "metric",
    "horizon",
    "champion_family",
    "Family",
    "Representative",
    "modeling_track",
    "configuration_id",
    "runner_status",
}

missing_binding_columns = sorted(
    required_binding_columns
    - set(holdout_execution_manifest_df.columns)
)

if missing_binding_columns:
    raise KeyError(
        "Frozen holdout execution manifest is missing required fields: "
        f"{missing_binding_columns}"
    )

saved_runner_contract_df = pd.read_parquet(
    HOLDOUT_RUNNER_CONTRACT_PATH
)

holdout_runner_binding_df = holdout_execution_manifest_df.merge(
    saved_runner_contract_df,
    on="champion_family",
    how="left",
    validate="many_to_one",
)

if len(holdout_runner_binding_df) != EXPECTED_JOB_COUNT:
    raise AssertionError(
        f"Expected {EXPECTED_JOB_COUNT} frozen holdout jobs; "
        f"found {len(holdout_runner_binding_df)}."
    )

if holdout_runner_binding_df.duplicated(
    ["metric", "horizon"]
).any():
    raise AssertionError(
        "Holdout runner bindings are not unique by Target × Horizon."
    )

if holdout_runner_binding_df["runner_contract_id"].isna().any():
    raise AssertionError(
        "At least one champion is missing a source runner contract."
    )

family_label_mismatches = (
    holdout_runner_binding_df["Family"]
    != holdout_runner_binding_df["runner_family_label"]
).sum()

if family_label_mismatches:
    raise AssertionError(
        f"Found {family_label_mismatches} family-label mismatch(es) "
        "in runner binding."
    )

expected_champion_counts = {
    "tree": 8,
    "transformer": 4,
    "neural": 3,
}

observed_champion_counts = (
    holdout_runner_binding_df["champion_family"]
    .value_counts()
    .to_dict()
)

if observed_champion_counts != expected_champion_counts:
    raise AssertionError(
        "Frozen champion counts changed before holdout execution: "
        f"{observed_champion_counts}"
    )

unexpected_prior_status = ~holdout_runner_binding_df[
    "runner_status"
].eq("pending_source_runner_binding")

if unexpected_prior_status.any():
    unexpected = sorted(
        holdout_runner_binding_df.loc[
            unexpected_prior_status,
            "runner_status",
        ]
        .astype(str)
        .unique()
    )

    raise AssertionError(
        "Holdout roster does not have the expected pre-binding status: "
        f"{unexpected}"
    )

holdout_runner_binding_df = (
    holdout_runner_binding_df
    .rename(columns={"runner_status": "prior_runner_status"})
)

holdout_runner_binding_df["runner_status"] = (
    "source_contract_bound_pending_implementation"
)

write_small_contract(
    holdout_runner_binding_df,
    HOLDOUT_RUNNER_BINDING_PATH,
)

binding_display_df = (
    holdout_runner_binding_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS),
        Horizon=lambda frame:
            "h=" + frame["horizon"].astype(int).astype(str),
    )[
        [
            "Target",
            "Horizon",
            "Family",
            "modeling_track",
            "Representative",
            "refit_cadence",
            "training_duration_policy",
            "runner_status",
        ]
    ]
    .rename(columns={
        "modeling_track": "Modeling track",
        "refit_cadence": "Refit cadence",
        "training_duration_policy": "Training duration",
        "runner_status": "Runner status",
    })
)

display(binding_display_df)

print(
    f"Frozen champion bindings: {len(holdout_runner_binding_df)} jobs\n"
    f"Output: {HOLDOUT_RUNNER_BINDING_PATH}"
)

,Target,Horizon,Family,Modeling track,Representative,Refit cadence,Training duration,Runner status
0,Taxi trips,h=1,Tree,full_history_regime_aware,LightGBM,One fresh fit at final-holdout start,Frozen LightGBM model definition,source_contract_bound_pending_implementation
1,Taxi trips,h=2,Tree,full_history_regime_aware,LightGBM,One fresh fit at final-holdout start,Frozen LightGBM model definition,source_contract_bound_pending_implementation
2,Taxi trips,h=5,Tree,full_history_regime_aware,LightGBM,One fresh fit at final-holdout start,Frozen LightGBM model definition,source_contract_bound_pending_implementation
3,Taxi average speed,h=1,Neural,post_cp_primary,Lag-window MLP system,Monthly expanding-history retraining,Historical inner-validation selected epoch,source_contract_bound_pending_implementation
4,Taxi average speed,h=2,Neural,post_cp_primary,Lag-window MLP system,Monthly expanding-history retraining,Historical inner-validation selected epoch,source_contract_bound_pending_implementation
5,Taxi average speed,h=5,Neural,post_cp_primary,Lag-window MLP system,Monthly expanding-history retraining,Historical inner-validation selected epoch,source_contract_bound_pending_implementation
6,FHVHV trips,h=1,Tree,full_history_regime_aware,LightGBM,One fresh fit at final-holdout start,Frozen LightGBM model definition,source_contract_bound_pending_implementation
7,FHVHV trips,h=2,Tree,full_history_regime_aware,LightGBM,One fresh fit at final-holdout start,Frozen LightGBM model definition,source_contract_bound_pending_implementation
8,FHVHV trips,h=5,Tree,full_history_regime_aware,LightGBM,One fresh fit at final-holdout start,Frozen LightGBM model definition,source_contract_bound_pending_implementation
9,FHVHV average speed,h=1,Transformer,full_history_regime_aware,PatchTST-inspired Transformer,Monthly expanding-history retraining,Historical inner-validation selected epoch,source_contract_bound_pending_implementation


Frozen champion bindings: 15 jobs
Output: pipeline_data/4.6.1.intermediate_tables/frozen_holdout_runner_bindings.parquet


**Findings.** All **15 frozen champions** are attached to exactly one source-specific runner contract: **8 Tree, 4 Transformer, and 3 Neural**.

Each binding preserves the champion's selected modeling track, upstream configuration, refit cadence, and training-duration policy.

No advanced-model final-holdout prediction has been created.

**4.6.1.5.2C — Verify the reusable source artifacts required by the active runners.**

Before fitting any champion, verify that the source notebooks provide the frozen definitions required to reproduce Tree, Neural, and Transformer.

Tree requires its selected model registry, ordered predictor roster, and output contract.

Neural requires its model, input, preprocessing, output, and training-policy contracts.

Transformer requires its architecture, context, input, output, production-schedule, and training-policy contracts.

Neural and Transformer also share the canonical mobility-history state used to reconstruct their sequence inputs.

In [40]:
# ---------------------------------------------------------------------
# 4.6.1.5.2C — Audit reusable source artifacts for active runners
# ---------------------------------------------------------------------

source_artifact_specs = [
    # Tree: selected LightGBM roster and exact input semantics.
    {
        "Family": "Tree",
        "Contract role": "Selected job registry",
        "Path": INPUT_441_FINAL_DIR / "lgbm_selected_model_registry.parquet",
        "Kind": "parquet",
        "Expected rows": 15,
    },
    {
        "Family": "Tree",
        "Contract role": "Ordered predictor registry",
        "Path": INPUT_441_FINAL_DIR / "lgbm_predictor_registry.parquet",
        "Kind": "parquet",
        "Expected rows": 425,
    },
    {
        "Family": "Tree",
        "Contract role": "Forecast-output contract",
        "Path": INPUT_441_FINAL_DIR / "lgbm_forecast_output_contract.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },

    # Neural: final MLP definitions plus input/training contracts.
    {
        "Family": "Neural",
        "Contract role": "Final model registry",
        "Path": INPUT_451_FINAL_DIR / "neural_final_model_registry.parquet",
        "Kind": "parquet",
        "Expected rows": 15,
    },
    {
        "Family": "Neural",
        "Contract role": "Model-definition contract",
        "Path": INPUT_451_FINAL_DIR / "neural_model_definition_contract.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Neural",
        "Contract role": "Input registry",
        "Path": INPUT_451_FINAL_DIR / "neural_input_registry.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Neural",
        "Contract role": "Sequence/preprocessing contract",
        "Path": (
            INPUT_451_FINAL_DIR
            / "neural_sequence_preprocessing_contract.parquet"
        ),
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Neural",
        "Contract role": "Forecast-output contract",
        "Path": INPUT_451_FINAL_DIR / "neural_forecast_output_contract.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Neural",
        "Contract role": "Training-policy evidence",
        "Path": INPUT_451_FINAL_DIR / "neural_training_policy_evidence.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },

    # Transformer: architecture/context definitions and training policy.
    {
        "Family": "Transformer",
        "Contract role": "Final model registry",
        "Path": INPUT_452_FINAL_DIR / "transformer_final_model_registry.parquet",
        "Kind": "parquet",
        "Expected rows": 15,
    },
    {
        "Family": "Transformer",
        "Contract role": "Model-definition contract",
        "Path": (
            INPUT_452_FINAL_DIR
            / "transformer_model_definition_contract.parquet"
        ),
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Transformer",
        "Contract role": "Context contract",
        "Path": INPUT_452_FINAL_DIR / "transformer_context_contract.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Transformer",
        "Contract role": "Input registry",
        "Path": INPUT_452_FINAL_DIR / "transformer_input_registry.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Transformer",
        "Contract role": "Forecast-output contract",
        "Path": (
            INPUT_452_FINAL_DIR
            / "transformer_forecast_output_contract.parquet"
        ),
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Transformer",
        "Contract role": "Production schedule",
        "Path": INPUT_452_FINAL_DIR / "transformer_production_schedule.parquet",
        "Kind": "parquet",
        "Expected rows": np.nan,
    },
    {
        "Family": "Transformer",
        "Contract role": "Training-policy evidence",
        "Path": (
            INPUT_452_FINAL_DIR
            / "transformer_training_policy_evidence.parquet"
        ),
        "Kind": "parquet",
        "Expected rows": np.nan,
    },

    # Both sequence families use the same canonical mobility-history state.
    {
        "Family": "Neural + Transformer",
        "Contract role": "Shared canonical mobility-history state",
        "Path": INPUT_451_FINAL_DIR / "neural_mobility_history_state.npz",
        "Kind": "npz",
        "Expected rows": np.nan,
    },
]

missing_artifacts = [
    str(spec["Path"])
    for spec in source_artifact_specs
    if not Path(spec["Path"]).exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Holdout runner source audit found missing artifact(s):\n- "
        + "\n- ".join(missing_artifacts)
    )

artifact_audit_records = []

for spec in source_artifact_specs:
    path = Path(spec["Path"])
    kind = spec["Kind"]

    if kind == "parquet":
        rows = int(parquet_row_count(path))
        columns = int(len(parquet_columns(path)))

        if rows <= 0 or columns <= 0:
            raise AssertionError(
                f"Source contract is empty: {path}"
            )

    elif kind == "npz":
        rows = np.nan
        columns = np.nan

        # Member inspection verifies the reusable state contract without
        # loading the dense arrays into notebook memory.
        with np.load(path, allow_pickle=False) as state:
            state_members = set(state.files)

        required_state_members = {
            "values",
            "masks",
            "zone_ids",
            "sequence_ids",
        }

        missing_state_members = sorted(
            required_state_members - state_members
        )

        if missing_state_members:
            raise KeyError(
                f"{path.name} is missing required history-state members: "
                f"{missing_state_members}"
            )

    else:
        raise ValueError(
            f"Unsupported source-artifact kind: {kind}"
        )

    expected_rows = spec["Expected rows"]

    row_contract_ok = (
        True
        if pd.isna(expected_rows)
        else int(rows) == int(expected_rows)
    )

    artifact_audit_records.append({
        "Family": spec["Family"],
        "Contract role": spec["Contract role"],
        "Artifact": path.name,
        "Kind": kind,
        "Materialized": True,
        "Rows": rows,
        "Columns": columns,
        "Expected rows": expected_rows,
        "Row contract OK": bool(row_contract_ok),
        "Size MB": path.stat().st_size / (1024 ** 2),
    })

holdout_source_artifact_audit_df = pd.DataFrame(
    artifact_audit_records
)

if not holdout_source_artifact_audit_df["Row contract OK"].all():
    failed = holdout_source_artifact_audit_df.loc[
        ~holdout_source_artifact_audit_df["Row contract OK"],
        ["Artifact", "Rows", "Expected rows"],
    ]

    raise AssertionError(
        "One or more frozen source artifacts changed shape:\n"
        + failed.to_string(index=False)
    )

if len(holdout_source_artifact_audit_df) != 17:
    raise AssertionError(
        "Expected 17 reusable source artifacts for the three active runners."
    )

write_small_contract(
    holdout_source_artifact_audit_df,
    HOLDOUT_SOURCE_ARTIFACT_AUDIT_PATH,
)

artifact_display_df = (
    holdout_source_artifact_audit_df[
        [
            "Family",
            "Contract role",
            "Artifact",
            "Materialized",
            "Rows",
            "Columns",
            "Size MB",
        ]
    ]
    .copy()
)

artifact_display_df["Rows"] = artifact_display_df["Rows"].map(
    lambda value: "—" if pd.isna(value) else f"{int(value):,}"
)
artifact_display_df["Columns"] = artifact_display_df["Columns"].map(
    lambda value: "—" if pd.isna(value) else f"{int(value):,}"
)
artifact_display_df["Size MB"] = (
    artifact_display_df["Size MB"].round(3)
)

display(artifact_display_df)

print(
    f"Required source artifacts verified: {len(artifact_display_df)}\n"
    f"Output: {HOLDOUT_SOURCE_ARTIFACT_AUDIT_PATH}"
)

,Family,Contract role,Artifact,Materialized,Rows,Columns,Size MB
0,Tree,Selected job registry,lgbm_selected_model_registry.parquet,True,15,51,0.037
1,Tree,Ordered predictor registry,lgbm_predictor_registry.parquet,True,425,6,0.006
2,Tree,Forecast-output contract,lgbm_forecast_output_contract.parquet,True,5,6,0.005
3,Neural,Final model registry,neural_final_model_registry.parquet,True,15,49,0.039
4,Neural,Model-definition contract,neural_model_definition_contract.parquet,True,3,14,0.011
5,Neural,Input registry,neural_input_registry.parquet,True,315,7,0.007
6,Neural,Sequence/preprocessing contract,neural_sequence_preprocessing_contract.parquet,True,5,20,0.021
7,Neural,Forecast-output contract,neural_forecast_output_contract.parquet,True,15,13,0.010
8,Neural,Training-policy evidence,neural_training_policy_evidence.parquet,True,150,58,0.069
9,Transformer,Final model registry,transformer_final_model_registry.parquet,True,15,34,0.026


Required source artifacts verified: 17
Output: pipeline_data/4.6.1.intermediate_tables/holdout_source_artifact_audit.parquet


**Findings.** All **17 source artifacts** required by the three active holdout runners are materialized and pass their structural checks.

Tree retains its selected LightGBM registry, ordered **425-row predictor registry**, and forecast-output contract. Neural and Transformer retain their frozen model, input, preprocessing/context, output, and training-policy definitions.

The shared canonical mobility-history state also passes its required-member check, so both sequence runners have the reusable history representation needed for final-holdout execution.

No final-holdout model is trained in this audit.

**4.6.1.5.2D — Summarize the planned holdout execution pattern.**

The 15 champion jobs use two execution patterns.

Tree performs one global-panel fit per selected Target × Horizon.

Neural and Transformer retrain monthly from expanding legal history. Every monthly restart unit contains one historical selector fit followed by one fresh final-model fit.

The next subsection converts those rules into the exact restart-unit and training-call counts.

In [41]:
# ---------------------------------------------------------------------
# 4.6.1.5.2D — Summarize the planned holdout execution pattern
# ---------------------------------------------------------------------

saved_binding_df = pd.read_parquet(
    HOLDOUT_RUNNER_BINDING_PATH
)

champion_job_counts = (
    saved_binding_df.groupby(
        ["champion_family", "runner_family_label"],
        as_index=False,
        observed=True,
    )
    .size()
    .rename(columns={"size": "Champion jobs"})
)

execution_pattern_df = pd.DataFrame([
    {
        "champion_family": "tree",
        "Execution pattern": "One global-panel fit per champion job",
        "Restart-unit interpretation": "1 Target × Horizon unit per job",
        "Fresh selector before final fit": "No",
    },
    {
        "champion_family": "neural",
        "Execution pattern": "Monthly expanding-history refits",
        "Restart-unit interpretation": (
            "1 Target × Horizon × holdout-origin month unit"
        ),
        "Fresh selector before final fit": "Yes",
    },
    {
        "champion_family": "transformer",
        "Execution pattern": "Monthly expanding-history refits",
        "Restart-unit interpretation": (
            "1 Target × Horizon × holdout-origin month unit"
        ),
        "Fresh selector before final fit": "Yes",
    },
])

holdout_execution_plan_df = champion_job_counts.merge(
    execution_pattern_df,
    on="champion_family",
    how="left",
    validate="one_to_one",
)

if holdout_execution_plan_df["Champion jobs"].sum() != EXPECTED_JOB_COUNT:
    raise AssertionError(
        "Holdout execution plan does not cover all 15 champions."
    )

required_plan_fields = [
    "Execution pattern",
    "Restart-unit interpretation",
    "Fresh selector before final fit",
]

if holdout_execution_plan_df[
    required_plan_fields
].isna().any().any():
    raise AssertionError(
        "At least one champion family is missing execution-plan metadata."
    )

write_small_contract(
    holdout_execution_plan_df,
    HOLDOUT_EXECUTION_PLAN_PATH,
)

execution_display_df = (
    holdout_execution_plan_df[
        [
            "runner_family_label",
            "Champion jobs",
            "Execution pattern",
            "Restart-unit interpretation",
            "Fresh selector before final fit",
        ]
    ]
    .rename(columns={"runner_family_label": "Family"})
)

display(execution_display_df)

print(
    "Execution-plan coverage: "
    f"{int(holdout_execution_plan_df['Champion jobs'].sum())} / "
    f"{EXPECTED_JOB_COUNT} champion jobs\n"
    f"Output: {HOLDOUT_EXECUTION_PLAN_PATH}"
)

,Family,Champion jobs,Execution pattern,Restart-unit interpretation,Fresh selector before final fit
0,Neural,3,Monthly expanding-history refits,1 Target × Horizon × holdout-origin month unit,Yes
1,Transformer,4,Monthly expanding-history refits,1 Target × Horizon × holdout-origin month unit,Yes
2,Tree,8,One global-panel fit per champion job,1 Target × Horizon unit per job,No


Execution-plan coverage: 15 / 15 champion jobs
Output: pipeline_data/4.6.1.intermediate_tables/holdout_execution_plan_summary.parquet


**Findings.** The execution plan covers all **15 champion jobs** across three active families.

Tree contributes **8 jobs**, each requiring one global-panel fit. Neural contributes **3 jobs** and Transformer contributes **4 jobs**; both sequence families retrain by holdout-origin month and use one selector plus one fresh final fit at every refresh.

The next subsection calculates the exact monthly schedules and checkpoint workload before any missing model fit is allowed to run.

### 4.6.1.5.3 Freeze the Exact Final-Holdout Workload and Restart Contract

The execution contracts can now be translated into physical work.

The eight Tree champions each require one Target × Horizon fit. Neural and Transformer retrain by holdout-origin month, with one selector and one fresh final model inside each monthly restart unit.

The next two blocks calculate the exact workload, define the durable checkpoint grain, and identify which units are already complete versus pending.

No missing model fit is launched unless its family-specific execution control is explicitly enabled.

**4.6.1.5.3A — Count the physical fitting work behind the 15 champions.**

Tree uses one restart unit per champion job.

Neural and Transformer use one restart unit per Target × Horizon × holdout-origin month. Monthly schedules are reconstructed from the actual **forecast origin**, preserving the chronology of multi-step forecasts near month boundaries.

Each sequence-model monthly unit contains two training calls: one historical training-duration selector and one fresh final model.

In [42]:
# ---------------------------------------------------------------------
# 4.6.1.5.3A — Count the physical fitting work behind the 15 champions
# ---------------------------------------------------------------------

HOLDOUT_EXACT_WORKLOAD_PATH = (
    INTERMEDIATE_DIR / "holdout_exact_workload_summary.parquet"
)
HOLDOUT_MONTHLY_REFIT_PLAN_PATH = (
    INTERMEDIATE_DIR / "holdout_monthly_refit_plan.parquet"
)

saved_binding_df = pd.read_parquet(
    HOLDOUT_RUNNER_BINDING_PATH
)

require_columns(
    saved_binding_df,
    [
        "metric",
        "horizon",
        "champion_family",
        "Family",
        "modeling_track",
        "configuration_id",
        "runner_contract_id",
    ],
    "frozen holdout runner bindings",
)

expected_champion_counts = {
    "tree": 8,
    "neural": 3,
    "transformer": 4,
}

observed_champion_counts = (
    saved_binding_df["champion_family"]
    .value_counts()
    .to_dict()
)

if observed_champion_counts != expected_champion_counts:
    raise RuntimeError(
        "Holdout workload does not match the frozen champion composition: "
        f"{observed_champion_counts}"
    )

expected_holdout_months = {
    str(period)
    for period in pd.period_range(
        FINAL_HOLDOUT_START_DATE,
        FINAL_HOLDOUT_END_DATE,
        freq="M",
    )
}

if len(expected_holdout_months) != 3:
    raise RuntimeError(
        "The final holdout no longer spans the expected three "
        "origin months."
    )


def read_holdout_origin_months(track, metric, horizon):
    """
    Build one monthly schedule from legal final-holdout forecast origins.

    Only one Track × Target × Horizon slice is needed, so Pandas reads that
    slice directly from Parquet instead of introducing a SQL query layer.
    """
    matrix_path = FORECASTING_MATRIX_PATHS[track]

    required_columns = [
        "modeling_track",
        "stage",
        "metric",
        "horizon",
        "origin_observation_sequence_id",
        "origin_date",
        "target_date",
        "target_observed_value",
        "target_label_available",
        "native_track_eligible",
    ]

    require_holdout_source_schema(
        matrix_path,
        required_columns,
        f"{TRACK_LABELS[track]} forecasting matrix",
    )

    schedule_source_df = pd.read_parquet(
        matrix_path,
        columns=[
            "origin_observation_sequence_id",
            "origin_date",
            "target_date",
            "target_observed_value",
            "target_label_available",
            "native_track_eligible",
        ],
        filters=[
            ("modeling_track", "==", str(track)),
            ("stage", "==", "final_holdout"),
            ("metric", "==", str(metric)),
            ("horizon", "==", int(horizon)),
        ],
    )

    target_values = pd.to_numeric(
        schedule_source_df["target_observed_value"],
        errors="coerce",
    )

    valid_mask = (
        schedule_source_df["native_track_eligible"].fillna(False).astype(bool)
        & schedule_source_df["target_label_available"].fillna(False).astype(bool)
        & np.isfinite(target_values.fillna(np.nan).to_numpy(dtype=float))
    )

    schedule_source_df = schedule_source_df.loc[valid_mask].copy()

    if schedule_source_df.empty:
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: no legal holdout origins."
        )

    schedule_source_df["origin_observation_sequence_id"] = pd.to_numeric(
        schedule_source_df["origin_observation_sequence_id"],
        errors="raise",
    ).astype("int64")

    schedule_source_df["origin_date"] = pd.to_datetime(
        schedule_source_df["origin_date"]
    ).dt.normalize()

    schedule_source_df["target_date"] = pd.to_datetime(
        schedule_source_df["target_date"]
    ).dt.normalize()

    schedule_source_df["refit_month_start"] = (
        schedule_source_df["origin_date"]
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    schedule_df = (
        schedule_source_df.groupby(
            "refit_month_start",
            observed=True,
            as_index=False,
        )
        .agg(
            refit_origin_observation_sequence_id=(
                "origin_observation_sequence_id",
                "min",
            ),
            refit_origin_date=("origin_date", "min"),
            first_target_date=("target_date", "min"),
            last_target_date=("target_date", "max"),
            score_rows=("origin_observation_sequence_id", "size"),
        )
        .sort_values("refit_month_start")
        .reset_index(drop=True)
    )

    schedule_df["refit_month"] = (
        schedule_df["refit_month_start"]
        .dt.to_period("M")
        .astype(str)
    )
    schedule_df["refit_step"] = np.arange(
        len(schedule_df),
        dtype=int,
    )

    schedule_df["next_refit_origin_observation_sequence_id"] = (
        schedule_df["refit_origin_observation_sequence_id"].shift(-1)
    )

    observed_months = set(schedule_df["refit_month"])

    if observed_months != expected_holdout_months:
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: expected origin months "
            f"{sorted(expected_holdout_months)}, found {sorted(observed_months)}."
        )

    return schedule_df


# ---------------------------------------------------------------------
# Neural + Transformer — selector + fresh final fit each month
# ---------------------------------------------------------------------

monthly_plan_frames = []

for family in ["neural", "transformer"]:
    family_jobs_df = saved_binding_df.loc[
        saved_binding_df["champion_family"].eq(family)
    ]

    for job in family_jobs_df.itertuples(index=False):
        schedule_df = read_holdout_origin_months(
            job.modeling_track,
            job.metric,
            int(job.horizon),
        )

        schedule_df = schedule_df.assign(
            champion_family=family,
            Family=job.Family,
            modeling_track=job.modeling_track,
            metric=job.metric,
            horizon=int(job.horizon),
            configuration_id=job.configuration_id,
            runner_contract_id=job.runner_contract_id,
            selector_fits=1,
            final_model_fits=1,
            planned_fit_calls=2,
        )

        monthly_plan_frames.append(schedule_df)

if not monthly_plan_frames:
    raise RuntimeError(
        "No Neural or Transformer monthly holdout work was found."
    )

holdout_monthly_refit_plan_df = (
    pd.concat(
        monthly_plan_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "champion_family",
            "metric",
            "horizon",
            "refit_step",
        ]
    )
    .reset_index(drop=True)
)

monthly_key = [
    "champion_family",
    "metric",
    "horizon",
    "refit_step",
]

if holdout_monthly_refit_plan_df.duplicated(monthly_key).any():
    raise RuntimeError(
        "Sequence-model holdout plan contains duplicate "
        "Target × Horizon × Month units."
    )

monthly_counts = (
    holdout_monthly_refit_plan_df.groupby(
        ["champion_family", "metric", "horizon"],
        observed=True,
    )
    .size()
)

if not monthly_counts.eq(len(expected_holdout_months)).all():
    raise RuntimeError(
        "At least one Neural/Transformer job has an incomplete "
        "monthly holdout schedule."
    )

write_small_contract(
    holdout_monthly_refit_plan_df,
    HOLDOUT_MONTHLY_REFIT_PLAN_PATH,
)


# ---------------------------------------------------------------------
# Summarize work at each family's natural restart grain
# ---------------------------------------------------------------------

tree_jobs = int(
    saved_binding_df["champion_family"].eq("tree").sum()
)

neural_plan_df = holdout_monthly_refit_plan_df.loc[
    holdout_monthly_refit_plan_df["champion_family"].eq("neural")
]

transformer_plan_df = holdout_monthly_refit_plan_df.loc[
    holdout_monthly_refit_plan_df["champion_family"].eq("transformer")
]

holdout_exact_workload_df = pd.DataFrame([
    {
        "champion_family": "tree",
        "Family": "Tree",
        "Champion jobs": tree_jobs,
        "Restart units": tree_jobs,
        "Selector fits": 0,
        "Production fits / refreshes": tree_jobs,
        "Planned training calls": tree_jobs,
    },
    {
        "champion_family": "neural",
        "Family": "Neural",
        "Champion jobs": int(
            saved_binding_df["champion_family"].eq("neural").sum()
        ),
        "Restart units": len(neural_plan_df),
        "Selector fits": int(neural_plan_df["selector_fits"].sum()),
        "Production fits / refreshes": int(
            neural_plan_df["final_model_fits"].sum()
        ),
        "Planned training calls": int(
            neural_plan_df["planned_fit_calls"].sum()
        ),
    },
    {
        "champion_family": "transformer",
        "Family": "Transformer",
        "Champion jobs": int(
            saved_binding_df["champion_family"].eq("transformer").sum()
        ),
        "Restart units": len(transformer_plan_df),
        "Selector fits": int(transformer_plan_df["selector_fits"].sum()),
        "Production fits / refreshes": int(
            transformer_plan_df["final_model_fits"].sum()
        ),
        "Planned training calls": int(
            transformer_plan_df["planned_fit_calls"].sum()
        ),
    },
])

if int(
    holdout_exact_workload_df["Champion jobs"].sum()
) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Exact holdout workload does not cover all 15 champions."
    )

total_restart_units = int(
    holdout_exact_workload_df["Restart units"].sum()
)
total_training_calls = int(
    holdout_exact_workload_df["Planned training calls"].sum()
)

if total_restart_units != 29:
    raise RuntimeError(
        f"Expected 29 physical restart units; found {total_restart_units}."
    )

if total_training_calls != 50:
    raise RuntimeError(
        f"Expected 50 planned training calls; found {total_training_calls}."
    )

write_small_contract(
    holdout_exact_workload_df,
    HOLDOUT_EXACT_WORKLOAD_PATH,
)

display(
    holdout_exact_workload_df[
        [
            "Family",
            "Champion jobs",
            "Restart units",
            "Selector fits",
            "Production fits / refreshes",
            "Planned training calls",
        ]
    ]
)

print(
    "PASS — exact final-holdout workload frozen before fitting.\n"
    f"  Tree units:              {tree_jobs:,}\n"
    f"  Neural monthly units:    {len(neural_plan_df):,}\n"
    f"  Transformer monthly units: {len(transformer_plan_df):,}\n"
    f"  Total restart units:     {total_restart_units:,}\n"
    f"  Planned training calls:  {total_training_calls:,}\n"
    f"  Summary: {HOLDOUT_EXACT_WORKLOAD_PATH}",
    flush=True,
)

,Family,Champion jobs,Restart units,Selector fits,Production fits / refreshes,Planned training calls
0,Tree,8,8,0,8,8
1,Neural,3,9,9,9,18
2,Transformer,4,12,12,12,24


PASS — exact final-holdout workload frozen before fitting.
  Tree units:              8
  Neural monthly units:    9
  Transformer monthly units: 12
  Total restart units:     29
  Planned training calls:  50
  Summary: pipeline_data/4.6.1.intermediate_tables/holdout_exact_workload_summary.parquet


**Findings.** The 15-job champion system expands into **29 physical restart units** and **50 planned training calls** for the January–March 2026 final holdout.

Tree contributes **8 restart units and 8 one-time LightGBM fits**.

Neural contributes **9 monthly restart units** across its 3 champion jobs. Each unit contains one selector and one fresh final model, producing **18 training calls**.

Transformer contributes **12 monthly restart units** across its 4 champion jobs, producing **24 training calls**.

The full workload is therefore:

**8 Tree + 18 Neural + 24 Transformer = 50 planned training calls.**

**4.6.1.5.3B — Freeze restart units, artifact checks, and explicit execution controls.**

Each expensive logical unit receives a deterministic checkpoint identity.

Tree checkpoints once per Target × Horizon. Neural and Transformer checkpoint once per Target × Horizon × holdout-origin month.

Every completed unit owns both:

- a row-level prediction artifact; and
- a compact fit-audit artifact.

A unit is recoverable only when both files exist, are readable, expose the required schema, and carry the expected runtime identity. Missing work remains pending until its family-specific execution control is deliberately enabled.

The destructive reset control remains separate from the execution controls.

In [43]:
# ---------------------------------------------------------------------
# 4.6.1.5.3B — Freeze restart units, artifact checks, and controls
# ---------------------------------------------------------------------

import shutil
import uuid

# Unit identity is based on the physical model execution contract. This allows
# compatible completed Tree jobs to remain reusable when their exact model,
# track, horizon, and configuration are unchanged.
HOLDOUT_RUNTIME_CONTRACT_ID = "final_holdout_runtime_v1"
HOLDOUT_RUNTIME_ROOT = (
    INTERMEDIATE_DIR / HOLDOUT_RUNTIME_CONTRACT_ID
)
HOLDOUT_CHECKPOINT_MANIFEST_PATH = (
    INTERMEDIATE_DIR / "holdout_checkpoint_manifest.parquet"
)

ACTIVE_HOLDOUT_FAMILIES = ["tree", "neural", "transformer"]

# Expensive computation is always opt-in. Recovery is safe to perform on every
# rerun because it only accepts checkpoint pairs that satisfy the contract.
RUN_TREE_HOLDOUT = True
RUN_NEURAL_HOLDOUT = False
RUN_TRANSFORMER_HOLDOUT = False
RESET_FINAL_HOLDOUT_RUNTIME = False

if RESET_FINAL_HOLDOUT_RUNTIME:
    if HOLDOUT_RUNTIME_ROOT.exists():
        shutil.rmtree(HOLDOUT_RUNTIME_ROOT)

    HOLDOUT_CHECKPOINT_MANIFEST_PATH.unlink(
        missing_ok=True
    )

for family in ACTIVE_HOLDOUT_FAMILIES:
    family_root = HOLDOUT_RUNTIME_ROOT / family

    (family_root / "predictions").mkdir(
        parents=True,
        exist_ok=True,
    )
    (family_root / "audit").mkdir(
        parents=True,
        exist_ok=True,
    )


HOLDOUT_UNIT_PREDICTION_COLUMNS = [
    "runtime_unit_id",
    "holdout_runtime_contract_id",
    "champion_family",
    "modeling_track",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "target_date",
    "actual",
    "benchmark_prediction",
    "prediction_raw",
    "prediction_final",
    "model_prediction_available",
    "benchmark_prediction_available",
]

HOLDOUT_UNIT_AUDIT_COLUMNS = [
    "runtime_unit_id",
    "holdout_runtime_contract_id",
    "champion_family",
    "unit_complete",
]


def stable_holdout_unit_id(*parts):
    """
    Return a deterministic ID for one source-native restart unit.

    The same logical model work receives the same identity across notebook
    reruns, allowing completed compatible units to be recovered safely.
    """
    token = "|".join(str(part) for part in parts)

    return uuid.uuid5(
        uuid.NAMESPACE_URL,
        f"{HOLDOUT_RUNTIME_CONTRACT_ID}|{token}",
    ).hex


def holdout_unit_paths(family, runtime_unit_id):
    """Return the prediction/audit checkpoint pair for one restart unit."""
    family_root = HOLDOUT_RUNTIME_ROOT / family

    prediction_path = (
        family_root
        / "predictions"
        / f"{runtime_unit_id}.parquet"
    )
    audit_path = (
        family_root
        / "audit"
        / f"{runtime_unit_id}.parquet"
    )

    return prediction_path, audit_path


def completed_holdout_unit_is_valid(row):
    """
    Decide whether one checkpoint pair can be recovered.

    Recovery checks durable identity and schema without rescanning the complete
    prediction table. Full row-level integrity remains the responsibility of
    the family runner when the unit is created and of final reconciliation.
    """
    prediction_path = Path(row["prediction_path"])
    audit_path = Path(row["audit_path"])

    if not prediction_path.exists() or not audit_path.exists():
        return False

    try:
        prediction_file = pq.ParquetFile(
            prediction_path
        )
        prediction_columns = set(
            prediction_file.schema_arrow.names
        )

        if prediction_file.metadata.num_rows <= 0:
            return False

        if not set(
            HOLDOUT_UNIT_PREDICTION_COLUMNS
        ).issubset(prediction_columns):
            return False

        audit_df = pd.read_parquet(
            audit_path,
            columns=HOLDOUT_UNIT_AUDIT_COLUMNS,
        )

    except Exception:
        return False

    if audit_df.empty:
        return False

    audit_ok = (
        audit_df["runtime_unit_id"]
        .astype(str)
        .eq(str(row["runtime_unit_id"]))
        .all()
        and audit_df["holdout_runtime_contract_id"]
        .astype(str)
        .eq(HOLDOUT_RUNTIME_CONTRACT_ID)
        .all()
        and audit_df["champion_family"]
        .astype(str)
        .eq(str(row["champion_family"]))
        .all()
        and audit_df["unit_complete"]
        .fillna(False)
        .astype(bool)
        .all()
    )

    return bool(audit_ok)


# ---------------------------------------------------------------------
# Build restart units at each active family's natural execution grain
# ---------------------------------------------------------------------

monthly_refit_plan_df = pd.read_parquet(
    HOLDOUT_MONTHLY_REFIT_PLAN_PATH
)
saved_binding_df = pd.read_parquet(
    HOLDOUT_RUNNER_BINDING_PATH
)

manifest_records = []


# Tree: one complete Target × Horizon global-panel fit is one restart unit.
tree_binding_df = saved_binding_df.loc[
    saved_binding_df["champion_family"].eq("tree")
]

for row in tree_binding_df.itertuples(index=False):
    unit_id = stable_holdout_unit_id(
        "tree",
        row.modeling_track,
        row.metric,
        int(row.horizon),
        row.configuration_id,
    )

    prediction_path, audit_path = holdout_unit_paths(
        "tree",
        unit_id,
    )

    manifest_records.append({
        "runtime_unit_id": unit_id,
        "champion_family": "tree",
        "Family": "Tree",
        "modeling_track": row.modeling_track,
        "metric": row.metric,
        "horizon": int(row.horizon),
        "selected_horizons": str(int(row.horizon)),
        "configuration_id": row.configuration_id,
        "runner_contract_id": row.runner_contract_id,
        "forecast_sequence_id": None,
        "taxi_zone_id": np.nan,
        "refit_month": None,
        "refit_step": 0,
        "refit_origin_observation_sequence_id": np.nan,
        "next_refit_origin_observation_sequence_id": np.nan,
        "planned_fit_calls": 1,
        "checkpoint_grain": "Target × Horizon global-panel fit",
        "prediction_path": str(prediction_path),
        "audit_path": str(audit_path),
    })


# Neural + Transformer: selector and fresh final fit belong to the same
# Target × Horizon × Month restart unit.
for row in monthly_refit_plan_df.itertuples(index=False):
    family = str(row.champion_family)

    if family not in {"neural", "transformer"}:
        raise RuntimeError(
            f"Unexpected monthly holdout family: {family}"
        )

    unit_id = stable_holdout_unit_id(
        family,
        row.modeling_track,
        row.metric,
        int(row.horizon),
        row.configuration_id,
        row.refit_month,
    )

    prediction_path, audit_path = holdout_unit_paths(
        family,
        unit_id,
    )

    manifest_records.append({
        "runtime_unit_id": unit_id,
        "champion_family": family,
        "Family": row.Family,
        "modeling_track": row.modeling_track,
        "metric": row.metric,
        "horizon": int(row.horizon),
        "selected_horizons": str(int(row.horizon)),
        "configuration_id": row.configuration_id,
        "runner_contract_id": row.runner_contract_id,
        "forecast_sequence_id": None,
        "taxi_zone_id": np.nan,
        "refit_month": row.refit_month,
        "refit_step": int(row.refit_step),
        "refit_origin_observation_sequence_id": int(
            row.refit_origin_observation_sequence_id
        ),
        "next_refit_origin_observation_sequence_id":
            row.next_refit_origin_observation_sequence_id,
        "planned_fit_calls": int(row.planned_fit_calls),
        "checkpoint_grain": "Target × Horizon × holdout month",
        "prediction_path": str(prediction_path),
        "audit_path": str(audit_path),
    })


holdout_checkpoint_manifest_df = pd.DataFrame(
    manifest_records
)

holdout_checkpoint_manifest_df[
    "holdout_runtime_contract_id"
] = HOLDOUT_RUNTIME_CONTRACT_ID

holdout_checkpoint_manifest_df[
    "selection_contract_id"
] = CHAMPION_SELECTION_CONTRACT_ID

holdout_checkpoint_manifest_df[
    "prediction_output_rule"
] = "prediction_final = max(0, prediction_raw)"


# ---------------------------------------------------------------------
# Validate the physical restart surface
# ---------------------------------------------------------------------

if holdout_checkpoint_manifest_df[
    "runtime_unit_id"
].duplicated().any():
    raise RuntimeError(
        "Final-holdout runtime manifest contains duplicate restart-unit IDs."
    )

observed_runtime_families = set(
    holdout_checkpoint_manifest_df["champion_family"]
)

if observed_runtime_families != set(ACTIVE_HOLDOUT_FAMILIES):
    raise RuntimeError(
        "Final-holdout runtime manifest does not contain exactly "
        "Tree, Neural, and Transformer."
    )

total_restart_units = len(
    holdout_checkpoint_manifest_df
)
total_planned_calls = int(
    holdout_checkpoint_manifest_df["planned_fit_calls"].sum()
)

if total_restart_units != 29:
    raise RuntimeError(
        f"Expected 29 checkpoint units; found {total_restart_units}."
    )

if total_planned_calls != 50:
    raise RuntimeError(
        f"Expected 50 planned training calls; found {total_planned_calls}."
    )


# ---------------------------------------------------------------------
# Recover completed compatible units; leave all missing work pending
# ---------------------------------------------------------------------

artifact_status = []

for row in holdout_checkpoint_manifest_df.to_dict("records"):
    prediction_exists = Path(
        row["prediction_path"]
    ).exists()
    audit_exists = Path(
        row["audit_path"]
    ).exists()

    if not prediction_exists and not audit_exists:
        status = "pending"

    elif prediction_exists != audit_exists:
        status = "partial_invalid"

    else:
        status = (
            "recovered"
            if completed_holdout_unit_is_valid(row)
            else "invalid"
        )

    artifact_status.append(status)

holdout_checkpoint_manifest_df[
    "artifact_status"
] = artifact_status

invalid_mask = holdout_checkpoint_manifest_df[
    "artifact_status"
].isin(["partial_invalid", "invalid"])

if invalid_mask.any():
    invalid_units = holdout_checkpoint_manifest_df.loc[
        invalid_mask,
        [
            "Family",
            "metric",
            "selected_horizons",
            "refit_month",
            "artifact_status",
            "runtime_unit_id",
        ],
    ]

    raise RuntimeError(
        "Holdout runtime contains incomplete or incompatible "
        "checkpoint pairs. Review them before using "
        "RESET_FINAL_HOLDOUT_RUNTIME=True:\n"
        + invalid_units.to_string(index=False)
    )

write_small_contract(
    holdout_checkpoint_manifest_df,
    HOLDOUT_CHECKPOINT_MANIFEST_PATH,
)


# ---------------------------------------------------------------------
# Reader-facing restart summary
# ---------------------------------------------------------------------

checkpoint_summary_df = (
    holdout_checkpoint_manifest_df.assign(
        Recovered=lambda frame:
            frame["artifact_status"].eq("recovered").astype(int),
        Pending=lambda frame:
            frame["artifact_status"].eq("pending").astype(int),
    )
    .groupby(
        ["champion_family", "Family"],
        as_index=False,
        observed=True,
    )
    .agg(**{
        "Restart units": ("runtime_unit_id", "size"),
        "Recovered": ("Recovered", "sum"),
        "Pending": ("Pending", "sum"),
        "Planned training calls": ("planned_fit_calls", "sum"),
    })
)

family_order = {
    "tree": 0,
    "neural": 1,
    "transformer": 2,
}

checkpoint_summary_df = (
    checkpoint_summary_df.assign(
        _family_order=lambda frame:
            frame["champion_family"].map(family_order)
    )
    .sort_values("_family_order")
    .drop(columns="_family_order")
    .reset_index(drop=True)
)

display(
    checkpoint_summary_df[
        [
            "Family",
            "Restart units",
            "Recovered",
            "Pending",
            "Planned training calls",
        ]
    ]
)

print(
    "PASS — final-holdout restart contract is ready.\n"
    f"  Runtime contract: {HOLDOUT_RUNTIME_CONTRACT_ID}\n"
    f"  Checkpoint units: {len(holdout_checkpoint_manifest_df):,}\n"
    f"  Planned training calls: {total_planned_calls:,}\n"
    f"  Manifest: {HOLDOUT_CHECKPOINT_MANIFEST_PATH}\n\n"
    "Execution controls:\n"
    f"  RUN_TREE_HOLDOUT={RUN_TREE_HOLDOUT}\n"
    f"  RUN_NEURAL_HOLDOUT={RUN_NEURAL_HOLDOUT}\n"
    f"  RUN_TRANSFORMER_HOLDOUT={RUN_TRANSFORMER_HOLDOUT}\n"
    f"  RESET_FINAL_HOLDOUT_RUNTIME={RESET_FINAL_HOLDOUT_RUNTIME}",
    flush=True,
)

,Family,Restart units,Recovered,Pending,Planned training calls
0,Tree,8,8,0,8
1,Neural,9,9,0,18
2,Transformer,12,12,0,24


PASS — final-holdout restart contract is ready.
  Runtime contract: final_holdout_runtime_v1
  Checkpoint units: 29
  Planned training calls: 50
  Manifest: pipeline_data/4.6.1.intermediate_tables/holdout_checkpoint_manifest.parquet

Execution controls:
  RUN_TREE_HOLDOUT=True
  RUN_NEURAL_HOLDOUT=False
  RUN_TRANSFORMER_HOLDOUT=False
  RESET_FINAL_HOLDOUT_RUNTIME=False


**Findings.** The final-holdout runtime contains **29 restart units**: 8 Tree, 9 Neural, and 12 Transformer.

The full champion system represents **50 planned training calls**, while the recovered Tree checkpoints reduce the remaining execution to **45 calls**: 3 Tree fits, 18 Neural selector/final fits, and 24 Transformer selector/final fits.

All family execution controls and the destructive reset control remain disabled, so no missing model fit is launched automatically.

### Section Summary

Section 5 freezes the unseen-holdout evaluation and execution contract before forecasting begins. All **15 validation-selected champion jobs** have legal benchmark-supported holdout rows, all champions are bound to their source-native runner contracts, and all **17 required source artifacts** pass their structural checks. The frozen champion system expands into **29 restart units and 50 planned training calls**, with durable recovery semantics fixed before any unseen-period accuracy is evaluated.

## 4.6.1.6 Execute and Recover the Frozen Champion Forecasts

The frozen holdout contract now becomes actual unseen-period forecasts.

This section preserves each winning family's source-native training lifecycle while giving all champions one durable forecast-output and recovery contract. Tree champions use fresh global-panel LightGBM fits, while Neural and Transformer champions retain their frozen monthly selector-then-final sequence-model lifecycles.

Execution remains restartable: compatible completed work is recovered automatically, while expensive missing work runs only through explicit family-specific controls.

### 4.6.1.6.1 Execute and Recover Tree Champions

The eight Tree-owned forecasting jobs use the frozen global-panel LightGBM contract. This subsection reconstructs that selected model definition, validates all Tree restart units without fitting, and then recovers or executes only the durable work required for those eight champions.

**4.6.1.6.1A — Give every champion family the same durable output contract.**

Tree, Neural, and Transformer train differently, but every completed restart unit must produce the same two durable artifacts:

1. a row-level forecast file containing physical forecast identity, observed mobility, frozen benchmark, raw prediction, final nonnegative prediction, and availability flags; and
2. a compact audit file confirming that the restart unit completed under the frozen final-holdout runtime contract.

These helpers standardize persistence and output semantics only. They do not determine how a family trains or constructs its inputs.

The audit is published only after the prediction artifact is complete, so a restart unit counts as recoverable only when both artifacts satisfy the runtime contract.

In [44]:
# ---------------------------------------------------------------------
# 4.6.1.6.1A — Shared final-holdout execution and recovery helpers
# ---------------------------------------------------------------------

from time import perf_counter

HOLDOUT_EXECUTION_AUDIT_VERSION = "holdout_unit_audit_v1"


def apply_frozen_output_rule(prediction_raw):
    """Apply the already-frozen physical zero floor while preserving raw forecasts."""
    values = np.asarray(prediction_raw, dtype=float)
    return np.maximum(0.0, values)


def build_holdout_prediction_frame(
    manifest_row,
    source_frame,
    prediction_raw,
    actual_column="target_observed_value",
    benchmark_column=None,
):
    """
    Standardize one family-specific forecast into the shared holdout schema.

    The source runner remains responsible for fitting and prediction. This
    helper only attaches the common physical-row identity and frozen output
    semantics required for cross-family evaluation.
    """
    required_source_columns = [
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
        actual_column,
    ]

    if benchmark_column is None:
        raise ValueError("A frozen benchmark column must be supplied.")

    require_columns(
        source_frame,
        [*required_source_columns, benchmark_column],
        f"{manifest_row['champion_family']} holdout scoring rows",
    )

    raw = np.asarray(prediction_raw, dtype=float)

    if len(raw) != len(source_frame):
        raise ValueError(
            f"{manifest_row['runtime_unit_id']}: prediction length "
            f"{len(raw):,} does not match {len(source_frame):,} scoring rows."
        )

    actual = pd.to_numeric(source_frame[actual_column], errors="raise").to_numpy(dtype=float)
    benchmark = pd.to_numeric(
        source_frame[benchmark_column], errors="coerce"
    ).to_numpy(dtype=float)

    if not np.isfinite(actual).all():
        raise ValueError(
            f"{manifest_row['runtime_unit_id']}: non-finite actual values "
            "entered the final-holdout scoring surface."
        )

    prediction_df = source_frame[
        [
            "forecast_sequence_id",
            "taxi_zone_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
            "target_date",
        ]
    ].copy()

    prediction_df["runtime_unit_id"] = manifest_row["runtime_unit_id"]
    prediction_df["holdout_runtime_contract_id"] = HOLDOUT_RUNTIME_CONTRACT_ID
    prediction_df["champion_family"] = manifest_row["champion_family"]
    prediction_df["modeling_track"] = manifest_row["modeling_track"]
    prediction_df["metric"] = manifest_row["metric"]
    prediction_df["horizon"] = int(manifest_row["horizon"])

    prediction_df["actual"] = actual
    prediction_df["benchmark_prediction"] = benchmark
    prediction_df["prediction_raw"] = raw
    prediction_df["prediction_final"] = apply_frozen_output_rule(raw)

    # Keep availability explicit because the final system uses the advanced
    # prediction when available and the frozen benchmark as its fallback.
    prediction_df["model_prediction_available"] = (
        np.isfinite(prediction_df["prediction_raw"].to_numpy(dtype=float))
        & np.isfinite(prediction_df["prediction_final"].to_numpy(dtype=float))
    )
    prediction_df["benchmark_prediction_available"] = np.isfinite(benchmark)

    return prediction_df[HOLDOUT_UNIT_PREDICTION_COLUMNS]


def build_holdout_unit_audit(
    manifest_row,
    prediction_df,
    started_at_seconds,
    **family_metadata,
):
    """Create the success record written only after the prediction artifact is durable."""
    elapsed_seconds = perf_counter() - started_at_seconds

    audit_record = {
        "runtime_unit_id": manifest_row["runtime_unit_id"],
        "holdout_runtime_contract_id": HOLDOUT_RUNTIME_CONTRACT_ID,
        "champion_family": manifest_row["champion_family"],
        "unit_complete": True,
        "execution_audit_version": HOLDOUT_EXECUTION_AUDIT_VERSION,
        "modeling_track": manifest_row["modeling_track"],
        "metric": manifest_row["metric"],
        "selected_horizons": manifest_row["selected_horizons"],
        "configuration_id": manifest_row["configuration_id"],
        "runner_contract_id": manifest_row["runner_contract_id"],
        "prediction_rows": int(len(prediction_df)),
        "model_available_rows": int(
            prediction_df["model_prediction_available"].fillna(False).sum()
        ),
        "benchmark_available_rows": int(
            prediction_df["benchmark_prediction_available"].fillna(False).sum()
        ),
        "runtime_seconds": float(elapsed_seconds),
        **family_metadata,
    }

    return pd.DataFrame([audit_record])

def validate_holdout_unit_before_commit(manifest_row, prediction_df, audit_df):
    """
    Validate one completed runtime unit while its rows are already in memory.

    This is the cheapest place to perform row-level integrity checks: fitting
    just produced the DataFrames, so there is no reason to write them to disk
    and repeatedly rescan the Parquet files during every later notebook rerun.
    """
    missing_prediction = sorted(
        set(HOLDOUT_UNIT_PREDICTION_COLUMNS) - set(prediction_df.columns)
    )
    missing_audit = sorted(
        set(HOLDOUT_UNIT_AUDIT_COLUMNS) - set(audit_df.columns)
    )

    if missing_prediction:
        raise RuntimeError(
            "Holdout prediction artifact is missing required columns: "
            f"{missing_prediction}"
        )

    if missing_audit:
        raise RuntimeError(
            "Holdout audit artifact is missing required columns: "
            f"{missing_audit}"
        )

    if prediction_df.empty:
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: prediction artifact is empty."
        )

    if audit_df.empty:
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: audit artifact is empty."
        )

    # Every row in one checkpoint must belong to the logical unit that owns it.
    identity_checks = {
        "runtime_unit_id": str(manifest_row["runtime_unit_id"]),
        "holdout_runtime_contract_id": HOLDOUT_RUNTIME_CONTRACT_ID,
        "champion_family": str(manifest_row["champion_family"]),
        "modeling_track": str(manifest_row["modeling_track"]),
        "metric": str(manifest_row["metric"]),
    }

    for column, expected in identity_checks.items():
        if not prediction_df[column].astype(str).eq(expected).all():
            raise RuntimeError(
                f"{manifest_row['runtime_unit_id']}: prediction rows do not "
                f"match expected {column}={expected!r}."
            )

    expected_horizons = {
        int(value)
        for value in str(manifest_row["selected_horizons"]).split(",")
        if str(value).strip()
    }
    observed_horizons = {
        int(value)
        for value in prediction_df["horizon"].dropna().unique()
    }

    if observed_horizons != expected_horizons:
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: expected horizons "
            f"{sorted(expected_horizons)}, found {sorted(observed_horizons)}."
        )

    target_dates = pd.to_datetime(
        prediction_df["target_date"],
        errors="raise",
    ).dt.normalize()

    if (
        target_dates.min() < FINAL_HOLDOUT_START_DATE
        or target_dates.max() > FINAL_HOLDOUT_END_DATE
    ):
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: prediction dates fall outside "
            "the frozen final-holdout window."
        )

    actual = pd.to_numeric(
        prediction_df["actual"],
        errors="coerce",
    ).to_numpy(dtype=float)

    prediction_raw = pd.to_numeric(
        prediction_df["prediction_raw"],
        errors="coerce",
    ).to_numpy(dtype=float)

    prediction_final = pd.to_numeric(
        prediction_df["prediction_final"],
        errors="coerce",
    ).to_numpy(dtype=float)

    benchmark = pd.to_numeric(
        prediction_df["benchmark_prediction"],
        errors="coerce",
    ).to_numpy(dtype=float)

    if not np.isfinite(actual).all():
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: actual contains non-finite values."
        )

    expected_model_available = (
        np.isfinite(prediction_raw)
        & np.isfinite(prediction_final)
    )
    observed_model_available = (
        prediction_df["model_prediction_available"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    if not np.array_equal(
        expected_model_available,
        observed_model_available,
    ):
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: model-availability flags "
            "do not match the stored predictions."
        )

    expected_benchmark_available = np.isfinite(benchmark)
    observed_benchmark_available = (
        prediction_df["benchmark_prediction_available"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    if not np.array_equal(
        expected_benchmark_available,
        observed_benchmark_available,
    ):
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: benchmark-availability flags "
            "do not match the stored benchmark predictions."
        )

    if expected_model_available.any():
        floor_error = np.abs(
            prediction_final[expected_model_available]
            - np.maximum(0.0, prediction_raw[expected_model_available])
        )

        if np.any(floor_error > OUTPUT_RULE_TOLERANCE):
            raise RuntimeError(
                f"{manifest_row['runtime_unit_id']}: prediction_final violates "
                "the frozen nonnegative output rule."
            )

    if prediction_df.duplicated(COMMON_FORECAST_KEY_COLUMNS).any():
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: prediction artifact contains "
            "duplicate physical forecast keys."
        )

    # The audit is the success marker written last, so it must itself belong to
    # the same current-contract logical unit before either artifact is committed.
    audit_ok = (
        audit_df["runtime_unit_id"]
        .astype(str)
        .eq(str(manifest_row["runtime_unit_id"]))
        .all()
        and audit_df["holdout_runtime_contract_id"]
        .astype(str)
        .eq(HOLDOUT_RUNTIME_CONTRACT_ID)
        .all()
        and audit_df["champion_family"]
        .astype(str)
        .eq(str(manifest_row["champion_family"]))
        .all()
        and audit_df["unit_complete"]
        .fillna(False)
        .astype(bool)
        .all()
    )

    if not audit_ok:
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: audit artifact does not satisfy "
            "the current completion contract."
        )

def commit_holdout_unit(manifest_row, prediction_df, audit_df):
    """
    Validate and durably publish one completed runtime unit.

    The larger prediction artifact is written first. The small audit artifact
    is written last and therefore acts as the durable success marker.
    """
    validate_holdout_unit_before_commit(
        manifest_row,
        prediction_df,
        audit_df,
    )

    prediction_path = Path(manifest_row["prediction_path"])
    audit_path = Path(manifest_row["audit_path"])

    atomic_write_parquet(prediction_df, prediction_path)
    atomic_write_parquet(audit_df, audit_path)


def refresh_holdout_checkpoint_status():
    """Re-audit every restart unit without launching any expensive model work."""
    manifest_df = pd.read_parquet(HOLDOUT_CHECKPOINT_MANIFEST_PATH)
    statuses = []

    for row in manifest_df.to_dict("records"):
        prediction_exists = Path(row["prediction_path"]).exists()
        audit_exists = Path(row["audit_path"]).exists()

        if not prediction_exists and not audit_exists:
            status = "pending"
        elif prediction_exists != audit_exists:
            status = "partial_invalid"
        else:
            status = "recovered" if completed_holdout_unit_is_valid(row) else "invalid"

        statuses.append(status)

    manifest_df["artifact_status"] = statuses

    invalid_mask = manifest_df["artifact_status"].isin(["partial_invalid", "invalid"])

    if invalid_mask.any():
        failed_df = manifest_df.loc[
            invalid_mask,
            [
                "Family",
                "metric",
                "selected_horizons",
                "refit_month",
                "artifact_status",
                "runtime_unit_id",
            ],
        ]

        raise RuntimeError(
            "Final-holdout recovery found an incomplete or incompatible "
            "checkpoint pair. Review it before using the destructive reset:\n"
            + failed_df.to_string(index=False)
        )

    write_small_contract(manifest_df, HOLDOUT_CHECKPOINT_MANIFEST_PATH)
    return manifest_df


def summarize_holdout_runtime(manifest_df):
    """Return one compact family-level view of completed and remaining work."""
    status_df = (
        manifest_df.assign(
            Recovered=manifest_df["artifact_status"].eq("recovered").astype(int),
            Pending=manifest_df["artifact_status"].eq("pending").astype(int),
        )
        .groupby(["champion_family", "Family"], as_index=False, observed=True)
        .agg(
            **{
                "Restart units": ("runtime_unit_id", "size"),
                "Recovered": ("Recovered", "sum"),
                "Pending": ("Pending", "sum"),
                "Planned training calls": ("planned_fit_calls", "sum"),
            }
        )
    )

    family_order = {family: i for i, family in enumerate(FAMILY_ORDER)}

    return (
        status_df.assign(
            _family_order=lambda frame: frame["champion_family"].map(family_order)
        )
        .sort_values("_family_order")
        .drop(columns="_family_order")
        .reset_index(drop=True)
    )


holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()
holdout_runtime_status_df = summarize_holdout_runtime(holdout_checkpoint_manifest_df)

display(
    holdout_runtime_status_df[
        [
            "Family",
            "Restart units",
            "Recovered",
            "Pending",
            "Planned training calls",
        ]
    ]
)

print(
    "PASS — shared final-holdout recovery helpers are ready.\n"
    "No model fitting was performed.",
    flush=True,
)


,Family,Restart units,Recovered,Pending,Planned training calls
0,Tree,8,8,0,8
1,Neural,9,9,0,18
2,Transformer,12,12,0,24


PASS — shared final-holdout recovery helpers are ready.
No model fitting was performed.


**Findings.** This block adds only the shared **finish-and-recovery layer** for final-holdout execution. It does not fit a forecasting model and does not alter the frozen 15-job champion roster.

A completed unit must still satisfy the `final_holdout_runtime_v1` contract established in 5.3B. Predictions are written first, the completion audit is committed last, and a rerun independently revalidates the pair before treating it as recovered. Missing checkpoints remain pending rather than triggering computation automatically.

The shared layer also preserves the project's forecasting output rule exactly: `prediction_raw` remains auditable, while `prediction_final = max(0, prediction_raw)` is derived separately. Model and benchmark availability remain explicit fields rather than being hidden by row filtering.


**4.6.1.6.1B — Reconstruct the selected LightGBM for each Tree-owned job.**

Tree owns **8 of the 15 final-holdout jobs**. Each uses one fresh global-panel LightGBM fit at the beginning of the final holdout.

For each selected Target × Horizon, the runner:

1. loads the exact ordered predictor roster exported by 4.4.1;
2. identifies the first legal final-holdout forecast origin;
3. allows only training outcomes observable by that origin;
4. fits the frozen Full-History LightGBM definition across the eligible Taxi Zone panel;
5. scores all legal holdout rows for that job; and
6. preserves both the unconstrained prediction and the frozen nonnegative final prediction.

Taxi Zone remains categorical. Its levels are learned from the legal training history and applied unchanged to the holdout rows.

In [45]:
# ---------------------------------------------------------------------
# 4.6.1.6.1B — Tree / LightGBM final-holdout runner
# ---------------------------------------------------------------------

from lightgbm import LGBMRegressor

TREE_REGISTRY_PATH = INPUT_441_FINAL_DIR / "lgbm_selected_model_registry.parquet"
TREE_PREDICTOR_REGISTRY_PATH = INPUT_441_FINAL_DIR / "lgbm_predictor_registry.parquet"

TREE_PARAM_COLUMNS = [
    "param_objective", "param_n_estimators", "param_learning_rate",
    "param_num_leaves", "param_min_child_samples", "param_subsample",
    "param_subsample_freq", "param_colsample_bytree", "param_reg_lambda",
    "param_random_state", "param_n_jobs", "param_verbosity",
]

tree_registry_df = pd.read_parquet(TREE_REGISTRY_PATH)
tree_predictor_registry_df = pd.read_parquet(TREE_PREDICTOR_REGISTRY_PATH)

# These are boundary assertions against contracts already frozen in 4.4.1.
# They are not schema-discovery logic: unexpected drift should fail loudly.
require_columns(
    tree_registry_df,
    [
        "modeling_track", "metric", "horizon", "production_config_id",
        "capacity_profile", "target_representation", "refit_cadence",
        "benchmark_id", "benchmark_label", *TREE_PARAM_COLUMNS,
    ],
    "selected LightGBM model registry",
)

require_columns(
    tree_predictor_registry_df,
    ["metric", "predictor_order", "predictor_name", "is_categorical"],
    "LightGBM predictor registry",
)

if not tree_registry_df["refit_cadence"].eq("validation_start_only").all():
    raise RuntimeError(
        "The selected LightGBM handoff no longer uses the frozen "
        "validation-start-only refit contract."
    )

if not tree_registry_df["target_representation"].eq("raw").all():
    raise RuntimeError(
        "The selected LightGBM handoff no longer uses raw target representation."
    )


def resolve_tree_predictor_contract(metric):
    """Return the exact ordered inputs and categorical roles exported by 4.4.1."""
    contract_df = (
        tree_predictor_registry_df.loc[
            tree_predictor_registry_df["metric"].eq(metric)
        ]
        .sort_values("predictor_order")
        .reset_index(drop=True)
    )

    if contract_df.empty:
        raise KeyError(f"No LightGBM predictor contract found for {metric}.")

    if contract_df["predictor_order"].duplicated().any():
        raise RuntimeError(f"{metric}: duplicate LightGBM predictor order.")

    predictors = contract_df["predictor_name"].astype(str).tolist()
    categorical = (
        contract_df.loc[contract_df["is_categorical"].astype(bool), "predictor_name"]
        .astype(str)
        .tolist()
    )

    return predictors, categorical


def resolve_tree_model_params(registry_row):
    """Restore the frozen LightGBM constructor from the exported param_* fields."""
    params = {}

    for column in TREE_PARAM_COLUMNS:
        value = registry_row[column]

        if pd.isna(value):
            raise RuntimeError(
                f"{registry_row['metric']} · h={int(registry_row['horizon'])}: "
                f"missing frozen LightGBM parameter {column}."
            )

        if isinstance(value, np.generic):
            value = value.item()

        params[column.removeprefix("param_")] = value

    return params


def prepare_holdout_lightgbm_frames(X_train, X_score, categorical_names):
    """
    Apply the categorical treatment used by the source LightGBM runner.

    Category levels come only from legal training history. Values first seen
    during the holdout map to an explicit unseen level instead of influencing
    how the training data are encoded.
    """
    train = X_train.copy()
    score = X_score.copy()

    for column in categorical_names:
        train_values = train[column].astype("string").fillna("__MISSING__")
        score_values = score[column].astype("string").fillna("__MISSING__")

        categories = train_values.drop_duplicates().tolist()
        if "__UNSEEN__" not in categories:
            categories.append("__UNSEEN__")

        score_values = score_values.where(
            score_values.isin(categories),
            "__UNSEEN__",
        )

        train[column] = pd.Categorical(train_values, categories=categories)
        score[column] = pd.Categorical(score_values, categories=categories)

    return train, score


def selected_tree_registry_row(metric, horizon):
    """Resolve the one frozen LightGBM definition for a Target × Horizon job."""
    rows = tree_registry_df.loc[
        tree_registry_df["metric"].eq(metric)
        & tree_registry_df["horizon"].astype(int).eq(int(horizon))
    ]

    if len(rows) != 1:
        raise RuntimeError(
            f"{metric} · h={horizon}: expected one selected LightGBM "
            f"registry row; found {len(rows)}."
        )

    return rows.iloc[0]


def load_tree_holdout_surface(manifest_row):
    """
    Load legal pre-holdout training history and the final-holdout rows.

    The one Tree fit uses outcomes observable by the first holdout forecast
    origin, matching 4.4.1's validation-start-only chronology. The frozen
    weekly benchmark is joined separately because it belongs to the 4.2.4
    baseline artifact rather than the forecasting matrix.
    """
    track = str(manifest_row["modeling_track"])
    metric = str(manifest_row["metric"])
    horizon = int(manifest_row["horizon"])

    predictors, categorical_names = resolve_tree_predictor_contract(metric)
    matrix_path = FORECASTING_MATRIX_PATHS[track]
    baseline_path = BASELINE_PREDICTION_PATHS[track]

    matrix_columns = list(dict.fromkeys([
        "modeling_track", "stage", "forecast_sequence_id", "taxi_zone_id",
        "metric", "origin_observation_sequence_id",
        "target_observation_sequence_id", "horizon", "target_date",
        "target_observed_value", "target_label_available",
        "native_track_eligible", *predictors,
    ]))

    require_holdout_source_schema(
        matrix_path,
        matrix_columns,
        f"{TRACK_LABELS[track]} Tree forecasting matrix",
    )

    job_df = pd.read_parquet(
        matrix_path,
        columns=matrix_columns,
        filters=[("metric", "==", metric), ("horizon", "==", horizon)],
    )

    if not job_df["modeling_track"].eq(track).all():
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: matrix returned another track."
        )

    job_df["target_date"] = pd.to_datetime(job_df["target_date"]).dt.normalize()

    for column in ["origin_observation_sequence_id", "target_observation_sequence_id"]:
        job_df[column] = pd.to_numeric(job_df[column], errors="raise").astype("int64")

    target_values = pd.to_numeric(
        job_df["target_observed_value"], errors="coerce"
    ).to_numpy(dtype=float)

    job_df["_label_available"] = (
        job_df["target_label_available"].fillna(False).astype(bool)
        & np.isfinite(target_values)
    )

    score_df = job_df.loc[
        job_df["stage"].eq("final_holdout")
        & job_df["native_track_eligible"].fillna(False).astype(bool)
        & job_df["_label_available"]
    ].copy()

    if score_df.empty:
        raise ValueError(f"{track} · {metric} · h={horizon}: no legal holdout rows.")

    first_holdout_origin = int(
        score_df["origin_observation_sequence_id"].min()
    )

    # Later holdout outcomes are never allowed into this single production fit.
    train_df = job_df.loc[
        job_df["_label_available"]
        & job_df["stage"].ne("final_holdout")
        & job_df["target_observation_sequence_id"].le(first_holdout_origin)
    ].copy()

    if train_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: no legal training rows."
        )

    if train_df["target_observation_sequence_id"].max() > first_holdout_origin:
        raise RuntimeError("Tree training chronology guard failed.")


    # -----------------------------------------------------------------
    # Attach the frozen Previous week · same Daypart benchmark
    # -----------------------------------------------------------------

    baseline_columns = [
        "stage", "forecast_sequence_id", "origin_observation_sequence_id",
        "target_observation_sequence_id", "horizon", "metric", "target_date",
        "target_label_available", HOLDOUT_BASELINE_COLUMN,
    ]

    require_holdout_source_schema(
        baseline_path,
        baseline_columns,
        f"{TRACK_LABELS[track]} frozen benchmark predictions",
    )

    baseline_df = pd.read_parquet(
        baseline_path,
        columns=baseline_columns,
        filters=[
            ("stage", "==", "final_holdout"),
            ("metric", "==", metric),
            ("horizon", "==", horizon),
        ],
    )

    baseline_df["target_date"] = pd.to_datetime(
        baseline_df["target_date"]
    ).dt.normalize()

    baseline_df = baseline_df.loc[
        baseline_df["target_label_available"].fillna(False).astype(bool),
        [
            "forecast_sequence_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
            "horizon",
            "metric",
            "target_date",
            HOLDOUT_BASELINE_COLUMN,
        ],
    ].rename(columns={HOLDOUT_BASELINE_COLUMN: "benchmark_prediction"})

    join_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
        "metric",
        "target_date",
    ]

    # Keep all otherwise-legal model rows. Benchmark gaps stay explicit and
    # will be represented by benchmark_prediction_available downstream.
    score_df = score_df.merge(
        baseline_df,
        on=join_keys,
        how="left",
        validate="one_to_one",
    )

    return {
        "train_df": train_df,
        "score_df": score_df,
        "predictors": predictors,
        "categorical_names": categorical_names,
        "first_holdout_origin": first_holdout_origin,
    }


def execute_tree_holdout_unit(manifest_row):
    """Fit, score, and checkpoint one frozen LightGBM champion job."""
    started = perf_counter()
    metric = str(manifest_row["metric"])
    horizon = int(manifest_row["horizon"])

    registry_row = selected_tree_registry_row(metric, horizon)

    if registry_row["modeling_track"] != manifest_row["modeling_track"]:
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: Tree modeling track no longer "
            "matches the frozen champion binding."
        )

    source_configuration_id = (
        f"{registry_row['production_config_id']} · "
        f"{registry_row['target_representation']}"
    )

    if source_configuration_id != manifest_row["configuration_id"]:
        raise RuntimeError(
            f"{manifest_row['runtime_unit_id']}: Tree configuration no longer "
            "matches the frozen champion binding."
        )

    payload = load_tree_holdout_surface(manifest_row)
    train_df = payload["train_df"]
    score_df = payload["score_df"]
    predictors = payload["predictors"]
    categorical_names = payload["categorical_names"]

    X_train, X_score = prepare_holdout_lightgbm_frames(
        train_df[predictors],
        score_df[predictors],
        categorical_names,
    )

    y_train = pd.to_numeric(
        train_df["target_observed_value"], errors="raise"
    ).astype(float)

    model = LGBMRegressor(**resolve_tree_model_params(registry_row))

    fit_started = perf_counter()
    model.fit(X_train, y_train, categorical_feature=categorical_names)
    fit_seconds = perf_counter() - fit_started

    predict_started = perf_counter()
    prediction_raw = np.asarray(model.predict(X_score), dtype=float)
    predict_seconds = perf_counter() - predict_started

    if not np.isfinite(prediction_raw).all():
        raise ValueError(
            f"{metric} · h={horizon}: LightGBM produced non-finite "
            "final-holdout predictions."
        )

    prediction_df = build_holdout_prediction_frame(
        manifest_row,
        score_df,
        prediction_raw,
        actual_column="target_observed_value",
        benchmark_column="benchmark_prediction",
    )

    audit_df = build_holdout_unit_audit(
        manifest_row,
        prediction_df,
        started,
        fit_calls=1,
        fit_seconds=float(fit_seconds),
        predict_seconds=float(predict_seconds),
        train_rows=int(len(train_df)),
        score_rows=int(len(score_df)),
        first_holdout_origin=int(payload["first_holdout_origin"]),
        refit_cadence=str(registry_row["refit_cadence"]),
        production_config_id=str(registry_row["production_config_id"]),
    )

    commit_holdout_unit(manifest_row, prediction_df, audit_df)

    return {
        "runtime_unit_id": manifest_row["runtime_unit_id"],
        "metric": metric,
        "horizon": horizon,
        "status": "success",
        "train_rows": int(len(train_df)),
        "score_rows": int(len(score_df)),
        "runtime_seconds": float(perf_counter() - started),
    }

**4.6.1.6.1C — Preflight all 8 Tree restart units without fitting.**

Before enabling the LightGBM fit, verify that every Tree champion can still be reconstructed from the frozen 4.4.1 contracts.

For each job, the preflight confirms the selected model definition, ordered predictors, categorical inputs, legal training population, and final-holdout scoring population.

Checkpoint status is shown alongside structural readiness so already-complete units remain distinguishable from work that still requires fitting.

In [46]:
# ---------------------------------------------------------------------
# 4.6.1.6.1C — Preflight all Tree restart units without fitting
# ---------------------------------------------------------------------

tree_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("tree")
    ]
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(tree_manifest_df) != 8:
    raise RuntimeError(
        f"Expected 8 frozen Tree restart units; found {len(tree_manifest_df)}."
    )

tree_preflight_records = []

for manifest_row in tree_manifest_df.to_dict("records"):
    registry_row = selected_tree_registry_row(
        manifest_row["metric"],
        int(manifest_row["horizon"]),
    )

    params = resolve_tree_model_params(registry_row)
    payload = load_tree_holdout_surface(manifest_row)

    tree_preflight_records.append({
        "Target": TARGET_LABELS.get(
            manifest_row["metric"],
            manifest_row["metric"],
        ),
        "Horizon": int(manifest_row["horizon"]),
        "Track": TRACK_LABELS.get(
            manifest_row["modeling_track"],
            manifest_row["modeling_track"],
        ),
        "Predictors": len(payload["predictors"]),
        "Categorical": len(payload["categorical_names"]),
        "Train rows": len(payload["train_df"]),
        "Holdout rows": len(payload["score_df"]),
        "Trees": int(params["n_estimators"]),
        "Checkpoint": manifest_row["artifact_status"],
        "Ready": True,
    })

tree_preflight_df = pd.DataFrame(tree_preflight_records)

if len(tree_preflight_df) != 8 or not tree_preflight_df["Ready"].all():
    raise RuntimeError(
        "At least one frozen Tree champion failed structural preflight."
    )

display(tree_preflight_df)

print(
    "PASS — all 8 Tree champions satisfy the frozen LightGBM "
    "reconstruction contract.\n"
    f"RUN_TREE_HOLDOUT={RUN_TREE_HOLDOUT}; no LightGBM fit was run.",
    flush=True,
)

,Target,Horizon,Track,Predictors,Categorical,Train rows,Holdout rows,Trees,Checkpoint,Ready
0,FHVHV trips,1,Full-History Regime-Aware,85,9,1415790,110693,400,recovered,True
1,FHVHV trips,2,Full-History Regime-Aware,85,9,1415277,110693,400,recovered,True
2,FHVHV trips,5,Full-History Regime-Aware,85,9,1413732,110693,400,recovered,True
3,Subway ridership,1,Full-History Regime-Aware,85,9,844275,64511,400,recovered,True
4,Subway ridership,5,Full-History Regime-Aware,85,9,843043,64511,400,recovered,True
5,Taxi trips,1,Full-History Regime-Aware,85,9,1415790,110693,400,recovered,True
6,Taxi trips,2,Full-History Regime-Aware,85,9,1415277,110693,400,recovered,True
7,Taxi trips,5,Full-History Regime-Aware,85,9,1413732,110693,400,recovered,True


PASS — all 8 Tree champions satisfy the frozen LightGBM reconstruction contract.
RUN_TREE_HOLDOUT=True; no LightGBM fit was run.


**Findings.** All **8 Tree champion jobs pass structural preflight** under the frozen Full-History LightGBM contract. Each uses **85 predictors**, including **9 categorical features**, and the selected **400-tree** model definition.

The demand jobs contain approximately **1.41 million legal training rows** and **110,693 final-holdout rows** per Target × Horizon. Subway contains approximately **843,000–844,000 training rows** and **64,511 holdout rows**.

Checkpoint status separates the workload into **5 recovered units and 3 pending units**. The pending jobs are **Subway ridership h=1** and **Taxi trips h=1 and h=2**. All eight jobs are structurally ready for final-holdout forecasting.

**4.6.1.6.1D — Recover completed Tree forecasts and optionally execute the pending jobs.**

The Tree runtime reads checkpoint status from durable artifacts before doing any work.

Completed jobs require no new fit. Pending jobs are fitted only when `RUN_TREE_HOLDOUT = True`; otherwise they remain explicitly pending.

This makes recovery the default behavior and expensive execution an explicit choice.

In [47]:
# ---------------------------------------------------------------------
# 4.6.1.6.1D — Recover or execute frozen Tree holdout champions
# ---------------------------------------------------------------------

holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

tree_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("tree")
    ]
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

tree_total = len(tree_manifest_df)

if tree_total != 8:
    raise RuntimeError(
        f"Expected 8 frozen Tree units; found {tree_total}."
    )

tree_completed = int(
    tree_manifest_df["artifact_status"].eq("recovered").sum()
)
tree_pending = int(
    tree_manifest_df["artifact_status"].eq("pending").sum()
)

print(
    "Tree final-holdout restart state\n"
    f"  Recovered: {tree_completed}/{tree_total}\n"
    f"  Pending:   {tree_pending}/{tree_total}\n"
    f"  RUN_TREE_HOLDOUT={RUN_TREE_HOLDOUT}",
    flush=True,
)

if tree_pending and not RUN_TREE_HOLDOUT:
    print(
        "\nExecution is disabled; pending Tree units remain unchanged.",
        flush=True,
    )

elif tree_pending:
    pending_tree_df = (
        tree_manifest_df.loc[
            tree_manifest_df["artifact_status"].eq("pending")
        ]
        .reset_index(drop=True)
    )

    for position, manifest_row in enumerate(
        pending_tree_df.to_dict("records"),
        start=1,
    ):
        print(
            f"\nTree holdout [{position}/{len(pending_tree_df)}] "
            f"{TARGET_LABELS.get(manifest_row['metric'], manifest_row['metric'])} "
            f"· h={int(manifest_row['horizon'])}",
            flush=True,
        )

        result = execute_tree_holdout_unit(manifest_row)

        print(
            f"  complete · train={result['train_rows']:,} "
            f"· score={result['score_rows']:,} "
            f"· runtime={result['runtime_seconds']:.1f}s",
            flush=True,
        )


# Reconcile status from the durable checkpoint pairs rather than from
# in-memory completion messages.
holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

tree_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("tree")
    ]
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

tree_status_df = (
    tree_manifest_df.groupby(
        "artifact_status",
        as_index=False,
        observed=True,
    )
    .agg(Units=("runtime_unit_id", "size"))
    .rename(columns={"artifact_status": "Status"})
)

display(tree_status_df)

if RUN_TREE_HOLDOUT and not tree_manifest_df[
    "artifact_status"
].eq("recovered").all():
    raise RuntimeError(
        "Tree execution was enabled, but at least one restart unit "
        "did not produce a valid checkpoint pair."
    )

print(
    "Tree checkpoint status reconciled from durable artifacts.",
    flush=True,
)

Tree final-holdout restart state
  Recovered: 8/8
  Pending:   0/8
  RUN_TREE_HOLDOUT=True


,Status,Units
0,recovered,8


Tree checkpoint status reconciled from durable artifacts.


**Findings.** All **8 Tree restart units have valid durable checkpoints**, with **0 units pending**.

The three executed Tree champions complete successfully: **Subway ridership h=1** in **15.0 seconds**, **Taxi trips h=1** in **21.6 seconds**, and **Taxi trips h=2** in **21.0 seconds**. Their final-holdout scoring populations contain **64,511 Subway rows** and **110,693 Taxi-trip rows** per horizon.

The complete Tree branch is therefore recoverable from its prediction and audit artifacts and requires no additional LightGBM fitting.

### 4.6.1.6.2 Prepare the Frozen Sequence-Model Runtime

Neural and Transformer champions share a monthly sequence-model execution pattern but retain distinct frozen architectures. Before either family executes, this subsection validates the monthly chronology, binds the saved sequence/context contracts, reconstructs leakage-safe preprocessing, restores each model's frozen training lifecycle, and defines one legal monthly restart unit.

**4.6.1.6.2A — Preflight the 21 monthly Neural and Transformer restart units.**

Neural and Transformer use the same monthly execution pattern: each restart unit begins at a legal forecast origin, selects a training duration using historical inner validation, fits a fresh final model on all legally available history, and then scores the forecasts assigned to that month.

This preflight verifies the chronology without training either network.

For every monthly restart unit it confirms:

- a nonempty legal expanding training population;
- an earlier selector-training population;
- a nonempty 28-day historical inner-validation tail;
- a nonempty final-holdout scoring population;
- enough stored mobility history for the family's sequence lookback; and
- agreement between the monthly runtime unit and the frozen champion schedule.

Neural requires 35 Dayparts of recent mobility history. Transformer requires the frozen 70-Daypart patched history.

In [48]:
# ---------------------------------------------------------------------
# 4.6.1.6.2A — Preflight Neural and Transformer monthly chronology
# ---------------------------------------------------------------------

SEQUENCE_LOOKBACK_STEPS = {
    "neural": 35,
    "transformer": 70,
}

SEQUENCE_SELECTOR_MAX_EPOCHS = {
    "neural": 24,
    "transformer": 30,
}

SEQUENCE_INNER_VALIDATION_DAYS = 28

HISTORY_STATE_PATH = (
    INPUT_451_FINAL_DIR / "neural_mobility_history_state.npz"
)


# ---------------------------------------------------------------------
# 1) Load only the compact identities needed to validate history coverage
# ---------------------------------------------------------------------
# The dense values and masks are intentionally not loaded here. Preflight
# needs only to know whether each forecast Zone and origin can address a
# complete legal history window in the frozen mobility-history state.

with np.load(HISTORY_STATE_PATH, allow_pickle=False) as history_state:
    history_zone_ids = np.asarray(
        history_state["zone_ids"],
        dtype=np.int64,
    )
    history_sequence_ids = np.asarray(
        history_state["sequence_ids"],
        dtype=np.int64,
    )

if len(np.unique(history_zone_ids)) != len(history_zone_ids):
    raise RuntimeError(
        "Mobility-history state contains duplicate Taxi Zone IDs."
    )

if len(np.unique(history_sequence_ids)) != len(history_sequence_ids):
    raise RuntimeError(
        "Mobility-history state contains duplicate sequence IDs."
    )

if np.any(np.diff(history_sequence_ids) <= 0):
    raise RuntimeError(
        "Mobility-history sequence IDs are not strictly chronological."
    )

history_zone_set = set(history_zone_ids.tolist())

history_sequence_position = {
    int(sequence_id): position
    for position, sequence_id in enumerate(history_sequence_ids)
}


PREFLIGHT_TRACK_START_CACHE = {}


def resolve_preflight_track_start(track, matrix_path):
    """
    Return the first legal history sequence ID for one modeling track.

    This boundary is invariant across monthly restart units, so resolve it once
    from a narrow Parquet slice and cache it for the rest of the preflight.
    """
    if track == "full_history_regime_aware":
        return int(history_sequence_ids[0])

    if track != "post_cp_primary":
        raise ValueError(f"Unsupported sequence modeling track: {track}")

    cache_key = (str(track), str(Path(matrix_path)))

    if cache_key in PREFLIGHT_TRACK_START_CACHE:
        return PREFLIGHT_TRACK_START_CACHE[cache_key]

    boundary_df = pd.read_parquet(
        matrix_path,
        columns=[
            "target_observation_sequence_id",
            "native_track_eligible",
        ],
        filters=[
            ("modeling_track", "==", str(track)),
            ("target_pre_post_cp", "==", "post_cp"),
        ],
    )

    eligible_mask = (
        boundary_df["native_track_eligible"]
        .fillna(False)
        .astype(bool)
    )

    eligible_ids = pd.to_numeric(
        boundary_df.loc[
            eligible_mask,
            "target_observation_sequence_id",
        ],
        errors="coerce",
    ).dropna()

    if eligible_ids.empty:
        raise RuntimeError(
            f"{track}: legal history boundary could not be resolved."
        )

    start_id = int(eligible_ids.min())
    PREFLIGHT_TRACK_START_CACHE[cache_key] = start_id

    return start_id


def preflight_sequence_holdout_unit(manifest_row):
    """
    Validate one Neural/Transformer Job × Month execution unit without fitting.

    Training labels may enter only after their target observation exists.
    The latest 28 historical days are reserved for selector validation, while
    scoring rows begin at this month's refit origin and stop before the next.
    """
    family = str(manifest_row["champion_family"])

    if family not in SEQUENCE_LOOKBACK_STEPS:
        raise ValueError(
            f"Unsupported sequence holdout family: {family}"
        )

    track = str(manifest_row["modeling_track"])
    metric = str(manifest_row["metric"])
    horizon = int(manifest_row["horizon"])

    refit_origin = int(
        manifest_row[
            "refit_origin_observation_sequence_id"
        ]
    )

    next_refit_origin = (
        None
        if pd.isna(
            manifest_row[
                "next_refit_origin_observation_sequence_id"
            ]
        )
        else int(
            manifest_row[
                "next_refit_origin_observation_sequence_id"
            ]
        )
    )

    lookback_steps = SEQUENCE_LOOKBACK_STEPS[family]
    matrix_path = FORECASTING_MATRIX_PATHS[track]

    required_columns = [
        "modeling_track",
        "stage",
        "forecast_sequence_id",
        "taxi_zone_id",
        "metric",
        "origin_observation_sequence_id",
        "origin_date",
        "target_observation_sequence_id",
        "horizon",
        "target_date",
        "target_observed_value",
        "target_label_available",
        "native_track_eligible",
        "target_pre_post_cp",
    ]

    require_holdout_source_schema(
        matrix_path,
        required_columns,
        f"{FAMILY_LABELS[family]} holdout forecasting matrix",
    )

    path_sql = sql_path(matrix_path)
    track_sql = track.replace("'", "''")
    metric_sql = metric.replace("'", "''")

    # The summary query reads from a normalized CTE where the origin column
    # is called `origin_id`. The identity query reads the source Parquet
    # directly and therefore uses the original physical column name.
    score_upper_bound_alias = (
        ""
        if next_refit_origin is None
        else f"AND origin_id < {next_refit_origin}"
    )

    score_upper_bound_source = (
        ""
        if next_refit_origin is None
        else (
            "AND CAST(origin_observation_sequence_id AS BIGINT) "
            f"< {next_refit_origin}"
        )
    )

    # Resolve the first legal history position for this modeling track.
    # A model-ready origin must occur only after the full lookback window
    # fits inside that legal track-specific history.
    track_start_id = resolve_preflight_track_start(
        track,
        matrix_path,
    )

    if track_start_id not in history_sequence_position:
        raise RuntimeError(
            f"{track}: legal history boundary is absent from "
            "the frozen mobility-history state."
        )

    track_start_position = history_sequence_position[
        track_start_id
    ]

    first_legal_position = (
        track_start_position
        + lookback_steps
        - 1
    )

    if first_legal_position >= len(history_sequence_ids):
        raise RuntimeError(
            f"{family} · {metric} · h={horizon}: "
            "lookback extends beyond the frozen history state."
        )

    minimum_history_origin = int(
        history_sequence_ids[first_legal_position]
    )

    # Both audits use the same legal history boundary. The first query checks
    # training, selector-validation, and monthly scoring populations. The
    # second verifies that every scoring Zone/origin exists in the dense state.
    with duckdb.connect() as connection:
        summary = connection.execute(f"""
            WITH eligible AS (
                SELECT
                    CAST(stage AS VARCHAR) AS stage,
                    CAST(taxi_zone_id AS BIGINT) AS taxi_zone_id,
                    CAST(origin_observation_sequence_id AS BIGINT)
                        AS origin_id,
                    CAST(target_observation_sequence_id AS BIGINT)
                        AS target_id,
                    CAST(target_date AS DATE) AS target_date

                FROM read_parquet('{path_sql}')

                WHERE CAST(modeling_track AS VARCHAR) = '{track_sql}'
                  AND CAST(metric AS VARCHAR) = '{metric_sql}'
                  AND CAST(horizon AS INTEGER) = {horizon}
                  AND COALESCE(
                        CAST(native_track_eligible AS BOOLEAN),
                        FALSE
                      )
                  AND COALESCE(
                        CAST(target_label_available AS BOOLEAN),
                        FALSE
                      )
                  AND COALESCE(
                        isfinite(
                            CAST(target_observed_value AS DOUBLE)
                        ),
                        FALSE
                      )
                  AND CAST(
                        origin_observation_sequence_id AS BIGINT
                      ) >= {minimum_history_origin}
            ),

            train AS (
                SELECT *
                FROM eligible
                WHERE target_id <= {refit_origin}
            ),

            train_bounds AS (
                SELECT
                    MAX(target_date) AS inner_end_date
                FROM train
            ),

            score AS (
                SELECT *
                FROM eligible
                WHERE stage = 'final_holdout'
                  AND origin_id >= {refit_origin}
                  {score_upper_bound_alias}
            )

            SELECT
                (
                    SELECT COUNT(*)
                    FROM train
                ) AS train_rows,

                (
                    SELECT COUNT(*)
                    FROM train, train_bounds
                    WHERE target_date <
                          inner_end_date
                          - INTERVAL '{SEQUENCE_INNER_VALIDATION_DAYS - 1}' DAY
                ) AS selector_train_rows,

                (
                    SELECT COUNT(*)
                    FROM train, train_bounds
                    WHERE target_date >=
                          inner_end_date
                          - INTERVAL '{SEQUENCE_INNER_VALIDATION_DAYS - 1}' DAY
                ) AS selector_validation_rows,

                (
                    SELECT COUNT(*)
                    FROM score
                ) AS score_rows,

                (
                    SELECT COUNT(DISTINCT taxi_zone_id)
                    FROM score
                ) AS score_zones,

                (
                    SELECT MIN(target_date)
                    FROM score
                ) AS first_score_date,

                (
                    SELECT MAX(target_date)
                    FROM score
                ) AS last_score_date
        """).df().iloc[0]

        score_identity_df = connection.execute(f"""
            SELECT DISTINCT
                CAST(taxi_zone_id AS BIGINT)
                    AS taxi_zone_id,
                CAST(origin_observation_sequence_id AS BIGINT)
                    AS origin_id

            FROM read_parquet('{path_sql}')

            WHERE CAST(modeling_track AS VARCHAR) = '{track_sql}'
              AND CAST(metric AS VARCHAR) = '{metric_sql}'
              AND CAST(horizon AS INTEGER) = {horizon}
              AND CAST(stage AS VARCHAR) = 'final_holdout'
              AND COALESCE(
                    CAST(native_track_eligible AS BOOLEAN),
                    FALSE
                  )
              AND COALESCE(
                    CAST(target_label_available AS BOOLEAN),
                    FALSE
                  )
              AND COALESCE(
                    isfinite(
                        CAST(target_observed_value AS DOUBLE)
                    ),
                    FALSE
                  )
              AND CAST(
                    origin_observation_sequence_id AS BIGINT
                  ) >= {minimum_history_origin}
              AND CAST(
                    origin_observation_sequence_id AS BIGINT
                  ) >= {refit_origin}
              {score_upper_bound_source}
        """).df()

    if score_identity_df.empty:
        raise RuntimeError(
            f"{family} · {metric} · h={horizon} · "
            f"{manifest_row['refit_month']}: "
            "no legal holdout scoring rows."
        )

    missing_zones = (
        set(
            score_identity_df[
                "taxi_zone_id"
            ].astype(int)
        )
        - history_zone_set
    )

    missing_origins = [
        origin
        for origin in (
            score_identity_df[
                "origin_id"
            ]
            .astype(int)
            .unique()
        )
        if origin not in history_sequence_position
    ]

    if missing_zones:
        raise RuntimeError(
            f"{family} · {metric} · h={horizon}: "
            "score rows contain Taxi Zones outside the history state: "
            f"{sorted(missing_zones)}"
        )

    if missing_origins:
        raise RuntimeError(
            f"{family} · {metric} · h={horizon}: "
            "score origins fall outside the frozen history state."
        )

    # The earliest scoring origin must occur at or after the first origin
    # capable of supplying the complete legal track-specific lookback.
    minimum_history_position = min(
        history_sequence_position[int(origin)]
        for origin in (
            score_identity_df[
                "origin_id"
            ]
            .astype(int)
            .unique()
        )
    )

    history_ready = (
        minimum_history_position
        >= first_legal_position
    )

    ready = all([
        int(summary["train_rows"]) > 0,
        int(summary["selector_train_rows"]) > 0,
        int(summary["selector_validation_rows"]) > 0,
        int(summary["score_rows"]) > 0,
        history_ready,
    ])

    return {
        "Family": FAMILY_LABELS[family],
        "Target": TARGET_LABELS.get(
            metric,
            metric,
        ),
        "Horizon": horizon,
        "Month": str(
            manifest_row["refit_month"]
        ),
        "Track": TRACK_LABELS.get(
            track,
            track,
        ),
        "Lookback": lookback_steps,
        "Selector max epochs":
            SEQUENCE_SELECTOR_MAX_EPOCHS[family],
        "Train rows":
            int(summary["train_rows"]),
        "Selector train rows":
            int(summary["selector_train_rows"]),
        "Selector validation rows":
            int(summary["selector_validation_rows"]),
        "Score rows":
            int(summary["score_rows"]),
        "Score zones":
            int(summary["score_zones"]),
        "Checkpoint":
            str(manifest_row["artifact_status"]),
        "Ready":
            bool(ready),
    }


# ---------------------------------------------------------------------
# 2) Preflight every monthly Neural and Transformer restart unit
# ---------------------------------------------------------------------

sequence_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df[
            "champion_family"
        ].isin(
            ["neural", "transformer"]
        )
    ]
    .sort_values(
        [
            "champion_family",
            "metric",
            "horizon",
            "refit_step",
        ]
    )
    .reset_index(drop=True)
)

if len(sequence_manifest_df) != 21:
    raise RuntimeError(
        "Expected 21 Neural/Transformer monthly restart units; "
        f"found {len(sequence_manifest_df)}."
    )

sequence_preflight_df = pd.DataFrame([
    preflight_sequence_holdout_unit(
        manifest_row
    )
    for manifest_row
    in sequence_manifest_df.to_dict("records")
])

if not sequence_preflight_df["Ready"].all():
    failed_sequence_preflight_df = (
        sequence_preflight_df.loc[
            ~sequence_preflight_df["Ready"]
        ]
    )

    raise RuntimeError(
        "At least one Neural/Transformer monthly unit "
        "failed preflight:\n"
        + failed_sequence_preflight_df.to_string(
            index=False
        )
    )

display(sequence_preflight_df)

print(
    "PASS — all 21 monthly Neural/Transformer restart units satisfy "
    "the legal-history and scoring chronology checks.\n"
    f"RUN_NEURAL_HOLDOUT={RUN_NEURAL_HOLDOUT}\n"
    f"RUN_TRANSFORMER_HOLDOUT={RUN_TRANSFORMER_HOLDOUT}\n"
    "No neural-network fit was run.",
    flush=True,
)

,Family,Target,Horizon,Month,Track,Lookback,Selector max epochs,Train rows,Selector train rows,Selector validation rows,Score rows,Score zones,Checkpoint,Ready
0,Neural,Taxi average speed,1,2026-01,Post-CP Primary,35,24,330215,304188,26027,25211,189,recovered,True
1,Neural,Taxi average speed,1,2026-02,Post-CP Primary,35,24,355426,330215,25211,25967,189,recovered,True
2,Neural,Taxi average speed,1,2026-03,Post-CP Primary,35,24,381393,356174,25219,28474,189,recovered,True
3,Neural,Taxi average speed,2,2026-01,Post-CP Primary,35,24,329852,304006,25846,25397,189,recovered,True
4,Neural,Taxi average speed,2,2026-02,Post-CP Primary,35,24,355244,330033,25211,25968,189,recovered,True
5,Neural,Taxi average speed,2,2026-03,Post-CP Primary,35,24,381211,355992,25219,28287,189,recovered,True
6,Neural,Taxi average speed,5,2026-01,Post-CP Primary,35,24,328762,303471,25291,25959,189,recovered,True
7,Neural,Taxi average speed,5,2026-02,Post-CP Primary,35,24,354709,329498,25211,25964,189,recovered,True
8,Neural,Taxi average speed,5,2026-03,Post-CP Primary,35,24,380676,355457,25219,27729,189,recovered,True
9,Transformer,FHVHV average speed,1,2026-01,Full-History Regime-Aware,70,30,1392367,1356465,35902,34861,257,recovered,True


PASS — all 21 monthly Neural/Transformer restart units satisfy the legal-history and scoring chronology checks.
RUN_NEURAL_HOLDOUT=False
RUN_TRANSFORMER_HOLDOUT=False
No neural-network fit was run.


**Findings.** All **21 monthly sequence-model restart units** pass the legal-history and scoring chronology checks: **9 Neural units** for Taxi average speed and **12 Transformer units** covering FHVHV average speed and Subway ridership.

Neural uses the frozen **35-Daypart lookback** and may examine up to **24 selector epochs**. Its legal training population expands from roughly **335,000–336,000 rows in January** to about **387,000 rows in March**, while each monthly selector retains roughly **25,000–26,000 historical validation rows**. Each monthly scoring population contains approximately **25,000–28,000 rows across 189 Taxi Zones**.

Transformer uses the frozen **70-Daypart lookback** and may examine up to **30 selector epochs**. FHVHV-speed training expands from about **1.39 million rows in January** to **1.46 million in March**, with roughly **35,000 historical selector-validation rows** per month and **257 scoring zones**. Subway h=2 expands from about **833,000 to 876,000 training rows**, with approximately **21,000 historical selector-validation rows** and **153 scoring zones**.

Every unit has a nonempty legal training population, historical selector split, complete sequence-history support, and nonempty monthly holdout population. All 21 units are structurally ready for execution.

**4.6.1.6.2B — Consolidate final-holdout execution readiness.**

The final preflight combines structural readiness with durable checkpoint status.

A restart unit is ready only when its source-specific contracts and legal chronology are valid. A recovered unit already has a complete prediction/audit pair; a pending unit still requires its family-specific training procedure.

The remaining workload is measured in actual training calls, excluding work already recovered from valid checkpoints.

In [49]:
# ---------------------------------------------------------------------
# 4.6.1.6.2B — Consolidate three-family execution readiness
# ---------------------------------------------------------------------

holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

preflight_unit_counts = {
    "tree": len(tree_preflight_df),
    "neural": int(
        sequence_preflight_df["Family"].eq("Neural").sum()
    ),
    "transformer": int(
        sequence_preflight_df["Family"].eq("Transformer").sum()
    ),
}

expected_unit_counts = {
    "tree": 8,
    "neural": 9,
    "transformer": 12,
}

if preflight_unit_counts != expected_unit_counts:
    raise RuntimeError(
        "Preflight unit counts do not match the frozen workload: "
        f"{preflight_unit_counts}"
    )

readiness_records = []

for family in ["tree", "neural", "transformer"]:
    family_df = holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq(family)
    ]

    recovered = int(
        family_df["artifact_status"].eq("recovered").sum()
    )
    pending = int(
        family_df["artifact_status"].eq("pending").sum()
    )

    remaining_calls = int(
        family_df.loc[
            family_df["artifact_status"].eq("pending"),
            "planned_fit_calls",
        ].sum()
    )

    readiness_records.append({
        "Family": FAMILY_LABELS[family],
        "Restart units": len(family_df),
        "Preflight ready": preflight_unit_counts[family],
        "Recovered": recovered,
        "Pending": pending,
        "Remaining training calls": remaining_calls,
    })

holdout_execution_readiness_df = pd.DataFrame(
    readiness_records
)

total_row = pd.DataFrame([{
    "Family": "TOTAL",
    "Restart units":
        int(holdout_execution_readiness_df["Restart units"].sum()),
    "Preflight ready":
        int(holdout_execution_readiness_df["Preflight ready"].sum()),
    "Recovered":
        int(holdout_execution_readiness_df["Recovered"].sum()),
    "Pending":
        int(holdout_execution_readiness_df["Pending"].sum()),
    "Remaining training calls":
        int(
            holdout_execution_readiness_df[
                "Remaining training calls"
            ].sum()
        ),
}])

holdout_execution_readiness_df = pd.concat(
    [holdout_execution_readiness_df, total_row],
    ignore_index=True,
)

total = holdout_execution_readiness_df.iloc[-1]

# ---------------------------------------------------------------------
# Validate the frozen workload while allowing checkpoint state to advance
# ---------------------------------------------------------------------
# The physical workload is fixed at 29 restart units and 50 possible training
# calls. Recovery state is intentionally dynamic: completed units should reduce
# both the pending-unit count and the remaining training workload on rerun.

if int(total["Restart units"]) != 29:
    raise RuntimeError(
        "Execution readiness no longer contains 29 restart units."
    )

if int(total["Preflight ready"]) != 29:
    raise RuntimeError(
        "Not all 29 restart units passed preflight."
    )

if (
    int(total["Recovered"])
    + int(total["Pending"])
    != 29
):
    raise RuntimeError(
        "Recovered and pending restart units do not partition "
        "the 29-unit final-holdout workload."
    )

total_planned_calls = int(
    holdout_checkpoint_manifest_df["planned_fit_calls"].sum()
)

if total_planned_calls != 50:
    raise RuntimeError(
        "Final-holdout manifest no longer contains the expected "
        f"50 planned training calls; found {total_planned_calls}."
    )

if int(total["Remaining training calls"]) < 0:
    raise RuntimeError(
        "Remaining training-call count cannot be negative."
    )

display(holdout_execution_readiness_df)

print(
    "PASS — the frozen champion system is structurally ready for "
    "final-holdout execution.\n"
    f"  RUN_TREE_HOLDOUT={RUN_TREE_HOLDOUT}\n"
    f"  RUN_NEURAL_HOLDOUT={RUN_NEURAL_HOLDOUT}\n"
    f"  RUN_TRANSFORMER_HOLDOUT={RUN_TRANSFORMER_HOLDOUT}\n"
    f"  RESET_FINAL_HOLDOUT_RUNTIME={RESET_FINAL_HOLDOUT_RUNTIME}",
    flush=True,
)

,Family,Restart units,Preflight ready,Recovered,Pending,Remaining training calls
0,Tree,8,8,8,0,0
1,Neural,9,9,9,0,0
2,Transformer,12,12,12,0,0
3,TOTAL,29,29,29,0,0


PASS — the frozen champion system is structurally ready for final-holdout execution.
  RUN_TREE_HOLDOUT=True
  RUN_NEURAL_HOLDOUT=False
  RUN_TRANSFORMER_HOLDOUT=False
  RESET_FINAL_HOLDOUT_RUNTIME=False


**Findings.** All **29 final-holdout restart units** pass structural preflight and are now recoverable from valid durable checkpoints: **8 Tree, 9 Neural, and 12 Transformer**.

No unit remains pending and no additional training call is required. The complete three-family execution surface is therefore ready for final-holdout assembly and scoring.

**4.6.1.6.2C — Load the frozen Neural and Transformer execution contracts.**

The sequence champions require more than a model name. Their final-holdout runners must reproduce the exact input representation, preprocessing roles, target-specific architecture, and training-duration policy selected upstream.

This block therefore binds the active Neural and Transformer jobs to:

- the finalized model registries;
- the published 52-value context contract;
- the shared dense five-modality mobility-history state;
- the 35-Daypart Neural history window;
- the 70-Daypart overlapping Transformer representation; and
- each family's frozen selector policy.

These contracts determine how the models execute; no architecture or training-policy choice is reopened in the final holdout.

In [50]:
# ---------------------------------------------------------------------
# 4.6.1.6.2C — Load frozen Neural and Transformer execution contracts
# ---------------------------------------------------------------------

import gc
import hashlib
import random
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


# ---------------------------------------------------------------------
# Frozen source artifacts
# ---------------------------------------------------------------------
# These files carry decisions already made in 4.5.1 and 4.5.2. The
# final holdout consumes them as authoritative contracts rather than
# reconstructing or retuning either sequence-model family.

NEURAL_FINAL_REGISTRY_PATH = (
    INPUT_451_FINAL_DIR / "neural_final_model_registry.parquet"
)
NEURAL_MODEL_DEFINITION_PATH = (
    INPUT_451_FINAL_DIR / "neural_model_definition_contract.parquet"
)

TRANSFORMER_FINAL_REGISTRY_PATH = (
    INPUT_452_FINAL_DIR / "transformer_final_model_registry.parquet"
)
TRANSFORMER_CONTEXT_CONTRACT_PATH = (
    INPUT_452_FINAL_DIR / "transformer_context_contract.parquet"
)

SEQUENCE_HISTORY_STATE_PATH = (
    INPUT_451_FINAL_DIR / "neural_mobility_history_state.npz"
)

for required_path in [
    NEURAL_FINAL_REGISTRY_PATH,
    NEURAL_MODEL_DEFINITION_PATH,
    TRANSFORMER_FINAL_REGISTRY_PATH,
    TRANSFORMER_CONTEXT_CONTRACT_PATH,
    SEQUENCE_HISTORY_STATE_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required sequence-model contract is missing: {required_path}"
        )


# ---------------------------------------------------------------------
# Shared execution constants
# ---------------------------------------------------------------------
# The permanent notebook remains portable across CPU/GPU environments.
# Device choice changes runtime only; it does not alter the frozen model
# definitions or training policy.

SEQUENCE_HOLDOUT_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEQUENCE_DAYPART_ORDER = [
    "overnight",
    "am_peak",
    "midday",
    "pm_peak",
    "evening",
]

# Neural and Transformer intentionally retain different historical
# representations because those choices were resolved upstream.
NEURAL_HOLDOUT_LOOKBACK = 35
NEURAL_HOLDOUT_BATCH_SIZE = 1024
NEURAL_SELECTOR_MAX_EPOCHS = 24
NEURAL_SELECTOR_PATIENCE = 4
NEURAL_SELECTOR_MIN_DELTA = 1e-4

TRANSFORMER_HOLDOUT_LOOKBACK = 70
TRANSFORMER_HOLDOUT_BATCH_SIZE = 1024
TRANSFORMER_SELECTOR_MAX_EPOCHS = 30
TRANSFORMER_SELECTOR_PATIENCE = 5
TRANSFORMER_SELECTOR_MIN_DELTA = 1e-4

SEQUENCE_GRAD_CLIP_NORM = 1.0
SEQUENCE_INNER_VALIDATION_DAYS = 28


# ---------------------------------------------------------------------
# Frozen Transformer representation and coherent architecture profiles
# ---------------------------------------------------------------------
# These are coherent upstream-selected profiles. Keeping them bundled
# prevents the final holdout from quietly becoming another tuning stage.

TRANSFORMER_HOLDOUT_REPRESENTATIONS = {
    "two_week_overlapping_two_day": {
        "representation_id": "two_week_overlapping_two_day",
        "lookback_steps": 70,
        "patch_length": 10,
        "patch_stride": 5,
    },
}

TRANSFORMER_HOLDOUT_PROFILES = {
    "compact": {
        "profile_id": "compact",
        "d_model": 48,
        "n_heads": 4,
        "n_layers": 2,
        "d_ff": 96,
        "dropout": 0.10,
        "zone_embedding_dim": 8,
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
    },
    "balanced": {
        "profile_id": "balanced",
        "d_model": 64,
        "n_heads": 4,
        "n_layers": 2,
        "d_ff": 128,
        "dropout": 0.10,
        "zone_embedding_dim": 16,
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
    },
    "higher_capacity_regularized": {
        "profile_id": "higher_capacity_regularized",
        "d_model": 96,
        "n_heads": 6,
        "n_layers": 3,
        "d_ff": 192,
        "dropout": 0.15,
        "zone_embedding_dim": 16,
        "learning_rate": 2e-4,
        "weight_decay": 5e-4,
    },
}


# ---------------------------------------------------------------------
# Read and normalize the frozen registries
# ---------------------------------------------------------------------

neural_final_registry_df = pd.read_parquet(
    NEURAL_FINAL_REGISTRY_PATH
)
neural_model_definition_df = pd.read_parquet(
    NEURAL_MODEL_DEFINITION_PATH
)

transformer_final_registry_df = pd.read_parquet(
    TRANSFORMER_FINAL_REGISTRY_PATH
)

# The Transformer source registry publishes the modeling track as `track`.
# Normalize the column once here so both sequence families can share the
# same downstream runtime interface.
if (
    "track" in transformer_final_registry_df.columns
    and "modeling_track" not in transformer_final_registry_df.columns
):
    transformer_final_registry_df = transformer_final_registry_df.rename(
        columns={"track": "modeling_track"}
    )

transformer_context_contract_df = (
    pd.read_parquet(TRANSFORMER_CONTEXT_CONTRACT_PATH)
    .sort_values("input_index")
    .reset_index(drop=True)
)

require_columns(
    neural_final_registry_df,
    ["modeling_track", "metric", "horizon", "profile_id"],
    "Neural final model registry",
)

require_columns(
    neural_model_definition_df,
    [
        "profile_id",
        "sequence_width_1",
        "sequence_width_2",
        "dropout",
        "learning_rate",
        "weight_decay",
        "zone_embedding_dim",
    ],
    "Neural model-definition contract",
)

require_columns(
    transformer_final_registry_df,
    [
        "modeling_track",
        "metric",
        "horizon",
        "representation_id",
        "profile_id",
    ],
    "Transformer final model registry",
)

require_columns(
    transformer_context_contract_df,
    ["feature", "role", "input_index"],
    "Transformer context contract",
)


# ---------------------------------------------------------------------
# The 52-value context contract is shared by both sequence families
# ---------------------------------------------------------------------
# Using one ordered list here protects the exact feature position contract
# expected by both the finalized MLP and Transformer implementations.

SEQUENCE_CONTEXT_FEATURE_ORDER = (
    transformer_context_contract_df["feature"].astype(str).tolist()
)

if len(SEQUENCE_CONTEXT_FEATURE_ORDER) != 52:
    raise RuntimeError(
        "Sequence-model context contract must contain exactly 52 inputs."
    )

context_role_counts = (
    transformer_context_contract_df["role"]
    .astype(str)
    .value_counts()
    .to_dict()
)

expected_context_roles = {
    "ordinary_continuous": 24,
    "weekly_stl_offset": 1,
    "passthrough": 27,
}

if context_role_counts != expected_context_roles:
    raise RuntimeError(
        "Sequence-model context roles do not match the frozen 52-value "
        f"contract: {context_role_counts}"
    )


# ---------------------------------------------------------------------
# Load the shared dense mobility-history state once
# ---------------------------------------------------------------------
# The dense state stores raw mobility values and observation masks by
# Taxi Zone × temporal position × mobility measure. History windows are
# sliced from this state later rather than duplicated for every fit.

with np.load(
    SEQUENCE_HISTORY_STATE_PATH,
    allow_pickle=False,
) as history_state:
    SEQUENCE_HISTORY_VALUES = np.asarray(
        history_state["values"],
        dtype=np.float32,
    )
    SEQUENCE_HISTORY_MASKS = np.asarray(
        history_state["masks"],
        dtype=bool,
    )
    SEQUENCE_HISTORY_ZONE_IDS = np.asarray(
        history_state["zone_ids"],
        dtype=np.int64,
    )
    SEQUENCE_HISTORY_SEQUENCE_IDS = np.asarray(
        history_state["sequence_ids"],
        dtype=np.int64,
    )

if SEQUENCE_HISTORY_VALUES.shape != SEQUENCE_HISTORY_MASKS.shape:
    raise RuntimeError(
        "Shared sequence-history values and masks do not align."
    )

if (
    SEQUENCE_HISTORY_VALUES.ndim != 3
    or SEQUENCE_HISTORY_VALUES.shape[-1] != len(FORECAST_TARGETS)
):
    raise RuntimeError(
        "Shared sequence history must have shape "
        "[Taxi Zone, temporal position, five mobility measures]."
    )

# Integer-position lookups keep repeated history slicing cheap while also
# failing cleanly if a requested Zone or temporal position is absent.
SEQUENCE_ZONE_TO_POSITION = {
    int(zone_id): position
    for position, zone_id in enumerate(SEQUENCE_HISTORY_ZONE_IDS)
}
SEQUENCE_ID_TO_POSITION = {
    int(sequence_id): position
    for position, sequence_id in enumerate(SEQUENCE_HISTORY_SEQUENCE_IDS)
}

SEQUENCE_METRIC_TO_POSITION = {
    metric: position
    for position, metric in enumerate(FORECAST_TARGETS)
}


# ---------------------------------------------------------------------
# Resolve the legal history boundary of each modeling track
# ---------------------------------------------------------------------

def resolve_sequence_track_start(track):
    """
    Return the earliest sequence ID legal for this modeling track.

    Full-History begins at the first shared history position. Post-CP resolves
    the boundary from only the two physical fields needed for that decision.
    """
    if track == "full_history_regime_aware":
        return int(SEQUENCE_HISTORY_SEQUENCE_IDS[0])

    if track != "post_cp_primary":
        raise ValueError(f"Unsupported sequence modeling track: {track}")

    boundary_df = pd.read_parquet(
        FORECASTING_MATRIX_PATHS[track],
        columns=[
            "target_observation_sequence_id",
            "native_track_eligible",
        ],
        filters=[
            ("target_pre_post_cp", "==", "post_cp"),
        ],
    )

    eligible_ids = pd.to_numeric(
        boundary_df.loc[
            boundary_df["native_track_eligible"]
            .fillna(False)
            .astype(bool),
            "target_observation_sequence_id",
        ],
        errors="coerce",
    ).dropna()

    if eligible_ids.empty:
        raise RuntimeError(
            "Post-CP sequence-history boundary could not be resolved."
        )

    return int(eligible_ids.min())


SEQUENCE_TRACK_HISTORY_START = {
    track: resolve_sequence_track_start(track)
    for track in ORDINARY_TRACKS
}


# ---------------------------------------------------------------------
# Audit only the active champion configurations
# ---------------------------------------------------------------------
# The holdout executes champions only. Counting these jobs here catches
# accidental expansion back into the full upstream candidate tournaments.

active_sequence_jobs_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].isin(
            ["neural", "transformer"]
        ),
        [
            "champion_family",
            "modeling_track",
            "metric",
            "horizon",
        ],
    ]
    .drop_duplicates()
    .sort_values(["champion_family", "metric", "horizon"])
    .reset_index(drop=True)
)

if len(active_sequence_jobs_df) != 7:
    raise RuntimeError(
        "Expected 7 active Neural/Transformer champion jobs."
    )

display(active_sequence_jobs_df.assign(
    Family=lambda frame: frame["champion_family"].map(FAMILY_LABELS),
    Target=lambda frame: frame["metric"].map(TARGET_LABELS),
)[["Family", "Target", "horizon", "modeling_track"]].rename(columns={
    "horizon": "Horizon",
    "modeling_track": "Modeling track",
}))

print(
    "PASS — frozen sequence-model contracts loaded.\n"
    f"  Context inputs: {len(SEQUENCE_CONTEXT_FEATURE_ORDER)}\n"
    f"  History state: {SEQUENCE_HISTORY_VALUES.shape[0]:,} zones × "
    f"{SEQUENCE_HISTORY_VALUES.shape[1]:,} temporal positions × "
    f"{SEQUENCE_HISTORY_VALUES.shape[2]:,} mobility measures\n"
    f"  Active jobs: {len(active_sequence_jobs_df)}",
    flush=True,
)

,Family,Target,Horizon,Modeling track
0,Neural,Taxi average speed,1,post_cp_primary
1,Neural,Taxi average speed,2,post_cp_primary
2,Neural,Taxi average speed,5,post_cp_primary
3,Transformer,FHVHV average speed,1,full_history_regime_aware
4,Transformer,FHVHV average speed,2,full_history_regime_aware
5,Transformer,FHVHV average speed,5,full_history_regime_aware
6,Transformer,Subway ridership,2,full_history_regime_aware


PASS — frozen sequence-model contracts loaded.
  Context inputs: 52
  History state: 258 zones × 5,930 temporal positions × 5 mobility measures
  Active jobs: 7


**Findings.** The sequence-model runtime binds **7 frozen champion jobs** to one shared **52-value supporting-context contract** and one five-modality mobility-history state.

Neural owns the three Taxi-speed horizons and uses a **35-Daypart** history window. Transformer owns the three FHVHV-speed horizons plus Subway h=2 and uses the frozen **70-Daypart overlapping-patch** representation.

The supporting context contains **24 training-scaled continuous inputs, 1 specially scaled weekly-STL offset, and 27 deterministic/binary/missingness inputs**. No final-holdout outcome contributes to these frozen structural definitions.

**4.6.1.6.2D — Reproduce the shared leakage-safe sequence preprocessing.**

Both sequence families consume the same underlying information but encode the temporal history differently.

For every monthly refit, preprocessing is learned from legal training history only:

- observed mobility values determine the five history-channel means and scales;
- missing historical observations become zero only after standardization and remain distinguishable through their observation masks;
- ordinary continuous context features are median-completed and standardized from training rows;
- weekly STL learns its scale only from enabled, legally available fitted offsets;
- deterministic, binary, and missingness indicators pass through unchanged; and
- the forecasting target is standardized from the same legal training sample.

The history itself is streamed from the shared dense state in mini-batches so the full expanding-history tensor does not need to be duplicated in memory.

In [51]:
# ---------------------------------------------------------------------
# 4.6.1.6.2D — Shared sequence preprocessing and streaming helpers
# ---------------------------------------------------------------------

# These source groups describe the raw forecast-matrix columns needed to
# reconstruct the already-frozen 52-value model-facing context. Keeping
# the groups named makes the information supplied to the models auditable.

MULTIMODAL_CONTEXT_SOURCES = [
    "mm_taxi_avg_trip_duration_origin_value",
    "mm_fhvhv_avg_trip_duration_origin_value",
    "mm_subway_transfers_origin_value",
    "mm_bus_trip_count_origin_value",
    "mm_avg_bus_speed_origin_value",
]

SPATIAL_CONTEXT_SOURCES = [
    f"spatial_connected__{metric}"
    for metric in FORECAST_TARGETS
]

WEATHER_CONTEXT_SOURCES = [
    "weather_origin__temperature",
    "weather_origin__precipitation",
    "weather_origin__visibility",
    "weather_origin__wind_speed",
    "weather_target_proxy__temperature",
    "weather_target_proxy__precipitation",
    "weather_target_proxy__visibility",
    "weather_target_proxy__wind_speed",
]

CALENDAR_CONTEXT_SOURCES = [
    "target_daypart",
    "target_weekday_index",
    "target_is_weekend",
    "annual_fourier_sin",
    "annual_fourier_cos",
    "target_pre_post_cp",
]

STL_CONTEXT_SOURCES = [
    "weekly_stl_enabled",
    "weekly_stl_seasonal_offset",
]

GEOGRAPHIC_CONTEXT_SOURCES = [
    "is_cbd_zone",
    "is_cbd_gateway_zone",
    "is_cbd_adjacent_zone",
    "distance_to_cbd_miles",
    "distance_to_gateway_miles",
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

SEQUENCE_CONTEXT_SOURCE_FEATURES = list(dict.fromkeys(
    MULTIMODAL_CONTEXT_SOURCES
    + SPATIAL_CONTEXT_SOURCES
    + WEATHER_CONTEXT_SOURCES
    + CALENDAR_CONTEXT_SOURCES
    + STL_CONTEXT_SOURCES
    + GEOGRAPHIC_CONTEXT_SOURCES
))


def derive_sequence_context_frame(frame):
    """
    Rebuild the frozen 52-value sequence-model context deterministically.

    Continuous source values remain missing until the fitted preprocessor
    completes them from training-only statistics. Static binary geography is
    different: missingness is recorded explicitly first, then zero is used as
    the finite tensor placeholder, matching the finalized 4.5.1 contract.
    """
    missing_sources = sorted(
        set(SEQUENCE_CONTEXT_SOURCE_FEATURES)
        - set(frame.columns)
    )

    if missing_sources:
        raise ValueError(
            "Forecast matrix lacks final context sources: "
            f"{missing_sources}"
        )

    result = pd.DataFrame(index=frame.index)

    static_binary_features = [
        "is_cbd_zone",
        "is_cbd_gateway_zone",
        "is_cbd_adjacent_zone",
    ]

    derived_sources = {
        "target_daypart",
        "target_weekday_index",
        "target_pre_post_cp",
    }

    # -----------------------------------------------------------------
    # 1) Preserve raw numeric context in native units
    # -----------------------------------------------------------------
    # Missing continuous values intentionally remain missing here. Their
    # training-only completion belongs to FrozenSequencePreprocessor.fit().
    for feature in SEQUENCE_CONTEXT_SOURCE_FEATURES:
        if feature not in derived_sources:
            result[feature] = pd.to_numeric(
                frame[feature],
                errors="coerce",
            )


    # -----------------------------------------------------------------
    # 2) Record missingness BEFORE applying any placeholder
    # -----------------------------------------------------------------
    # This is essential for inherited static geography: a placeholder zero
    # must remain distinguishable from a genuinely observed zero-valued flag.
    for feature in SEQUENCE_CONTEXT_FEATURE_ORDER:
        if not feature.startswith("missing_mask__"):
            continue

        source = feature.removeprefix(
            "missing_mask__"
        )

        if source not in frame.columns:
            raise ValueError(
                "Missingness source is absent from forecast matrix: "
                f"{source}"
            )

        numeric_source = pd.to_numeric(
            frame[source],
            errors="coerce",
        ).replace(
            [np.inf, -np.inf],
            np.nan,
        )

        result[feature] = (
            numeric_source.isna()
            .astype("float32")
        )


    # -----------------------------------------------------------------
    # 3) Encode target Daypart cyclically
    # -----------------------------------------------------------------
    daypart_lookup = {
        name: index
        for index, name in enumerate(
            SEQUENCE_DAYPART_ORDER
        )
    }

    daypart_index = frame[
        "target_daypart"
    ].map(daypart_lookup)

    if daypart_index.isna().any():
        unknown = sorted(
            frame.loc[
                daypart_index.isna(),
                "target_daypart",
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            f"Unknown target Dayparts: {unknown}"
        )

    daypart_angle = (
        2.0
        * np.pi
        * daypart_index.astype(float)
        / len(SEQUENCE_DAYPART_ORDER)
    )

    result["target_daypart_sin"] = (
        np.sin(daypart_angle)
    )
    result["target_daypart_cos"] = (
        np.cos(daypart_angle)
    )


    # -----------------------------------------------------------------
    # 4) Encode weekday cyclically
    # -----------------------------------------------------------------
    weekday = pd.to_numeric(
        frame["target_weekday_index"],
        errors="raise",
    ).astype(int)

    if not weekday.between(0, 6).all():
        raise ValueError(
            "target_weekday_index must use the canonical "
            "Monday=0 ... Sunday=6 range."
        )

    weekday_angle = (
        2.0
        * np.pi
        * weekday.astype(float)
        / 7.0
    )

    result["target_weekday_sin"] = (
        np.sin(weekday_angle)
    )
    result["target_weekday_cos"] = (
        np.cos(weekday_angle)
    )


    # -----------------------------------------------------------------
    # 5) Preserve the deterministic annual Fourier coordinates
    # -----------------------------------------------------------------
    for feature in [
        "annual_fourier_sin",
        "annual_fourier_cos",
    ]:
        values = pd.to_numeric(
            frame[feature],
            errors="raise",
        )

        if not np.isfinite(
            values.to_numpy(dtype=float)
        ).all():
            raise ValueError(
                f"{feature} contains non-finite values."
            )

        result[feature] = values.astype(
            "float32"
        )


    # -----------------------------------------------------------------
    # 6) Validate weekend and congestion-pricing regime
    # -----------------------------------------------------------------
    target_is_weekend = pd.to_numeric(
        frame["target_is_weekend"],
        errors="raise",
    ).astype("float32")

    if not target_is_weekend.isin(
        [0.0, 1.0]
    ).all():
        raise ValueError(
            "target_is_weekend must be binary."
        )

    result["target_is_weekend"] = (
        target_is_weekend
    )

    track_values = (
        frame["modeling_track"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if len(track_values) != 1:
        raise ValueError(
            "Each sequence-model payload must contain exactly "
            f"one modeling track; found {track_values}."
        )

    track = track_values[0]

    target_regime = (
        frame["target_pre_post_cp"]
        .astype("string")
    )

    valid_regime = target_regime.isin(
        ["pre_cp", "post_cp"]
    )

    if not valid_regime.all():
        unknown = sorted(
            target_regime.loc[
                ~valid_regime
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            "Unrecognized target policy-regime values: "
            f"{unknown}"
        )

    if track == "post_cp_primary":
        if not target_regime.eq(
            "post_cp"
        ).fillna(False).all():
            raise ValueError(
                "Post-CP Primary sequence context contains "
                "a non-post-CP target regime."
            )

    elif track != "full_history_regime_aware":
        raise ValueError(
            f"Unsupported sequence modeling track: {track}"
        )

    result["target_cp_regime"] = (
        target_regime.map({
            "pre_cp": 0.0,
            "post_cp": 1.0,
        })
        .astype("float32")
    )


    # -----------------------------------------------------------------
    # 7) Preserve weekly-STL state semantics
    # -----------------------------------------------------------------
    weekly_stl_enabled = pd.to_numeric(
        frame["weekly_stl_enabled"],
        errors="raise",
    ).astype("float32")

    if not weekly_stl_enabled.isin(
        [0.0, 1.0]
    ).all():
        raise ValueError(
            "weekly_stl_enabled must be binary."
        )

    result["weekly_stl_enabled"] = (
        weekly_stl_enabled
    )

    weekly_stl_offset = pd.to_numeric(
        frame["weekly_stl_seasonal_offset"],
        errors="coerce",
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    )

    result["weekly_stl_seasonal_offset"] = (
        weekly_stl_offset
    )

    result[
        "missing_mask__weekly_stl_seasonal_offset"
    ] = (
        weekly_stl_offset.isna()
        .astype("float32")
    )

    # When STL is disabled, zero is a semantic value—not missing data.
    disabled_offsets = weekly_stl_offset.loc[
        weekly_stl_enabled.eq(0.0)
    ]

    if (
        disabled_offsets.isna().any()
        or not np.isclose(
            disabled_offsets.to_numpy(
                dtype=float
            ),
            0.0,
        ).all()
    ):
        raise ValueError(
            "Disabled weekly-STL rows must retain "
            "their semantic zero offset."
        )


    # -----------------------------------------------------------------
    # 8) Convert missing static binary geography to finite placeholders
    # -----------------------------------------------------------------
    # Missing static geography is legitimate upstream state. The matching
    # missingness masks were already created above, so zero can now serve as
    # a safe tensor placeholder without pretending the value was observed.
    for feature in static_binary_features:
        numeric = (
            pd.to_numeric(
                frame[feature],
                errors="raise",
            )
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
        )

        observed = numeric.dropna()

        if not observed.isin(
            [0.0, 1.0]
        ).all():
            invalid_values = sorted(
                observed.loc[
                    ~observed.isin(
                        [0.0, 1.0]
                    )
                ]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"{feature} contains non-binary observed "
                f"values: {invalid_values}"
            )

        result[feature] = (
            numeric.fillna(0.0)
            .astype("float32")
        )


    # -----------------------------------------------------------------
    # 9) Require the exact published 52-value model-facing contract
    # -----------------------------------------------------------------
    missing_final = sorted(
        set(SEQUENCE_CONTEXT_FEATURE_ORDER)
        - set(result.columns)
    )

    if missing_final:
        raise ValueError(
            "Derived context is missing final model inputs: "
            f"{missing_final}"
        )

    final_context = result[
        SEQUENCE_CONTEXT_FEATURE_ORDER
    ].copy()

    return final_context.astype(
        "float32",
        copy=False,
    )


def sequence_channel_positions(target_metric):
    """Return target-first mobility-channel order for one forecasting job."""
    if target_metric not in SEQUENCE_METRIC_TO_POSITION:
        raise KeyError(
            f"Unknown sequence target metric: {target_metric}"
        )

    target_position = SEQUENCE_METRIC_TO_POSITION[target_metric]

    # The target channel always occupies the first pair of value/mask
    # positions, matching the upstream sequence-model training contract.
    return [
        target_position,
        *[
            position
            for position in range(len(FORECAST_TARGETS))
            if position != target_position
        ],
    ]


def sequence_frame_positions(frame, track, lookback_steps):
    """
    Resolve dense-history coordinates for a model-ready forecasting frame.

    The first history position may not precede the modeling track's legal
    history boundary. This is particularly important for the Post-CP track,
    whose sequence input must not reach backward into pre-policy observations.
    """
    zones = pd.to_numeric(
        frame["taxi_zone_id"],
        errors="raise",
    ).astype(int)

    origins = pd.to_numeric(
        frame["origin_observation_sequence_id"],
        errors="raise",
    ).astype(int)

    zone_positions = zones.map(SEQUENCE_ZONE_TO_POSITION)
    origin_positions = origins.map(SEQUENCE_ID_TO_POSITION)

    if zone_positions.isna().any() or origin_positions.isna().any():
        raise ValueError(
            "Sequence rows fall outside the shared mobility-history state."
        )

    zone_positions = zone_positions.to_numpy(dtype=np.int64)
    origin_positions = origin_positions.to_numpy(dtype=np.int64)

    start_positions = origin_positions - int(lookback_steps) + 1

    track_start_id = SEQUENCE_TRACK_HISTORY_START[track]

    if track_start_id not in SEQUENCE_ID_TO_POSITION:
        raise RuntimeError(
            f"{track}: legal history boundary is absent from the dense state."
        )

    track_start_position = SEQUENCE_ID_TO_POSITION[track_start_id]

    if np.any(start_positions < track_start_position):
        raise ValueError(
            "At least one sequence window crosses its modeling-track "
            "history boundary."
        )

    return zone_positions, origin_positions


def raw_sequence_batch(
    zone_positions,
    origin_positions,
    row_indices,
    target_metric,
    lookback_steps,
):
    """Extract one target-first raw history batch from the dense state."""
    row_indices = np.asarray(row_indices, dtype=np.int64)

    zones = zone_positions[row_indices]
    origins = origin_positions[row_indices]
    starts = origins - int(lookback_steps) + 1

    # Build all temporal offsets for this mini-batch at once so NumPy can
    # slice the dense state vectorially instead of looping row by row.
    offsets = np.arange(int(lookback_steps), dtype=np.int64)
    positions = starts[:, None] + offsets[None, :]

    channels = sequence_channel_positions(target_metric)

    values = SEQUENCE_HISTORY_VALUES[
        zones[:, None],
        positions,
        :,
    ][:, :, channels]

    masks = SEQUENCE_HISTORY_MASKS[
        zones[:, None],
        positions,
        :,
    ][:, :, channels]

    return values, masks


class FrozenSequencePreprocessor:
    """
    Shared training-only preprocessor for Neural and Transformer champions.

    Sequence scaling, continuous-context completion/scaling, weekly-STL
    scaling, and target normalization are learned from the legal training
    sample supplied to fit(). Evaluation rows never update this state.
    """

    def __init__(self, context_contract):
        contract = (
            context_contract.sort_values("input_index")
            .reset_index(drop=True)
            .copy()
        )

        self.feature_order = contract["feature"].astype(str).tolist()

        self.continuous_features = (
            contract.loc[
                contract["role"].eq("ordinary_continuous"),
                "feature",
            ]
            .astype(str)
            .tolist()
        )

        stl_features = (
            contract.loc[
                contract["role"].eq("weekly_stl_offset"),
                "feature",
            ]
            .astype(str)
            .tolist()
        )

        if stl_features != ["weekly_stl_seasonal_offset"]:
            raise ValueError(
                "Expected one weekly-STL offset in the context contract."
            )

        self.stl_feature = stl_features[0]

        self.passthrough_features = (
            contract.loc[
                contract["role"].eq("passthrough"),
                "feature",
            ]
            .astype(str)
            .tolist()
        )

        self.is_fitted_ = False

    def fit(
        self,
        training_frame,
        track,
        target_metric,
        lookback_steps,
        moment_batch_size=8192,
    ):
        """Fit every learned preprocessing statistic from training rows only."""
        zone_positions, origin_positions = sequence_frame_positions(
            training_frame,
            track,
            lookback_steps,
        )

        # Accumulate history moments in chunks so the expanding-history windows
        # never need to be materialized as one large tensor.
        channel_count = len(FORECAST_TARGETS)
        sums = np.zeros(channel_count, dtype=np.float64)
        squared_sums = np.zeros(channel_count, dtype=np.float64)
        counts = np.zeros(channel_count, dtype=np.int64)

        for start in range(0, len(training_frame), moment_batch_size):
            stop = min(start + moment_batch_size, len(training_frame))
            batch_indices = np.arange(start, stop, dtype=np.int64)

            values, masks = raw_sequence_batch(
                zone_positions,
                origin_positions,
                batch_indices,
                target_metric,
                lookback_steps,
            )

            for channel in range(channel_count):
                observed = values[..., channel][
                    masks[..., channel]
                ]
                observed = observed[np.isfinite(observed)]

                if observed.size:
                    observed = observed.astype(
                        np.float64,
                        copy=False,
                    )

                    sums[channel] += observed.sum()
                    squared_sums[channel] += np.square(
                        observed
                    ).sum()
                    counts[channel] += observed.size

        if np.any(counts == 0):
            raise ValueError(
                "At least one mobility-history channel has no observed "
                "training values."
            )

        self.sequence_means_ = sums / counts

        variances = (
            squared_sums / counts
            - np.square(self.sequence_means_)
        )
        variances = np.maximum(
            variances,
            0.0,
        )

        self.sequence_stds_ = np.sqrt(
            variances
        )

        # np.where returns a new writable array, avoiding assumptions about the
        # mutability of arrays produced by pandas / Arrow-backed data.
        self.sequence_stds_ = np.where(
            self.sequence_stds_ <= 1e-8,
            1.0,
            self.sequence_stds_,
        )

        context_frame = derive_sequence_context_frame(
            training_frame
        )

        # Do not mutate the array returned by pandas.to_numpy(). Arrow-backed
        # frames may expose a read-only NumPy view. np.where creates a new array
        # while converting non-finite values to NaN for median estimation.
        continuous = (
            context_frame[self.continuous_features]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float64)
        )

        continuous = np.where(
            np.isfinite(continuous),
            continuous,
            np.nan,
        )

        self.context_medians_ = np.nanmedian(
            continuous,
            axis=0,
        )

        if not np.isfinite(
            self.context_medians_
        ).all():
            raise ValueError(
                "At least one continuous context feature has no finite "
                "training support."
            )

        # Median completion creates a fresh writable array and uses statistics
        # learned only from the legal training population.
        completed = np.where(
            np.isfinite(continuous),
            continuous,
            self.context_medians_[None, :],
        )

        self.context_means_ = completed.mean(
            axis=0
        )

        self.context_stds_ = completed.std(
            axis=0,
            ddof=0,
        )

        self.context_stds_ = np.where(
            self.context_stds_ <= 1e-8,
            1.0,
            self.context_stds_,
        )

        # Weekly STL is scaled only from enabled, legally fitted states.
        # Disabled semantic zeros therefore do not distort the fitted scale.
        stl_enabled = pd.to_numeric(
            context_frame["weekly_stl_enabled"],
            errors="raise",
        ).to_numpy(dtype=float)

        stl_missing = pd.to_numeric(
            context_frame[
                "missing_mask__weekly_stl_seasonal_offset"
            ],
            errors="raise",
        ).to_numpy(dtype=float)

        stl_values = pd.to_numeric(
            context_frame[self.stl_feature],
            errors="coerce",
        ).to_numpy(dtype=float)

        fitted_stl = (
            np.isclose(stl_enabled, 1.0)
            & np.isclose(stl_missing, 0.0)
            & np.isfinite(stl_values)
        )

        fitted_values = stl_values[
            fitted_stl
        ]

        self.stl_training_support_ = int(
            fitted_values.size
        )

        if fitted_values.size >= 2:
            self.stl_mean_ = float(
                fitted_values.mean()
            )
            self.stl_std_ = float(
                fitted_values.std(ddof=0)
            )
        else:
            self.stl_mean_ = 0.0
            self.stl_std_ = 1.0

        self.stl_scaler_active_ = bool(
            fitted_values.size >= 2
            and self.stl_std_ > 1e-8
        )

        if not self.stl_scaler_active_:
            self.stl_std_ = 1.0

        # Target scaling is learned from the same legal training population.
        target = pd.to_numeric(
            training_frame[
                "target_observed_value"
            ],
            errors="raise",
        ).to_numpy(dtype=np.float64)

        if not np.isfinite(target).all():
            raise ValueError(
                "Training target contains non-finite values."
            )

        self.target_mean_ = float(
            target.mean()
        )
        self.target_std_ = float(
            target.std(ddof=0)
        )

        if self.target_std_ <= 1e-8:
            raise ValueError(
                "Training target has effectively zero variance."
            )

        self.is_fitted_ = True

        return self

    def _require_fitted(self):
        if not self.is_fitted_:
            raise RuntimeError(
                "Sequence preprocessor must be fitted before transform."
            )

    def transform_sequence(self, values, masks):
        """Standardize observed mobility and append observation masks."""
        self._require_fitted()

        values = np.asarray(values, dtype=np.float32)
        masks = np.asarray(masks, dtype=bool)

        standardized = (
            values
            - self.sequence_means_.astype(np.float32)
        ) / self.sequence_stds_.astype(np.float32)

        # Missing observations become zero only AFTER standardization,
        # so zero means the training mean. The appended mask tells the
        # model whether that zero was observed or merely a placeholder.
        standardized = np.where(
            masks & np.isfinite(standardized),
            standardized,
            0.0,
        )

        return np.concatenate(
            [
                standardized.astype(np.float32),
                masks.astype(np.float32),
            ],
            axis=-1,
        )

    def transform_context(self, context_frame):
        """Apply the fitted 52-value context transformation."""
        self._require_fitted()

        context_frame = context_frame[
            self.feature_order
        ].copy()

        continuous = (
            context_frame[self.continuous_features]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float64)
        )

        continuous = np.where(
            np.isfinite(continuous),
            continuous,
            self.context_medians_[None, :],
        )
        continuous = (
            continuous - self.context_means_
        ) / self.context_stds_

        stl_enabled = pd.to_numeric(
            context_frame["weekly_stl_enabled"],
            errors="raise",
        ).to_numpy(dtype=float)

        stl_missing = pd.to_numeric(
            context_frame[
                "missing_mask__weekly_stl_seasonal_offset"
            ],
            errors="raise",
        ).to_numpy(dtype=float)

        raw_stl = pd.to_numeric(
            context_frame[self.stl_feature],
            errors="coerce",
        ).to_numpy(dtype=float)

        # Default zero preserves both disabled STL and unavailable STL as
        # neutral numeric states; the separate indicators distinguish them.
        transformed_stl = np.zeros(len(context_frame), dtype=np.float32)
        fitted_stl = (
            np.isclose(stl_enabled, 1.0)
            & np.isclose(stl_missing, 0.0)
            & np.isfinite(raw_stl)
        )

        if self.stl_scaler_active_:
            transformed_stl[fitted_stl] = (
                raw_stl[fitted_stl] - self.stl_mean_
            ) / self.stl_std_

        passthrough = (
            context_frame[self.passthrough_features]
            .apply(pd.to_numeric, errors="raise")
            .to_numpy(dtype=np.float32)
        )

        if not np.isfinite(passthrough).all():
            raise ValueError(
                "Passthrough context contains non-finite values."
            )

        # Reassemble by the authoritative feature order rather than by role,
        # preserving the exact 52-position input contract.
        transformed_by_feature = {}

        for index, feature in enumerate(self.continuous_features):
            transformed_by_feature[feature] = (
                continuous[:, index].astype(np.float32)
            )

        transformed_by_feature[self.stl_feature] = transformed_stl

        for index, feature in enumerate(self.passthrough_features):
            transformed_by_feature[feature] = passthrough[:, index]

        transformed = np.column_stack([
            transformed_by_feature[feature]
            for feature in self.feature_order
        ]).astype(np.float32)

        return transformed

    def transform_target(self, values):
        """Standardize native-unit targets with training-only moments."""
        self._require_fitted()
        values = np.asarray(values, dtype=np.float32)

        return (
            (values - self.target_mean_) / self.target_std_
        ).astype(np.float32)

    def inverse_target(self, values):
        """Return standardized predictions to the target's native units."""
        self._require_fitted()
        values = np.asarray(values, dtype=np.float32)

        return (
            values * self.target_std_ + self.target_mean_
        ).astype(np.float32)


class StreamingSequenceDataset(Dataset):
    """Stream raw history windows from the shared dense state per mini-batch."""

    def __init__(
        self,
        frame,
        track,
        target_metric,
        lookback_steps,
        preprocessor,
    ):
        self.frame = frame.reset_index(drop=True)
        self.track = track
        self.target_metric = target_metric
        self.lookback_steps = int(lookback_steps)
        self.preprocessor = preprocessor

        (
            self.zone_positions,
            self.origin_positions,
        ) = sequence_frame_positions(
            self.frame,
            track,
            self.lookback_steps,
        )

        # Context is much smaller than the temporal-history tensor, so it
        # can be transformed once while history remains streamed by batch.
        context_frame = derive_sequence_context_frame(self.frame)

        self.context = preprocessor.transform_context(
            context_frame
        )

        target_raw = pd.to_numeric(
            self.frame["target_observed_value"],
            errors="raise",
        ).to_numpy(dtype=np.float32)

        self.target = preprocessor.transform_target(
            target_raw
        )

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        # Return only the row position. The collate function performs the
        # expensive vectorized history extraction for the complete batch.
        return int(index)

    def collate_batch(self, indices):
        indices = np.asarray(indices, dtype=np.int64)

        values, masks = raw_sequence_batch(
            self.zone_positions,
            self.origin_positions,
            indices,
            self.target_metric,
            self.lookback_steps,
        )

        sequence = self.preprocessor.transform_sequence(
            values,
            masks,
        )

        return (
            torch.from_numpy(sequence),
            torch.from_numpy(self.context[indices]),
            torch.from_numpy(
                self.zone_positions[indices].astype(
                    np.int64,
                    copy=False,
                )
            ),
            torch.from_numpy(self.target[indices]),
        )


def make_sequence_loader(
    frame,
    track,
    target_metric,
    lookback_steps,
    preprocessor,
    batch_size,
    shuffle,
):
    """Build one mini-batch loader without materializing all history windows."""
    dataset = StreamingSequenceDataset(
        frame=frame,
        track=track,
        target_metric=target_metric,
        lookback_steps=lookback_steps,
        preprocessor=preprocessor,
    )

    # num_workers=0 keeps the large shared NumPy history in one process and
    # avoids accidental copies of the dense state on constrained runtimes.
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        num_workers=0,
        collate_fn=dataset.collate_batch,
    )


print(
    "PASS — shared 52-context preprocessing and streamed-history "
    "helpers are ready.",
    flush=True,
)

PASS — shared 52-context preprocessing and streamed-history helpers are ready.


**4.6.1.6.2E — Reconstruct the frozen MLP and patch-based Transformer training lifecycles.**

The two sequence families share the same legal data and preprocessing contract but use different history encoders.

The Neural champion flattens its 35 × 10 history and passes it through the target-specific two-layer MLP selected in 4.5.1.

The Transformer groups its 70 × 10 history into overlapping temporal patches, projects each patch into the frozen model width, applies encoder self-attention across the historical tokens, mean-pools the encoded history, and combines that state with the 52 supporting inputs and Taxi Zone embedding.

Both families use the same training lifecycle: a historical selector chooses the training duration, that selector is discarded, and a fresh model is trained from scratch for exactly the selected number of epochs.

In [52]:
# ---------------------------------------------------------------------
# 4.6.1.6.2E — Frozen Neural / Transformer model and training helpers
# ---------------------------------------------------------------------

def seed_sequence_holdout(runtime_unit_id, phase):
    """Give each logical training phase a stable stochastic identity."""
    # Include the runtime contract, monthly unit, and phase so a restarted
    # selector or final refit receives the same seed without forcing every
    # independent fit to share one stochastic trajectory.
    token = (
        f"696|{HOLDOUT_RUNTIME_CONTRACT_ID}|"
        f"{runtime_unit_id}|{phase}"
    )
    digest = hashlib.sha256(token.encode("utf-8")).digest()
    seed = int.from_bytes(digest[:4], "big") % (2**31 - 1)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    return seed


def clear_sequence_device_memory():
    """Release disposable model state between independent monthly fits."""
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class FrozenForecastHead(nn.Module):
    """Fuse learned history, supporting context, and Taxi Zone identity."""

    def __init__(
        self,
        sequence_dim,
        context_dim,
        zone_count,
        dropout,
        zone_dim,
    ):
        super().__init__()

        # Taxi Zone remains a learned categorical embedding rather than an
        # arbitrary numeric ID, matching the sequence-family design.
        self.zone_embedding = nn.Embedding(
            int(zone_count),
            int(zone_dim),
        )

        fused_dim = (
            int(sequence_dim)
            + int(context_dim)
            + int(zone_dim)
        )

        self.head = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.ReLU(),
            nn.Dropout(float(dropout)),
            nn.Linear(64, 1),
        )

    def forward(self, sequence_state, context, zone_index):
        zone_state = self.zone_embedding(zone_index)

        combined = torch.cat(
            [sequence_state, context, zone_state],
            dim=1,
        )

        return self.head(combined).squeeze(-1)


class FrozenLagWindowMLP(nn.Module):
    """Target-specific 35-step MLP champion from the Neural family."""

    def __init__(
        self,
        sequence_input_dim,
        context_dim,
        zone_count,
        spec,
    ):
        super().__init__()

        # The MLP treats the complete 35-step value+mask history as one
        # ordered feature vector, exactly as selected upstream.
        flattened_dim = (
            NEURAL_HOLDOUT_LOOKBACK
            * int(sequence_input_dim)
        )

        width_1 = int(spec["sequence_width_1"])
        width_2 = int(spec["sequence_width_2"])
        dropout = float(spec["dropout"])

        self.sequence_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_dim, width_1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(width_1, width_2),
            nn.ReLU(),
        )

        self.forecast_head = FrozenForecastHead(
            sequence_dim=width_2,
            context_dim=context_dim,
            zone_count=zone_count,
            dropout=dropout,
            zone_dim=int(spec["zone_embedding_dim"]),
        )

    def forward(self, sequence, context, zone_index):
        history_state = self.sequence_encoder(sequence)

        return self.forecast_head(
            history_state,
            context,
            zone_index,
        )


class FrozenPatchTransformer(nn.Module):
    """Patch-based encoder-only Transformer used by the selected family."""

    def __init__(
        self,
        sequence_input_dim,
        context_dim,
        zone_count,
        representation,
        profile,
    ):
        super().__init__()

        lookback = int(representation["lookback_steps"])
        patch_length = int(representation["patch_length"])
        patch_stride = int(representation["patch_stride"])

        # The frozen representation must tile deterministically under its
        # published overlap rule; otherwise token count would silently change.
        if (lookback - patch_length) % patch_stride != 0:
            raise ValueError(
                "Transformer representation does not exactly tile its "
                "history under the frozen stride."
            )

        self.patch_length = patch_length
        self.patch_stride = patch_stride
        self.patch_count = (
            1 + (lookback - patch_length) // patch_stride
        )

        d_model = int(profile["d_model"])

        # Each patch contains every value and observation-mask channel.
        # Projection converts that local history block into one Transformer token.
        self.patch_projection = nn.Linear(
            patch_length * int(sequence_input_dim),
            d_model,
        )

        self.position = nn.Parameter(
            torch.zeros(1, self.patch_count, d_model)
        )
        nn.init.normal_(self.position, mean=0.0, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=int(profile["n_heads"]),
            dim_feedforward=int(profile["d_ff"]),
            dropout=float(profile["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=int(profile["n_layers"]),
        )

        self.history_norm = nn.LayerNorm(d_model)

        self.zone_embedding = nn.Embedding(
            int(zone_count),
            int(profile["zone_embedding_dim"]),
        )

        fused_dim = (
            d_model
            + int(context_dim)
            + int(profile["zone_embedding_dim"])
        )

        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, d_model),
            nn.GELU(),
            nn.Dropout(float(profile["dropout"])),
            nn.Linear(d_model, 1),
        )

    def forward(self, sequence, context, zone_index):
        # Convert [batch, history, channels] into overlapping temporal
        # patches while keeping all channels inside each patch token.
        patches = sequence.unfold(
            dimension=1,
            size=self.patch_length,
            step=self.patch_stride,
        ).permute(0, 1, 3, 2)

        patches = patches.contiguous().view(
            sequence.shape[0],
            self.patch_count,
            -1,
        )

        tokens = self.patch_projection(patches)
        tokens = tokens + self.position

        encoded = self.encoder(tokens)

        # Mean pooling gives one history representation for the forecast
        # head while allowing self-attention to mix information across patches.
        history_state = self.history_norm(
            encoded.mean(dim=1)
        )

        zone_state = self.zone_embedding(zone_index)

        fused = torch.cat(
            [history_state, context, zone_state],
            dim=1,
        )

        return self.head(fused).squeeze(-1)


def resolve_neural_holdout_spec(track, metric, horizon):
    """Recover the exact selected MLP profile for one Neural champion."""
    selected = neural_final_registry_df.loc[
        neural_final_registry_df["modeling_track"].eq(track)
        & neural_final_registry_df["metric"].eq(metric)
        & neural_final_registry_df["horizon"].astype(int).eq(int(horizon))
    ]

    if len(selected) != 1:
        raise RuntimeError(
            f"Expected one Neural registry row for "
            f"{track} · {metric} · h={horizon}; found {len(selected)}."
        )

    profile_id = str(selected.iloc[0]["profile_id"])

    definition = neural_model_definition_df.loc[
        neural_model_definition_df["profile_id"].astype(str).eq(profile_id)
    ]

    if len(definition) != 1:
        raise RuntimeError(
            f"Neural profile {profile_id} does not resolve uniquely."
        )

    return definition.iloc[0].to_dict()


def resolve_transformer_holdout_spec(track, metric, horizon):
    """Recover the selected representation/profile for one Transformer job."""
    selected = transformer_final_registry_df.loc[
        transformer_final_registry_df["modeling_track"].eq(track)
        & transformer_final_registry_df["metric"].eq(metric)
        & transformer_final_registry_df["horizon"].astype(int).eq(int(horizon))
    ]

    if len(selected) != 1:
        raise RuntimeError(
            f"Expected one Transformer registry row for "
            f"{track} · {metric} · h={horizon}; found {len(selected)}."
        )

    row = selected.iloc[0]
    representation_id = str(row["representation_id"])
    profile_id = str(row["profile_id"])

    if representation_id not in TRANSFORMER_HOLDOUT_REPRESENTATIONS:
        raise RuntimeError(
            f"Unsupported frozen Transformer representation: "
            f"{representation_id}"
        )

    if profile_id not in TRANSFORMER_HOLDOUT_PROFILES:
        raise RuntimeError(
            f"Unsupported frozen Transformer profile: {profile_id}"
        )

    return (
        TRANSFORMER_HOLDOUT_REPRESENTATIONS[representation_id],
        TRANSFORMER_HOLDOUT_PROFILES[profile_id],
    )


def train_sequence_epoch(model, loader, optimizer):
    """Run one Huber-loss training epoch and return row-weighted mean loss."""
    criterion = nn.HuberLoss()
    model.train()

    weighted_loss = 0.0
    rows = 0

    for sequence, context, zones, target in loader:
        sequence = sequence.to(SEQUENCE_HOLDOUT_DEVICE)
        context = context.to(SEQUENCE_HOLDOUT_DEVICE)
        zones = zones.to(SEQUENCE_HOLDOUT_DEVICE)
        target = target.to(SEQUENCE_HOLDOUT_DEVICE)

        optimizer.zero_grad(set_to_none=True)

        prediction = model(
            sequence,
            context,
            zones,
        )
        loss = criterion(prediction, target)
        loss.backward()

        # Gradient clipping is part of the frozen sequence-family training
        # policy and protects individual mini-batches from unstable updates.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            SEQUENCE_GRAD_CLIP_NORM,
        )

        optimizer.step()

        batch_rows = len(target)
        weighted_loss += float(loss.item()) * batch_rows
        rows += batch_rows

    return weighted_loss / rows


def standardized_sequence_mae(model, loader):
    """Evaluate selector MAE in the standardized target space."""
    model.eval()

    absolute_error_sum = 0.0
    rows = 0

    with torch.no_grad():
        for sequence, context, zones, target in loader:
            sequence = sequence.to(SEQUENCE_HOLDOUT_DEVICE)
            context = context.to(SEQUENCE_HOLDOUT_DEVICE)
            zones = zones.to(SEQUENCE_HOLDOUT_DEVICE)
            target = target.to(SEQUENCE_HOLDOUT_DEVICE)

            prediction = model(
                sequence,
                context,
                zones,
            )

            absolute_error_sum += float(
                torch.abs(prediction - target).sum().item()
            )
            rows += len(target)

    return absolute_error_sum / rows


def select_sequence_training_epoch(
    model,
    train_loader,
    validation_loader,
    learning_rate,
    weight_decay,
    max_epochs,
    patience,
    min_delta,
):
    """Select training duration using only the historical inner-validation tail."""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(learning_rate),
        weight_decay=float(weight_decay),
    )

    best_epoch = None
    best_validation_mae = np.inf
    stale_epochs = 0
    curve_records = []

    for epoch in range(1, int(max_epochs) + 1):
        train_loss = train_sequence_epoch(
            model,
            train_loader,
            optimizer,
        )

        validation_mae = standardized_sequence_mae(
            model,
            validation_loader,
        )

        improved = (
            validation_mae
            < best_validation_mae - float(min_delta)
        )

        if improved:
            best_epoch = epoch
            best_validation_mae = validation_mae
            stale_epochs = 0
        else:
            stale_epochs += 1

        curve_records.append({
            "epoch": epoch,
            "training_huber_loss": train_loss,
            "validation_standardized_mae": validation_mae,
            "is_best_epoch": bool(improved),
        })

        # Only the historical selector sees this early-stopping decision.
        # The later final model receives the chosen epoch count as fixed.
        if stale_epochs >= int(patience):
            break

    if best_epoch is None:
        raise RuntimeError(
            "Sequence-model selector did not choose a training epoch."
        )

    return int(best_epoch), pd.DataFrame(curve_records)


def fit_sequence_fixed_epochs(
    model,
    train_loader,
    learning_rate,
    weight_decay,
    epochs,
):
    """Train a fresh final model for exactly the selector-chosen duration."""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(learning_rate),
        weight_decay=float(weight_decay),
    )

    # No second early-stopping rule is allowed here. The historical selector
    # already chose the duration, so the final refit follows that choice exactly.
    for _ in range(int(epochs)):
        train_sequence_epoch(
            model,
            train_loader,
            optimizer,
        )

    return model


def predict_sequence_model(model, loader, preprocessor):
    """Return native-unit predictions for one monthly scoring slice."""
    model.eval()
    batches = []

    with torch.no_grad():
        for sequence, context, zones, _ in loader:
            prediction = model(
                sequence.to(SEQUENCE_HOLDOUT_DEVICE),
                context.to(SEQUENCE_HOLDOUT_DEVICE),
                zones.to(SEQUENCE_HOLDOUT_DEVICE),
            )

            batches.append(
                prediction.cpu().numpy()
            )

    if not batches:
        raise RuntimeError(
            "Sequence-model scoring loader produced no predictions."
        )

    standardized = np.concatenate(batches)

    # Scoring is performed in native mobility units, not standardized units.
    return preprocessor.inverse_target(
        standardized
    )


print(
    "PASS — frozen MLP, patch-Transformer, selector, and fresh-refit "
    "helpers are ready.",
    flush=True,
)

PASS — frozen MLP, patch-Transformer, selector, and fresh-refit helpers are ready.


**4.6.1.6.2F — Build and execute one legal monthly sequence-model restart unit.**

A monthly restart unit owns exactly one forecast month for one Target × Horizon.

Its legal history is split twice for two different purposes:

1. the latest 28 historical days are withheld from the selector-training sample so they can choose the training duration; then
2. after that duration is fixed, preprocessing and model weights start again from scratch using **all** history legally observable by the monthly refit origin.

The final model then forecasts only the rows assigned to that monthly unit. Its predictions are written through the same durable final-holdout contract used by Tree, preserving raw predictions, nonnegative final predictions, model availability, benchmark availability, and physical forecast identity.

In [53]:
# ---------------------------------------------------------------------
# 4.6.1.6.2F — Build and execute one monthly sequence-model restart unit
# ---------------------------------------------------------------------

# Keep one filtered matrix per Target × Horizon in memory while its monthly
# units execute. Reusing this frame avoids rereading the same Parquet data
# for January, February, and March.
SEQUENCE_JOB_CACHE = {}


def load_sequence_holdout_job(track, metric, horizon, lookback_steps):
    """
    Load one champion job once and retain every row needed across its three
    monthly refits.

    Predicate pushdown limits the read to one Track × Target × Horizon while
    retaining the complete published context schema.
    """
    cache_key = (
        track,
        metric,
        int(horizon),
        int(lookback_steps),
    )

    if cache_key in SEQUENCE_JOB_CACHE:
        return SEQUENCE_JOB_CACHE[cache_key]

    matrix_path = FORECASTING_MATRIX_PATHS[track]

    matrix_df = pd.read_parquet(
        matrix_path,
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
            ("native_track_eligible", "==", True),
        ],
    )

    matrix_df["target_date"] = pd.to_datetime(
        matrix_df["target_date"]
    ).dt.normalize()
    matrix_df["origin_date"] = pd.to_datetime(
        matrix_df["origin_date"]
    ).dt.normalize()

    # A sequence-model row is usable only after the complete history window
    # lies inside the modeling track's legal history. This reproduces the
    # upstream 4.5.1 / 4.5.2 eligibility rule rather than allowing an early
    # Post-CP forecast to reach backward into pre-policy observations.
    first_legal_window_origin = (
        SEQUENCE_TRACK_HISTORY_START[track]
        + int(lookback_steps)
        - 1
    )

    window_eligible = pd.to_numeric(
        matrix_df["origin_observation_sequence_id"],
        errors="raise",
    ).astype(int).ge(first_legal_window_origin)

    # Training and scoring also require an observed target. Apply both
    # eligibility conditions before constructing any temporal window.
    labeled = (
        matrix_df["target_label_available"]
        .fillna(False)
        .astype(bool)
    )

    target = pd.to_numeric(
        matrix_df["target_observed_value"],
        errors="coerce",
    )

    matrix_df = matrix_df.loc[
        window_eligible
        & labeled
        & np.isfinite(target)
    ].copy()

    if matrix_df.empty:
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: no rows remain after "
            "legal sequence-history eligibility."
        )

    # This is now an invariant check rather than the mechanism that discovers
    # and removes early rows. Every retained sequence must fit wholly within
    # the track's legal history boundary.
    sequence_frame_positions(
        matrix_df,
        track,
        lookback_steps,
    )

    SEQUENCE_JOB_CACHE[cache_key] = matrix_df.reset_index(
        drop=True
    )

    return SEQUENCE_JOB_CACHE[cache_key]


def attach_sequence_holdout_benchmark(score_df, track, metric, horizon):
    """Attach the frozen weekly benchmark without shrinking model support."""
    baseline_path = BASELINE_PREDICTION_PATHS[track]

    baseline_columns = [
        "stage",
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
        "metric",
        "target_date",
        "target_label_available",
        HOLDOUT_BASELINE_COLUMN,
    ]

    baseline_df = pd.read_parquet(
        baseline_path,
        columns=baseline_columns,
        filters=[
            ("stage", "==", "final_holdout"),
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
        ],
    )

    baseline_df["target_date"] = pd.to_datetime(
        baseline_df["target_date"]
    ).dt.normalize()

    baseline_df = baseline_df.loc[
        baseline_df["target_label_available"]
        .fillna(False)
        .astype(bool)
    ].rename(
        columns={
            HOLDOUT_BASELINE_COLUMN: "benchmark_prediction",
        }
    )

    join_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
        "metric",
        "target_date",
    ]

    if baseline_df.duplicated(join_keys).any():
        raise RuntimeError(
            "Frozen benchmark contains duplicate physical forecast keys."
        )

    # LEFT JOIN is intentional: model-support rows remain visible even if
    # benchmark support is absent, allowing later support auditing rather
    # than silently shrinking the advanced model's output population.
    return score_df.merge(
        baseline_df[join_keys + ["benchmark_prediction"]],
        on=join_keys,
        how="left",
        validate="one_to_one",
    )


def build_sequence_month_splits(manifest_row):
    """Resolve selector, final-training, and owned scoring rows for one unit."""
    family = str(manifest_row["champion_family"])
    track = str(manifest_row["modeling_track"])
    metric = str(manifest_row["metric"])
    horizon = int(manifest_row["horizon"])

    lookback_steps = (
        NEURAL_HOLDOUT_LOOKBACK
        if family == "neural"
        else TRANSFORMER_HOLDOUT_LOOKBACK
    )

    refit_origin = int(
        manifest_row["refit_origin_observation_sequence_id"]
    )

    next_refit_origin = (
        None
        if pd.isna(
            manifest_row[
                "next_refit_origin_observation_sequence_id"
            ]
        )
        else int(
            manifest_row[
                "next_refit_origin_observation_sequence_id"
            ]
        )
    )

    matrix_df = load_sequence_holdout_job(
        track,
        metric,
        horizon,
        lookback_steps,
    )

    target_ids = pd.to_numeric(
        matrix_df["target_observation_sequence_id"],
        errors="raise",
    ).astype(int)

    # The final training population contains every labeled target legally
    # observable by this monthly refit origin—never later information.
    legal_train_df = matrix_df.loc[
        target_ids.le(refit_origin)
    ].copy()

    if legal_train_df.empty:
        raise RuntimeError(
            f"{family} · {metric} · h={horizon}: no legal training rows."
        )

    # The latest 28 historical calendar days are used only to select the
    # training duration. They are not mixed into selector training.
    inner_end_date = legal_train_df["target_date"].max()
    inner_start_date = (
        inner_end_date
        - pd.Timedelta(
            days=SEQUENCE_INNER_VALIDATION_DAYS - 1
        )
    )

    selector_train_df = legal_train_df.loc[
        legal_train_df["target_date"].lt(inner_start_date)
    ].copy()

    selector_validation_df = legal_train_df.loc[
        legal_train_df["target_date"].between(
            inner_start_date,
            inner_end_date,
            inclusive="both",
        )
    ].copy()

    # Each restart unit owns forecasts from its refit origin until the next
    # monthly refit becomes available. This keeps monthly outputs disjoint.
    score_mask = (
        matrix_df["stage"].astype(str).eq("final_holdout")
        & pd.to_numeric(
            matrix_df["origin_observation_sequence_id"],
            errors="raise",
        ).astype(int).ge(refit_origin)
    )

    if next_refit_origin is not None:
        score_mask &= pd.to_numeric(
            matrix_df["origin_observation_sequence_id"],
            errors="raise",
        ).astype(int).lt(next_refit_origin)

    score_df = matrix_df.loc[score_mask].copy()

    if (
        selector_train_df.empty
        or selector_validation_df.empty
        or score_df.empty
    ):
        raise RuntimeError(
            f"{family} · {metric} · h={horizon}: "
            "monthly sequence split is incomplete."
        )

    if (
        selector_train_df["target_observation_sequence_id"].max()
        >= selector_validation_df[
            "target_observation_sequence_id"
        ].min()
    ):
        raise RuntimeError(
            "Selector training and historical validation are not "
            "strictly ordered."
        )

    score_df = attach_sequence_holdout_benchmark(
        score_df,
        track,
        metric,
        horizon,
    )

    return {
        "selector_train": selector_train_df,
        "selector_validation": selector_validation_df,
        "final_train": legal_train_df,
        "score": score_df,
        "lookback_steps": lookback_steps,
    }


def build_sequence_model(family, track, metric, horizon):
    """Instantiate one fresh model from the authoritative family contract."""
    context_dim = len(SEQUENCE_CONTEXT_FEATURE_ORDER)

    # Zone embeddings index the same dense Zone state used for history.
    zone_count = len(SEQUENCE_HISTORY_ZONE_IDS)

    # Five mobility values plus five corresponding observation masks.
    sequence_input_dim = len(FORECAST_TARGETS) * 2

    if family == "neural":
        spec = resolve_neural_holdout_spec(
            track,
            metric,
            horizon,
        )

        model = FrozenLagWindowMLP(
            sequence_input_dim=sequence_input_dim,
            context_dim=context_dim,
            zone_count=zone_count,
            spec=spec,
        )

        return model, {
            **spec,
            "max_epochs": NEURAL_SELECTOR_MAX_EPOCHS,
            "patience": NEURAL_SELECTOR_PATIENCE,
            "min_delta": NEURAL_SELECTOR_MIN_DELTA,
            "batch_size": NEURAL_HOLDOUT_BATCH_SIZE,
        }

    if family == "transformer":
        representation, profile = (
            resolve_transformer_holdout_spec(
                track,
                metric,
                horizon,
            )
        )

        model = FrozenPatchTransformer(
            sequence_input_dim=sequence_input_dim,
            context_dim=context_dim,
            zone_count=zone_count,
            representation=representation,
            profile=profile,
        )

        return model, {
            **profile,
            **representation,
            "max_epochs": TRANSFORMER_SELECTOR_MAX_EPOCHS,
            "patience": TRANSFORMER_SELECTOR_PATIENCE,
            "min_delta": TRANSFORMER_SELECTOR_MIN_DELTA,
            "batch_size": TRANSFORMER_HOLDOUT_BATCH_SIZE,
        }

    raise ValueError(
        f"Unsupported sequence-model family: {family}"
    )


def persist_sequence_holdout_unit(
    manifest_row,
    score_df,
    prediction_raw,
    selected_epoch,
    selector_curve_df,
    runtime_seconds,
):
    """Publish one completed sequence-model restart unit atomically."""
    prediction_raw = np.asarray(
        prediction_raw,
        dtype=float,
    )

    prediction_available = np.isfinite(prediction_raw)

    # Preserve raw model behavior for diagnostics, then apply the same
    # nonnegative production rule used throughout the forecasting project.
    prediction_final = np.where(
        prediction_available,
        np.maximum(prediction_raw, 0.0),
        np.nan,
    )

    benchmark_prediction = pd.to_numeric(
        score_df["benchmark_prediction"],
        errors="coerce",
    ).to_numpy(dtype=float)

    prediction_df = pd.DataFrame({
        "runtime_unit_id":
            str(manifest_row["runtime_unit_id"]),
        "holdout_runtime_contract_id":
            HOLDOUT_RUNTIME_CONTRACT_ID,
        "champion_family":
            str(manifest_row["champion_family"]),
        "modeling_track":
            str(manifest_row["modeling_track"]),
        "metric":
            str(manifest_row["metric"]),
        "horizon":
            int(manifest_row["horizon"]),
        "forecast_sequence_id":
            score_df["forecast_sequence_id"].astype(str).to_numpy(),
        "taxi_zone_id":
            pd.to_numeric(
                score_df["taxi_zone_id"],
                errors="raise",
            ).astype(int).to_numpy(),
        "origin_observation_sequence_id":
            pd.to_numeric(
                score_df["origin_observation_sequence_id"],
                errors="raise",
            ).astype(int).to_numpy(),
        "target_observation_sequence_id":
            pd.to_numeric(
                score_df["target_observation_sequence_id"],
                errors="raise",
            ).astype(int).to_numpy(),
        "target_date":
            pd.to_datetime(
                score_df["target_date"]
            ).dt.normalize().to_numpy(),
        "actual":
            pd.to_numeric(
                score_df["target_observed_value"],
                errors="raise",
            ).to_numpy(dtype=float),
        "benchmark_prediction":
            benchmark_prediction,
        "prediction_raw":
            prediction_raw,
        "prediction_final":
            prediction_final,
        "model_prediction_available":
            prediction_available,
        "benchmark_prediction_available":
            np.isfinite(benchmark_prediction),
    })

    if len(prediction_df) != len(score_df):
        raise RuntimeError(
            "Sequence prediction output changed row count."
        )

    physical_key = [
        "metric",
        "horizon",
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]

    if prediction_df.duplicated(physical_key).any():
        raise RuntimeError(
            "Sequence restart unit produced duplicate physical forecast rows."
        )

    available = prediction_df["model_prediction_available"]

    if not np.allclose(
        prediction_df.loc[available, "prediction_final"],
        np.maximum(
            prediction_df.loc[available, "prediction_raw"],
            0.0,
        ),
        atol=OUTPUT_RULE_TOLERANCE,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Sequence predictions violate the frozen nonnegative output rule."
        )

    prediction_path = Path(
        manifest_row["prediction_path"]
    )
    audit_path = Path(
        manifest_row["audit_path"]
    )

    # Publish predictions first. A success audit is written only after the
    # durable row-level artifact exists, making restart detection conservative.
    atomic_write_parquet(
        prediction_df,
        prediction_path,
    )

    audit_df = pd.DataFrame([{
        "runtime_unit_id":
            str(manifest_row["runtime_unit_id"]),
        "holdout_runtime_contract_id":
            HOLDOUT_RUNTIME_CONTRACT_ID,
        "champion_family":
            str(manifest_row["champion_family"]),
        "modeling_track":
            str(manifest_row["modeling_track"]),
        "metric":
            str(manifest_row["metric"]),
        "horizon":
            int(manifest_row["horizon"]),
        "refit_month":
            str(manifest_row["refit_month"]),
        "selected_epoch":
            int(selected_epoch),
        "selector_epochs_run":
            int(len(selector_curve_df)),
        "score_rows":
            int(len(prediction_df)),
        "model_available_rows":
            int(available.sum()),
        "runtime_seconds":
            float(runtime_seconds),
        "unit_complete":
            True,
    }])

    atomic_write_parquet(
        audit_df,
        audit_path,
    )

    return audit_df.iloc[0].to_dict()


def execute_sequence_holdout_unit(manifest_row):
    """Run selector → fresh final refit → monthly forecast for one unit."""
    family = str(manifest_row["champion_family"])
    track = str(manifest_row["modeling_track"])
    metric = str(manifest_row["metric"])
    horizon = int(manifest_row["horizon"])

    if family not in {"neural", "transformer"}:
        raise ValueError(
            f"Sequence executor cannot run family {family}."
        )

    started = time.perf_counter()

    splits = build_sequence_month_splits(
        manifest_row
    )

    # -------------------------------------------------------------
    # Historical selector
    # -------------------------------------------------------------
    # Preprocessing is fitted ONLY on selector-training history. The held-out
    # historical tail chooses duration without influencing fitted scalers.

    selector_preprocessor = FrozenSequencePreprocessor(
        transformer_context_contract_df
    ).fit(
        training_frame=splits["selector_train"],
        track=track,
        target_metric=metric,
        lookback_steps=splits["lookback_steps"],
    )

    selector_model, model_contract = build_sequence_model(
        family,
        track,
        metric,
        horizon,
    )

    seed_sequence_holdout(
        manifest_row["runtime_unit_id"],
        "selector",
    )

    selector_model = selector_model.to(
        SEQUENCE_HOLDOUT_DEVICE
    )

    selector_train_loader = make_sequence_loader(
        splits["selector_train"],
        track,
        metric,
        splits["lookback_steps"],
        selector_preprocessor,
        model_contract["batch_size"],
        shuffle=True,
    )

    selector_validation_loader = make_sequence_loader(
        splits["selector_validation"],
        track,
        metric,
        splits["lookback_steps"],
        selector_preprocessor,
        model_contract["batch_size"],
        shuffle=False,
    )

    selected_epoch, selector_curve_df = (
        select_sequence_training_epoch(
            model=selector_model,
            train_loader=selector_train_loader,
            validation_loader=selector_validation_loader,
            learning_rate=model_contract["learning_rate"],
            weight_decay=model_contract["weight_decay"],
            max_epochs=model_contract["max_epochs"],
            patience=model_contract["patience"],
            min_delta=model_contract["min_delta"],
        )
    )

    # The selector is intentionally discarded. Reusing its fitted weights or
    # preprocessing would contaminate the fresh production-style final refit.
    del selector_model
    del selector_train_loader
    del selector_validation_loader
    del selector_preprocessor
    clear_sequence_device_memory()

    # -------------------------------------------------------------
    # Fresh final model on all legally observable history
    # -------------------------------------------------------------
    # After the epoch count is frozen, preprocessing is relearned on all
    # legally available history and the model starts from fresh weights.

    final_preprocessor = FrozenSequencePreprocessor(
        transformer_context_contract_df
    ).fit(
        training_frame=splits["final_train"],
        track=track,
        target_metric=metric,
        lookback_steps=splits["lookback_steps"],
    )

    final_model, final_contract = build_sequence_model(
        family,
        track,
        metric,
        horizon,
    )

    seed_sequence_holdout(
        manifest_row["runtime_unit_id"],
        "final",
    )

    final_model = final_model.to(
        SEQUENCE_HOLDOUT_DEVICE
    )

    final_train_loader = make_sequence_loader(
        splits["final_train"],
        track,
        metric,
        splits["lookback_steps"],
        final_preprocessor,
        final_contract["batch_size"],
        shuffle=True,
    )

    final_model = fit_sequence_fixed_epochs(
        model=final_model,
        train_loader=final_train_loader,
        learning_rate=final_contract["learning_rate"],
        weight_decay=final_contract["weight_decay"],
        epochs=selected_epoch,
    )

    # The scoring loader contains only this monthly restart unit's assigned
    # final-holdout rows. No later holdout row influences model fitting.
    score_loader = make_sequence_loader(
        splits["score"],
        track,
        metric,
        splits["lookback_steps"],
        final_preprocessor,
        final_contract["batch_size"],
        shuffle=False,
    )

    prediction_raw = predict_sequence_model(
        final_model,
        score_loader,
        final_preprocessor,
    )

    runtime_seconds = time.perf_counter() - started

    result = persist_sequence_holdout_unit(
        manifest_row=manifest_row,
        score_df=splits["score"],
        prediction_raw=prediction_raw,
        selected_epoch=selected_epoch,
        selector_curve_df=selector_curve_df,
        runtime_seconds=runtime_seconds,
    )

    # Monthly fits are independent. Free the model before the next unit so
    # memory does not grow across the 21 sequence-model restart units.
    del final_model
    del final_train_loader
    del score_loader
    del final_preprocessor
    clear_sequence_device_memory()

    return result


print(
    "PASS — monthly sequence-model execution unit is fully defined.",
    flush=True,
)

PASS — monthly sequence-model execution unit is fully defined.


### 4.6.1.6.3 Execute Sequence Champions and Reconcile the Full System

With the shared sequence runtime frozen, the remaining work is execution and recovery. Neural and Transformer restart units are completed under their own saved contracts, then all three active champion families are reconciled into one complete 29-unit final-holdout forecasting system before any accuracy metric is computed.

Note:
new 6.2F currently has no substantive Findings cell and does not require one
merely for a self-explanatory readiness/status print.

**4.6.1.6.3A — Execute or recover the 9 Neural final-holdout units.**

Neural owns Taxi average speed at h=1, h=2, and h=5. Each job receives one fresh monthly refit in January, February, and March.

Every pending unit executes the frozen Neural lifecycle:

1. fit selector preprocessing on the earlier legal history;
2. choose the training duration from the latest 28 historical days;
3. discard the selector;
4. refit preprocessing on all legal history;
5. initialize a fresh target-specific MLP;
6. train for exactly the selected number of epochs; and
7. forecast only that month's assigned final-holdout rows.

Completed units are recovered from their durable prediction/audit pair rather than fitted again.

In [54]:
# ---------------------------------------------------------------------
# 4.6.1.6.3A — Execute or recover Neural final-holdout units
# ---------------------------------------------------------------------

# Refresh first so completed artifacts from an earlier interrupted run are
# recognized before deciding which expensive monthly units still need work.
holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

neural_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("neural")
    ]
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

if len(neural_manifest_df) != 9:
    raise RuntimeError(
        f"Expected 9 Neural restart units; found {len(neural_manifest_df)}."
    )

neural_recovered = int(
    neural_manifest_df["artifact_status"].eq("recovered").sum()
)
neural_pending = int(
    neural_manifest_df["artifact_status"].eq("pending").sum()
)

print(
    "Neural final-holdout restart state\n"
    f"  Recovered: {neural_recovered}/9\n"
    f"  Pending:   {neural_pending}/9\n"
    f"  RUN_NEURAL_HOLDOUT={RUN_NEURAL_HOLDOUT}",
    flush=True,
)

if neural_pending and RUN_NEURAL_HOLDOUT:
    # Execute only missing units. Each successful unit becomes durable before
    # control moves to the next one, making the long run naturally restartable.
    pending_neural_df = neural_manifest_df.loc[
        neural_manifest_df["artifact_status"].eq("pending")
    ].reset_index(drop=True)

    for position, manifest_row in enumerate(
        pending_neural_df.to_dict("records"),
        start=1,
    ):
        print(
            f"\nNeural holdout [{position}/{len(pending_neural_df)}] "
            f"{TARGET_LABELS.get(manifest_row['metric'], manifest_row['metric'])} "
            f"· h={int(manifest_row['horizon'])} "
            f"· {manifest_row['refit_month']}",
            flush=True,
        )

        result = execute_sequence_holdout_unit(
            manifest_row
        )

        print(
            f"  complete · selected epoch={result['selected_epoch']} "
            f"· score={result['score_rows']:,} "
            f"· runtime={result['runtime_seconds'] / 60:.1f} min",
            flush=True,
        )

elif neural_pending:
    print(
        "\nExecution is disabled; pending Neural units remain unchanged.",
        flush=True,
    )


# Re-audit from disk rather than trusting only the in-memory loop results.
# This is the same recovery path a restarted notebook will use.
holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

neural_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("neural")
    ]
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

display(
    neural_manifest_df[
        [
            "metric",
            "horizon",
            "refit_month",
            "artifact_status",
        ]
    ]
    .assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS)
    )[
        ["Target", "horizon", "refit_month", "artifact_status"]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "refit_month": "Month",
        "artifact_status": "Status",
    })
)

if (
    RUN_NEURAL_HOLDOUT
    and not neural_manifest_df["artifact_status"].eq("recovered").all()
):
    raise RuntimeError(
        "Neural execution was enabled, but at least one monthly unit "
        "did not produce a valid checkpoint pair."
    )

Neural final-holdout restart state
  Recovered: 9/9
  Pending:   0/9
  RUN_NEURAL_HOLDOUT=False


,Target,Horizon,Month,Status
0,Taxi average speed,1,2026-01,recovered
1,Taxi average speed,1,2026-02,recovered
2,Taxi average speed,1,2026-03,recovered
3,Taxi average speed,2,2026-01,recovered
4,Taxi average speed,2,2026-02,recovered
5,Taxi average speed,2,2026-03,recovered
6,Taxi average speed,5,2026-01,recovered
7,Taxi average speed,5,2026-02,recovered
8,Taxi average speed,5,2026-03,recovered


**Findings.** All **9 Neural final-holdout restart units** are complete and recoverable across Taxi average speed at h=1, h=2, and h=5.

The historical selectors choose training durations ranging from **3 to 20 epochs**, with a **median of 11 epochs**. Across the nine monthly units, Neural produces **238,956 final-holdout forecasts** with a total recorded runtime of approximately **9.4 minutes**.

All three Taxi-speed horizons therefore have complete monthly Neural forecast coverage under the frozen selector-then-fresh-refit contract.

**4.6.1.6.3B — Execute or recover the 12 Transformer final-holdout units.**

Transformer owns all three FHVHV average-speed horizons and Subway h=2. Each champion is refreshed monthly across the three-month final holdout.

Every pending unit uses the frozen 70-Daypart overlapping representation. Historical values and observation masks are grouped into 10-step patches with a 5-step stride, encoded with the target-specific Transformer profile, and fused with the 52 supporting inputs and Taxi Zone embedding.

Training follows the same leakage-safe selector-then-fresh-final lifecycle used to validate the family. Completed monthly units are recovered rather than retrained.

In [55]:
# ---------------------------------------------------------------------
# 4.6.1.6.3B — Execute or recover Transformer final-holdout units
# ---------------------------------------------------------------------

# Refresh the manifest so this cell resumes from whatever durable monthly
# Transformer artifacts already exist.
holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

transformer_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("transformer")
    ]
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

if len(transformer_manifest_df) != 12:
    raise RuntimeError(
        "Expected 12 Transformer restart units; "
        f"found {len(transformer_manifest_df)}."
    )

transformer_recovered = int(
    transformer_manifest_df["artifact_status"].eq("recovered").sum()
)
transformer_pending = int(
    transformer_manifest_df["artifact_status"].eq("pending").sum()
)

print(
    "Transformer final-holdout restart state\n"
    f"  Recovered: {transformer_recovered}/12\n"
    f"  Pending:   {transformer_pending}/12\n"
    f"  RUN_TRANSFORMER_HOLDOUT={RUN_TRANSFORMER_HOLDOUT}",
    flush=True,
)

if transformer_pending and RUN_TRANSFORMER_HOLDOUT:
    # Run only incomplete monthly units. Durable per-unit writes preserve
    # completed work if the long execution is interrupted.
    pending_transformer_df = transformer_manifest_df.loc[
        transformer_manifest_df["artifact_status"].eq("pending")
    ].reset_index(drop=True)

    for position, manifest_row in enumerate(
        pending_transformer_df.to_dict("records"),
        start=1,
    ):
        print(
            f"\nTransformer holdout "
            f"[{position}/{len(pending_transformer_df)}] "
            f"{TARGET_LABELS.get(manifest_row['metric'], manifest_row['metric'])} "
            f"· h={int(manifest_row['horizon'])} "
            f"· {manifest_row['refit_month']}",
            flush=True,
        )

        result = execute_sequence_holdout_unit(
            manifest_row
        )

        print(
            f"  complete · selected epoch={result['selected_epoch']} "
            f"· score={result['score_rows']:,} "
            f"· runtime={result['runtime_seconds'] / 60:.1f} min",
            flush=True,
        )

elif transformer_pending:
    print(
        "\nExecution is disabled; pending Transformer units remain unchanged.",
        flush=True,
    )


# Reconstruct completion state from durable files rather than assuming that
# successful function returns alone establish a recoverable experiment.
holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

transformer_manifest_df = (
    holdout_checkpoint_manifest_df.loc[
        holdout_checkpoint_manifest_df["champion_family"].eq("transformer")
    ]
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

display(
    transformer_manifest_df[
        [
            "metric",
            "horizon",
            "refit_month",
            "artifact_status",
        ]
    ]
    .assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS)
    )[
        ["Target", "horizon", "refit_month", "artifact_status"]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "refit_month": "Month",
        "artifact_status": "Status",
    })
)

if (
    RUN_TRANSFORMER_HOLDOUT
    and not transformer_manifest_df[
        "artifact_status"
    ].eq("recovered").all()
):
    raise RuntimeError(
        "Transformer execution was enabled, but at least one monthly "
        "unit did not produce a valid checkpoint pair."
    )

Transformer final-holdout restart state
  Recovered: 12/12
  Pending:   0/12
  RUN_TRANSFORMER_HOLDOUT=False


,Target,Horizon,Month,Status
0,FHVHV average speed,1,2026-01,recovered
1,FHVHV average speed,1,2026-02,recovered
2,FHVHV average speed,1,2026-03,recovered
3,FHVHV average speed,2,2026-01,recovered
4,FHVHV average speed,2,2026-02,recovered
5,FHVHV average speed,2,2026-03,recovered
6,FHVHV average speed,5,2026-01,recovered
7,FHVHV average speed,5,2026-02,recovered
8,FHVHV average speed,5,2026-03,recovered
9,Subway ridership,2,2026-01,recovered


**Findings.** All **12 Transformer final-holdout restart units** are complete and recoverable across FHVHV average speed at h=1, h=2, and h=5 and Subway ridership at h=2.

The historical selectors choose training durations ranging from **4 to 23 epochs**, with a **median of 13 epochs**. Across the twelve monthly units, Transformer produces **395,204 final-holdout forecasts** with a total recorded runtime of approximately **238.8 minutes**.

The Transformer branch therefore provides complete monthly forecast coverage for all four frozen champion jobs under the 70-Daypart overlapping-patch representation.

**4.6.1.6.3C — Require one complete 29-unit final-holdout forecast system.**

The holdout can be evaluated only after every frozen champion restart unit has a valid durable prediction/audit pair.

This reconciliation treats Tree, Neural, and Transformer identically: family-specific training is complete only when its outputs satisfy the shared runtime contract.

No accuracy metric is calculated until all 29 units are complete.

In [56]:
# ---------------------------------------------------------------------
# 4.6.1.6.3C — Reconcile all 29 final-holdout restart units
# ---------------------------------------------------------------------

# Re-read durable status for every family in one place before allowing any
# holdout metric to be computed. Partial family completion cannot silently
# produce a smaller or easier evaluation population.
holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

holdout_completion_df = (
    holdout_checkpoint_manifest_df.assign(
        Recovered=lambda frame:
            frame["artifact_status"].eq("recovered").astype(int),
        Pending=lambda frame:
            frame["artifact_status"].eq("pending").astype(int),
    )
    .groupby(
        ["champion_family", "Family"],
        as_index=False,
        observed=True,
    )
    .agg(**{
        "Restart units": ("runtime_unit_id", "size"),
        "Recovered": ("Recovered", "sum"),
        "Pending": ("Pending", "sum"),
    })
)

display(
    holdout_completion_df[
        ["Family", "Restart units", "Recovered", "Pending"]
    ]
)

# The 29-unit total is part of the frozen runtime contract:
# Tree 8 + Neural 9 + Transformer 12.
if len(holdout_checkpoint_manifest_df) != 29:
    raise RuntimeError(
        "Final-holdout manifest no longer contains 29 restart units."
    )

incomplete_holdout_df = holdout_checkpoint_manifest_df.loc[
    ~holdout_checkpoint_manifest_df[
        "artifact_status"
    ].eq("recovered")
]

if not incomplete_holdout_df.empty:
    raise RuntimeError(
        "Final-holdout forecasting is incomplete. "
        f"{len(incomplete_holdout_df)} of 29 restart units still lack "
        "a valid prediction/audit checkpoint pair."
    )

print(
    "PASS — all 29 final-holdout restart units are complete and "
    "recoverable.",
    flush=True,
)

,Family,Restart units,Recovered,Pending
0,Neural,9,9,0
1,Transformer,12,12,0
2,Tree,8,8,0


PASS — all 29 final-holdout restart units are complete and recoverable.


**Findings.** All **29 final-holdout restart units** are complete and recoverable: **8 Tree, 9 Neural, and 12 Transformer**.

Every frozen champion job now has a valid durable prediction/audit checkpoint pair. The complete 15-job forecasting system is therefore ready for unseen final-holdout evaluation.

### Section Summary

Section 6 completes the frozen champion forecasting workload without changing model selection. All **29 final-holdout restart units** are complete and recoverable: **8 Tree, 9 Neural, and 12 Transformer** units. The three active families preserve their source-native fitting lifecycles while producing one compatible row-level forecast contract, so unseen-period scoring can proceed only after the entire champion system is complete.

## 4.6.1.7 Evaluate the Frozen Champion System on Unseen Data

Model selection and final-holdout execution are now complete. This section asks the primary unseen-data question: whether the already-selected 15-job system retains forecasting value on January–March 2026 mobility.

The holdout remains evaluation data, not a second tournament. Champions stay frozen while the notebook assembles the common scoring surface, audits advanced-model coverage, scores holdout accuracy, and compares validation with unseen performance.

### 4.6.1.7.1 Assemble and Audit the Holdout Evaluation Surface

Before interpreting accuracy, the completed restart units must reproduce the exact predeclared holdout population. This subsection assembles the row-level champion forecasts and verifies that advanced-model coverage—not accidental row loss—defines what is being evaluated.

**4.6.1.7.1A — Assemble one row-level final-holdout surface from all 29 restart units.**

The family runners write independent restart-unit artifacts, but evaluation needs one physical forecast table.

This block concatenates those artifacts, verifies that every physical Target × Horizon × Zone × forecast row appears at most once, and constructs the deployable system prediction:

`advanced prediction when available; frozen weekly benchmark otherwise`.

Rows without a valid benchmark remain visible for coverage auditing but cannot enter the primary benchmark-relative scorecard.

In [57]:
# ---------------------------------------------------------------------
# 4.6.1.7.1A — Materialize the row-level champion holdout surface
# ---------------------------------------------------------------------

FINAL_HOLDOUT_CHAMPION_PREDICTION_PATH = (
    FINAL_DIR / "champion_final_holdout_predictions.parquet"
)
FINAL_HOLDOUT_SUPPORT_AUDIT_PATH = (
    FINAL_DIR / "champion_final_holdout_support_audit.parquet"
)

holdout_checkpoint_manifest_df = refresh_holdout_checkpoint_status()

if not holdout_checkpoint_manifest_df[
    "artifact_status"
].eq("recovered").all():
    raise RuntimeError(
        "All 29 restart units must be recovered before holdout assembly."
    )


# ---------------------------------------------------------------------
# 1) Read every durable monthly prediction artifact
# ---------------------------------------------------------------------

prediction_frames = []

for row in holdout_checkpoint_manifest_df.to_dict("records"):
    frame = pd.read_parquet(
        row["prediction_path"]
    )

    require_columns(
        frame,
        HOLDOUT_UNIT_PREDICTION_COLUMNS,
        f"holdout unit {row['runtime_unit_id']}",
    )

    prediction_frames.append(frame)

champion_holdout_prediction_df = pd.concat(
    prediction_frames,
    ignore_index=True,
)


# ---------------------------------------------------------------------
# 2) Prove that monthly ownership produced no duplicate physical forecasts
# ---------------------------------------------------------------------

physical_key = [
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

if champion_holdout_prediction_df.duplicated(
    physical_key
).any():
    duplicate_rows = champion_holdout_prediction_df.loc[
        champion_holdout_prediction_df.duplicated(
            physical_key,
            keep=False,
        ),
        physical_key,
    ]

    raise RuntimeError(
        "Champion holdout surface contains duplicate physical rows:\n"
        + duplicate_rows.head(20).to_string(index=False)
    )


# ---------------------------------------------------------------------
# 3) Construct the deployable system prediction
# ---------------------------------------------------------------------
# Selection used a complete forecasting-system interpretation: advanced
# model when available, otherwise the frozen weekly benchmark. The unseen
# holdout must preserve that exact denominator and fallback semantics.

actual = pd.to_numeric(
    champion_holdout_prediction_df["actual"],
    errors="coerce",
).to_numpy(dtype=float)

benchmark = pd.to_numeric(
    champion_holdout_prediction_df["benchmark_prediction"],
    errors="coerce",
).to_numpy(dtype=float)

advanced = pd.to_numeric(
    champion_holdout_prediction_df["prediction_final"],
    errors="coerce",
).to_numpy(dtype=float)

model_available = (
    champion_holdout_prediction_df[
        "model_prediction_available"
    ]
    .fillna(False)
    .astype(bool)
    .to_numpy()
    & np.isfinite(advanced)
)

benchmark_available = (
    champion_holdout_prediction_df[
        "benchmark_prediction_available"
    ]
    .fillna(False)
    .astype(bool)
    .to_numpy()
    & np.isfinite(benchmark)
)

champion_holdout_prediction_df[
    "model_prediction_available"
] = model_available
champion_holdout_prediction_df[
    "benchmark_prediction_available"
] = benchmark_available

champion_holdout_prediction_df[
    "used_benchmark_fallback"
] = ~model_available

champion_holdout_prediction_df[
    "system_prediction"
] = np.where(
    model_available,
    advanced,
    benchmark,
)

# Primary benchmark-relative evaluation requires both an observed outcome
# and a finite frozen benchmark. Rows outside that support remain in the
# assembled artifact for transparent coverage accounting.
champion_holdout_prediction_df[
    "primary_evaluation_eligible"
] = (
    np.isfinite(actual)
    & benchmark_available
)

eligible = champion_holdout_prediction_df[
    "primary_evaluation_eligible"
].to_numpy(dtype=bool)

if not np.isfinite(
    champion_holdout_prediction_df.loc[
        eligible,
        "system_prediction",
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "At least one benchmark-supported holdout row lacks a finite "
        "system prediction."
    )

# When the champion is unavailable, the system prediction must literally
# equal the benchmark—not merely have the same aggregate error later.
fallback_mask = (
    eligible
    & ~champion_holdout_prediction_df[
        "model_prediction_available"
    ].to_numpy(dtype=bool)
)

if not np.allclose(
    champion_holdout_prediction_df.loc[
        fallback_mask,
        "system_prediction",
    ],
    champion_holdout_prediction_df.loc[
        fallback_mask,
        "benchmark_prediction",
    ],
    atol=CROSS_ARTIFACT_ATOL,
    rtol=CROSS_ARTIFACT_RTOL,
):
    raise RuntimeError(
        "Benchmark fallback identity failed on the final holdout."
    )

atomic_write_parquet(
    champion_holdout_prediction_df,
    FINAL_HOLDOUT_CHAMPION_PREDICTION_PATH,
)


# ---------------------------------------------------------------------
# 4) Compare assembled support with the population audited in 5.1A
# ---------------------------------------------------------------------
# This prevents a training or merge defect from quietly changing the
# denominator between the predeclared holdout contract and final scoring.

assembled_support_df = (
    champion_holdout_prediction_df.groupby(
        ["metric", "horizon"],
        observed=True,
        as_index=False,
    )
    .agg(
        assembled_labeled_rows=("actual", "size"),
        assembled_benchmark_rows=(
            "primary_evaluation_eligible",
            "sum",
        ),
    )
)

expected_support_df = (
    holdout_surface_audit_df[
        [
            "metric",
            "horizon",
            "labeled_rows",
            "benchmark_scoreable_rows",
        ]
    ]
    .rename(columns={
        "labeled_rows": "expected_labeled_rows",
        "benchmark_scoreable_rows":
            "expected_benchmark_rows",
    })
)

holdout_support_reconciliation_df = (
    expected_support_df.merge(
        assembled_support_df,
        on=["metric", "horizon"],
        how="outer",
        validate="one_to_one",
    )
)

if not (
    holdout_support_reconciliation_df[
        "expected_labeled_rows"
    ].eq(
        holdout_support_reconciliation_df[
            "assembled_labeled_rows"
        ]
    )
    & holdout_support_reconciliation_df[
        "expected_benchmark_rows"
    ].eq(
        holdout_support_reconciliation_df[
            "assembled_benchmark_rows"
        ]
    )
).all():
    raise RuntimeError(
        "Assembled champion predictions do not reproduce the audited "
        "final-holdout population."
    )

atomic_write_parquet(
    holdout_support_reconciliation_df,
    FINAL_HOLDOUT_SUPPORT_AUDIT_PATH,
)

display(
    holdout_support_reconciliation_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS)
    )[
        [
            "Target",
            "horizon",
            "assembled_labeled_rows",
            "assembled_benchmark_rows",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "assembled_labeled_rows": "Labeled rows",
        "assembled_benchmark_rows":
            "Benchmark-supported rows",
    })
)

print(
    "PASS — row-level final-holdout champion surface matches the "
    "predeclared evaluation population.",
    flush=True,
)

,Target,Horizon,Labeled rows,Benchmark-supported rows
0,Taxi trips,1,110693,110499
1,Taxi trips,2,110693,110499
2,Taxi trips,5,110693,110499
3,Taxi average speed,1,79652,78325
4,Taxi average speed,2,79652,78325
5,Taxi average speed,5,79652,78325
6,FHVHV trips,1,110693,110499
7,FHVHV trips,2,110693,110499
8,FHVHV trips,5,110693,110499
9,FHVHV average speed,1,110231,110012


PASS — row-level final-holdout champion surface matches the predeclared evaluation population.


**Findings.** The assembled final-holdout surface reproduces the predeclared evaluation population exactly across all **15 Target × Horizon jobs**, with no duplicate physical forecasts.

Across the full system, the holdout contains **1,427,340 labeled forecast rows**, of which **1,421,436** have the frozen weekly benchmark required for primary evaluation.

Support remains consistent within each target across horizons: Taxi trips and FHVHV trips contribute **110,499 benchmark-supported rows per horizon**, Taxi average speed contributes **78,325**, FHVHV average speed contributes **110,012**, and Subway ridership contributes **64,477**.

The final scoring denominator therefore matches the holdout surface frozen before model execution.

**4.6.1.7.1B — Audit advanced-model coverage before looking at accuracy.**

Final-holdout accuracy is meaningful only if availability is kept explicit.

For each champion job, this block reports how often the selected advanced model produced a usable forecast and how often the complete system therefore used the frozen weekly fallback.

The identity

`Advanced-model rows + Fallback rows = Benchmark-supported rows`

must hold for all 15 jobs before any error metric is calculated.

In [58]:
# ---------------------------------------------------------------------
# 4.6.1.7.1B — Audit holdout model coverage and benchmark fallback
# ---------------------------------------------------------------------

# Primary evaluation is restricted only by actual/benchmark support—not by
# whether the advanced champion happened to produce a forecast.
holdout_selection_df = champion_holdout_prediction_df.loc[
    champion_holdout_prediction_df[
        "primary_evaluation_eligible"
    ]
].copy()

holdout_coverage_df = (
    holdout_selection_df.groupby(
        [
            "metric",
            "horizon",
            "champion_family",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        evaluation_rows=("actual", "size"),
        model_available_rows=(
            "model_prediction_available",
            "sum",
        ),
        fallback_rows=(
            "used_benchmark_fallback",
            "sum",
        ),
    )
)

holdout_coverage_df["model_coverage_pct"] = (
    100.0
    * holdout_coverage_df["model_available_rows"]
    / holdout_coverage_df["evaluation_rows"]
)

holdout_coverage_df["fallback_pct"] = (
    100.0
    * holdout_coverage_df["fallback_rows"]
    / holdout_coverage_df["evaluation_rows"]
)

# Availability and fallback must form an exact partition. Otherwise some
# physical rows would either disappear or receive two competing predictions.
if not (
    holdout_coverage_df["model_available_rows"]
    + holdout_coverage_df["fallback_rows"]
).eq(
    holdout_coverage_df["evaluation_rows"]
).all():
    raise RuntimeError(
        "Advanced-model and benchmark-fallback rows do not partition "
        "the final-holdout population."
    )

if len(holdout_coverage_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Holdout coverage audit does not contain exactly 15 jobs."
    )

display(
    holdout_coverage_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS),
        Champion=lambda frame:
            frame["champion_family"].map(FAMILY_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "evaluation_rows",
            "model_available_rows",
            "fallback_rows",
            "model_coverage_pct",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "evaluation_rows": "Evaluation rows",
        "model_available_rows": "Advanced-model rows",
        "fallback_rows": "Fallback rows",
        "model_coverage_pct": "Model coverage (%)",
    })
    .round(3)
)

,Target,Horizon,Champion,Evaluation rows,Advanced-model rows,Fallback rows,Model coverage (%)
0,FHVHV average speed,1,Transformer,110012,110012,0,100.000
1,FHVHV average speed,2,Transformer,110012,110012,0,100.000
2,FHVHV average speed,5,Transformer,110012,110012,0,100.000
3,FHVHV trips,1,Tree,110499,110499,0,100.000
4,FHVHV trips,2,Tree,110499,110499,0,100.000
5,FHVHV trips,5,Tree,110499,110499,0,100.000
6,Subway ridership,1,Tree,64477,64477,0,100.000
7,Subway ridership,2,Transformer,64477,64477,0,100.000
8,Subway ridership,5,Tree,64477,64477,0,100.000
9,Taxi average speed,1,Neural,78325,78325,0,100.000


### 4.6.1.7.2 Score Holdout Accuracy and Generalization

With the physical holdout surface fixed, the notebook can measure how well the frozen system actually performs. The first block scores each Target × Horizon against the frozen weekly benchmark; the second compares those unseen results with ordinary validation to test whether the selected system transfers beyond the period that chose it.

**Findings.** All **15 frozen champions achieve 100% advanced-model coverage** on their benchmark-supported final-holdout populations.

No job requires the frozen weekly benchmark as a fallback: **1,421,436 of 1,421,436 evaluation rows** receive a usable champion forecast.

The complete-system denominator nevertheless remains unchanged, preserving the same advanced-model-or-benchmark scoring contract used during model selection.

**4.6.1.7.2A — Score the frozen champion system on the unseen final holdout.**

The primary holdout score is the error of the **deployable forecasting system**, not model-only accuracy.

For every benchmark-supported row:

- use the champion's `prediction_final` when its advanced forecast is available;
- otherwise use the frozen weekly benchmark;
- calculate system MAE, RMSE, bias, Relative MAE, and benchmark skill on that common denominator.

These metrics test how well the frozen champion system generalized. They do not rank alternative families or reopen model selection.

In [59]:
# ---------------------------------------------------------------------
# 4.6.1.7.2A — Score the frozen champion system on final holdout
# ---------------------------------------------------------------------

FINAL_HOLDOUT_SCORECARD_PATH = (
    FINAL_DIR / "champion_final_holdout_scorecard.parquet"
)


def holdout_smape(actual, prediction):
    """Return mean symmetric absolute percentage error in percent."""
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)

    denominator = np.abs(actual) + np.abs(prediction)
    numerator = 200.0 * np.abs(prediction - actual)

    # np.divide evaluates only where the denominator is positive, so a
    # legitimate actual=prediction=0 row contributes zero without warning.
    contribution = np.zeros_like(denominator, dtype=float)

    np.divide(
        numerator,
        denominator,
        out=contribution,
        where=denominator > 0,
    )

    return float(np.mean(contribution))


holdout_score_records = []

for (
    metric,
    horizon,
    champion_family,
), job_df in holdout_selection_df.groupby(
    ["metric", "horizon", "champion_family"],
    observed=True,
):
    actual = job_df["actual"].to_numpy(dtype=float)
    system_prediction = job_df[
        "system_prediction"
    ].to_numpy(dtype=float)

    benchmark_prediction = job_df[
        "benchmark_prediction"
    ].to_numpy(dtype=float)

    system_error = system_prediction - actual
    benchmark_error = benchmark_prediction - actual

    system_mae = float(
        np.mean(np.abs(system_error))
    )
    benchmark_mae = float(
        np.mean(np.abs(benchmark_error))
    )

    # Relative MAE scales native-unit error by the observed target magnitude,
    # allowing the five differently scaled mobility measures to be compared.
    observed_abs_mean = float(
        np.mean(np.abs(actual))
    )

    model_available_rows = int(
        job_df["model_prediction_available"].sum()
    )
    evaluation_rows = len(job_df)

    holdout_score_records.append({
        "metric": metric,
        "horizon": int(horizon),
        "champion_family": champion_family,
        "evaluation_rows": evaluation_rows,
        "model_available_rows": model_available_rows,
        "fallback_rows":
            evaluation_rows - model_available_rows,
        "model_coverage_pct":
            100.0 * model_available_rows / evaluation_rows,
        "mae": system_mae,
        "rmse": float(
            np.sqrt(np.mean(np.square(system_error)))
        ),
        "bias": float(np.mean(system_error)),
        "abs_bias": float(abs(np.mean(system_error))),
        "smape_pct": holdout_smape(
            actual,
            system_prediction,
        ),
        "observed_abs_mean": observed_abs_mean,
        "relative_mae_pct": (
            100.0 * system_mae / observed_abs_mean
            if observed_abs_mean > 0
            else np.nan
        ),
        "benchmark_mae": benchmark_mae,

        # Positive skill means the complete deployed system improves on the
        # frozen weekly benchmark over the same physical evaluation rows.
        "benchmark_skill_pct": (
            100.0 * (1.0 - system_mae / benchmark_mae)
            if benchmark_mae > 0
            else np.nan
        ),
    })

final_holdout_scorecard_df = pd.DataFrame(
    holdout_score_records
)

if len(final_holdout_scorecard_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Final-holdout scorecard does not contain exactly 15 jobs."
    )

if final_holdout_scorecard_df[
    [
        "mae",
        "rmse",
        "bias",
        "relative_mae_pct",
        "benchmark_mae",
        "benchmark_skill_pct",
    ]
].isna().any().any():
    raise RuntimeError(
        "Final-holdout scorecard contains an unexpected missing metric."
    )

atomic_write_parquet(
    final_holdout_scorecard_df,
    FINAL_HOLDOUT_SCORECARD_PATH,
)

holdout_score_display_df = (
    final_holdout_scorecard_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS),
        Champion=lambda frame:
            frame["champion_family"].map(FAMILY_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "model_coverage_pct",
            "mae",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "bias",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "model_coverage_pct": "Model coverage (%)",
        "mae": "System MAE",
        "relative_mae_pct": "Relative MAE (%)",
        "benchmark_skill_pct": "Benchmark skill (%)",
        "bias": "System bias",
    })
)

display(
    holdout_score_display_df.round(3)
)

print(
    f"PASS — frozen 15-job champion system scored on final holdout.\n"
    f"Output: {FINAL_HOLDOUT_SCORECARD_PATH}",
    flush=True,
)

,Target,Horizon,Champion,Model coverage (%),System MAE,Relative MAE (%),Benchmark skill (%),System bias
0,FHVHV average speed,1,Transformer,100.000,0.954,5.835,36.460,0.031
1,FHVHV average speed,2,Transformer,100.000,1.014,6.205,32.428,0.001
2,FHVHV average speed,5,Transformer,100.000,1.041,6.369,30.638,-0.028
3,FHVHV trips,1,Tree,100.000,57.622,10.556,37.665,-3.895
4,FHVHV trips,2,Tree,100.000,65.738,12.043,28.885,-3.048
5,FHVHV trips,5,Tree,100.000,66.102,12.109,28.491,-17.387
6,Subway ridership,1,Tree,100.000,407.487,8.841,35.836,-18.931
7,Subway ridership,2,Transformer,100.000,512.990,11.130,19.223,-11.250
8,Subway ridership,5,Tree,100.000,454.958,9.871,28.361,34.783
9,Taxi average speed,1,Neural,100.000,2.137,16.013,28.525,-0.121


PASS — frozen 15-job champion system scored on final holdout.
Output: pipeline_data/4.6.1.final_tables/champion_final_holdout_scorecard.parquet


**Findings.** All **15 frozen champion jobs retain positive benchmark skill** on the unseen final holdout. The selected system improves on the frozen Previous week · same Daypart benchmark by **19.2% to 37.7%**, depending on the Target × Horizon.

The strongest benchmark-relative results are **FHVHV trips h=1 (37.7%)**, **FHVHV average speed h=1 (36.5%)**, and **Subway ridership h=1 (35.8%)**. The weakest is **Subway ridership h=2 at 19.2%**, but it still provides a substantial improvement over the benchmark.

Relative MAE is lowest for FHVHV average speed at **5.8%–6.4%**, followed by Subway ridership at **8.8%–11.1%** and FHVHV trips at **10.6%–12.1%**. Taxi trips reach **14.7%–15.8%**, while Taxi average speed is consistently near **16%**.

Both speed targets remain nearly unbiased across all horizons. Larger native-unit biases appear in several demand forecasts—most notably FHVHV trips h=5 and Subway h=5—but none prevents the corresponding system from retaining positive benchmark skill.

**4.6.1.7.2B — Compare validation and holdout as a generalization check.**

The final question is not whether another family would have won the holdout. Those alternatives are intentionally not evaluated for selection.

Instead, compare each frozen champion's **system-level behavior** between ordinary validation and the unseen final holdout.

Because mobility scale can shift between periods, the most interpretable comparisons are:

- change in Relative MAE;
- change in benchmark skill;
- change in advanced-model coverage; and
- change in absolute system bias.

Stable values indicate that the validation evidence transferred well. Large deterioration identifies forecasting jobs that require caution in downstream interpretation.

In [60]:
# ---------------------------------------------------------------------
# 4.6.1.7.2B — Compare validation and holdout generalization
# ---------------------------------------------------------------------

FINAL_HOLDOUT_GENERALIZATION_PATH = (
    FINAL_DIR / "champion_validation_holdout_generalization.parquet"
)


# ---------------------------------------------------------------------
# 1) Recover the already-selected champions' ordinary-validation metrics
# ---------------------------------------------------------------------
# Merge the frozen champion registry back to the complete validation
# scorecard, then keep only the family that was selected for each job.
# Holdout performance is never used to change this mapping.

validation_champion_df = (
    validation_scorecard_df.merge(
        champion_registry_df[
            [
                "metric",
                "horizon",
                "champion_family",
            ]
        ],
        on=["metric", "horizon"],
        how="inner",
        validate="many_to_one",
    )
    .loc[
        lambda frame:
            frame["family"].eq(frame["champion_family"])
    ]
    [
        [
            "metric",
            "horizon",
            "champion_family",
            "mae",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "model_coverage_pct",
            "abs_bias",
        ]
    ]
    .rename(columns={
        "mae": "validation_system_mae",
        "relative_mae_pct": "validation_relative_mae_pct",
        "benchmark_skill_pct": "validation_benchmark_skill_pct",
        "model_coverage_pct": "validation_model_coverage_pct",
        "abs_bias": "validation_abs_bias",
    })
)

if len(validation_champion_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Could not resolve exactly one validation champion row per job."
    )


# ---------------------------------------------------------------------
# 2) Join unseen holdout evidence to the frozen validation evidence
# ---------------------------------------------------------------------

holdout_generalization_df = (
    final_holdout_scorecard_df.rename(columns={
        "mae": "holdout_system_mae",
        "relative_mae_pct": "holdout_relative_mae_pct",
        "benchmark_skill_pct": "holdout_benchmark_skill_pct",
        "model_coverage_pct": "holdout_model_coverage_pct",
        "abs_bias": "holdout_abs_bias",
    })
    .merge(
        validation_champion_df,
        on=[
            "metric",
            "horizon",
            "champion_family",
        ],
        how="inner",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------------------
# 3) Express transfer from validation to unseen holdout
# ---------------------------------------------------------------------
# Percentage-point changes keep scale-free error, benchmark skill, and
# coverage interpretable even when the absolute target level shifts.

holdout_generalization_df[
    "relative_mae_change_pp"
] = (
    holdout_generalization_df["holdout_relative_mae_pct"]
    - holdout_generalization_df["validation_relative_mae_pct"]
)

holdout_generalization_df[
    "benchmark_skill_change_pp"
] = (
    holdout_generalization_df["holdout_benchmark_skill_pct"]
    - holdout_generalization_df["validation_benchmark_skill_pct"]
)

holdout_generalization_df[
    "model_coverage_change_pp"
] = (
    holdout_generalization_df["holdout_model_coverage_pct"]
    - holdout_generalization_df["validation_model_coverage_pct"]
)

holdout_generalization_df[
    "abs_bias_change"
] = (
    holdout_generalization_df["holdout_abs_bias"]
    - holdout_generalization_df["validation_abs_bias"]
)

if len(holdout_generalization_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Validation/holdout generalization table does not contain 15 jobs."
    )

atomic_write_parquet(
    holdout_generalization_df,
    FINAL_HOLDOUT_GENERALIZATION_PATH,
)

display(
    holdout_generalization_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_LABELS),
        Champion=lambda frame:
            frame["champion_family"].map(FAMILY_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "validation_relative_mae_pct",
            "holdout_relative_mae_pct",
            "relative_mae_change_pp",
            "validation_benchmark_skill_pct",
            "holdout_benchmark_skill_pct",
            "benchmark_skill_change_pp",
            "model_coverage_change_pp",
            "abs_bias_change",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "validation_relative_mae_pct":
            "Validation Relative MAE (%)",
        "holdout_relative_mae_pct":
            "Holdout Relative MAE (%)",
        "relative_mae_change_pp":
            "Relative MAE change (pp)",
        "validation_benchmark_skill_pct":
            "Validation skill (%)",
        "holdout_benchmark_skill_pct":
            "Holdout skill (%)",
        "benchmark_skill_change_pp":
            "Skill change (pp)",
        "model_coverage_change_pp":
            "Coverage change (pp)",
        "abs_bias_change":
            "Absolute-bias change",
    })
    .round(3)
)

,Target,Horizon,Champion,Validation Relative MAE (%),Holdout Relative MAE (%),Relative MAE change (pp),Validation skill (%),Holdout skill (%),Skill change (pp),Coverage change (pp),Absolute-bias change
0,FHVHV average speed,1,Transformer,5.460,5.835,0.375,30.726,36.460,5.734,0.000,-0.012
1,FHVHV average speed,2,Transformer,5.771,6.205,0.434,26.773,32.428,5.655,0.000,-0.006
2,FHVHV average speed,5,Transformer,5.872,6.369,0.497,25.492,30.638,5.146,0.000,0.010
3,FHVHV trips,1,Tree,9.201,10.556,1.355,31.391,37.665,6.274,0.000,1.782
4,FHVHV trips,2,Tree,10.129,12.043,1.913,24.467,28.885,4.418,0.000,-0.459
5,FHVHV trips,5,Tree,9.654,12.109,2.456,28.013,28.491,0.478,0.000,15.581
6,Subway ridership,1,Tree,8.780,8.841,0.061,21.683,35.836,14.152,0.000,-26.540
7,Subway ridership,2,Transformer,10.035,11.130,1.095,10.493,19.223,8.730,0.000,8.974
8,Subway ridership,5,Tree,9.308,9.871,0.562,16.971,28.361,11.389,0.000,-112.940
9,Taxi average speed,1,Neural,15.759,16.013,0.254,26.198,28.525,2.328,0.000,0.107


**Findings.** The frozen champion system transfers well from ordinary validation into the unseen final holdout.

Relative MAE increases for all 15 jobs, but the deterioration is generally modest: changes range from **+0.061 to +2.456 percentage points**. The speed forecasts are especially stable, with all six Taxi- and FHVHV-speed jobs changing by less than **0.5 percentage points**.

Benchmark-relative performance is even stronger. Holdout skill improves for **14 of 15 jobs** despite the modest increases in Relative MAE. The only decline is Taxi trips h=5, where skill decreases by just **0.617 percentage points** and remains strongly positive at **29.3%**.

The larger Relative-MAE shifts are concentrated in demand forecasting, particularly **FHVHV trips h=5 (+2.456 pp)**, **FHVHV trips h=2 (+1.913 pp)**, and the Taxi-trip horizons (**+1.167 to +1.612 pp**). Subway h=2 also remains the weakest benchmark-relative job, although its holdout skill improves from **10.5% to 19.2%**.

Advanced-model coverage remains unchanged at **100% for every champion job**. Bias is also highly stable for both speed targets; the more noticeable bias changes are confined to several demand forecasts.

Overall, the unseen holdout supports the frozen champion system rather than exposing a generalization failure. No Target × Horizon job loses benchmark value or becomes too unstable for downstream use, although the larger demand-forecast shifts warrant more cautious interpretation than the highly stable speed forecasts.

### Section Summary

The frozen 15-job champion system successfully completes its unseen January–March 2026 final-holdout evaluation.

All **29 execution units** are complete and recoverable, and the assembled forecasts reproduce the predeclared evaluation surface exactly. Every champion provides a usable advanced forecast on **100% of its benchmark-supported rows**, so no weekly-benchmark fallback is required during the final evaluation.

All **15 Target × Horizon jobs outperform the frozen Previous week · same Daypart benchmark**, with holdout skill ranging from **19.2% to 37.7%**. Relative MAE rises modestly from validation for every job, while benchmark skill improves for **14 of 15**, indicating that the selected system continues to add forecasting value under the changed unseen-period conditions.

Generalization is strongest and most stable for the speed forecasts. Demand forecasts show somewhat greater drift—especially at longer horizons—but no job exhibits evidence strong enough to invalidate its downstream use.

The final holdout therefore supports the frozen **8 Tree / 4 Transformer / 3 Neural** champion system for downstream forecast analysis, while preserving explicit caution around the comparatively less stable demand forecasts.

## 4.6.1.8 Audit Champions Against Pre-Holdout Runner-Ups

Beating the weekly benchmark establishes that the frozen system remains useful on unseen data. A stricter robustness question is whether each validation-selected champion also remains competitive with the strongest alternative that existed before the holdout was opened.

This section therefore freezes one pre-holdout runner-up per job, executes those alternatives under their own family contracts, and compares champion and runner-up on one identical unseen denominator. Holdout ranking reversals measure selection stability only; they do not reopen the frozen champion registry.

### 4.6.1.8.1 Freeze and Execute the Runner-Up Audit

The runner-up audit is defined entirely from pre-holdout evidence. This subsection freezes the 15 alternatives, binds their family-specific execution contracts, creates a separate restart surface, validates every planned unit, and recovers or executes the complete runner-up workload without contaminating the champion runtime.

**4.6.1.8.1A — Freeze one pre-holdout runner-up for every champion job.**

The robustness challenger must be determined without looking at final-holdout performance.

For clear primary wins, this block recovers the ordinary-validation system-MAE runner-up already identified by the tournament. For near ties, it selects the single surviving contender other than the frozen champion.

The resulting 15-row registry also attaches each runner-up's exact upstream modeling track and configuration. These are the definitions that would have been deployed had that family advanced, so the holdout audit does not introduce new tuning choices.

In [61]:
# ---------------------------------------------------------------------
# 4.6.1.8.1A — Freeze the pre-holdout runner-up registry
# ---------------------------------------------------------------------

RUNNER_UP_AUDIT_CONTRACT_ID = (
    "frozen_runner_up_holdout_robustness_v1"
)

RUNNER_UP_REGISTRY_PATH = (
    INTERMEDIATE_DIR / "frozen_pre_holdout_runner_up_registry.parquet"
)


# ---------------------------------------------------------------------
# 1) Recover exactly one alternative using PRE-HOLDOUT evidence only
# ---------------------------------------------------------------------
# The rule differs slightly by how the original champion was selected.
#
# Clear-primary jobs:
#   the point-estimate winner also cleared the uncertainty/practical-margin
#   screen, so its validation-MAE runner-up is the natural challenger.
#
# Near-tie jobs:
#   two families survived the primary screen. The frozen champion was chosen
#   by absolute system bias, so the other surviving family is the only fair
#   pre-holdout runner-up. We do NOT choose whichever family looks best now.

runner_up_records = []

for champion in frozen_champion_registry_df.itertuples(index=False):
    metric = str(champion.metric)
    horizon = int(champion.horizon)
    champion_family = str(champion.champion_family)

    if bool(champion.clear_primary_winner):
        point_row = point_winner_df.loc[
            point_winner_df["metric"].eq(metric)
            & point_winner_df["horizon"].astype(int).eq(horizon)
        ]

        if len(point_row) != 1:
            raise RuntimeError(
                f"{metric} · h={horizon}: expected one point-winner row."
            )

        runner_up_family = str(
            point_row.iloc[0]["runner_up_family"]
        )
        runner_up_basis = (
            "second_lowest_system_mae_after_clear_primary_win"
        )

    else:
        surviving_alternatives = (
            near_tie_evidence_df.loc[
                near_tie_evidence_df["metric"].eq(metric)
                & near_tie_evidence_df["horizon"].astype(int).eq(horizon)
                & ~near_tie_evidence_df["family"].eq(champion_family),
                "family",
            ]
            .astype(str)
            .drop_duplicates()
            .tolist()
        )

        if len(surviving_alternatives) != 1:
            raise RuntimeError(
                f"{metric} · h={horizon}: expected exactly one "
                "non-champion near-tie contender; found "
                f"{surviving_alternatives}."
            )

        runner_up_family = surviving_alternatives[0]
        runner_up_basis = "surviving_near_tie_alternative"

    if runner_up_family == champion_family:
        raise RuntimeError(
            f"{metric} · h={horizon}: runner-up equals the champion."
        )

    runner_up_records.append({
        "metric": metric,
        "horizon": horizon,
        "champion_family": champion_family,
        "runner_up_family": runner_up_family,
        "champion_selection_basis": champion.selection_basis,
        "runner_up_basis": runner_up_basis,
        "selection_frozen_before_holdout": True,
        "holdout_used_for_selection": False,
        "robustness_only": True,
        "runner_up_audit_contract_id":
            RUNNER_UP_AUDIT_CONTRACT_ID,
    })

runner_up_registry_df = pd.DataFrame(runner_up_records)


# ---------------------------------------------------------------------
# 2) Carry forward the validation evidence that defined the matchup
# ---------------------------------------------------------------------
# Keeping the pre-holdout MAEs beside the unseen-period results later makes
# ranking reversals easy to interpret without reconstructing the tournament.

champion_validation_df = (
    validation_scorecard_df[
        [
            "metric",
            "horizon",
            "family",
            "mae",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "model_coverage_pct",
            "abs_bias",
        ]
    ]
    .rename(columns={
        "family": "champion_family",
        "mae": "validation_champion_mae",
        "relative_mae_pct":
            "validation_champion_relative_mae_pct",
        "benchmark_skill_pct":
            "validation_champion_benchmark_skill_pct",
        "model_coverage_pct":
            "validation_champion_coverage_pct",
        "abs_bias":
            "validation_champion_abs_bias",
    })
)

runner_validation_df = (
    validation_scorecard_df[
        [
            "metric",
            "horizon",
            "family",
            "mae",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "model_coverage_pct",
            "abs_bias",
        ]
    ]
    .rename(columns={
        "family": "runner_up_family",
        "mae": "validation_runner_up_mae",
        "relative_mae_pct":
            "validation_runner_up_relative_mae_pct",
        "benchmark_skill_pct":
            "validation_runner_up_benchmark_skill_pct",
        "model_coverage_pct":
            "validation_runner_up_coverage_pct",
        "abs_bias":
            "validation_runner_up_abs_bias",
    })
)

runner_up_registry_df = (
    runner_up_registry_df
    .merge(
        champion_validation_df,
        on=["metric", "horizon", "champion_family"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        runner_validation_df,
        on=["metric", "horizon", "runner_up_family"],
        how="left",
        validate="one_to_one",
    )
)

# Positive = the frozen champion had LOWER validation MAE.
# A small negative value is legitimate for a near-tie job whose champion
# advanced through the frozen absolute-bias tie-break rather than point MAE.
runner_up_registry_df["validation_mae_advantage"] = (
    runner_up_registry_df["validation_runner_up_mae"]
    - runner_up_registry_df["validation_champion_mae"]
)


# ---------------------------------------------------------------------
# 3) Attach the exact frozen configuration owned by each runner-up family
# ---------------------------------------------------------------------
# A holdout robustness audit is meaningful only if the challenger is the same
# finalized family representative that existed before the holdout was opened.

runner_up_config_df = (
    finalist_registry_df[
        [
            "family",
            "metric",
            "horizon",
            "Family",
            "Representative",
            "modeling_track",
            "configuration_id",
        ]
    ]
    .rename(columns={
        "family": "runner_up_family",
        "Family": "runner_up_family_label",
        "Representative": "runner_up_representative",
        "modeling_track": "runner_up_modeling_track",
        "configuration_id": "runner_up_configuration_id",
    })
)

runner_up_registry_df = runner_up_registry_df.merge(
    runner_up_config_df,
    on=["runner_up_family", "metric", "horizon"],
    how="left",
    validate="one_to_one",
)

champion_label_df = (
    frozen_champion_registry_df[
        [
            "metric",
            "horizon",
            "Family",
            "Representative",
            "modeling_track",
        ]
    ]
    .rename(columns={
        "Family": "champion_family_label",
        "Representative": "champion_representative",
        "modeling_track": "champion_modeling_track",
    })
)

runner_up_registry_df = runner_up_registry_df.merge(
    champion_label_df,
    on=["metric", "horizon"],
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------------------
# 4) Prove the roster still represents the frozen pre-holdout tournament
# ---------------------------------------------------------------------

if len(runner_up_registry_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Runner-up robustness registry does not contain exactly "
        f"{EXPECTED_JOB_COUNT} jobs."
    )

if runner_up_registry_df.duplicated(["metric", "horizon"]).any():
    raise RuntimeError(
        "Runner-up registry is not unique by Target × Horizon."
    )

required_runner_up_fields = [
    "runner_up_family",
    "runner_up_family_label",
    "runner_up_representative",
    "runner_up_modeling_track",
    "runner_up_configuration_id",
    "validation_champion_mae",
    "validation_runner_up_mae",
]

if runner_up_registry_df[
    required_runner_up_fields
].isna().any().any():
    raise RuntimeError(
        "At least one runner-up job is missing its frozen "
        "configuration or validation evidence."
    )

if not runner_up_registry_df[
    "runner_up_modeling_track"
].isin(ORDINARY_TRACKS).all():
    raise RuntimeError(
        "At least one runner-up uses a non-ordinary modeling track."
    )

# These counts are not tuning choices. They are a compact integrity check
# against the already-completed tournament:
#   10 Transformer alternatives, 4 Tree alternatives, 1 Neural alternative.
expected_runner_up_counts = {
    "transformer": 10,
    "tree": 4,
    "neural": 1,
}

observed_runner_up_counts = (
    runner_up_registry_df["runner_up_family"]
    .value_counts()
    .to_dict()
)

if observed_runner_up_counts != expected_runner_up_counts:
    raise RuntimeError(
        "Pre-holdout runner-up composition changed unexpectedly: "
        f"{observed_runner_up_counts}"
    )

if runner_up_registry_df["runner_up_family"].eq("classical").any():
    raise RuntimeError(
        "A Classical runner-up unexpectedly entered the frozen "
        "robustness roster."
    )

atomic_write_parquet(
    runner_up_registry_df,
    RUNNER_UP_REGISTRY_PATH,
)


# ---------------------------------------------------------------------
# 5) Show the 15 frozen champion/challenger matchups
# ---------------------------------------------------------------------

runner_up_display_df = (
    runner_up_registry_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_SHORT_LABELS),
        Champion=lambda frame:
            frame["champion_family"].map(FAMILY_LABELS),
        **{
            "Runner-up": lambda frame:
                frame["runner_up_family"].map(FAMILY_LABELS),
        },
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "Runner-up",
            "runner_up_basis",
            "validation_champion_mae",
            "validation_runner_up_mae",
            "validation_mae_advantage",
            "runner_up_modeling_track",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "runner_up_basis": "Runner-up basis",
        "validation_champion_mae": "Champion validation MAE",
        "validation_runner_up_mae": "Runner-up validation MAE",
        "validation_mae_advantage": "Validation MAE advantage",
        "runner_up_modeling_track": "Runner-up track",
    })
)

display(runner_up_display_df.round(3))

print(
    "PASS — one pre-holdout runner-up frozen for every champion job.\n"
    "  Transformer runner-ups: 10\n"
    "  Tree runner-ups:         4\n"
    "  Neural runner-ups:       1\n"
    "  Classical runner-ups:    0\n"
    f"  Registry: {RUNNER_UP_REGISTRY_PATH}",
    flush=True,
)

,Target,Horizon,Champion,Runner-up,Runner-up basis,Champion validation MAE,Runner-up validation MAE,Validation MAE advantage,Runner-up track
0,Taxi trips,1,Tree,Transformer,second_lowest_system_mae_after_clear_primary_win,14.077,16.331,2.253,post_cp_primary
1,Taxi trips,2,Tree,Neural,second_lowest_system_mae_after_clear_primary_win,15.078,18.981,3.903,post_cp_primary
2,Taxi trips,5,Tree,Transformer,second_lowest_system_mae_after_clear_primary_win,14.151,17.794,3.643,post_cp_primary
3,Taxi speed,1,Neural,Transformer,surviving_near_tie_alternative,2.178,2.168,-0.010,post_cp_primary
4,Taxi speed,2,Neural,Transformer,surviving_near_tie_alternative,2.195,2.196,0.001,post_cp_primary
5,Taxi speed,5,Neural,Transformer,surviving_near_tie_alternative,2.220,2.227,0.007,post_cp_primary
6,FHVHV trips,1,Tree,Transformer,second_lowest_system_mae_after_clear_primary_win,47.632,58.265,10.632,post_cp_primary
7,FHVHV trips,2,Tree,Transformer,second_lowest_system_mae_after_clear_primary_win,52.439,61.866,9.427,post_cp_primary
8,FHVHV trips,5,Tree,Transformer,second_lowest_system_mae_after_clear_primary_win,49.978,62.447,12.470,post_cp_primary
9,FHVHV speed,1,Transformer,Tree,surviving_near_tie_alternative,0.921,0.938,0.017,full_history_regime_aware


PASS — one pre-holdout runner-up frozen for every champion job.
  Transformer runner-ups: 10
  Tree runner-ups:         4
  Neural runner-ups:       1
  Classical runner-ups:    0
  Registry: pipeline_data/4.6.1.intermediate_tables/frozen_pre_holdout_runner_up_registry.parquet


**Findings.** One pre-holdout runner-up is frozen for each of the **15 champion jobs**, producing a complete robustness roster of **10 Transformer, 4 Tree, and 1 Neural** alternatives. Classical is not the strongest pre-holdout alternative for any selected job.

For the **8 clear-primary wins**, the runner-up is the family with the second-lowest coverage-adjusted validation system MAE. For the **7 near ties**, the runner-up is the other family that survived the paired uncertainty and practical-equivalence screen before the secondary absolute-bias tie-break.

The clear-primary jobs generally show meaningful validation-MAE separation, including advantages of **9.4–12.5 MAE units across the three FHVHV-trip horizons** and **30.7–35.8 units for Subway h=1 and h=5**. The near-tie jobs are much closer by construction: several differ by only hundredths of an MAE unit.

Two near-tie champions—**Taxi average speed h=1** and **FHVHV average speed h=5**—do not have the lowest point validation MAE. This is expected because those champions were frozen using the secondary absolute-bias criterion after the primary uncertainty screen could not establish a practically decisive MAE winner.

The resulting roster therefore preserves the original pre-holdout selection logic while defining exactly one credible challenger for every frozen champion.

**4.6.1.8.1B — Bind the runner-ups to their frozen execution contracts and count the physical work.**

The runner-up audit must reproduce the same family-specific training lifecycle each alternative would have used if it had advanced.

Tree therefore receives one fresh global-panel fit per runner-up job. Neural and Transformer retain monthly expanding-history retraining, including the historical duration selector followed by a fresh final model for each holdout month.

Because the runner-up roster contains no Classical jobs, the existing Tree, Neural, and Transformer execution contracts cover the full robustness audit.

In [62]:
# ---------------------------------------------------------------------
# 4.6.1.8.1B — Bind runner-ups and freeze the robustness workload
# ---------------------------------------------------------------------

RUNNER_UP_BINDING_PATH = (
    INTERMEDIATE_DIR / "runner_up_holdout_runner_bindings.parquet"
)
RUNNER_UP_MONTHLY_PLAN_PATH = (
    INTERMEDIATE_DIR / "runner_up_holdout_monthly_refit_plan.parquet"
)
RUNNER_UP_WORKLOAD_PATH = (
    INTERMEDIATE_DIR / "runner_up_holdout_workload_summary.parquet"
)


# ---------------------------------------------------------------------
# 1) Reuse the same family execution contracts as the champion holdout
# ---------------------------------------------------------------------
# The model lifecycle belongs to the FAMILY, not to its champion/runner-up
# label. A Transformer challenger therefore uses the exact Transformer
# selector/refit contract already validated for champion execution.

saved_runner_contract_df = pd.read_parquet(
    HOLDOUT_RUNNER_CONTRACT_PATH
)

runner_contract_df = (
    saved_runner_contract_df
    .rename(columns={
        "champion_family": "runner_up_family",
        "runner_family_label": "contract_family_label",
    })
)

runner_up_binding_df = runner_up_registry_df.merge(
    runner_contract_df,
    on="runner_up_family",
    how="left",
    validate="many_to_one",
)

if runner_up_binding_df["runner_contract_id"].isna().any():
    missing = runner_up_binding_df.loc[
        runner_up_binding_df["runner_contract_id"].isna(),
        ["metric", "horizon", "runner_up_family"],
    ]

    raise RuntimeError(
        "At least one runner-up lacks an executable frozen family contract:\n"
        + missing.to_string(index=False)
    )

if not runner_up_binding_df[
    "runner_up_family_label"
].eq(
    runner_up_binding_df["contract_family_label"]
).all():
    raise RuntimeError(
        "Runner-up family labels disagree with the runtime registry."
    )

write_small_contract(
    runner_up_binding_df,
    RUNNER_UP_BINDING_PATH,
)


# ---------------------------------------------------------------------
# 2) Expand Neural/Transformer jobs to their natural monthly restart grain
# ---------------------------------------------------------------------
# One sequence restart unit owns BOTH:
#   1) the historical selector that chooses training duration, and
#   2) the fresh final fit trained for exactly that selected duration.
#
# Keeping those two calls together prevents a selector checkpoint from ever
# being mistaken for a production forecast model.

runner_up_monthly_frames = []

sequence_runner_up_df = runner_up_binding_df.loc[
    runner_up_binding_df["runner_up_family"].isin(
        ["neural", "transformer"]
    )
]

for job in sequence_runner_up_df.itertuples(index=False):
    schedule_df = read_holdout_origin_months(
        job.runner_up_modeling_track,
        job.metric,
        int(job.horizon),
    )

    schedule_df = schedule_df.assign(
        runner_up_family=job.runner_up_family,

        # Shared sequence runners call this field `champion_family`.
        # Here it means "family being executed"; robustness_role later makes
        # clear that the model is a pre-holdout runner-up, not a new champion.
        champion_family=job.runner_up_family,

        Family=job.runner_up_family_label,
        modeling_track=job.runner_up_modeling_track,
        metric=job.metric,
        horizon=int(job.horizon),
        configuration_id=job.runner_up_configuration_id,
        runner_contract_id=job.runner_contract_id,
        selector_fits=1,
        final_model_fits=1,
        planned_fit_calls=2,
    )

    runner_up_monthly_frames.append(schedule_df)

runner_up_monthly_refit_plan_df = pd.concat(
    runner_up_monthly_frames,
    ignore_index=True,
)

runner_up_monthly_refit_plan_df = (
    runner_up_monthly_refit_plan_df
    .sort_values(
        ["runner_up_family", "metric", "horizon", "refit_step"]
    )
    .reset_index(drop=True)
)

monthly_key = [
    "runner_up_family",
    "metric",
    "horizon",
    "refit_step",
]

if runner_up_monthly_refit_plan_df.duplicated(monthly_key).any():
    raise RuntimeError(
        "Runner-up monthly plan contains duplicate restart units."
    )

monthly_counts = (
    runner_up_monthly_refit_plan_df.groupby(
        ["runner_up_family", "metric", "horizon"],
        observed=True,
    )
    .size()
)

if not monthly_counts.eq(3).all():
    raise RuntimeError(
        "At least one sequence-model runner-up lacks all three "
        "monthly holdout refits."
    )

write_small_contract(
    runner_up_monthly_refit_plan_df,
    RUNNER_UP_MONTHLY_PLAN_PATH,
)


# ---------------------------------------------------------------------
# 3) Count the work before allowing any expensive model fit
# ---------------------------------------------------------------------
# Making the cost explicit before execution keeps the long run deliberate
# and gives restart/recovery checks a fixed target.

tree_jobs = int(
    runner_up_binding_df["runner_up_family"].eq("tree").sum()
)
neural_jobs = int(
    runner_up_binding_df["runner_up_family"].eq("neural").sum()
)
transformer_jobs = int(
    runner_up_binding_df["runner_up_family"].eq("transformer").sum()
)

neural_months = runner_up_monthly_refit_plan_df.loc[
    runner_up_monthly_refit_plan_df["runner_up_family"].eq("neural")
]
transformer_months = runner_up_monthly_refit_plan_df.loc[
    runner_up_monthly_refit_plan_df["runner_up_family"].eq("transformer")
]

runner_up_workload_df = pd.DataFrame([
    {
        "runner_up_family": "tree",
        "Family": "Tree",
        "Runner-up jobs": tree_jobs,
        "Restart units": tree_jobs,
        "Selector fits": 0,
        "Final fits": tree_jobs,
        "Planned training calls": tree_jobs,
    },
    {
        "runner_up_family": "neural",
        "Family": "Neural",
        "Runner-up jobs": neural_jobs,
        "Restart units": len(neural_months),
        "Selector fits": int(neural_months["selector_fits"].sum()),
        "Final fits": int(neural_months["final_model_fits"].sum()),
        "Planned training calls": int(
            neural_months["planned_fit_calls"].sum()
        ),
    },
    {
        "runner_up_family": "transformer",
        "Family": "Transformer",
        "Runner-up jobs": transformer_jobs,
        "Restart units": len(transformer_months),
        "Selector fits": int(
            transformer_months["selector_fits"].sum()
        ),
        "Final fits": int(
            transformer_months["final_model_fits"].sum()
        ),
        "Planned training calls": int(
            transformer_months["planned_fit_calls"].sum()
        ),
    },
])

total_runner_up_jobs = int(
    runner_up_workload_df["Runner-up jobs"].sum()
)
total_runner_up_units = int(
    runner_up_workload_df["Restart units"].sum()
)
total_runner_up_calls = int(
    runner_up_workload_df["Planned training calls"].sum()
)

if total_runner_up_jobs != 15:
    raise RuntimeError(
        f"Expected 15 runner-up jobs; found {total_runner_up_jobs}."
    )

if total_runner_up_units != 37:
    raise RuntimeError(
        f"Expected 37 runner-up restart units; "
        f"found {total_runner_up_units}."
    )

if total_runner_up_calls != 70:
    raise RuntimeError(
        f"Expected 70 runner-up training calls; "
        f"found {total_runner_up_calls}."
    )

write_small_contract(
    runner_up_workload_df,
    RUNNER_UP_WORKLOAD_PATH,
)

display(runner_up_workload_df)

print(
    "PASS — runner-up holdout workload frozen.\n"
    f"  Runner-up jobs:         {total_runner_up_jobs}\n"
    f"  Restart units:          {total_runner_up_units}\n"
    f"  Planned training calls: {total_runner_up_calls}\n"
    f"  Workload: {RUNNER_UP_WORKLOAD_PATH}",
    flush=True,
)

,runner_up_family,Family,Runner-up jobs,Restart units,Selector fits,Final fits,Planned training calls
0,tree,Tree,4,4,0,4,4
1,neural,Neural,1,3,3,3,6
2,transformer,Transformer,10,30,30,30,60


PASS — runner-up holdout workload frozen.
  Runner-up jobs:         15
  Restart units:          37
  Planned training calls: 70
  Workload: pipeline_data/4.6.1.intermediate_tables/runner_up_holdout_workload_summary.parquet


**Findings.** Evaluating all **15 pre-holdout runner-ups** requires **37 restart units and 70 training calls** under their frozen family-specific execution contracts.

The **4 Tree alternatives** require one final fit each. The single Neural alternative expands into **3 monthly restart units**, each containing a historical selector and fresh final refit, for **6 training calls**. The **10 Transformer alternatives** expand into **30 monthly restart units** and **60 training calls**.

The workload is therefore dominated by Transformer execution, while preserving the same restart grain and selector-then-fresh-refit semantics used by the corresponding family during champion evaluation.

**4.6.1.8.1C — Create an isolated restart contract for runner-up forecasts.**

Runner-up forecasts must never be confused with the champion checkpoints already used in the final evaluation.

This block therefore gives the robustness audit its own artifact root and manifest while preserving the same family runtime contract and deterministic logical fit identity.

Preserving the logical fit identity is important for the sequence models: a runner-up receives the same deterministic seed it would have received had that exact Family × Track × Target × Horizon × Month configuration entered the original holdout runtime. Only its storage location and analytical role differ.

In [63]:
# ---------------------------------------------------------------------
# 4.6.1.8.1C — Freeze runner-up restart units and recovery controls
# ---------------------------------------------------------------------

RUNNER_UP_RUNTIME_ROOT = (
    INTERMEDIATE_DIR / "final_holdout_runner_up_runtime_v1"
)
RUNNER_UP_CHECKPOINT_MANIFEST_PATH = (
    INTERMEDIATE_DIR / "runner_up_holdout_checkpoint_manifest.parquet"
)

RUN_RUNNER_UP_TREE = True
RUN_RUNNER_UP_NEURAL = True
RUN_RUNNER_UP_TRANSFORMER = False
RESET_RUNNER_UP_HOLDOUT_RUNTIME = False


# ---------------------------------------------------------------------
# 1) Keep destructive reset explicit and scoped ONLY to runner-up artifacts
# ---------------------------------------------------------------------
# Champion checkpoints are authoritative final-holdout evidence. A reset for
# this diagnostic must never touch their runtime directory.

if RESET_RUNNER_UP_HOLDOUT_RUNTIME:
    if RUNNER_UP_RUNTIME_ROOT.exists():
        shutil.rmtree(RUNNER_UP_RUNTIME_ROOT)

    RUNNER_UP_CHECKPOINT_MANIFEST_PATH.unlink(missing_ok=True)

for family in ["tree", "neural", "transformer"]:
    family_root = RUNNER_UP_RUNTIME_ROOT / family
    (family_root / "predictions").mkdir(parents=True, exist_ok=True)
    (family_root / "audit").mkdir(parents=True, exist_ok=True)


def runner_up_unit_paths(family, runtime_unit_id):
    """
    Return the isolated prediction/audit paths for one runner-up restart unit.

    The logical model identity stays compatible with the existing family
    runner, but the physical directory is different so diagnostic challenger
    forecasts can never overwrite authoritative champion forecasts.
    """
    family_root = RUNNER_UP_RUNTIME_ROOT / family

    return (
        family_root / "predictions" / f"{runtime_unit_id}.parquet",
        family_root / "audit" / f"{runtime_unit_id}.parquet",
    )


# ---------------------------------------------------------------------
# 2) Build restart IDs from the same hypothetical deployment identity
# ---------------------------------------------------------------------
# stable_holdout_unit_id() intentionally does NOT include "runner-up".
# That preserves the deterministic seed/model identity the family would have
# received had it originally advanced. The separate storage root supplies the
# isolation needed for this diagnostic.

runner_up_manifest_records = []

tree_runner_up_df = runner_up_binding_df.loc[
    runner_up_binding_df["runner_up_family"].eq("tree")
]

for row in tree_runner_up_df.itertuples(index=False):
    unit_id = stable_holdout_unit_id(
        "tree",
        row.runner_up_modeling_track,
        row.metric,
        int(row.horizon),
        row.runner_up_configuration_id,
    )

    prediction_path, audit_path = runner_up_unit_paths(
        "tree",
        unit_id,
    )

    runner_up_manifest_records.append({
        "runtime_unit_id": unit_id,

        # Shared family runners use champion_family as the executed-family
        # identity. robustness_role below records the analytical role.
        "champion_family": "tree",
        "runner_up_family": "tree",
        "Family": "Tree",

        "modeling_track": row.runner_up_modeling_track,
        "metric": row.metric,
        "horizon": int(row.horizon),
        "selected_horizons": str(int(row.horizon)),
        "configuration_id": row.runner_up_configuration_id,
        "runner_contract_id": row.runner_contract_id,

        "forecast_sequence_id": None,
        "taxi_zone_id": np.nan,
        "refit_month": None,
        "refit_step": 0,
        "refit_origin_observation_sequence_id": np.nan,
        "next_refit_origin_observation_sequence_id": np.nan,

        "planned_fit_calls": 1,
        "checkpoint_grain": "Target × Horizon global-panel fit",
        "prediction_path": str(prediction_path),
        "audit_path": str(audit_path),

        "robustness_role": "pre_holdout_runner_up",
        "runner_up_audit_contract_id":
            RUNNER_UP_AUDIT_CONTRACT_ID,
    })


for row in runner_up_monthly_refit_plan_df.itertuples(index=False):
    family = str(row.runner_up_family)

    unit_id = stable_holdout_unit_id(
        family,
        row.modeling_track,
        row.metric,
        int(row.horizon),
        row.configuration_id,
        row.refit_month,
    )

    prediction_path, audit_path = runner_up_unit_paths(
        family,
        unit_id,
    )

    runner_up_manifest_records.append({
        "runtime_unit_id": unit_id,
        "champion_family": family,
        "runner_up_family": family,
        "Family": row.Family,

        "modeling_track": row.modeling_track,
        "metric": row.metric,
        "horizon": int(row.horizon),
        "selected_horizons": str(int(row.horizon)),
        "configuration_id": row.configuration_id,
        "runner_contract_id": row.runner_contract_id,

        "forecast_sequence_id": None,
        "taxi_zone_id": np.nan,
        "refit_month": row.refit_month,
        "refit_step": int(row.refit_step),
        "refit_origin_observation_sequence_id": int(
            row.refit_origin_observation_sequence_id
        ),
        "next_refit_origin_observation_sequence_id":
            row.next_refit_origin_observation_sequence_id,

        "planned_fit_calls": int(row.planned_fit_calls),
        "checkpoint_grain": "Target × Horizon × holdout month",
        "prediction_path": str(prediction_path),
        "audit_path": str(audit_path),

        "robustness_role": "pre_holdout_runner_up",
        "runner_up_audit_contract_id":
            RUNNER_UP_AUDIT_CONTRACT_ID,
    })


runner_up_checkpoint_manifest_df = pd.DataFrame(
    runner_up_manifest_records
)

# The shared runners and their artifact validators intentionally keep the
# existing runtime contract ID. The separate runner-up audit contract records
# WHY these otherwise-compatible family outputs were produced.
runner_up_checkpoint_manifest_df[
    "holdout_runtime_contract_id"
] = HOLDOUT_RUNTIME_CONTRACT_ID
runner_up_checkpoint_manifest_df[
    "selection_contract_id"
] = CHAMPION_SELECTION_CONTRACT_ID
runner_up_checkpoint_manifest_df[
    "prediction_output_rule"
] = "prediction_final = max(0, prediction_raw)"


# ---------------------------------------------------------------------
# 3) Validate the complete restart surface before recovery/execution
# ---------------------------------------------------------------------

if len(runner_up_checkpoint_manifest_df) != 37:
    raise RuntimeError(
        "Runner-up manifest no longer contains 37 restart units."
    )

if runner_up_checkpoint_manifest_df[
    "runtime_unit_id"
].duplicated().any():
    raise RuntimeError(
        "Runner-up runtime contains duplicate restart-unit IDs."
    )

if int(
    runner_up_checkpoint_manifest_df[
        "planned_fit_calls"
    ].sum()
) != 70:
    raise RuntimeError(
        "Runner-up manifest no longer contains 70 training calls."
    )

if set(
    runner_up_checkpoint_manifest_df["runner_up_family"]
) != {"tree", "neural", "transformer"}:
    raise RuntimeError(
        "Runner-up runtime does not contain exactly the three "
        "expected executable families."
    )


# ---------------------------------------------------------------------
# 4) Recover only the 37 EXACT artifact pairs named by the manifest
# ---------------------------------------------------------------------
# This intentionally does no directory traversal.
#
# Missing prediction + missing audit  -> pending work.
# One file without its partner        -> invalid interrupted checkpoint.
# Both files + matching completion ID -> safely recoverable unit.

def refresh_runner_up_checkpoint_status():
    """
    Reconcile the runner-up manifest with its exact durable checkpoint pairs.

    Recovery is intentionally path-driven rather than directory-driven so a
    huge Deepnote project cannot trigger an expensive recursive filesystem
    scan, and unrelated Parquet files can never be mistaken for checkpoints.
    """
    manifest_df = pd.read_parquet(
        RUNNER_UP_CHECKPOINT_MANIFEST_PATH
    )
    statuses = []

    for row in manifest_df.to_dict("records"):
        prediction_exists = Path(
            row["prediction_path"]
        ).exists()
        audit_exists = Path(
            row["audit_path"]
        ).exists()

        if not prediction_exists and not audit_exists:
            status = "pending"

        elif prediction_exists != audit_exists:
            status = "partial_invalid"

        else:
            status = (
                "recovered"
                if completed_holdout_unit_is_valid(row)
                else "invalid"
            )

        statuses.append(status)

    manifest_df["artifact_status"] = statuses

    invalid_mask = manifest_df[
        "artifact_status"
    ].isin(["partial_invalid", "invalid"])

    if invalid_mask.any():
        invalid_df = manifest_df.loc[
            invalid_mask,
            [
                "Family",
                "metric",
                "horizon",
                "refit_month",
                "artifact_status",
                "runtime_unit_id",
            ],
        ]

        raise RuntimeError(
            "Runner-up recovery found an incomplete or incompatible "
            "checkpoint pair:\n"
            + invalid_df.to_string(index=False)
        )

    write_small_contract(
        manifest_df,
        RUNNER_UP_CHECKPOINT_MANIFEST_PATH,
    )

    return manifest_df


write_small_contract(
    runner_up_checkpoint_manifest_df,
    RUNNER_UP_CHECKPOINT_MANIFEST_PATH,
)

runner_up_checkpoint_manifest_df = (
    refresh_runner_up_checkpoint_status()
)

runner_up_runtime_summary_df = (
    runner_up_checkpoint_manifest_df.assign(
        Recovered=lambda frame:
            frame["artifact_status"].eq("recovered").astype(int),
        Pending=lambda frame:
            frame["artifact_status"].eq("pending").astype(int),
    )
    .groupby(
        ["runner_up_family", "Family"],
        as_index=False,
        observed=True,
    )
    .agg(
        **{
            "Restart units": ("runtime_unit_id", "size"),
            "Recovered": ("Recovered", "sum"),
            "Pending": ("Pending", "sum"),
            "Planned training calls":
                ("planned_fit_calls", "sum"),
        }
    )
)

display(
    runner_up_runtime_summary_df[
        [
            "Family",
            "Restart units",
            "Recovered",
            "Pending",
            "Planned training calls",
        ]
    ]
)

print(
    "PASS — isolated runner-up restart contract is ready.\n"
    "  Restart units: 37\n"
    "  Training calls: 70\n"
    f"  Runtime root: {RUNNER_UP_RUNTIME_ROOT}\n\n"
    "Execution controls:\n"
    f"  RUN_RUNNER_UP_TREE={RUN_RUNNER_UP_TREE}\n"
    f"  RUN_RUNNER_UP_NEURAL={RUN_RUNNER_UP_NEURAL}\n"
    f"  RUN_RUNNER_UP_TRANSFORMER={RUN_RUNNER_UP_TRANSFORMER}\n"
    f"  RESET_RUNNER_UP_HOLDOUT_RUNTIME="
    f"{RESET_RUNNER_UP_HOLDOUT_RUNTIME}",
    flush=True,
)

,Family,Restart units,Recovered,Pending,Planned training calls
0,Neural,3,3,0,6
1,Transformer,30,30,0,60
2,Tree,4,4,0,4


PASS — isolated runner-up restart contract is ready.
  Restart units: 37
  Training calls: 70
  Runtime root: pipeline_data/4.6.1.intermediate_tables/final_holdout_runner_up_runtime_v1

Execution controls:
  RUN_RUNNER_UP_TREE=True
  RUN_RUNNER_UP_NEURAL=True
  RUN_RUNNER_UP_TRANSFORMER=False
  RESET_RUNNER_UP_HOLDOUT_RUNTIME=False


**Findings.** The runner-up audit has an isolated restart surface containing exactly **37 units**: **4 Tree, 3 Neural, and 30 Transformer**.

Each unit has its own prediction and audit destinations under the runner-up runtime root, preventing the robustness exercise from colliding with the frozen champion artifacts. No compatible runner-up checkpoint is currently recovered, so all **37 units remain pending** at this stage.

Execution controls enable the comparatively small **Tree and Neural** branches while leaving the **Transformer branch disabled**. The reset control also remains off, preserving restartability and preventing completed compatible work from being discarded unintentionally.

**4.6.1.8.1D — Preflight all 37 runner-up restart units without fitting.**

Before spending any additional compute, reconstruct every challenger at its natural execution grain.

Tree preflight verifies the exact LightGBM registry definition, predictor contract, legal training history, and holdout surface. Neural and Transformer preflight verify their family-specific lookback, legal modeling-track history boundary, historical selector populations, and monthly scoring ownership.

This is a structural test only. No model is fitted.

In [64]:
# ---------------------------------------------------------------------
# 4.6.1.8.1D — Preflight all runner-up restart units
# ---------------------------------------------------------------------

runner_up_checkpoint_manifest_df = (
    refresh_runner_up_checkpoint_status()
)


# ---------------------------------------------------------------------
# 1) Reconstruct the four Tree alternatives without fitting LightGBM
# ---------------------------------------------------------------------

tree_runner_manifest_df = (
    runner_up_checkpoint_manifest_df.loc[
        runner_up_checkpoint_manifest_df[
            "runner_up_family"
        ].eq("tree")
    ]
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(tree_runner_manifest_df) != 4:
    raise RuntimeError(
        f"Expected 4 Tree runner-up units; "
        f"found {len(tree_runner_manifest_df)}."
    )

tree_runner_preflight_records = []

for manifest_row in tree_runner_manifest_df.to_dict("records"):
    registry_row = selected_tree_registry_row(
        manifest_row["metric"],
        int(manifest_row["horizon"]),
    )

    source_configuration_id = (
        f"{registry_row['production_config_id']} · "
        f"{registry_row['target_representation']}"
    )

    # The challenger must resolve to the exact Tree finalist configuration
    # frozen before holdout; merely finding "a" valid Tree model is not enough.
    if source_configuration_id != manifest_row["configuration_id"]:
        raise RuntimeError(
            f"{manifest_row['metric']} · h={manifest_row['horizon']}: "
            "Tree runner-up configuration no longer matches "
            "the frozen source registry."
        )

    if str(registry_row["modeling_track"]) != str(
        manifest_row["modeling_track"]
    ):
        raise RuntimeError(
            f"{manifest_row['metric']} · h={manifest_row['horizon']}: "
            "Tree runner-up track no longer matches the source registry."
        )

    params = resolve_tree_model_params(registry_row)
    payload = load_tree_holdout_surface(manifest_row)

    tree_runner_preflight_records.append({
        "Family": "Tree",
        "Target": TARGET_LABELS.get(
            manifest_row["metric"],
            manifest_row["metric"],
        ),
        "Horizon": int(manifest_row["horizon"]),
        "Track": TRACK_LABELS.get(
            manifest_row["modeling_track"],
            manifest_row["modeling_track"],
        ),
        "Restart units": 1,
        "Train rows min": len(payload["train_df"]),
        "Train rows max": len(payload["train_df"]),
        "Score rows": len(payload["score_df"]),
        "Checkpoint": manifest_row["artifact_status"],
        "Ready": True,
        "Trees": int(params["n_estimators"]),
    })

tree_runner_preflight_df = pd.DataFrame(
    tree_runner_preflight_records
)


# ---------------------------------------------------------------------
# 2) Reconstruct all 33 monthly Neural/Transformer alternatives
# ---------------------------------------------------------------------
# Calling the frozen family resolvers first proves that every challenger maps
# to one supported source definition before the more expensive chronology scan.

sequence_runner_manifest_df = (
    runner_up_checkpoint_manifest_df.loc[
        runner_up_checkpoint_manifest_df[
            "runner_up_family"
        ].isin(["neural", "transformer"])
    ]
    .sort_values(
        ["runner_up_family", "metric", "horizon", "refit_step"]
    )
    .reset_index(drop=True)
)

if len(sequence_runner_manifest_df) != 33:
    raise RuntimeError(
        "Expected 33 Neural/Transformer runner-up units; "
        f"found {len(sequence_runner_manifest_df)}."
    )

for manifest_row in sequence_runner_manifest_df.to_dict("records"):
    family = str(manifest_row["runner_up_family"])
    track = str(manifest_row["modeling_track"])
    metric = str(manifest_row["metric"])
    horizon = int(manifest_row["horizon"])

    if family == "neural":
        resolve_neural_holdout_spec(track, metric, horizon)

    elif family == "transformer":
        resolve_transformer_holdout_spec(track, metric, horizon)

    else:
        raise RuntimeError(
            f"Unexpected sequence runner-up family: {family}"
        )

sequence_runner_unit_preflight_df = pd.DataFrame([
    preflight_sequence_holdout_unit(manifest_row)
    for manifest_row
    in sequence_runner_manifest_df.to_dict("records")
])

if not sequence_runner_unit_preflight_df["Ready"].all():
    failed_df = sequence_runner_unit_preflight_df.loc[
        ~sequence_runner_unit_preflight_df["Ready"]
    ]

    raise RuntimeError(
        "At least one sequence-model runner-up failed preflight:\n"
        + failed_df.to_string(index=False)
    )

# Reader-facing output stays at the 15-job level. Monthly units are execution
# details, so summarize their legal row ranges rather than dumping 33 rows.
sequence_runner_preflight_df = (
    sequence_runner_unit_preflight_df.groupby(
        ["Family", "Target", "Horizon", "Track"],
        as_index=False,
        observed=True,
    )
    .agg(
        **{
            "Restart units": ("Ready", "size"),
            "Train rows min": ("Train rows", "min"),
            "Train rows max": ("Train rows", "max"),
            "Score rows": ("Score rows", "sum"),
            "Ready": ("Ready", "all"),
        }
    )
)


# ---------------------------------------------------------------------
# 3) Present one compact 15-job readiness surface
# ---------------------------------------------------------------------

tree_display_df = tree_runner_preflight_df[
    [
        "Family",
        "Target",
        "Horizon",
        "Track",
        "Restart units",
        "Train rows min",
        "Train rows max",
        "Score rows",
        "Ready",
    ]
]

runner_up_preflight_df = pd.concat(
    [tree_display_df, sequence_runner_preflight_df],
    ignore_index=True,
)

target_order = {
    TARGET_LABELS[metric]: index
    for index, metric in enumerate(FORECAST_TARGETS)
}

runner_up_preflight_df = (
    runner_up_preflight_df.assign(
        _target_order=lambda frame:
            frame["Target"].map(target_order)
    )
    .sort_values(["_target_order", "Horizon"])
    .drop(columns="_target_order")
    .reset_index(drop=True)
)

if len(runner_up_preflight_df) != 15:
    raise RuntimeError(
        "Runner-up preflight no longer contains 15 jobs."
    )

if not runner_up_preflight_df["Ready"].all():
    raise RuntimeError(
        "At least one runner-up job failed structural preflight."
    )

if int(
    runner_up_preflight_df["Restart units"].sum()
) != 37:
    raise RuntimeError(
        "Runner-up preflight no longer covers 37 restart units."
    )

display(runner_up_preflight_df)

print(
    "PASS — all 37 runner-up restart units satisfy their frozen "
    "family execution contracts.\n"
    "No model fit was run.",
    flush=True,
)

,Family,Target,Horizon,Track,Restart units,Train rows min,Train rows max,Score rows,Ready
0,Transformer,Taxi trips,1,Post-CP Primary,3,450243,521043,110333,True
1,Neural,Taxi trips,2,Post-CP Primary,3,458707,529764,110333,True
2,Transformer,Taxi trips,5,Post-CP Primary,3,448189,520016,110333,True
3,Transformer,Taxi average speed,1,Post-CP Primary,3,324082,375260,79652,True
4,Transformer,Taxi average speed,2,Post-CP Primary,3,323716,375075,79652,True
5,Transformer,Taxi average speed,5,Post-CP Primary,3,322602,374516,79652,True
6,Transformer,FHVHV trips,1,Post-CP Primary,3,450243,521043,110333,True
7,Transformer,FHVHV trips,2,Post-CP Primary,3,449729,520786,110333,True
8,Transformer,FHVHV trips,5,Post-CP Primary,3,448189,520016,110333,True
9,Tree,FHVHV average speed,1,Full-History Regime-Aware,1,1410017,1410017,110231,True


PASS — all 37 runner-up restart units satisfy their frozen family execution contracts.
No model fit was run.


**Findings.** All **37 runner-up restart units pass structural preflight** under their frozen family-specific execution contracts. No model fit is performed by this check.

The Post-CP sequence jobs have substantial expanding-history support. Depending on Target × Horizon, January training populations begin at roughly **323,000–459,000 rows** and grow to approximately **375,000–530,000 rows** by the final monthly refit. Their holdout scoring populations contain about **79,652 Taxi-speed rows** or **110,333 trip rows** per job.

The Full-History alternatives also have ample support. The three Tree FHVHV-speed jobs each train on approximately **1.41 million rows**, while the Tree Subway h=2 job uses about **844,000 rows**. Transformer Subway jobs expand from approximately **832,000–834,000 training rows** to about **875,000–876,000**, with **64,511 holdout scoring rows** per horizon.

Every runner-up therefore reaches execution with nonzero training and scoring support, the expected modeling track, and a valid restart-unit structure. The robustness workload is ready for controlled fitting without changing any frozen selection or holdout-evaluation rule.

**4.6.1.8.1E — Execute or recover the frozen runner-up forecasts.**

Execution remains restartable and explicitly opt-in.

Each pending Tree unit uses the frozen LightGBM runner. Neural and Transformer units use the same monthly selector → discard → fresh-final-refit lifecycle used by their champion counterparts.

The runner-up artifacts live in their own runtime directory, so completing this audit cannot overwrite or alter any champion forecast used in the primary final-holdout evaluation.

When a valid prediction/audit pair already exists, the unit is recovered rather than fitted again.

In [65]:
# ---------------------------------------------------------------------
# 4.6.1.8.1E — Execute or recover runner-up holdout forecasts
# ---------------------------------------------------------------------

runner_up_checkpoint_manifest_df = (
    refresh_runner_up_checkpoint_status()
)

execution_controls = {
    "tree": RUN_RUNNER_UP_TREE,
    "neural": RUN_RUNNER_UP_NEURAL,
    "transformer": RUN_RUNNER_UP_TRANSFORMER,
}


# ---------------------------------------------------------------------
# Run only families that the reader explicitly enabled
# ---------------------------------------------------------------------
# Recovery happens first, so rerunning this cell never refits a compatible
# completed unit. This makes long GPU/CPU runs safely restartable.

for family in ["tree", "neural", "transformer"]:
    family_manifest_df = (
        runner_up_checkpoint_manifest_df.loc[
            runner_up_checkpoint_manifest_df[
                "runner_up_family"
            ].eq(family)
        ]
        .sort_values(["metric", "horizon", "refit_step"])
        .reset_index(drop=True)
    )

    recovered = int(
        family_manifest_df["artifact_status"].eq("recovered").sum()
    )
    pending = int(
        family_manifest_df["artifact_status"].eq("pending").sum()
    )

    print(
        f"\n{FAMILY_LABELS[family]} runner-up restart state\n"
        f"  Recovered: {recovered}/{len(family_manifest_df)}\n"
        f"  Pending:   {pending}/{len(family_manifest_df)}\n"
        f"  Execution enabled: {execution_controls[family]}",
        flush=True,
    )

    if not pending or not execution_controls[family]:
        continue

    pending_df = family_manifest_df.loc[
        family_manifest_df["artifact_status"].eq("pending")
    ].reset_index(drop=True)

    for position, manifest_row in enumerate(
        pending_df.to_dict("records"),
        start=1,
    ):
        target = TARGET_LABELS.get(
            manifest_row["metric"],
            manifest_row["metric"],
        )

        month_text = (
            ""
            if pd.isna(manifest_row["refit_month"])
            else f" · {manifest_row['refit_month']}"
        )

        print(
            f"\n{FAMILY_LABELS[family]} runner-up "
            f"[{position}/{len(pending_df)}] "
            f"{target} · h={int(manifest_row['horizon'])}"
            f"{month_text}",
            flush=True,
        )

        if family == "tree":
            result = execute_tree_holdout_unit(manifest_row)

            print(
                f"  complete · score={result['score_rows']:,} "
                f"· runtime={result['runtime_seconds'] / 60:.1f} min",
                flush=True,
            )

        else:
            result = execute_sequence_holdout_unit(manifest_row)

            print(
                f"  complete · selected epoch="
                f"{int(result['selected_epoch'])} "
                f"· score={int(result['score_rows']):,} "
                f"· runtime={float(result['runtime_seconds']) / 60:.1f} min",
                flush=True,
            )


# ---------------------------------------------------------------------
# Reconstruct status from durable artifacts rather than in-memory counters
# ---------------------------------------------------------------------

runner_up_checkpoint_manifest_df = (
    refresh_runner_up_checkpoint_status()
)

runner_up_execution_status_df = (
    runner_up_checkpoint_manifest_df.assign(
        Recovered=lambda frame:
            frame["artifact_status"].eq("recovered").astype(int),
        Pending=lambda frame:
            frame["artifact_status"].eq("pending").astype(int),
    )
    .groupby(
        ["runner_up_family", "Family"],
        as_index=False,
        observed=True,
    )
    .agg(
        **{
            "Restart units": ("runtime_unit_id", "size"),
            "Recovered": ("Recovered", "sum"),
            "Pending": ("Pending", "sum"),
            "Training calls": ("planned_fit_calls", "sum"),
        }
    )
)

display(
    runner_up_execution_status_df[
        [
            "Family",
            "Restart units",
            "Recovered",
            "Pending",
            "Training calls",
        ]
    ]
)

enabled_families = [
    family
    for family, enabled in execution_controls.items()
    if enabled
]

for family in enabled_families:
    family_pending = runner_up_checkpoint_manifest_df.loc[
        runner_up_checkpoint_manifest_df[
            "runner_up_family"
        ].eq(family),
        "artifact_status",
    ].ne("recovered").sum()

    if family_pending:
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} runner-up execution was enabled, "
            f"but {family_pending} unit(s) remain incomplete."
        )


Tree runner-up restart state
  Recovered: 4/4
  Pending:   0/4
  Execution enabled: True

Neural runner-up restart state
  Recovered: 3/3
  Pending:   0/3
  Execution enabled: True

Transformer runner-up restart state
  Recovered: 30/30
  Pending:   0/30
  Execution enabled: False


,Family,Restart units,Recovered,Pending,Training calls
0,Neural,3,3,0,6
1,Transformer,30,30,0,60
2,Tree,4,4,0,4


**Findings.** The isolated runner-up workload is fully recovered: all **37 of 37 restart units** are complete, with **0 pending**. This includes all **30 Transformer units**, **4 Tree units**, and **3 Neural units**, representing the full planned **70 training calls**.

The recovery result is especially important for the locally executed Transformer workload. Deepnote successfully rediscovered all 30 returned Transformer checkpoints under the frozen 8.1C contract, so the locally completed work re-entered the notebook without rerunning any model fits.

The runner-up audit can therefore proceed on the complete frozen workload rather than a partial recovery surface.

### 4.6.1.8.2 Test Champion-vs-Runner-Up Ranking Stability

Once both systems exist on unseen data, each champion/runner-up pair is placed on the same benchmark-supported population. Point-estimate differences are then tested with the paired calendar-week bootstrap and the frozen ±1% benchmark-MAE practical margin so the notebook can distinguish clear wins, reversals, practical equivalence, and unresolved comparisons.

**4.6.1.8.2A — Put each champion and runner-up on one identical unseen denominator.**

Raw runner-up output is not compared on whatever rows happen to be convenient for that family.

Instead, the comparison starts from the champion's already-predeclared benchmark-supported holdout population. The runner-up prediction is left-joined onto those physical rows. Where its advanced forecast is unavailable, the same frozen weekly benchmark fills the gap.

This reproduces the coverage-adjusted system logic of the original tournament:

`advanced runner-up prediction when available; weekly benchmark otherwise`.

Champion and runner-up MAE, Relative MAE, bias, and benchmark skill can therefore be compared on exactly the same unseen observations.

In [66]:
# ---------------------------------------------------------------------
# 4.6.1.8.2A — Score champion vs runner-up on one unseen denominator
# ---------------------------------------------------------------------

RUNNER_UP_HOLDOUT_COMPARISON_PATH = (
    FINAL_DIR / "champion_runner_up_holdout_rows.parquet"
)
RUNNER_UP_HOLDOUT_SCORECARD_PATH = (
    FINAL_DIR / "champion_runner_up_holdout_scorecard.parquet"
)

runner_up_checkpoint_manifest_df = (
    refresh_runner_up_checkpoint_status()
)

if not runner_up_checkpoint_manifest_df[
    "artifact_status"
].eq("recovered").all():
    pending = int(
        runner_up_checkpoint_manifest_df[
            "artifact_status"
        ].ne("recovered").sum()
    )

    raise RuntimeError(
        "Runner-up robustness scoring requires all 37 restart units. "
        f"{pending} unit(s) remain incomplete."
    )


# ---------------------------------------------------------------------
# 1) Point DuckDB at the EXACT 37 runner-up prediction artifacts
# ---------------------------------------------------------------------
# The row-level comparison contains millions of values across champion and
# challenger systems. Keep that work in DuckDB/Parquet rather than expanding
# both complete surfaces in Pandas.

runner_prediction_paths = [
    str(path)
    for path in runner_up_checkpoint_manifest_df[
        "prediction_path"
    ]
]

runner_path_sql = ", ".join(
    "'" + sql_path(path) + "'"
    for path in runner_prediction_paths
)

champion_path_sql = sql_path(
    FINAL_HOLDOUT_CHAMPION_PREDICTION_PATH
)

temporary_comparison_path = (
    RUNNER_UP_HOLDOUT_COMPARISON_PATH.with_name(
        f".{RUNNER_UP_HOLDOUT_COMPARISON_PATH.stem}.tmp"
        f"{RUNNER_UP_HOLDOUT_COMPARISON_PATH.suffix}"
    )
)

temporary_comparison_path.unlink(missing_ok=True)
temporary_comparison_sql = sql_path(
    temporary_comparison_path
)


# ---------------------------------------------------------------------
# 2) Register only the tiny 15-row matchup table in DuckDB
# ---------------------------------------------------------------------
# runner_up_registry_df tells the large join which executed family belongs
# to each Target × Horizon. The physical forecasts themselves remain on disk.

runner_matchup_df = runner_up_registry_df[
    [
        "metric",
        "horizon",
        "champion_family",
        "runner_up_family",
    ]
].copy()


with duckdb.connect() as connection:
    connection.register(
        "runner_matchup_df",
        runner_matchup_df,
    )

    # -----------------------------------------------------------------
    # 3) Refuse duplicate runner-up physical forecasts before joining
    # -----------------------------------------------------------------
    duplicate_runner_rows = connection.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT
                metric,
                horizon,
                forecast_sequence_id,
                taxi_zone_id,
                origin_observation_sequence_id,
                target_observation_sequence_id,
                target_date,
                COUNT(*) AS row_count

            FROM read_parquet([{runner_path_sql}])

            GROUP BY
                metric,
                horizon,
                forecast_sequence_id,
                taxi_zone_id,
                origin_observation_sequence_id,
                target_observation_sequence_id,
                target_date

            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    if duplicate_runner_rows:
        raise RuntimeError(
            "Runner-up artifacts contain duplicate physical forecasts."
        )

    # -----------------------------------------------------------------
    # 4) Join onto the champion's PREDECLARED benchmark-supported rows
    # -----------------------------------------------------------------
    # LEFT JOIN is deliberate: absence of an advanced challenger prediction
    # must not shrink the evaluation denominator. Such a row receives the same
    # weekly benchmark fallback used by the original tournament.
    connection.execute(f"""
        COPY (
            WITH
            champion_rows AS (
                SELECT
                    metric,
                    CAST(horizon AS INTEGER) AS horizon,
                    champion_family,
                    forecast_sequence_id,
                    CAST(taxi_zone_id AS BIGINT) AS taxi_zone_id,
                    CAST(origin_observation_sequence_id AS BIGINT)
                        AS origin_observation_sequence_id,
                    CAST(target_observation_sequence_id AS BIGINT)
                        AS target_observation_sequence_id,
                    CAST(target_date AS DATE) AS target_date,
                    CAST(actual AS DOUBLE) AS actual,
                    CAST(benchmark_prediction AS DOUBLE)
                        AS benchmark_prediction,
                    CAST(system_prediction AS DOUBLE)
                        AS champion_system_prediction,
                    CAST(model_prediction_available AS BOOLEAN)
                        AS champion_model_available

                FROM read_parquet('{champion_path_sql}')

                WHERE COALESCE(
                    CAST(primary_evaluation_eligible AS BOOLEAN),
                    FALSE
                )
            ),

            runner_rows AS (
                SELECT
                    metric,
                    CAST(horizon AS INTEGER) AS horizon,

                    -- Shared runner artifacts call the executed family
                    -- `champion_family`; in this diagnostic it is the runner-up.
                    CAST(champion_family AS VARCHAR)
                        AS runner_up_family,

                    forecast_sequence_id,
                    CAST(taxi_zone_id AS BIGINT) AS taxi_zone_id,
                    CAST(origin_observation_sequence_id AS BIGINT)
                        AS origin_observation_sequence_id,
                    CAST(target_observation_sequence_id AS BIGINT)
                        AS target_observation_sequence_id,
                    CAST(target_date AS DATE) AS target_date,

                    CAST(actual AS DOUBLE) AS runner_up_actual,
                    CAST(benchmark_prediction AS DOUBLE)
                        AS runner_up_benchmark_prediction,
                    CAST(prediction_raw AS DOUBLE)
                        AS runner_up_prediction_raw,
                    CAST(prediction_final AS DOUBLE)
                        AS runner_up_prediction_final,
                    COALESCE(
                        CAST(model_prediction_available AS BOOLEAN),
                        FALSE
                    ) AS stored_runner_up_model_available

                FROM read_parquet([{runner_path_sql}])
            ),

            aligned AS (
                SELECT
                    c.*,
                    m.runner_up_family,

                    r.runner_up_actual,
                    r.runner_up_benchmark_prediction,
                    r.runner_up_prediction_raw,
                    r.runner_up_prediction_final,

                    (
                        r.stored_runner_up_model_available
                        AND isfinite(r.runner_up_prediction_final)
                    ) AS runner_up_model_available

                FROM champion_rows AS c

                INNER JOIN runner_matchup_df AS m
                  ON c.metric = m.metric
                 AND c.horizon = m.horizon
                 AND c.champion_family = m.champion_family

                LEFT JOIN runner_rows AS r
                  ON c.metric = r.metric
                 AND c.horizon = r.horizon
                 AND m.runner_up_family = r.runner_up_family
                 AND c.forecast_sequence_id = r.forecast_sequence_id
                 AND c.taxi_zone_id = r.taxi_zone_id
                 AND c.origin_observation_sequence_id
                     = r.origin_observation_sequence_id
                 AND c.target_observation_sequence_id
                     = r.target_observation_sequence_id
                 AND c.target_date = r.target_date
            )

            SELECT
                *,
                NOT COALESCE(
                    runner_up_model_available,
                    FALSE
                ) AS runner_up_used_benchmark_fallback,

                CASE
                    WHEN COALESCE(
                        runner_up_model_available,
                        FALSE
                    )
                    THEN runner_up_prediction_final
                    ELSE benchmark_prediction
                END AS runner_up_system_prediction

            FROM aligned
        )
        TO '{temporary_comparison_sql}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    # -----------------------------------------------------------------
    # 5) Audit actual/benchmark identity wherever both artifacts have a row
    # -----------------------------------------------------------------
    # Different modeling tracks may make a runner-up row absent. That is
    # allowed because fallback preserves the denominator. But if the row does
    # exist, its observed outcome and weekly benchmark must describe the same
    # physical forecast as the champion artifact.

    alignment_audit = connection.execute(f"""
        SELECT
            COUNT(*) AS evaluation_rows,

            SUM(
                CASE
                    WHEN runner_up_actual IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS matched_runner_rows,

            MAX(
                CASE
                    WHEN runner_up_actual IS NOT NULL
                    THEN ABS(actual - runner_up_actual)
                    ELSE 0
                END
            ) AS max_actual_delta,

            MAX(
                CASE
                    WHEN runner_up_benchmark_prediction IS NOT NULL
                    THEN ABS(
                        benchmark_prediction
                        - runner_up_benchmark_prediction
                    )
                    ELSE 0
                END
            ) AS max_benchmark_delta

        FROM read_parquet('{temporary_comparison_sql}')
    """).df().iloc[0]

    if float(alignment_audit["max_actual_delta"]) > CROSS_ARTIFACT_ATOL:
        raise RuntimeError(
            "Champion and runner-up artifacts disagree on observed "
            "holdout outcomes."
        )

    if (
        float(alignment_audit["max_benchmark_delta"])
        > CROSS_ARTIFACT_ATOL
    ):
        raise RuntimeError(
            "Champion and runner-up artifacts disagree on the frozen "
            "weekly benchmark."
        )

    # Every canonical row must still have a finite system prediction after the
    # runner-up fallback rule is applied.
    nonfinite_system_rows = connection.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{temporary_comparison_sql}')
        WHERE NOT isfinite(runner_up_system_prediction)
    """).fetchone()[0]

    if nonfinite_system_rows:
        raise RuntimeError(
            "At least one canonical row lacks a usable runner-up system "
            "prediction after benchmark fallback."
        )

    # -----------------------------------------------------------------
    # 6) Score both systems in DuckDB on the SAME physical rows
    # -----------------------------------------------------------------
    runner_up_holdout_scorecard_df = connection.execute(f"""
        SELECT
            metric,
            CAST(horizon AS INTEGER) AS horizon,
            champion_family,
            runner_up_family,

            COUNT(*) AS evaluation_rows,

            AVG(
                ABS(champion_system_prediction - actual)
            ) AS champion_mae,

            AVG(
                ABS(runner_up_system_prediction - actual)
            ) AS runner_up_mae,

            AVG(
                ABS(benchmark_prediction - actual)
            ) AS benchmark_mae,

            AVG(ABS(actual)) AS observed_abs_mean,

            100.0
            * AVG(
                CAST(
                    COALESCE(runner_up_model_available, FALSE)
                    AS DOUBLE
                )
            ) AS runner_up_model_coverage_pct,

            ABS(
                AVG(champion_system_prediction - actual)
            ) AS champion_abs_bias,

            ABS(
                AVG(runner_up_system_prediction - actual)
            ) AS runner_up_abs_bias

        FROM read_parquet('{temporary_comparison_sql}')

        GROUP BY
            metric,
            horizon,
            champion_family,
            runner_up_family

        ORDER BY metric, horizon
    """).df()


# Publish only after all row-level integrity checks have passed.
temporary_comparison_path.replace(
    RUNNER_UP_HOLDOUT_COMPARISON_PATH
)


# ---------------------------------------------------------------------
# 7) Derive the compact reader-facing performance measures
# ---------------------------------------------------------------------

runner_up_holdout_scorecard_df[
    "champion_relative_mae_pct"
] = (
    100.0
    * runner_up_holdout_scorecard_df["champion_mae"]
    / runner_up_holdout_scorecard_df["observed_abs_mean"]
)

runner_up_holdout_scorecard_df[
    "runner_up_relative_mae_pct"
] = (
    100.0
    * runner_up_holdout_scorecard_df["runner_up_mae"]
    / runner_up_holdout_scorecard_df["observed_abs_mean"]
)

runner_up_holdout_scorecard_df[
    "champion_benchmark_skill_pct"
] = (
    100.0
    * (
        1.0
        - runner_up_holdout_scorecard_df["champion_mae"]
        / runner_up_holdout_scorecard_df["benchmark_mae"]
    )
)

runner_up_holdout_scorecard_df[
    "runner_up_benchmark_skill_pct"
] = (
    100.0
    * (
        1.0
        - runner_up_holdout_scorecard_df["runner_up_mae"]
        / runner_up_holdout_scorecard_df["benchmark_mae"]
    )
)

# Positive advantage = runner-up error is larger, so the champion is better.
# Negative advantage = the pre-holdout runner-up has lower unseen MAE.
runner_up_holdout_scorecard_df[
    "holdout_mae_advantage"
] = (
    runner_up_holdout_scorecard_df["runner_up_mae"]
    - runner_up_holdout_scorecard_df["champion_mae"]
)

runner_up_holdout_scorecard_df[
    "holdout_advantage_benchmark_pct"
] = (
    100.0
    * runner_up_holdout_scorecard_df["holdout_mae_advantage"]
    / runner_up_holdout_scorecard_df["benchmark_mae"]
)

runner_up_holdout_scorecard_df = (
    runner_up_holdout_scorecard_df
    .merge(
        runner_up_registry_df[
            [
                "metric",
                "horizon",
                "runner_up_basis",
                "validation_champion_mae",
                "validation_runner_up_mae",
                "validation_mae_advantage",
            ]
        ],
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
)

practical_margin = (
    PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION
    * runner_up_holdout_scorecard_df["benchmark_mae"]
)

runner_up_holdout_scorecard_df[
    "point_practical_outcome"
] = np.select(
    [
        runner_up_holdout_scorecard_df[
            "holdout_mae_advantage"
        ].gt(practical_margin),

        runner_up_holdout_scorecard_df[
            "holdout_mae_advantage"
        ].lt(-practical_margin),
    ],
    [
        "champion_practical_edge",
        "runner_up_practical_edge",
    ],
    default="within_practical_margin",
)

atomic_write_parquet(
    runner_up_holdout_scorecard_df,
    RUNNER_UP_HOLDOUT_SCORECARD_PATH,
)

display(
    runner_up_holdout_scorecard_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_SHORT_LABELS),
        Champion=lambda frame:
            frame["champion_family"].map(FAMILY_LABELS),
        **{
            "Runner-up": lambda frame:
                frame["runner_up_family"].map(FAMILY_LABELS),
        },
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "Runner-up",
            "runner_up_model_coverage_pct",
            "champion_mae",
            "runner_up_mae",
            "benchmark_mae",
            "holdout_advantage_benchmark_pct",
            "champion_benchmark_skill_pct",
            "runner_up_benchmark_skill_pct",
            "point_practical_outcome",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "runner_up_model_coverage_pct":
            "Runner-up coverage (%)",
        "champion_mae": "Champion MAE",
        "runner_up_mae": "Runner-up MAE",
        "benchmark_mae": "Benchmark MAE",
        "holdout_advantage_benchmark_pct":
            "Champion advantage (% benchmark MAE)",
        "champion_benchmark_skill_pct":
            "Champion skill (%)",
        "runner_up_benchmark_skill_pct":
            "Runner-up skill (%)",
        "point_practical_outcome":
            "Point-estimate outcome",
    })
    .round(3)
)

print(
    "PASS — all 15 champion/runner-up pairs scored on one "
    "identical unseen denominator.\n"
    f"  Comparison rows: "
    f"{int(alignment_audit['evaluation_rows']):,}\n"
    f"  Matched advanced-runner rows: "
    f"{int(alignment_audit['matched_runner_rows']):,}\n"
    f"  Output: {RUNNER_UP_HOLDOUT_SCORECARD_PATH}",
    flush=True,
)

,Target,Horizon,Champion,Runner-up,Runner-up coverage (%),Champion MAE,Runner-up MAE,Benchmark MAE,Champion advantage (% benchmark MAE),Champion skill (%),Runner-up skill (%),Point-estimate outcome
0,FHVHV speed,1,Transformer,Tree,100.000,0.954,0.985,1.501,2.046,36.460,34.414,champion_practical_edge
1,FHVHV speed,2,Transformer,Tree,100.000,1.014,1.054,1.501,2.631,32.428,29.797,champion_practical_edge
2,FHVHV speed,5,Transformer,Tree,100.000,1.041,1.037,1.501,-0.297,30.638,30.934,within_practical_margin
3,FHVHV trips,1,Tree,Transformer,99.720,57.622,64.763,92.439,7.726,37.665,29.939,champion_practical_edge
4,FHVHV trips,2,Tree,Transformer,99.720,65.738,74.299,92.439,9.261,28.885,19.624,champion_practical_edge
5,FHVHV trips,5,Tree,Transformer,99.720,66.102,81.820,92.439,17.003,28.491,11.487,champion_practical_edge
6,Subway,1,Tree,Transformer,100.000,407.487,436.723,635.067,4.604,35.836,31.232,champion_practical_edge
7,Subway,2,Transformer,Tree,100.000,512.990,441.433,635.067,-11.268,19.223,30.490,runner_up_practical_edge
8,Subway,5,Tree,Transformer,100.000,454.958,557.758,635.067,16.187,28.361,12.173,champion_practical_edge
9,Taxi speed,1,Neural,Transformer,100.000,2.137,2.159,2.990,0.754,28.525,27.771,within_practical_margin


PASS — all 15 champion/runner-up pairs scored on one identical unseen denominator.
  Comparison rows: 1,421,436
  Matched advanced-runner rows: 1,419,582
  Output: pipeline_data/4.6.1.final_tables/champion_runner_up_holdout_scorecard.parquet


**Findings.** All **15 frozen champion/runner-up pairs** were scored on the same unseen holdout denominator, covering **1,421,436 benchmark-supported rows**. Runner-up advanced forecasts were available for essentially the entire surface: coverage was **100%** for the speed and Subway comparisons and **99.720%** for the affected trip-volume comparisons. The point-estimate comparison is therefore driven overwhelmingly by the competing advanced models rather than by benchmark fallback.

At the point-estimate level, the frozen champion retained a **practically meaningful advantage in 11 of 15 jobs** under the ±1% benchmark-MAE margin. Three additional jobs fell within that practical margin: **FHVHV speed at horizon 5** and **Taxi speed at horizons 1 and 2**. Only **Subway ridership at horizon 2** showed a practically meaningful runner-up advantage, where Tree achieved MAE **441.433** versus **512.990** for the frozen Transformer champion.

The largest champion advantages appeared in several longer-horizon demand forecasts. Relative to benchmark MAE, the champion led by **22.532%** for Taxi trips at horizon 5, **17.003%** for FHVHV trips at horizon 5, and **16.187%** for Subway ridership at horizon 5. This suggests that several of the frozen validation choices continue to separate meaningfully on unseen data even as the forecast horizon expands.

These are still point estimates, however. A lower holdout MAE on one realized history does not by itself establish that the ordering is stable across plausible reweightings of the holdout weeks. The paired calendar-week bootstrap in the next step tests that stronger claim.

**4.6.1.8.2B — Distinguish genuine holdout reversals from small point-estimate noise.**

A runner-up can post a slightly lower MAE by chance, especially when the original selection was already close.

The final robustness check therefore repeats the tournament's paired calendar-week block bootstrap on the unseen period. Champion and runner-up absolute errors are resampled together by week, preserving the shared shocks that make their errors correlated.

The same practical margin remains fixed at **1% of frozen benchmark MAE**.

For each job, the result is classified as:

- **Champion clear win** — the paired interval supports an advantage larger than the practical margin;
- **Runner-up clear win** — the reverse advantage is larger than the practical margin;
- **Practical equivalence** — the full interval lies inside the equivalence band; or
- **Uncertain difference** — the holdout does not cleanly resolve the pair.

These labels describe robustness only. They cannot alter the frozen champion registry.

In [67]:
# ---------------------------------------------------------------------
# 4.6.1.8.2B — Paired weekly runner-up robustness bootstrap
# ---------------------------------------------------------------------

RUNNER_UP_ROBUSTNESS_FINAL_PATH = (
    FINAL_DIR / "champion_runner_up_holdout_robustness.parquet"
)


# ---------------------------------------------------------------------
# 1) Reduce the unseen comparison surface to calendar-week blocks
# ---------------------------------------------------------------------
# The saved comparison artifact is already compact enough for Pandas. Reading
# only the six required columns keeps the operation transparent while avoiding
# a second query language for a straightforward groupby.

runner_up_rows_df = pd.read_parquet(
    RUNNER_UP_HOLDOUT_COMPARISON_PATH,
    columns=[
        "metric",
        "horizon",
        "target_date",
        "actual",
        "champion_system_prediction",
        "runner_up_system_prediction",
    ],
)

runner_up_rows_df["target_date"] = pd.to_datetime(
    runner_up_rows_df["target_date"]
)

# DuckDB DATE_TRUNC('week') used Monday as the start; reproduce that contract
# explicitly so the bootstrap blocks remain identical.
runner_up_rows_df["week_start"] = (
    runner_up_rows_df["target_date"]
    - pd.to_timedelta(
        runner_up_rows_df["target_date"].dt.weekday,
        unit="D",
    )
).dt.normalize()

runner_up_rows_df["champion_absolute_error"] = (
    runner_up_rows_df["champion_system_prediction"]
    - runner_up_rows_df["actual"]
).abs()

runner_up_rows_df["runner_up_absolute_error"] = (
    runner_up_rows_df["runner_up_system_prediction"]
    - runner_up_rows_df["actual"]
).abs()

runner_up_weekly_df = (
    runner_up_rows_df.groupby(
        ["metric", "horizon", "week_start"],
        observed=True,
        as_index=False,
    )
    .agg(
        evaluation_rows=("actual", "size"),
        champion_absolute_error_sum=(
            "champion_absolute_error",
            "sum",
        ),
        runner_up_absolute_error_sum=(
            "runner_up_absolute_error",
            "sum",
        ),
    )
    .sort_values(
        ["metric", "horizon", "week_start"]
    )
    .reset_index(drop=True)
)

del runner_up_rows_df

def bootstrap_runner_up_robustness(
    weekly_df,
    metric,
    horizon,
    benchmark_mae,
    repeats,
    seed,
):
    """
    Bootstrap runner-up MAE minus champion MAE by calendar-week block.

    Positive advantage means the runner-up has MORE error and therefore the
    frozen champion performs better. Negative advantage means the runner-up
    has lower unseen MAE.

    Whole calendar weeks are resampled together so champion and runner-up see
    the same traffic, weather, holidays, and other shared weekly shocks in
    every bootstrap draw.
    """
    job_df = (
        weekly_df.loc[
            weekly_df["metric"].eq(metric)
            & weekly_df["horizon"].astype(int).eq(int(horizon))
        ]
        .sort_values("week_start")
        .reset_index(drop=True)
    )

    if len(job_df) < 2:
        raise RuntimeError(
            f"{metric} · h={horizon}: too few calendar-week blocks."
        )

    rows = job_df["evaluation_rows"].to_numpy(dtype=float)

    weekly_difference_sum = (
        job_df["runner_up_absolute_error_sum"].to_numpy(dtype=float)
        - job_df["champion_absolute_error_sum"].to_numpy(dtype=float)
    )

    point_advantage = (
        weekly_difference_sum.sum()
        / rows.sum()
    )

    rng = np.random.default_rng(seed)
    n_blocks = len(job_df)

    sample_indices = rng.integers(
        low=0,
        high=n_blocks,
        size=(repeats, n_blocks),
    )

    sampled_difference = (
        weekly_difference_sum[sample_indices]
        .sum(axis=1)
    )
    sampled_rows = (
        rows[sample_indices]
        .sum(axis=1)
    )

    bootstrap_advantage = (
        sampled_difference
        / sampled_rows
    )

    alpha = 1.0 - BOOTSTRAP_CONFIDENCE_LEVEL

    ci_low, ci_high = np.quantile(
        bootstrap_advantage,
        [alpha / 2.0, 1.0 - alpha / 2.0],
    )

    practical_margin = (
        PRACTICAL_EQUIVALENCE_BENCHMARK_FRACTION
        * benchmark_mae
    )

    # The original selection bootstrap only needed to prove whether its point
    # leader won. This diagnostic is symmetric because an unseen runner-up can
    # genuinely reverse the ordering.
    if ci_low > practical_margin:
        status = "champion_clear_win"

    elif ci_high < -practical_margin:
        status = "runner_up_clear_win"

    elif (
        ci_low >= -practical_margin
        and ci_high <= practical_margin
    ):
        status = "practical_equivalence"

    else:
        status = "uncertain_difference"

    return {
        "metric": metric,
        "horizon": int(horizon),
        "calendar_week_blocks": n_blocks,
        "point_advantage_mae": float(point_advantage),
        "ci_low_mae": float(ci_low),
        "ci_high_mae": float(ci_high),
        "practical_margin_mae": float(practical_margin),

        "point_advantage_benchmark_pct":
            100.0 * point_advantage / benchmark_mae,
        "ci_low_benchmark_pct":
            100.0 * ci_low / benchmark_mae,
        "ci_high_benchmark_pct":
            100.0 * ci_high / benchmark_mae,

        "comparison_status": status,
    }


# ---------------------------------------------------------------------
# 2) Apply the same paired test systematically to all 15 matchups
# ---------------------------------------------------------------------

bootstrap_records = []

for job_number, row in enumerate(
    runner_up_holdout_scorecard_df.itertuples(index=False),
    start=1,
):
    bootstrap_records.append(
        bootstrap_runner_up_robustness(
            weekly_df=runner_up_weekly_df,
            metric=row.metric,
            horizon=int(row.horizon),
            benchmark_mae=float(row.benchmark_mae),
            repeats=BOOTSTRAP_REPEATS,
            seed=BOOTSTRAP_RANDOM_SEED + job_number,
        )
    )

runner_up_bootstrap_df = pd.DataFrame(
    bootstrap_records
)

runner_up_robustness_df = (
    runner_up_holdout_scorecard_df
    .merge(
        runner_up_bootstrap_df,
        on=["metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
)

runner_up_robustness_df[
    "runner_up_lower_holdout_mae"
] = (
    runner_up_robustness_df[
        "holdout_mae_advantage"
    ].lt(0)
)

runner_up_robustness_df[
    "clear_runner_up_win"
] = (
    runner_up_robustness_df[
        "comparison_status"
    ].eq("runner_up_clear_win")
)

atomic_write_parquet(
    runner_up_robustness_df,
    RUNNER_UP_ROBUSTNESS_FINAL_PATH,
)


# ---------------------------------------------------------------------
# 3) Show both the 15-job evidence and its compact outcome mix
# ---------------------------------------------------------------------

status_labels = {
    "champion_clear_win": "Champion clear win",
    "runner_up_clear_win": "Runner-up clear win",
    "practical_equivalence": "Practical equivalence",
    "uncertain_difference": "Uncertain difference",
}

runner_up_robustness_display_df = (
    runner_up_robustness_df.assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_SHORT_LABELS),
        Champion=lambda frame:
            frame["champion_family"].map(FAMILY_LABELS),
        **{
            "Runner-up": lambda frame:
                frame["runner_up_family"].map(FAMILY_LABELS),
            "Bootstrap result": lambda frame:
                frame["comparison_status"].map(status_labels),
        },
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "Runner-up",
            "validation_mae_advantage",
            "holdout_mae_advantage",
            "point_advantage_benchmark_pct",
            "ci_low_benchmark_pct",
            "ci_high_benchmark_pct",
            "runner_up_model_coverage_pct",
            "Bootstrap result",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "validation_mae_advantage":
            "Validation MAE advantage",
        "holdout_mae_advantage":
            "Holdout MAE advantage",
        "point_advantage_benchmark_pct":
            "Holdout advantage (% benchmark MAE)",
        "ci_low_benchmark_pct":
            "95% low (% benchmark MAE)",
        "ci_high_benchmark_pct":
            "95% high (% benchmark MAE)",
        "runner_up_model_coverage_pct":
            "Runner-up coverage (%)",
    })
)

display(
    runner_up_robustness_display_df.round(3)
)

robustness_status_summary_df = (
    runner_up_robustness_df.assign(
        Status=lambda frame:
            frame["comparison_status"].map(status_labels)
    )
    .groupby(
        "Status",
        as_index=False,
        observed=True,
    )
    .size()
    .rename(columns={"size": "Jobs"})
    .sort_values("Jobs", ascending=False)
    .reset_index(drop=True)
)

display(robustness_status_summary_df)

print(
    "PASS — runner-up robustness audit complete.\n"
    f"  Jobs compared: {len(runner_up_robustness_df)}\n"
    f"  Runner-up lower point MAE: "
    f"{int(runner_up_robustness_df['runner_up_lower_holdout_mae'].sum())}\n"
    f"  Clear runner-up wins: "
    f"{int(runner_up_robustness_df['clear_runner_up_win'].sum())}\n"
    f"  Output: {RUNNER_UP_ROBUSTNESS_FINAL_PATH}",
    flush=True,
)

,Target,Horizon,Champion,Runner-up,Validation MAE advantage,Holdout MAE advantage,Holdout advantage (% benchmark MAE),95% low (% benchmark MAE),95% high (% benchmark MAE),Runner-up coverage (%),Bootstrap result
0,FHVHV speed,1,Transformer,Tree,0.017,0.031,2.046,-0.074,4.062,100.000,Uncertain difference
1,FHVHV speed,2,Transformer,Tree,0.009,0.039,2.631,-0.599,6.090,100.000,Uncertain difference
2,FHVHV speed,5,Transformer,Tree,-0.012,-0.004,-0.297,-3.053,2.779,100.000,Uncertain difference
3,FHVHV trips,1,Tree,Transformer,10.632,7.142,7.726,0.145,15.424,99.720,Uncertain difference
4,FHVHV trips,2,Tree,Transformer,9.427,8.561,9.261,-0.095,19.588,99.720,Uncertain difference
5,FHVHV trips,5,Tree,Transformer,12.470,15.718,17.003,8.059,27.642,99.720,Champion clear win
6,Subway,1,Tree,Transformer,30.730,29.236,4.604,-3.721,14.599,100.000,Uncertain difference
7,Subway,2,Transformer,Tree,2.148,-71.557,-11.268,-19.486,-4.266,100.000,Runner-up clear win
8,Subway,5,Tree,Transformer,35.843,102.801,16.187,9.201,24.741,100.000,Champion clear win
9,Taxi speed,1,Neural,Transformer,-0.010,0.023,0.754,0.158,1.379,100.000,Uncertain difference


,Status,Jobs
0,Uncertain difference,9
1,Champion clear win,4
2,Practical equivalence,1
3,Runner-up clear win,1


PASS — runner-up robustness audit complete.
  Jobs compared: 15
  Runner-up lower point MAE: 2
  Clear runner-up wins: 1
  Output: pipeline_data/4.6.1.final_tables/champion_runner_up_holdout_robustness.parquet


**Findings.** The paired calendar-week bootstrap produces a more cautious robustness picture than the point estimates alone. Of the **15 champion/runner-up comparisons**, **4 are clear champion wins**, **1 is practically equivalent**, **9 remain uncertain**, and **1 is a clear runner-up win** under the same ±1% benchmark-MAE practical margin used during validation.

The four clear champion wins are concentrated in demand forecasts: **FHVHV trips at horizon 5**, **Subway ridership at horizon 5**, and **Taxi trips at horizons 2 and 5**. Their 95% bootstrap intervals remain entirely beyond the +1% practical threshold, providing the strongest holdout evidence that the frozen champion advantage is both stable and large enough to matter.

**Taxi speed at horizon 2** is the only case classified as practical equivalence. Its holdout advantage is just **0.134% of benchmark MAE**, and the full 95% interval remains inside the ±1% practical-equivalence band. For the other nine uncertain comparisons, the observed point advantage is not supported strongly enough to declare either practical equivalence or a stable practical winner. This includes several cases with sizable point-estimate champion leads, illustrating why the bootstrap adds information beyond a single MAE ranking.

There is one clear holdout ranking reversal: for **Subway ridership at horizon 2**, the Tree runner-up outperforms the frozen Transformer champion by **11.268% of benchmark MAE**, with a 95% interval from **-19.486% to -4.266%**. Because this section is a pre-specified robustness audit rather than a second model-selection tournament, that result is recorded as evidence about ranking stability; it does **not** replace the already frozen champion.

Overall, the audit does not support a simplistic claim that every selected champion decisively dominates its runner-up on unseen data. Instead, it shows a mixed but informative pattern: several champion choices transfer with strong evidence, many remain statistically unresolved at the practical threshold, one pair is effectively indistinguishable, and one job reverses clearly. That is the appropriate robustness interpretation to carry forward into the final comparison visuals and the later diagnostic analysis.

**4.6.1.8.2C — Explain why some holdout comparisons remain uncertain.**

An **uncertain difference** does not mean that the bootstrap failed or that the two forecasting systems are necessarily tied. It means the 95% bootstrap interval does not fit wholly inside one of the predeclared decision regions.

A clear champion win requires the full interval to remain above the **+1% benchmark-MAE practical margin**; a clear runner-up win requires it to remain below **−1%**; and practical equivalence requires the entire interval to remain inside that ±1% band. An interval that crosses one of those boundaries therefore remains unresolved.

There are two substantively different reasons this can happen. Some intervals cross **zero**, meaning plausible reweightings of the holdout weeks can change which system has lower MAE. Others remain entirely on one side of zero but still cross the **practical threshold**, meaning the direction of the advantage is stable while its practical importance is not. The figure below isolates the uncertain jobs so those two cases are visible directly.

In [68]:
# ---------------------------------------------------------------------
# 4.6.1.8.2C — Visualize why holdout comparisons remain uncertain
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from project_branding import BRAND_COLORS, apply_branding


ROBUSTNESS_PATH = Path(
    "pipeline_data/4.6.1.final_tables/"
    "champion_runner_up_holdout_robustness.parquet"
)

PRACTICAL_MARGIN_PCT = 1.0

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway",
}


# ---------------------------------------------------------------------
# Load the compact final 8.2B artifact rather than reconstructing evidence.
# ---------------------------------------------------------------------
# This visualization explains the already-frozen robustness result. It does
# not rerun the bootstrap or reopen any model-selection logic.

robustness_df = pd.read_parquet(ROBUSTNESS_PATH)

required_columns = {
    "metric",
    "horizon",
    "champion_family",
    "runner_up_family",
    "point_advantage_benchmark_pct",
    "ci_low_benchmark_pct",
    "ci_high_benchmark_pct",
    "comparison_status",
}

missing_columns = required_columns - set(robustness_df.columns)

if missing_columns:
    raise RuntimeError(
        "8.2B robustness artifact is missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Normalize only the display representation of the saved status. The underlying
# 8.2B classification remains authoritative and is not recomputed here.

status_key = (
    robustness_df["comparison_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

uncertain_df = (
    robustness_df.loc[status_key.eq("uncertain_difference")]
    .copy()
    .reset_index(drop=True)
)

if uncertain_df.empty:
    raise RuntimeError(
        "8.2B contains no uncertain comparisons to visualize."
    )


# ---------------------------------------------------------------------
# Validate the interval geometry before interpreting why a job is uncertain.
# ---------------------------------------------------------------------
# The point estimate should fall inside its own bootstrap interval. Checking
# that here prevents a malformed saved result from becoming a persuasive plot.

numeric_columns = [
    "point_advantage_benchmark_pct",
    "ci_low_benchmark_pct",
    "ci_high_benchmark_pct",
]

if not np.isfinite(uncertain_df[numeric_columns].to_numpy(dtype=float)).all():
    raise RuntimeError(
        "The uncertain-comparison surface contains non-finite bootstrap values."
    )

point = uncertain_df["point_advantage_benchmark_pct"].astype(float)
ci_low = uncertain_df["ci_low_benchmark_pct"].astype(float)
ci_high = uncertain_df["ci_high_benchmark_pct"].astype(float)

if not ((ci_low <= point) & (point <= ci_high)).all():
    raise RuntimeError(
        "At least one saved point estimate falls outside its 95% interval."
    )


# ---------------------------------------------------------------------
# Separate the two reasons an already-uncertain comparison can remain open.
# ---------------------------------------------------------------------
# 1) If the interval crosses zero, even the direction of the MAE advantage can
#    change across plausible calendar-week resamples.
# 2) If the interval stays on one side of zero but overlaps ±1%, the direction
#    is stable while the size of the advantage is not securely practical.

direction_is_stable = (ci_low > 0) | (ci_high < 0)

uncertain_df["uncertainty_reason"] = np.where(
    direction_is_stable,
    "Direction stable; practical size uncertain",
    "Direction itself uncertain",
)

if uncertain_df["metric"].map(TARGET_LABELS).isna().any():
    unknown_metrics = sorted(
        uncertain_df.loc[
            uncertain_df["metric"].map(TARGET_LABELS).isna(),
            "metric",
        ].astype(str).unique()
    )
    raise RuntimeError(
        f"Missing display labels for targets: {unknown_metrics}"
    )

uncertain_df["job_label"] = (
    uncertain_df["metric"].map(TARGET_LABELS)
    + " · h="
    + uncertain_df["horizon"].astype(int).astype(str)
    + " · "
    + uncertain_df["champion_family"].astype(str).str.title()
    + " vs "
    + uncertain_df["runner_up_family"].astype(str).str.title()
)

# Sorting by the point estimate makes the chart read naturally from stronger
# runner-up evidence on the left toward stronger champion evidence on the right.
uncertain_df = uncertain_df.sort_values(
    ["point_advantage_benchmark_pct", "metric", "horizon"]
).reset_index(drop=True)

direction_unresolved_n = int(
    uncertain_df["uncertainty_reason"].eq(
        "Direction itself uncertain"
    ).sum()
)

practical_size_uncertain_n = int(
    uncertain_df["uncertainty_reason"].eq(
        "Direction stable; practical size uncertain"
    ).sum()
)

print(
    f"Uncertain jobs: {len(uncertain_df)}\n"
    f"  Direction itself uncertain:                  {direction_unresolved_n}\n"
    f"  Direction stable; practical size uncertain:  {practical_size_uncertain_n}",
    flush=True,
)


# ---------------------------------------------------------------------
# Draw an uncertain-only forest plot around the frozen ±1% decision margin.
# ---------------------------------------------------------------------
# Terracotta highlights jobs where even the winning direction can change.
# Dark teal marks jobs where the champion remains directionally ahead but the
# interval does not establish an advantage larger than the practical margin.

reason_colors = {
    "Direction itself uncertain": BRAND_COLORS["terracotta"],
    "Direction stable; practical size uncertain": BRAND_COLORS["dark_teal"],
}

fig = go.Figure()

# The shaded band is not a confidence interval. It is the predeclared region in
# which an MAE difference is too small to treat as practically meaningful.
fig.add_vrect(
    x0=-PRACTICAL_MARGIN_PCT,
    x1=PRACTICAL_MARGIN_PCT,
    fillcolor=BRAND_COLORS["pale_peach"],
    opacity=0.45,
    line_width=0,
    layer="below",
)

fig.add_vline(
    x=0,
    line_width=1.5,
    line_color=BRAND_COLORS["dark_teal"],
)

for threshold in (-PRACTICAL_MARGIN_PCT, PRACTICAL_MARGIN_PCT):
    fig.add_vline(
        x=threshold,
        line_width=1.5,
        line_dash="dot",
        line_color=BRAND_COLORS["terracotta"],
    )


# Plot each reason separately so the legend teaches the distinction directly.
for reason in [
    "Direction itself uncertain",
    "Direction stable; practical size uncertain",
]:
    subset = uncertain_df.loc[
        uncertain_df["uncertainty_reason"].eq(reason)
    ].copy()

    if subset.empty:
        continue

    point_values = subset["point_advantage_benchmark_pct"].astype(float)
    lower_values = subset["ci_low_benchmark_pct"].astype(float)
    upper_values = subset["ci_high_benchmark_pct"].astype(float)

    fig.add_trace(
        go.Scatter(
            x=point_values,
            y=subset["job_label"],
            mode="markers",
            name=reason,
            marker={
                "size": 10,
                "color": reason_colors[reason],
            },
            error_x={
                "type": "data",
                "symmetric": False,
                "array": (upper_values - point_values).to_numpy(),
                "arrayminus": (point_values - lower_values).to_numpy(),
                "color": reason_colors[reason],
                "thickness": 2,
                "width": 5,
            },
            customdata=subset[
                [
                    "ci_low_benchmark_pct",
                    "ci_high_benchmark_pct",
                    "champion_family",
                    "runner_up_family",
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Point advantage: %{x:.3f}% of benchmark MAE<br>"
                "95% interval: [%{customdata[0]:.3f}, "
                "%{customdata[1]:.3f}]<br>"
                "Champion: %{customdata[2]}<br>"
                "Runner-up: %{customdata[3]}"
                "<extra></extra>"
            ),
        )
    )

fig = apply_branding(fig)

fig.update_layout(
    title={
        "text": (
            f"Why {len(uncertain_df)} holdout comparisons remain uncertain"
            "<br><sup>Crossing 0 leaves the winner unresolved; crossing ±1% "
            "leaves practical importance unresolved.</sup>"
        ),
        "x": 0.01,
        "xanchor": "left",
    },
    xaxis_title=(
        "Holdout MAE advantage (% of benchmark MAE) "
        "— positive favors the frozen champion"
    ),
    yaxis_title="",
    height=max(590, 54 * len(uncertain_df) + 170),
    margin={"l": 265, "r": 45, "t": 105, "b": 75},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.01,
        "xanchor": "left",
        "x": 0,
    },
)

fig.update_xaxes(
    zeroline=False,
    gridcolor="rgba(0, 109, 119, 0.12)",
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=uncertain_df["job_label"].tolist(),
    gridcolor="rgba(0, 0, 0, 0)",
)

fig.show()

Uncertain jobs: 9
  Direction itself uncertain:                  6
  Direction stable; practical size uncertain:  3


**Findings.** The nine uncertain comparisons separate into two different forms of uncertainty. For **six jobs**, the 95% weekly-block interval crosses zero, so plausible reweightings of the unseen holdout weeks can change which forecasting system has lower MAE. These jobs therefore lack a stable directional winner on the observed holdout period.

The remaining **three jobs**—**FHVHV trips at h=1, Taxi speed at h=1, and Taxi trips at h=1**—have intervals that remain entirely on the champion-positive side of zero. Their direction is therefore stable, but their lower confidence bound does not clear the **+1% benchmark-MAE practical threshold**. In these cases the evidence supports a champion advantage, but not one that can be declared reliably large enough to matter under the notebook's predeclared margin.

This explains why some sizable point-estimate advantages in 8.2A become `uncertain_difference` after the weekly bootstrap. The classification requires both **stability across plausible weekly histories** and, for a clear win, an advantage that remains beyond the practical threshold—not merely a favorable point estimate.

### Section Summary

Section 8 tests ranking stability without reopening model selection. All **15 champion/runner-up pairs** are evaluated on the same unseen denominator, and the paired weekly bootstrap yields **4 clear champion wins, 1 practical equivalence, 9 uncertain differences, and 1 clear runner-up win**. Six uncertain intervals cross zero, while three preserve the champion's direction without reliably clearing the +1% practical threshold. The one clear reversal is **Subway h=2**; it is retained as robustness evidence rather than used to replace the frozen champion.

## 4.6.1.9 Visualize, Diagnose, and Freeze the Forecast-Evaluation Handoff

The tournament and unseen-data tests are complete. This final analytical section turns those results into reader-facing evidence: what won, how well validation transferred, how stable the champion ranking remained, how horizon affects forecasting, what the forecasts look like through time, and where reliability is geographically strong or weak.

These diagnostics do not create another selection stage. They make the frozen forecasting system inspectable and package the evidence that notebook 4.6.2 will use to explain its successes and failures.

### 4.6.1.9.1 Summarize Tournament and Generalization Results

The first views summarize the system at the Target × Horizon level. They show which family won each job, whether validation performance transferred to unseen data, how decisively each champion separated from its strongest alternative, and how forecast quality changes as the horizon moves farther away.

**4.6.1.9.1A — Show the final 5 × 3 champion board.**

The complete forecasting system contains one frozen champion for each of the **15 Target × Horizon jobs**.

Each cell below identifies the family selected from ordinary validation and reports that champion's **benchmark skill on the genuinely unseen final holdout**. Positive skill means the deployed forecasting system improved on **Previous week · same Daypart** over the exact same physical rows.

This gives the reader one compact answer to two questions: **which model did we select, and did that frozen choice still add value when it encountered new data?**

In [69]:
# ---------------------------------------------------------------------
# 4.6.1.9.1A — Show the final 5 × 3 champion board
# ---------------------------------------------------------------------

FINAL_CHAMPION_BOARD_PATH = (
    FINAL_DIR / "final_champion_board.parquet"
)


# ---------------------------------------------------------------------
# 1) Join the frozen selection to unseen-holdout performance
# ---------------------------------------------------------------------
# Durable artifacts remain authoritative here. The visualization should not
# depend on whatever DataFrames happen to remain in notebook memory.

champion_board_df = (
    pd.read_parquet(CHAMPION_REGISTRY_PATH)
    .merge(
        pd.read_parquet(FINAL_HOLDOUT_SCORECARD_PATH),
        on=["metric", "horizon", "champion_family"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

require_columns(
    champion_board_df,
    [
        "metric", "horizon", "champion_family",
        "champion_frozen", "holdout_used_for_selection",
        "mae", "relative_mae_pct", "benchmark_skill_pct", "bias",
    ],
    "final champion board",
)

if len(champion_board_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Final champion board does not contain exactly 15 jobs."
    )

if not champion_board_df["champion_frozen"].fillna(False).all():
    raise RuntimeError(
        "Final champion board contains an unfrozen selection."
    )

if champion_board_df["holdout_used_for_selection"].fillna(True).any():
    raise RuntimeError(
        "Final champion board indicates that holdout evidence "
        "influenced model selection."
    )

atomic_write_parquet(
    champion_board_df,
    FINAL_CHAMPION_BOARD_PATH,
)


# ---------------------------------------------------------------------
# 2) Arrange the 15 jobs as a reader-facing Target × Horizon board
# ---------------------------------------------------------------------

target_order = [
    TARGET_SHORT_LABELS[metric]
    for metric in FORECAST_TARGETS
]
horizon_order = [int(horizon) for horizon in FORECAST_HORIZONS]

board_plot_df = champion_board_df.assign(
    Target=lambda frame: frame["metric"].map(TARGET_SHORT_LABELS),
    Champion=lambda frame: frame["champion_family"].map(FAMILY_LABELS),
)

board_plot_df["cell_text"] = board_plot_df.apply(
    lambda row: (
        f"{row['Champion']}<br>"
        f"{row['benchmark_skill_pct']:.1f}% skill"
    ),
    axis=1,
)

skill_matrix = (
    board_plot_df.pivot(
        index="Target",
        columns="horizon",
        values="benchmark_skill_pct",
    )
    .reindex(index=target_order, columns=horizon_order)
)

text_matrix = (
    board_plot_df.pivot(
        index="Target",
        columns="horizon",
        values="cell_text",
    )
    .reindex(index=target_order, columns=horizon_order)
)

hover_matrix = (
    board_plot_df.assign(
        hover=lambda frame: frame.apply(
            lambda row: (
                f"Champion: {row['Champion']}<br>"
                f"Holdout MAE: {row['mae']:.3f}<br>"
                f"Relative MAE: {row['relative_mae_pct']:.3f}%<br>"
                f"Benchmark skill: {row['benchmark_skill_pct']:.3f}%<br>"
                f"Bias: {row['bias']:.3f}"
            ),
            axis=1,
        )
    )
    .pivot(index="Target", columns="horizon", values="hover")
    .reindex(index=target_order, columns=horizon_order)
)

fig = go.Figure(
    go.Heatmap(
        z=skill_matrix.to_numpy(),
        x=[f"h={horizon}" for horizon in horizon_order],
        y=target_order,
        text=text_matrix.to_numpy(),
        texttemplate="%{text}",
        customdata=hover_matrix.to_numpy(),
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        colorbar=dict(title="Holdout skill (%)"),
        hovertemplate=(
            "<b>%{y} · %{x}</b><br>"
            "%{customdata}<extra></extra>"
        ),
    )
)

apply_branding(fig)

fig.update_layout(
    title=(
        "Frozen 15-job forecasting system"
        "<br><sup>Cell color = unseen-holdout skill versus "
        "Previous week · same Daypart</sup>"
    ),
    xaxis_title="Forecast horizon",
    yaxis_title="",
    height=520,
)

fig.update_yaxes(autorange="reversed")
fig.show()

print(
    "PASS — final 15-job champion board is frozen.\n"
    f"Output: {FINAL_CHAMPION_BOARD_PATH}",
    flush=True,
)

PASS — final 15-job champion board is frozen.
Output: pipeline_data/4.6.1.final_tables/final_champion_board.parquet


**Findings.** The final forecasting system remains a deliberately mixed **8 Tree / 4 Transformer / 3 Neural** set of Target × Horizon champions rather than forcing one model family across every mobility problem.

Every selected job retains positive value on unseen data: final-holdout benchmark skill ranges from approximately **19.2% to 37.7%** across the 15 jobs.

The board therefore summarizes both the model-selection result and its unseen-period value: **which family ordinary validation selected, and how much that frozen forecasting system subsequently improved on the weekly benchmark.**

**4.6.1.9.1B — Show how validation performance transferred to unseen data.**

A forecasting system can look strong during model development and still disappoint when conditions change.

For each frozen Target × Horizon champion, the next view connects its **validation benchmark skill** to its **final-holdout benchmark skill**. Moving to the right means the champion gained benchmark-relative advantage on unseen data; moving left means some of that advantage weakened.

Because benchmark skill is scale-free, the five mobility targets can appear in one figure without comparing incompatible native-unit MAEs.

In [70]:
# ---------------------------------------------------------------------
# 4.6.1.9.1B — Visualize validation-to-holdout transfer
# ---------------------------------------------------------------------

generalization_plot_df = pd.read_parquet(
    FINAL_HOLDOUT_GENERALIZATION_PATH
).copy()

require_columns(
    generalization_plot_df,
    [
        "metric", "horizon", "champion_family",
        "validation_benchmark_skill_pct",
        "holdout_benchmark_skill_pct",
        "validation_relative_mae_pct",
        "holdout_relative_mae_pct",
        "benchmark_skill_change_pp",
        "relative_mae_change_pp",
    ],
    "validation-to-holdout generalization table",
)

if len(generalization_plot_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Generalization plot input does not contain exactly 15 jobs."
    )

job_order = [
    f"{TARGET_SHORT_LABELS[metric]} · h={horizon}"
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
]

generalization_plot_df["job_label"] = generalization_plot_df.apply(
    lambda row: (
        f"{TARGET_SHORT_LABELS[row['metric']]} · h={int(row['horizon'])}"
    ),
    axis=1,
)

generalization_plot_df["job_label"] = pd.Categorical(
    generalization_plot_df["job_label"],
    categories=job_order,
    ordered=True,
)

generalization_plot_df = (
    generalization_plot_df
    .sort_values("job_label")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Build all 15 connecting segments as one lightweight Plotly trace
# ---------------------------------------------------------------------

connector_x = []
connector_y = []

for row in generalization_plot_df.itertuples(index=False):
    connector_x.extend([
        row.validation_benchmark_skill_pct,
        row.holdout_benchmark_skill_pct,
        None,
    ])
    connector_y.extend([
        row.job_label,
        row.job_label,
        None,
    ])

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=connector_x,
        y=connector_y,
        mode="lines",
        line=dict(
            width=2,
            color="rgba(90, 110, 115, 0.35)",
        ),
        hoverinfo="skip",
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=generalization_plot_df[
            "validation_benchmark_skill_pct"
        ],
        y=generalization_plot_df["job_label"],
        mode="markers",
        name="Validation",
        marker=dict(
            size=9,
            color=BRAND_COLOR_SEQUENCE[1],
        ),
        customdata=np.column_stack([
            generalization_plot_df[
                "validation_relative_mae_pct"
            ],
            generalization_plot_df[
                "champion_family"
            ].map(FAMILY_LABELS),
        ]),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Champion: %{customdata[1]}<br>"
            "Validation skill: %{x:.3f}%<br>"
            "Validation Relative MAE: %{customdata[0]:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=generalization_plot_df[
            "holdout_benchmark_skill_pct"
        ],
        y=generalization_plot_df["job_label"],
        mode="markers",
        name="Unseen holdout",
        marker=dict(
            size=10,
            color=BRAND_COLOR_SEQUENCE[0],
        ),
        customdata=np.column_stack([
            generalization_plot_df[
                "holdout_relative_mae_pct"
            ],
            generalization_plot_df[
                "benchmark_skill_change_pp"
            ],
            generalization_plot_df[
                "champion_family"
            ].map(FAMILY_LABELS),
        ]),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Champion: %{customdata[2]}<br>"
            "Holdout skill: %{x:.3f}%<br>"
            "Holdout Relative MAE: %{customdata[0]:.3f}%<br>"
            "Skill change: %{customdata[1]:+.3f} pp"
            "<extra></extra>"
        ),
    )
)

apply_branding(fig)

fig.update_layout(
    title=(
        "Validation-selected champions transfer strongly to unseen data"
        "<br><sup>Lines connect each job's validation and "
        "final-holdout benchmark skill</sup>"
    ),
    xaxis_title="Benchmark skill (%)",
    yaxis_title="",
    height=650,
    legend=dict(
        orientation="h",
        y=-0.10,
    ),
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=job_order[::-1],
)

fig.show()

**Findings.** The validation-to-holdout picture is broadly stable rather than showing a collapse outside the selection period.

Holdout benchmark skill improves for **14 of the 15 frozen jobs**. The only decline is Taxi trips h=5, where skill decreases by about **0.6 percentage points** while remaining strongly positive at roughly **29.3%**.

Relative MAE increases modestly across all 15 jobs, with the larger shifts concentrated in demand forecasting rather than speed forecasting. That pattern motivates the deeper failure analysis in 4.6.2 without undermining the overall unseen-period value of the frozen system.

**4.6.1.9.1C — Show how decisively each champion beat its strongest pre-holdout alternative.**

A first-place model and second-place model can have different labels while still being practically indistinguishable.

The runner-up audit therefore needs more than another winner column. The next figure expresses the champion's unseen-holdout MAE advantage as a **percentage of the weekly benchmark MAE** and places its weekly-block bootstrap interval around that difference.

The shaded ±1% region is the same practical-equivalence margin frozen before the comparison. Points to the right favor the champion; points to the left favor the runner-up. The champion mapping itself remains unchanged regardless of the holdout result.

In [71]:
# ---------------------------------------------------------------------
# 4.6.1.9.1C — Visualize champion-versus-runner-up robustness
# ---------------------------------------------------------------------

runner_up_robustness_df = pd.read_parquet(
    RUNNER_UP_ROBUSTNESS_FINAL_PATH
).copy()

# 8.2B already converts the native-unit bootstrap result into % of benchmark
# MAE. Reuse that authoritative output instead of rebuilding the normalization
# here under a second set of column names.
require_columns(
    runner_up_robustness_df,
    [
        "metric", "horizon",
        "champion_family", "runner_up_family",
        "holdout_mae_advantage",
        "point_advantage_mae", "ci_low_mae", "ci_high_mae",
        "point_advantage_benchmark_pct",
        "ci_low_benchmark_pct", "ci_high_benchmark_pct",
        "comparison_status",
    ],
    "champion-versus-runner-up holdout robustness",
)

if len(runner_up_robustness_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Runner-up robustness artifact does not contain exactly 15 jobs."
    )

# 8.2A and 8.2B derive the same unseen-holdout MAE difference through two
# equivalent routes. Keeping this lightweight assertion protects the handoff
# without rerunning the bootstrap or reopening the underlying row-level data.
if not np.allclose(
    runner_up_robustness_df["holdout_mae_advantage"],
    runner_up_robustness_df["point_advantage_mae"],
    atol=CROSS_ARTIFACT_ATOL,
    rtol=CROSS_ARTIFACT_RTOL,
    equal_nan=False,
):
    raise RuntimeError(
        "Runner-up point advantage disagrees between the holdout scorecard "
        "and weekly-bootstrap artifact."
    )


# ---------------------------------------------------------------------
# 1) Put the 15 jobs in the notebook's canonical Target × Horizon order
# ---------------------------------------------------------------------

runner_up_robustness_df["job_label"] = runner_up_robustness_df.apply(
    lambda row: (
        f"{TARGET_SHORT_LABELS[row['metric']]} · "
        f"h={int(row['horizon'])}"
    ),
    axis=1,
)

job_order = [
    f"{TARGET_SHORT_LABELS[metric]} · h={horizon}"
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
]

runner_up_robustness_df["job_label"] = pd.Categorical(
    runner_up_robustness_df["job_label"],
    categories=job_order,
    ordered=True,
)

runner_up_robustness_df = (
    runner_up_robustness_df
    .sort_values("job_label")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 2) Draw the forest plot using the frozen ±1% practical-equivalence band
# ---------------------------------------------------------------------

status_labels = {
    "champion_clear_win": "Champion clear win",
    "runner_up_clear_win": "Runner-up clear win",
    "practical_equivalence": "Practical equivalence",
    "uncertain_difference": "Uncertain difference",
}

# Semantic colors make the statistical interpretation visible at a glance:
# champion-favoring evidence is teal, runner-up reversals are terracotta, and
# unresolved/equivalent cases use the softer supporting project colors.
status_colors = {
    "champion_clear_win": BRAND_COLORS["dark_teal"],
    "runner_up_clear_win": BRAND_COLORS["terracotta"],
    "practical_equivalence": BRAND_COLORS["seafoam"],
    "uncertain_difference": BRAND_COLORS["pale_peach"],
}

fig = go.Figure()

# Because 8.2B expresses the interval as % of benchmark MAE, the frozen
# practical margin is exactly ±1 on this axis.
fig.add_vrect(
    x0=-1.0,
    x1=1.0,
    fillcolor="rgba(120, 140, 145, 0.12)",
    line_width=0,
    annotation_text="Practical-equivalence band",
    annotation_position="top left",
)

fig.add_vline(
    x=0,
    line_width=1.5,
    line_dash="dash",
    line_color="rgba(60, 80, 85, 0.65)",
)

for status, status_label in status_labels.items():
    status_df = runner_up_robustness_df.loc[
        runner_up_robustness_df["comparison_status"].eq(status)
    ].copy()

    if status_df.empty:
        continue

    point = status_df["point_advantage_benchmark_pct"].to_numpy(dtype=float)
    low = status_df["ci_low_benchmark_pct"].to_numpy(dtype=float)
    high = status_df["ci_high_benchmark_pct"].to_numpy(dtype=float)

    fig.add_trace(
        go.Scatter(
            x=point,
            y=status_df["job_label"],
            mode="markers",
            name=status_label,
            marker=dict(
                size=10,
                color=status_colors[status],
            ),
            error_x=dict(
                type="data",
                symmetric=False,
                array=np.maximum(0.0, high - point),
                arrayminus=np.maximum(0.0, point - low),
                thickness=1.3,
                width=3,
            ),
            customdata=np.column_stack([
                status_df["champion_family"].map(FAMILY_LABELS),
                status_df["runner_up_family"].map(FAMILY_LABELS),
                status_df["point_advantage_mae"],
                status_df["ci_low_mae"],
                status_df["ci_high_mae"],
            ]),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Champion: %{customdata[0]}<br>"
                "Runner-up: %{customdata[1]}<br>"
                "Advantage: %{x:+.3f}% of benchmark MAE<br>"
                "Native-unit advantage: %{customdata[2]:+.4f}<br>"
                "Native-unit CI: [%{customdata[3]:+.4f}, "
                "%{customdata[4]:+.4f}]"
                "<extra></extra>"
            ),
        )
    )


# ---------------------------------------------------------------------
# 3) Apply shared branding, then preserve this chart's semantic overrides
# ---------------------------------------------------------------------

apply_branding(fig)

fig.update_layout(
    title=(
        "How decisively did each frozen champion beat its runner-up?"
        "<br><sup>Positive values favor the champion; "
        "holdout results do not change selection</sup>"
    ),
    xaxis_title="Champion MAE advantage (% of weekly-benchmark MAE)",
    yaxis_title="",
    height=690,
    legend=dict(
        orientation="h",
        y=-0.1,
    ),
)

# Reverse the category display so the first canonical job appears at the top.
fig.update_yaxes(
    categoryorder="array",
    categoryarray=job_order[::-1],
)

fig.show()

**Findings.** The unseen champion-versus-runner-up audit is more cautious than a simple point-estimate ranking. Across the **15 forecasting jobs**, **4 champions earn a clear holdout win**, **1 comparison is practically equivalent**, **9 remain uncertain**, and **1 produces a clear runner-up win** under the frozen ±1% benchmark-MAE practical margin.

The four clear champion wins are **FHVHV trips h=5, Subway h=5, Taxi trips h=2, and Taxi trips h=5**. **Taxi speed h=2** is practically equivalent. The one clear holdout reversal occurs for **Subway h=2**, where the Tree runner-up beats the frozen Transformer champion.

The nine uncertain jobs should not be read as nine ties. Six have intervals that cross zero, so plausible weekly reweightings can change which system has lower MAE. Three preserve the champion's directional advantage but do not clear the +1% practical threshold with 95% bootstrap support. Because the holdout audit measures ranking stability rather than reopening selection, none of these results changes the frozen champion registry.

**4.6.1.9.1D — Show how forecast quality changes as the horizon moves farther away.**

Forecasting should usually become harder as the model is asked to look farther into the future, but different mobility targets need not degrade at the same rate.

The next two panels follow the final champion system from **h=1 to h=2 to h=5**. Relative MAE shows how large the typical error is compared with the observed mobility level, while benchmark skill shows whether the advanced system continues to add value over the weekly benchmark.

The connected points represent the final system at each horizon; the underlying champion family may change because model selection was performed independently for every Target × Horizon job.

In [72]:
# ---------------------------------------------------------------------
# 4.6.1.9.1D — Visualize horizon degradation
# ---------------------------------------------------------------------

horizon_plot_df = pd.read_parquet(
    FINAL_HOLDOUT_SCORECARD_PATH
).copy()

require_columns(
    horizon_plot_df,
    [
        "metric", "horizon", "champion_family",
        "relative_mae_pct", "benchmark_skill_pct",
    ],
    "final-holdout horizon profile",
)

target_colors = {
    metric: BRAND_COLOR_SEQUENCE[
        index % len(BRAND_COLOR_SEQUENCE)
    ]
    for index, metric in enumerate(FORECAST_TARGETS)
}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Relative error",
        "Value beyond the weekly benchmark",
    ],
    horizontal_spacing=0.12,
)

for metric in FORECAST_TARGETS:
    metric_df = (
        horizon_plot_df.loc[
            horizon_plot_df["metric"].eq(metric)
        ]
        .sort_values("horizon")
        .copy()
    )

    customdata = np.column_stack([
        metric_df[
            "champion_family"
        ].map(FAMILY_LABELS),
        metric_df["benchmark_skill_pct"],
        metric_df["relative_mae_pct"],
    ])

    fig.add_trace(
        go.Scatter(
            x=metric_df["horizon"],
            y=metric_df["relative_mae_pct"],
            mode="lines+markers",
            name=TARGET_SHORT_LABELS[metric],
            legendgroup=metric,
            line=dict(
                color=target_colors[metric],
                width=2.5,
            ),
            marker=dict(size=9),
            customdata=customdata,
            hovertemplate=(
                "<b>" + TARGET_LABELS[metric] + "</b><br>"
                "Horizon: h=%{x}<br>"
                "Champion: %{customdata[0]}<br>"
                "Relative MAE: %{y:.3f}%<br>"
                "Benchmark skill: %{customdata[1]:.3f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=metric_df["horizon"],
            y=metric_df["benchmark_skill_pct"],
            mode="lines+markers",
            name=TARGET_SHORT_LABELS[metric],
            legendgroup=metric,
            showlegend=False,
            line=dict(
                color=target_colors[metric],
                width=2.5,
            ),
            marker=dict(size=9),
            customdata=customdata,
            hovertemplate=(
                "<b>" + TARGET_LABELS[metric] + "</b><br>"
                "Horizon: h=%{x}<br>"
                "Champion: %{customdata[0]}<br>"
                "Benchmark skill: %{y:.3f}%<br>"
                "Relative MAE: %{customdata[2]:.3f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

fig.add_hline(
    y=0,
    line_dash="dash",
    line_width=1,
    row=1,
    col=2,
)

apply_branding(fig)

fig.update_xaxes(
    tickmode="array",
    tickvals=FORECAST_HORIZONS,
    ticktext=[
        f"h={horizon}"
        for horizon in FORECAST_HORIZONS
    ],
    title_text="Forecast horizon",
)

fig.update_yaxes(
    title_text="Relative MAE (%)",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Benchmark skill (%)",
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "How forecast quality changes with horizon"
        "<br><sup>The final system remains benchmark-positive "
        "at h=1, h=2, and h=5</sup>"
    ),
    height=540,
    legend=dict(
        orientation="h",
        y=-0.16,
    ),
)

fig.show()

**Findings.** Longer-horizon forecasting is generally harder, but degradation is not uniform across targets.

FHVHV speed and FHVHV trips show the clearest decline in benchmark skill as the horizon lengthens, while Taxi speed changes only modestly. Taxi trips and Subway ridership are non-monotonic: their h=5 results recover some of the benchmark-relative performance lost at h=2.

Most importantly, **all 15 Target × Horizon jobs remain benchmark-positive**, so the farther horizons still add forecasting value even where their absolute error is higher.

### 4.6.1.9.2 Inspect Temporal and Geographic Reliability

Aggregate scorecards cannot show how individual forecasts behave or where failures concentrate. This subsection therefore moves from an inspectable holdout time series to citywide geographic reliability, then narrows into the weakest support-qualified forecasting surfaces and one concrete failure example.

**4.6.1.9.2A — Inspect an actual holdout time series.**

Aggregate scores are useful, but a reader should also be able to look directly at the forecasting problem.

The helper below plots one **Target × Horizon × Taxi Zone** over the unseen holdout and overlays four series: the observed mobility, the frozen champion forecast, the strongest pre-holdout runner-up, and Previous week · same Daypart.

The x-axis follows the canonical ordered mobility sequence so multiple Dayparts on the same date remain distinct. Hover reveals the date, Daypart, and temporal bucket for each observation.

The default example is selected by **holdout support rather than forecasting accuracy**, avoiding a hand-picked showcase. Any legal Target × Horizon × Taxi Zone can then be inspected by changing the three arguments.

In [73]:
# ---------------------------------------------------------------------
# 4.6.1.9.2A — Build a reusable holdout time-series inspector
# ---------------------------------------------------------------------

FORECAST_INSPECTOR_REQUIRED_COLUMNS = [
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "target_date",
    "actual",
    "benchmark_prediction",
    "champion_system_prediction",
    "runner_up_system_prediction",
    "runner_up_actual",
    "runner_up_benchmark_prediction",
    "champion_family",
    "runner_up_family",
]


def load_holdout_inspection_rows(metric, horizon, taxi_zone_id):
    """
    Load one reader-selected holdout series with target-time context.

    Predicate pushdown keeps this interactive diagnostic lightweight:
    Pandas receives one Target × Horizon × Taxi Zone rather than the
    complete citywide runner-up comparison.
    """
    comparison_path = Path(RUNNER_UP_HOLDOUT_COMPARISON_PATH)

    require_holdout_source_schema(
        comparison_path,
        FORECAST_INSPECTOR_REQUIRED_COLUMNS,
        "champion/runner-up holdout comparison",
    )

    series_df = pd.read_parquet(
        comparison_path,
        columns=FORECAST_INSPECTOR_REQUIRED_COLUMNS,
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
            ("taxi_zone_id", "==", int(taxi_zone_id)),
        ],
    ).copy()

    if series_df.empty:
        raise ValueError(
            f"No supported holdout rows for {metric} · "
            f"h={horizon} · Taxi Zone {taxi_zone_id}."
        )

    # 8.2A guarantees one identical denominator for champion and runner-up.
    # Rechecking the selected series here is cheap and prevents a plotting bug
    # from hiding an unexpected mismatch in actuals or benchmark predictions.
    actual_match = np.allclose(
        series_df["actual"].to_numpy(dtype=float),
        series_df["runner_up_actual"].to_numpy(dtype=float),
        equal_nan=True,
    )
    benchmark_match = np.allclose(
        series_df["benchmark_prediction"].to_numpy(dtype=float),
        series_df["runner_up_benchmark_prediction"].to_numpy(dtype=float),
        equal_nan=True,
    )

    if not actual_match or not benchmark_match:
        raise RuntimeError(
            "Champion and runner-up rows do not share the same "
            "actual/benchmark holdout denominator."
        )

    champion_rows = pd.read_parquet(
        CHAMPION_REGISTRY_PATH,
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
        ],
    )

    if len(champion_rows) != 1:
        raise RuntimeError(
            f"{metric} · h={horizon}: expected one frozen champion row."
        )

    track = str(champion_rows.iloc[0]["modeling_track"])
    matrix_path = FORECASTING_MATRIX_PATHS[track]

    time_columns = [
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "metric",
        "horizon",
        "target_date",
        "target_daypart",
        "target_temporal_bucket",
    ]

    require_holdout_source_schema(
        matrix_path,
        ["stage", *time_columns],
        f"{TRACK_LABELS[track]} forecasting matrix",
    )

    time_df = pd.read_parquet(
        matrix_path,
        columns=["stage", *time_columns],
        filters=[
            ("stage", "==", "final_holdout"),
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
            ("taxi_zone_id", "==", int(taxi_zone_id)),
        ],
    ).drop(columns="stage")

    join_keys = [
        "forecast_sequence_id",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "metric",
        "horizon",
        "target_date",
    ]

    for frame in [series_df, time_df]:
        frame["target_date"] = pd.to_datetime(
            frame["target_date"]
        ).dt.normalize()

    if time_df.duplicated(join_keys).any():
        raise RuntimeError(
            "Target-time lookup is not unique for the selected holdout series."
        )

    series_df = series_df.merge(
        time_df,
        on=join_keys,
        how="left",
        validate="one_to_one",
    )

    if series_df[
        ["target_daypart", "target_temporal_bucket"]
    ].isna().any().any():
        raise RuntimeError(
            "Selected holdout series is missing target-time context."
        )

    return (
        series_df
        .sort_values("target_observation_sequence_id")
        .reset_index(drop=True)
    )


def plot_holdout_forecast_inspector(metric, horizon, taxi_zone_id):
    """
    Plot actual, champion, runner-up, and benchmark for one holdout series.
    """
    series_df = load_holdout_inspection_rows(
        metric,
        horizon,
        taxi_zone_id,
    )

    champion_family = FAMILY_LABELS[
        str(series_df["champion_family"].iloc[0])
    ]
    runner_family = FAMILY_LABELS[
        str(series_df["runner_up_family"].iloc[0])
    ]

    x_values = series_df["target_observation_sequence_id"]

    customdata = np.column_stack([
        series_df["target_date"].dt.strftime("%Y-%m-%d"),
        series_df["target_daypart"].astype(str),
        series_df["target_temporal_bucket"].astype(str),
    ])

    fig = go.Figure()

    # The four lines answer different questions:
    # actual = what happened; champion/runner-up = advanced systems being audited;
    # benchmark = the simple weekly reference both systems must beat.
    series_specs = [
        (
            "actual",
            "Actual",
            BRAND_COLOR_SEQUENCE[0],
            3.0,
            "solid",
        ),
        (
            "champion_system_prediction",
            f"Champion · {champion_family}",
            BRAND_COLOR_SEQUENCE[1],
            2.5,
            "solid",
        ),
        (
            "runner_up_system_prediction",
            f"Runner-up · {runner_family}",
            BRAND_COLOR_SEQUENCE[2],
            2.0,
            "dash",
        ),
        (
            "benchmark_prediction",
            "Previous week · same Daypart",
            BRAND_COLOR_SEQUENCE[3],
            1.8,
            "dot",
        ),
    ]

    for column, label, color, width, dash in series_specs:
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=series_df[column],
                mode="lines",
                name=label,
                line=dict(
                    color=color,
                    width=width,
                    dash=dash,
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>" + label + "</b><br>"
                    "Date: %{customdata[0]}<br>"
                    "Daypart: %{customdata[1]}<br>"
                    "Temporal bucket: %{customdata[2]}<br>"
                    "Value: %{y:,.3f}"
                    "<extra></extra>"
                ),
            )
        )

    apply_branding(fig)

    fig.update_layout(
        title=(
            f"{TARGET_LABELS[metric]} · "
            f"h={int(horizon)} · "
            f"Taxi Zone {int(taxi_zone_id)}"
            "<br><sup>Unseen final holdout · "
            "ordered target sequence</sup>"
        ),
        xaxis_title="Ordered holdout target position",
        yaxis_title=TARGET_LABELS[metric],
        hovermode="x unified",
        height=560,
        legend=dict(
            orientation="h",
            y=-0.16,
        ),
    )

    fig.show()
    return series_df


# ---------------------------------------------------------------------
# Choose a high-support default without looking at forecasting error
# ---------------------------------------------------------------------
# Selecting the example by row count avoids cherry-picking a zone because its
# champion happened to look unusually strong or weak on the unseen holdout.

default_zone_rows_df = pd.read_parquet(
    RUNNER_UP_HOLDOUT_COMPARISON_PATH,
    columns=["taxi_zone_id"],
    filters=[
        ("metric", "==", "taxi_trip_count"),
        ("horizon", "==", 1),
    ],
)

if default_zone_rows_df.empty:
    raise RuntimeError(
        "No Taxi-trip h=1 holdout rows are available for the inspector default."
    )

# Highest row support wins; Taxi Zone ID provides the same deterministic
# tie-break used by the original SQL query.
default_zone_support_df = (
    default_zone_rows_df.groupby(
        "taxi_zone_id",
        observed=True,
        as_index=False,
    )
    .size()
    .sort_values(
        ["size", "taxi_zone_id"],
        ascending=[False, True],
    )
)

default_zone_id = int(
    default_zone_support_df.iloc[0]["taxi_zone_id"]
)

del default_zone_rows_df, default_zone_support_df

INSPECT_METRIC = "taxi_trip_count"
INSPECT_HORIZON = 1
INSPECT_TAXI_ZONE_ID = default_zone_id

inspection_example_df = plot_holdout_forecast_inspector(
    INSPECT_METRIC,
    INSPECT_HORIZON,
    INSPECT_TAXI_ZONE_ID,
)

**Findings.** The forecast inspector makes the aggregate scorecards observable at the underlying mobility-sequence level. The default view uses **Taxi trips at h=1 in Taxi Zone 3**, selected because it has strong holdout support rather than because its forecast happens to look favorable.

The reader can therefore compare the observed mobility directly with the frozen champion, strongest pre-holdout runner-up, and weekly benchmark without changing the evaluation contract. The same helper can be reused for any supported Target × Horizon × Taxi Zone, making individual successes and failures inspectable rather than relying on hand-picked examples.

**Reading the inspector.** This view is intentionally configurable rather than a single hand-picked success case. The default Taxi Zone is chosen only for strong holdout support. Changing `INSPECT_METRIC`, `INSPECT_HORIZON`, and `INSPECT_TAXI_ZONE_ID` lets the reader inspect any supported forecasting surface using the same four-series comparison.

**4.6.1.9.2B — Map geographic holdout error in native units.**

This map shows **Champion MAE** for each supported Taxi Zone within a selected Target × Horizon forecasting job. Because every zone in one map uses the same target and unit, native-unit MAE provides a direct view of where forecast misses are largest without introducing scale distortions from small percentage denominators.

Hover details provide complementary context: benchmark MAE, MAE advantage over the weekly benchmark, share of the job's total absolute error, bias, and observation support.

The dropdown selects only the **Target × Horizon** forecasting job, producing 15 spatial views with one consistent visual meaning.

In [74]:
# ---------------------------------------------------------------------
# 4.6.1.9.2B — Build and map geographic holdout error
# ---------------------------------------------------------------------

import json

import geopandas as gpd
from shapely import wkb


FORECAST_ZONE_RELIABILITY_PATH = (
    FINAL_DIR / "champion_holdout_zone_reliability.parquet"
)
TAXI_ZONE_SPATIAL_CONTEXT_PATH = (
    PIPELINE_DIR
    / "1.3.1.final_tables"
    / "nyc_taxi_zones_harmonized.parquet"
)

ZONE_ID_COLUMN = "taxi_zone_id"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"

FORECAST_MAP_CENTER = {"lat": 40.71, "lon": -73.90}
FORECAST_MAP_ZOOM = 9.15
FORECAST_MAP_STYLE = "carto-positron"

# This implementation intentionally uses Plotly's MapLibre-backed trace so the
# notebook does not depend on an external map token.
if not hasattr(go, "Choroplethmap"):
    raise RuntimeError(
        "This map requires Plotly's MapLibre-backed go.Choroplethmap. "
        "Do not fall back to token-dependent Choroplethmapbox."
    )


# ---------------------------------------------------------------------
# 1) Aggregate the primary holdout surface at Taxi Zone grain
# ---------------------------------------------------------------------
# The zone-level artifact stores complementary measures of absolute error,
# benchmark comparison, proportional error, support, and error contribution.

zone_holdout_rows_df = pd.read_parquet(
    FINAL_HOLDOUT_CHAMPION_PREDICTION_PATH,
    columns=[
        "metric",
        "horizon",
        "taxi_zone_id",
        "champion_family",
        "actual",
        "benchmark_prediction",
        "system_prediction",
        "primary_evaluation_eligible",
    ],
    filters=[("primary_evaluation_eligible", "==", True)],
)

if zone_holdout_rows_df.empty:
    raise RuntimeError(
        "No primary holdout rows are available for geographic reliability."
    )

zone_holdout_rows_df["champion_absolute_error"] = (
    zone_holdout_rows_df["system_prediction"] - zone_holdout_rows_df["actual"]
).abs()

zone_holdout_rows_df["benchmark_absolute_error"] = (
    zone_holdout_rows_df["benchmark_prediction"] - zone_holdout_rows_df["actual"]
).abs()

zone_holdout_rows_df["observed_absolute_value"] = (
    zone_holdout_rows_df["actual"].abs()
)

zone_holdout_rows_df["signed_error"] = (
    zone_holdout_rows_df["system_prediction"] - zone_holdout_rows_df["actual"]
)

forecast_zone_reliability_df = (
    zone_holdout_rows_df.groupby(
        ["metric", "horizon", "taxi_zone_id", "champion_family"],
        observed=True,
        as_index=False,
    )
    .agg(
        evaluation_rows=("actual", "size"),
        champion_absolute_error_sum=("champion_absolute_error", "sum"),
        benchmark_absolute_error_sum=("benchmark_absolute_error", "sum"),
        champion_mae=("champion_absolute_error", "mean"),
        benchmark_mae=("benchmark_absolute_error", "mean"),
        observed_abs_mean=("observed_absolute_value", "mean"),
        bias=("signed_error", "mean"),
    )
)

del zone_holdout_rows_df

if forecast_zone_reliability_df.empty:
    raise RuntimeError(
        "Zone-level holdout reliability aggregation returned no rows."
    )

job_count = len(
    forecast_zone_reliability_df[["metric", "horizon"]].drop_duplicates()
)
if job_count != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_JOB_COUNT} geographic jobs; found {job_count}."
    )


# ---------------------------------------------------------------------
# 2) Preserve the original support contract and add native-unit context
# ---------------------------------------------------------------------
# Two support flags intentionally answer different questions:
#
# native_support_eligible:
#   Enough repeated holdout observations to report native-unit MAE.
#
# support_eligible:
#   The original scale-aware 4.6.1 contract, which also requires positive
#   benchmark and observed scales before percentage ratios are interpreted.

forecast_zone_reliability_df["job_median_rows"] = (
    forecast_zone_reliability_df.groupby(
        ["metric", "horizon"], observed=True
    )["evaluation_rows"]
    .transform("median")
)

forecast_zone_reliability_df["support_ratio"] = (
    forecast_zone_reliability_df["evaluation_rows"]
    / forecast_zone_reliability_df["job_median_rows"]
)

forecast_zone_reliability_df["native_support_eligible"] = (
    forecast_zone_reliability_df["support_ratio"].ge(0.50)
)

# Percentage-based diagnostics require positive benchmark and observed scales
# so their denominators remain well-defined.
forecast_zone_reliability_df["support_eligible"] = (
    forecast_zone_reliability_df["native_support_eligible"]
    & forecast_zone_reliability_df["benchmark_mae"].gt(0)
    & forecast_zone_reliability_df["observed_abs_mean"].gt(0)
)

forecast_zone_reliability_df["benchmark_skill_pct"] = np.where(
    forecast_zone_reliability_df["benchmark_mae"].gt(0),
    100.0
    * (
        1.0
        - forecast_zone_reliability_df["champion_mae"]
        / forecast_zone_reliability_df["benchmark_mae"]
    ),
    np.nan,
)

forecast_zone_reliability_df["relative_mae_pct"] = np.where(
    forecast_zone_reliability_df["observed_abs_mean"].gt(0),
    100.0
    * forecast_zone_reliability_df["champion_mae"]
    / forecast_zone_reliability_df["observed_abs_mean"],
    np.nan,
)

# Native-unit MAE advantage is the direct analogue of benchmark skill without
# dividing by benchmark error. Positive values favor the advanced champion.
forecast_zone_reliability_df["mae_advantage"] = (
    forecast_zone_reliability_df["benchmark_mae"]
    - forecast_zone_reliability_df["champion_mae"]
)

job_row_total = (
    forecast_zone_reliability_df.groupby(
        ["metric", "horizon"], observed=True
    )["evaluation_rows"]
    .transform("sum")
)

job_absolute_error_total = (
    forecast_zone_reliability_df.groupby(
        ["metric", "horizon"], observed=True
    )["champion_absolute_error_sum"]
    .transform("sum")
)

forecast_zone_reliability_df["forecast_share_pct"] = np.where(
    job_row_total.gt(0),
    100.0
    * forecast_zone_reliability_df["evaluation_rows"]
    / job_row_total,
    np.nan,
)

# Error burden directly allocates each Target × Horizon's absolute-error
# budget across geography. It sums to 100% within each forecasting job.
forecast_zone_reliability_df["error_burden_pct"] = np.where(
    job_absolute_error_total.gt(0),
    100.0
    * forecast_zone_reliability_df["champion_absolute_error_sum"]
    / job_absolute_error_total,
    np.nan,
)

# A concentration ratio above 1 means a zone consumes more error than its
# share of forecasts would imply; below 1 means it contributes less.
forecast_zone_reliability_df["error_concentration_ratio"] = np.where(
    forecast_zone_reliability_df["forecast_share_pct"].gt(0),
    forecast_zone_reliability_df["error_burden_pct"]
    / forecast_zone_reliability_df["forecast_share_pct"],
    np.nan,
)

# These quantities are accounting identities. Validate them before exporting
# because 9.2D and 4.6.2 will interpret the resulting error budget directly.
budget_qa_df = (
    forecast_zone_reliability_df.groupby(
        ["metric", "horizon"], observed=True, as_index=False
    )
    .agg(
        forecast_share_pct=("forecast_share_pct", "sum"),
        error_burden_pct=("error_burden_pct", "sum"),
    )
)

if not np.allclose(
    budget_qa_df["forecast_share_pct"], 100.0, atol=1e-8, rtol=0
):
    raise RuntimeError("Zone forecast shares do not sum to 100% within each job.")

if not np.allclose(
    budget_qa_df["error_burden_pct"], 100.0, atol=1e-8, rtol=0
):
    raise RuntimeError("Zone error burdens do not sum to 100% within each job.")


# ---------------------------------------------------------------------
# 3) Load and simplify the Taxi Zone polygons once
# ---------------------------------------------------------------------
# Geometry follows the established project contract:
# WKB -> EPSG:4326 -> simplify in EPSG:2263 -> return to EPSG:4326.

taxi_zone_source_df = pd.read_parquet(
    TAXI_ZONE_SPATIAL_CONTEXT_PATH
).copy()

if (
    "location_id" in taxi_zone_source_df.columns
    and ZONE_ID_COLUMN not in taxi_zone_source_df.columns
):
    taxi_zone_source_df = taxi_zone_source_df.rename(
        columns={"location_id": ZONE_ID_COLUMN}
    )

require_columns(
    taxi_zone_source_df,
    [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN, "geometry"],
    "Taxi Zone map source",
)

taxi_zone_source_df["geometry"] = taxi_zone_source_df["geometry"].apply(
    lambda value: (
        wkb.loads(value)
        if isinstance(value, (bytes, bytearray, memoryview))
        else value
    )
)

taxi_zone_map_gdf = gpd.GeoDataFrame(
    taxi_zone_source_df,
    geometry="geometry",
    crs="EPSG:4326",
)

taxi_zone_map_gdf = (
    taxi_zone_map_gdf.loc[
        taxi_zone_map_gdf["geometry"].notna()
        & taxi_zone_map_gdf[ZONE_ID_COLUMN].notna()
    ]
    .sort_values(ZONE_ID_COLUMN)
    .drop_duplicates(subset=[ZONE_ID_COLUMN])
    .reset_index(drop=True)
)

if taxi_zone_map_gdf[ZONE_ID_COLUMN].duplicated().any():
    raise RuntimeError(
        "Taxi Zone geometry scaffold contains duplicate physical zone IDs."
    )

# Simplify in feet rather than angular degrees while preserving neighboring
# polygon topology.
taxi_zone_plot_gdf = taxi_zone_map_gdf.to_crs("EPSG:2263").copy()
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=75,
    preserve_topology=True,
)
taxi_zone_plot_gdf = (
    taxi_zone_plot_gdf.to_crs("EPSG:4326").reset_index(drop=True)
)

taxi_zone_plot_gdf[ZONE_ID_COLUMN] = (
    pd.to_numeric(taxi_zone_plot_gdf[ZONE_ID_COLUMN], errors="raise")
    .astype("Int64")
    .astype(str)
)

forecast_zone_reliability_df[ZONE_ID_COLUMN] = (
    pd.to_numeric(
        forecast_zone_reliability_df[ZONE_ID_COLUMN], errors="raise"
    )
    .astype("Int64")
    .astype(str)
)

zone_metadata_df = taxi_zone_plot_gdf[
    [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN]
].copy()

forecast_zone_reliability_df = forecast_zone_reliability_df.merge(
    zone_metadata_df,
    on=ZONE_ID_COLUMN,
    how="left",
    validate="many_to_one",
)

# Export the expanded—but backward-compatible—zone contract before any
# visualization-only formatting changes are introduced.
atomic_write_parquet(
    forecast_zone_reliability_df,
    FORECAST_ZONE_RELIABILITY_PATH,
)

forecast_map_geojson = json.loads(
    taxi_zone_plot_gdf[
        [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN, "geometry"]
    ].to_json()
)


# ---------------------------------------------------------------------
# 4) Prepare the 15 Target × Horizon map states
# ---------------------------------------------------------------------

def normalize_color_sequence(colors):
    """Convert a project color sequence into an explicit Plotly colorscale."""
    if not colors:
        raise ValueError("Color sequence cannot be empty.")

    if isinstance(colors[0], (list, tuple)) and len(colors[0]) == 2:
        return colors

    if len(colors) == 1:
        return [[0.0, colors[0]], [1.0, colors[0]]]

    return [
        [index / (len(colors) - 1), color]
        for index, color in enumerate(colors)
    ]


mae_colorscale = normalize_color_sequence(
    BRAND_SEQUENTIAL_HEATMAP_SCALE
)

map_states = {}

for metric in FORECAST_TARGETS:
    for horizon in FORECAST_HORIZONS:
        job_df = forecast_zone_reliability_df.loc[
            forecast_zone_reliability_df["metric"].eq(metric)
            & forecast_zone_reliability_df["horizon"].eq(int(horizon))
        ].copy()

        # Start from every physical polygon so unsupported geography remains in
        # the location scaffold even when it has no analytical color value.
        job_map_df = zone_metadata_df.merge(
            job_df.drop(
                columns=[ZONE_COLUMN, BOROUGH_COLUMN],
                errors="ignore",
            ),
            on=ZONE_ID_COLUMN,
            how="left",
            validate="one_to_one",
        )

        supported = (
            job_map_df["native_support_eligible"]
            .fillna(False)
            .astype(bool)
        )

        supported_mae = job_map_df.loc[supported, "champion_mae"]
        if supported_mae.empty or not np.isfinite(supported_mae).any():
            raise RuntimeError(
                f"No native-unit map support for {metric} · h={horizon}."
            )

        mae_max = max(float(supported_mae.max()), 1e-12)

        # Pre-format mixed hover fields explicitly. This guarantees consistent
        # display precision instead of relying on Plotly to recover numeric
        # types from a mixed text/numeric customdata array.
        hover_df = pd.DataFrame(
            {
                "zone": job_map_df[ZONE_COLUMN].fillna("Unknown"),
                "borough": job_map_df[BOROUGH_COLUMN].fillna("Unknown"),
                "champion": (
                    job_map_df["champion_family"]
                    .map(FAMILY_LABELS)
                    .fillna("No supported forecast")
                ),
                "rows": job_map_df["evaluation_rows"].map(
                    lambda value: f"{value:,.0f}" if pd.notna(value) else "0"
                ),
                "support_ratio": job_map_df["support_ratio"].map(
                    lambda value: f"{value:.2f}×" if pd.notna(value) else "N/A"
                ),
                "champion_mae": job_map_df["champion_mae"].map(
                    lambda value: f"{value:,.3f}" if pd.notna(value) else "N/A"
                ),
                "benchmark_mae": job_map_df["benchmark_mae"].map(
                    lambda value: f"{value:,.3f}" if pd.notna(value) else "N/A"
                ),
                "mae_advantage": job_map_df["mae_advantage"].map(
                    lambda value: f"{value:+,.3f}" if pd.notna(value) else "N/A"
                ),
                "error_burden": job_map_df["error_burden_pct"].map(
                    lambda value: f"{value:.3f}%" if pd.notna(value) else "N/A"
                ),
                "bias": job_map_df["bias"].map(
                    lambda value: f"{value:+,.3f}" if pd.notna(value) else "N/A"
                ),
            }
        )

        map_states[(metric, int(horizon))] = {
            "locations": job_map_df[ZONE_ID_COLUMN].tolist(),
            "z": job_map_df["champion_mae"].where(supported).tolist(),
            "customdata": hover_df.to_numpy(dtype=object),
            "zmin": 0.0,
            "zmax": mae_max,
            "colorbar_title": "Champion MAE",
        }


# ---------------------------------------------------------------------
# 5) Render one trace across the 15 Target × Horizon states
# ---------------------------------------------------------------------

state_order = [
    (metric, int(horizon))
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
]

if len(state_order) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Forecast map must contain exactly 15 Target × Horizon states."
    )

initial_metric, initial_horizon = state_order[0]
initial_state = map_states[(initial_metric, initial_horizon)]

forecast_map_fig = go.Figure(
    go.Choroplethmap(
        geojson=forecast_map_geojson,
        locations=initial_state["locations"],
        z=initial_state["z"],
        featureidkey=f"properties.{ZONE_ID_COLUMN}",
        colorscale=mae_colorscale,
        zmin=initial_state["zmin"],
        zmax=initial_state["zmax"],
        marker={
            "opacity": 0.86,
            "line": {
                "width": 0.35,
                "color": "rgba(40, 75, 80, 0.55)",
            },
        },
        colorbar={
            "title": initial_state["colorbar_title"],
            "thickness": 14,
            "len": 0.72,
        },
        customdata=initial_state["customdata"],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Borough: %{customdata[1]}<br>"
            "Champion: %{customdata[2]}<br>"
            "Evaluation rows: %{customdata[3]}<br>"
            "Support vs job median: %{customdata[4]}<br>"
            "Champion MAE: %{customdata[5]}<br>"
            "Benchmark MAE: %{customdata[6]}<br>"
            "MAE advantage vs benchmark: %{customdata[7]}<br>"
            "Share of job absolute error: %{customdata[8]}<br>"
            "Bias: %{customdata[9]}"
            "<extra></extra>"
        ),
    )
)

map_buttons = []

for metric, horizon in state_order:
    state = map_states[(metric, horizon)]

    map_buttons.append(
        {
            "label": f"{TARGET_SHORT_LABELS[metric]} · h={horizon}",
            "method": "update",
            "args": [
                {
                    "locations": [state["locations"]],
                    "z": [state["z"]],
                    "customdata": [state["customdata"]],
                    "zmin": [state["zmin"]],
                    "zmax": [state["zmax"]],
                    "colorbar.title.text": [state["colorbar_title"]],
                },
                {
                    "title": {
                        "text": (
                            f"{TARGET_LABELS[metric]} · h={horizon}"
                            "<br><sup>Champion holdout MAE · native units</sup>"
                        )
                    }
                },
            ],
        }
    )

forecast_map_fig.update_layout(
    map={
        "style": FORECAST_MAP_STYLE,
        "center": FORECAST_MAP_CENTER,
        "zoom": FORECAST_MAP_ZOOM,
    },
    updatemenus=[
        {
            "buttons": map_buttons,
            "direction": "down",
            "showactive": True,
            "x": 0.01,
            "y": 1.03,
            "xanchor": "left",
            "yanchor": "top",
        }
    ],
    title=(
        f"{TARGET_LABELS[initial_metric]} · h={initial_horizon}"
        "<br><sup>Champion holdout MAE · native units</sup>"
    ),
    width=950,
    height=700,
    margin={"l": 15, "r": 20, "t": 105, "b": 15},
)

apply_branding(forecast_map_fig)


# ---------------------------------------------------------------------
# 6) Guard the notebook payload before rendering
# ---------------------------------------------------------------------

forecast_map_payload_mb = (
    len(forecast_map_fig.to_json().encode("utf-8")) / (1024 ** 2)
)

if forecast_map_payload_mb >= 20.0:
    raise RuntimeError(
        "Forecast map payload is unexpectedly large "
        f"({forecast_map_payload_mb:.2f} MB). "
        "Do not render until geometry duplication is removed."
    )

if len(forecast_map_fig.data) != 1:
    raise RuntimeError(
        "Forecast map must contain exactly one shared choropleth trace."
    )

print(
    "Forecast map scaffold\n"
    f"  Taxi Zone polygons: {len(taxi_zone_plot_gdf):,}\n"
    f"  GeoJSON features:   {len(forecast_map_geojson['features']):,}\n"
    f"  Plotly traces:       {len(forecast_map_fig.data)}\n"
    f"  Dropdown states:     {len(state_order)}\n"
    f"  Serialized payload:  {forecast_map_payload_mb:.2f} MB",
    flush=True,
)

forecast_map_fig.show()

Forecast map scaffold
  Taxi Zone polygons: 262
  GeoJSON features:   262
  Plotly traces:       1
  Dropdown states:     15
  Serialized payload:  0.94 MB


**Findings.** Geographic MAE is concentrated in high-activity forecasting surfaces rather than in the sparse zones that can appear extreme under percentage-based diagnostics.

The largest native-unit errors occur where mobility volumes themselves are substantial, especially for Subway ridership and high-volume Taxi activity. Sparse near-zero zones can still be locally difficult, but their misses are small in absolute units and therefore contribute little to the total error budget.

Champion MAE should therefore be read as **local absolute forecast error**. Benchmark advantage and error-burden measures provide the complementary views needed to determine whether the champion improves on the simple benchmark and how much a zone actually contributes to system error.

**4.6.1.9.2C — Show how broadly the champion improves on the weekly benchmark.**

The MAE map answers where the champion makes larger average errors. This complementary distribution asks a different question: across adequately supported Taxi Zones, **how often—and by how much—does the champion improve on the previous-week benchmark?**

The comparison uses **MAE advantage in native target units**:

**Benchmark MAE − Champion MAE**

Positive values favor the champion, negative values favor the weekly benchmark, and zero marks equal average error. Unlike percentage skill, this difference remains interpretable when benchmark error is very small.

In [75]:
# ---------------------------------------------------------------------
# 4.6.1.9.2C — Visualize the geographic distribution of MAE advantage
# ---------------------------------------------------------------------

zone_advantage_df = pd.read_parquet(
    FORECAST_ZONE_RELIABILITY_PATH
).copy()

zone_advantage_df = zone_advantage_df.loc[
    zone_advantage_df["native_support_eligible"].fillna(False)
    & np.isfinite(zone_advantage_df["mae_advantage"])
].copy()

if zone_advantage_df.empty:
    raise RuntimeError(
        "No support-qualified Taxi Zone MAE-advantage rows are available."
    )


def build_advantage_ecdf(metric, horizon):
    """
    Return one ordered zone-level MAE-advantage distribution and summary.

    Positive advantage means the frozen champion has lower MAE than the
    previous-week benchmark; negative advantage means the benchmark is better.
    """
    frame = (
        zone_advantage_df.loc[
            zone_advantage_df["metric"].eq(metric)
            & zone_advantage_df["horizon"].eq(int(horizon))
        ]
        .sort_values("mae_advantage")
        .reset_index(drop=True)
    )

    if frame.empty:
        raise ValueError(
            f"No geographic MAE-advantage rows for {metric} · h={horizon}."
        )

    frame["cumulative_zone_pct"] = (
        100.0 * np.arange(1, len(frame) + 1) / len(frame)
    )

    positive_share = 100.0 * frame["mae_advantage"].gt(0).mean()

    return frame, float(positive_share)


advantage_states = {}

for metric in FORECAST_TARGETS:
    for horizon in FORECAST_HORIZONS:
        state_df, positive_share = build_advantage_ecdf(metric, horizon)

        hover_df = pd.DataFrame(
            {
                "zone": state_df[ZONE_COLUMN].fillna("Unknown"),
                "borough": state_df[BOROUGH_COLUMN].fillna("Unknown"),
                "rows": state_df["evaluation_rows"].map(
                    lambda value: f"{value:,.0f}"
                ),
                "champion_mae": state_df["champion_mae"].map(
                    lambda value: f"{value:,.3f}"
                ),
                "benchmark_mae": state_df["benchmark_mae"].map(
                    lambda value: f"{value:,.3f}"
                ),
            }
        )

        advantage_states[(metric, int(horizon))] = {
            "x": state_df["mae_advantage"].tolist(),
            "y": state_df["cumulative_zone_pct"].tolist(),
            "customdata": hover_df.to_numpy(dtype=object),
            "positive_share": positive_share,
            "zone_count": len(state_df),
            "median_advantage": float(state_df["mae_advantage"].median()),
        }


advantage_state_order = [
    (metric, int(horizon))
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
]

initial_metric, initial_horizon = advantage_state_order[0]
initial_state = advantage_states[(initial_metric, initial_horizon)]

advantage_ecdf_fig = go.Figure(
    go.Scatter(
        x=initial_state["x"],
        y=initial_state["y"],
        mode="lines",
        line={"width": 3, "color": BRAND_COLOR_SEQUENCE[0]},
        customdata=initial_state["customdata"],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Borough: %{customdata[1]}<br>"
            "MAE advantage vs benchmark: %{x:+,.3f}<br>"
            "Champion MAE: %{customdata[3]}<br>"
            "Benchmark MAE: %{customdata[4]}<br>"
            "Evaluation rows: %{customdata[2]}<br>"
            "Cumulative zones: %{y:.1f}%"
            "<extra></extra>"
        ),
        showlegend=False,
    )
)

advantage_ecdf_fig.add_vline(
    x=0,
    line_dash="dash",
    line_width=1.5,
    line_color="rgba(70, 90, 95, 0.65)",
)

advantage_buttons = []

for metric, horizon in advantage_state_order:
    state = advantage_states[(metric, horizon)]

    advantage_buttons.append(
        {
            "label": f"{TARGET_SHORT_LABELS[metric]} · h={horizon}",
            "method": "update",
            "args": [
                {
                    "x": [state["x"]],
                    "y": [state["y"]],
                    "customdata": [state["customdata"]],
                },
                {
                    "title": {
                        "text": (
                            "Geographic benchmark comparison · "
                            f"{TARGET_LABELS[metric]} · h={horizon}"
                        )
                    },
                    "annotations": [
                        {
                            "text": (
                                f"{state['positive_share']:.1f}% of "
                                f"{state['zone_count']} supported zones "
                                "beat the weekly benchmark"
                                "<br>Median MAE advantage: "
                                f"{state['median_advantage']:+,.3f}"
                            ),
                            "x": 0.99,
                            "y": 0.03,
                            "xref": "paper",
                            "yref": "paper",
                            "xanchor": "right",
                            "yanchor": "bottom",
                            "showarrow": False,
                            "align": "right",
                        }
                    ],
                },
            ],
        }
    )

advantage_ecdf_fig.update_layout(
    title=(
        "Geographic benchmark comparison · "
        f"{TARGET_LABELS[initial_metric]} · h={initial_horizon}"
    ),
    xaxis_title=(
        "MAE advantage vs weekly benchmark · native target units "
        "(positive = champion better)"
    ),
    yaxis_title="Cumulative supported Taxi Zones (%)",
    updatemenus=[
        {
            "buttons": advantage_buttons,
            "direction": "down",
            "showactive": True,
            "x": 0.01,
            "y": 1.02,
            "xanchor": "left",
            "yanchor": "top",
        }
    ],
    annotations=[
        {
            "text": (
                f"{initial_state['positive_share']:.1f}% of "
                f"{initial_state['zone_count']} supported zones "
                "beat the weekly benchmark"
                "<br>Median MAE advantage: "
                f"{initial_state['median_advantage']:+,.3f}"
            ),
            "x": 0.99,
            "y": 0.03,
            "xref": "paper",
            "yref": "paper",
            "xanchor": "right",
            "yanchor": "bottom",
            "showarrow": False,
            "align": "right",
        }
    ],
    height=520,
)

apply_branding(advantage_ecdf_fig)
advantage_ecdf_fig.show()


# Keep the 15-job result visible without requiring the reader to click through
# every dropdown state before interpreting geographic breadth.
advantage_summary_records = []

for metric, horizon in advantage_state_order:
    state = advantage_states[(metric, horizon)]

    advantage_summary_records.append(
        {
            "Target": TARGET_LABELS[metric],
            "Horizon": horizon,
            "Supported zones": state["zone_count"],
            "Champion wins (%)": state["positive_share"],
            "Median MAE advantage": state["median_advantage"],
        }
    )

advantage_summary_df = pd.DataFrame(advantage_summary_records)

display(
    advantage_summary_df.style.format(
        {
            "Champion wins (%)": "{:.1f}",
            "Median MAE advantage": "{:+.3f}",
        }
    )
)

,Target,Horizon,Supported zones,Champion wins (%),Median MAE advantage
0,Taxi trips,1,258,80.2,+0.423
1,Taxi trips,2,258,73.3,+0.320
2,Taxi trips,5,258,86.8,+0.672
3,Taxi speed,1,189,100.0,+0.774
4,Taxi speed,2,189,100.0,+0.750
5,Taxi speed,5,189,100.0,+0.747
6,FHVHV trips,1,258,93.4,+22.483
7,FHVHV trips,2,258,91.1,+15.683
8,FHVHV trips,5,258,87.2,+16.843
9,FHVHV speed,1,257,100.0,+0.511


**Findings.** The frozen champion improves on the weekly benchmark across most supported geography, but the breadth of that improvement varies meaningfully by forecasting job.

Both speed targets are exceptionally consistent: **100% of adequately supported Taxi Zones beat the weekly benchmark at all three horizons** for Taxi speed and FHVHV speed. FHVHV-trip forecasts are also geographically broad, with **93.4% at h=1, 91.1% at h=2, and 87.2% at h=5** of supported zones favoring the champion.

Demand forecasts are less uniform. Taxi trips range from **73.3% to 86.8%** of supported zones beating the weekly benchmark, while Subway reaches **95.4% at h=1** and **96.1% at h=5** but falls to **70.6% at h=2**.

Median MAE advantage is positive for all 15 Target × Horizon jobs. Its magnitude should be interpreted within each target because trips, riders, and miles per hour use different native units.

The champion system is therefore geographically broad in its usefulness, but that advantage is not universal—particularly for Taxi-trip forecasts and Subway h=2.

**4.6.1.9.2D — Identify the geographic surfaces that contribute most to forecast error.**

Large local percentage errors do not necessarily imply a large contribution to total MAE. This diagnostic therefore measures geographic importance directly through two complementary quantities:

- **Error burden** is the percentage of a Target × Horizon's total absolute error contributed by one Taxi Zone.
- **Error concentration** compares that error share with the zone's share of forecasts. Values above 1× indicate that the zone contributes disproportionately more error than its forecast volume would suggest.

Because both quantities are normalized within each forecasting job, they can be compared across targets without mixing native units.

Surfaces are ranked first by **error burden** and then by **error concentration**.

In [76]:
# ---------------------------------------------------------------------
# 4.6.1.9.2D — Identify consequential geographic error surfaces
# ---------------------------------------------------------------------

TROUBLESOME_SURFACE_PATH = (
    FINAL_DIR
    / "champion_holdout_troublesome_zone_surfaces.parquet"
)

trouble_df = pd.read_parquet(
    FORECAST_ZONE_RELIABILITY_PATH
).copy()

# Keep the established scale-aware eligibility for the handoff's difficult-case
# population. This also preserves compatibility with the 9.2E inspector, while
# the ranking itself no longer depends on percentage skill or Relative MAE.
trouble_df = trouble_df.loc[
    trouble_df["support_eligible"].fillna(False)
    & np.isfinite(trouble_df["error_burden_pct"])
    & np.isfinite(trouble_df["error_concentration_ratio"])
    & np.isfinite(trouble_df["mae_advantage"])
].copy()

if trouble_df.empty:
    raise RuntimeError(
        "No support-qualified forecasting surfaces are available "
        "for geographic error-burden analysis."
    )

trouble_df["Target"] = trouble_df["metric"].map(
    TARGET_SHORT_LABELS
)
trouble_df["Champion"] = trouble_df["champion_family"].map(
    FAMILY_LABELS
)

# Rank systemic importance before local extremeness:
# 1) how much of the job's actual error budget the zone consumes;
# 2) whether that burden is disproportionate to its share of forecasts;
# 3) whether the weekly benchmark outperforms the champion in native units.
trouble_df = (
    trouble_df.sort_values(
        [
            "error_burden_pct",
            "error_concentration_ratio",
            "mae_advantage",
            "metric",
            "horizon",
            ZONE_ID_COLUMN,
        ],
        ascending=[False, False, True, True, True, True],
    )
    .reset_index(drop=True)
)

atomic_write_parquet(
    trouble_df,
    TROUBLESOME_SURFACE_PATH,
)


# ---------------------------------------------------------------------
# 1) Show error importance separately from forecast volume
# ---------------------------------------------------------------------

target_color_map = {
    TARGET_SHORT_LABELS[metric]: BRAND_COLOR_SEQUENCE[
        index % len(BRAND_COLOR_SEQUENCE)
    ]
    for index, metric in enumerate(FORECAST_TARGETS)
}

# Pre-format mixed hover values so all reader-facing native-unit quantities
# stay at three decimals regardless of Plotly customdata type coercion.
trouble_df["champion_mae_display"] = trouble_df["champion_mae"].map(
    lambda value: f"{value:,.3f}"
)
trouble_df["benchmark_mae_display"] = trouble_df["benchmark_mae"].map(
    lambda value: f"{value:,.3f}"
)
trouble_df["mae_advantage_display"] = trouble_df["mae_advantage"].map(
    lambda value: f"{value:+,.3f}"
)
trouble_df["forecast_share_display"] = trouble_df["forecast_share_pct"].map(
    lambda value: f"{value:.3f}%"
)
trouble_df["bias_display"] = trouble_df["bias"].map(
    lambda value: f"{value:+,.3f}"
)

trouble_fig = px.scatter(
    trouble_df,
    x="error_concentration_ratio",
    y="error_burden_pct",
    color="Target",
    symbol="horizon",
    color_discrete_map=target_color_map,
    custom_data=[
        ZONE_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
        "Champion",
        "evaluation_rows",
        "champion_mae_display",
        "benchmark_mae_display",
        "mae_advantage_display",
        "forecast_share_display",
        "bias_display",
    ],
    labels={
        "error_concentration_ratio":
            "Error concentration vs forecast share (×)",
        "error_burden_pct":
            "Share of job absolute error (%)",
        "horizon":
            "Horizon",
    },
    title=(
        "Which geographic surfaces contribute most to holdout error?"
        "<br><sup>Each point is one supported "
        "Target × Horizon × Taxi Zone surface</sup>"
    ),
)

# A 1× concentration means the zone contributes exactly as much error as its
# share of forecasts would predict. Values to the right over-contribute.
trouble_fig.add_vline(
    x=1.0,
    line_dash="dash",
    line_width=1.5,
    line_color="rgba(70, 90, 95, 0.65)",
    annotation_text="Error proportional to forecast share",
    annotation_position="top right",
)

trouble_fig.update_traces(
    marker={"size": 8, "opacity": 0.68},
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>"
        "Taxi Zone ID: %{customdata[0]}<br>"
        "Borough: %{customdata[2]}<br>"
        "Champion: %{customdata[3]}<br>"
        "Share of job absolute error: %{y:.3f}%<br>"
        "Error concentration: %{x:.3f}×<br>"
        "Forecast share: %{customdata[8]}<br>"
        "Champion MAE: %{customdata[5]}<br>"
        "Benchmark MAE: %{customdata[6]}<br>"
        "MAE advantage vs benchmark: %{customdata[7]}<br>"
        "Evaluation rows: %{customdata[4]:,.0f}<br>"
        "Bias: %{customdata[9]}"
        "<extra></extra>"
    ),
)

apply_branding(trouble_fig)

trouble_fig.update_layout(
    height=610,
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.24,
        "yanchor": "top",
    },
    margin={"l": 70, "r": 30, "t": 80, "b": 175},
)

trouble_fig.show()


# ---------------------------------------------------------------------
# 2) Keep the reader-facing ranking compact
# ---------------------------------------------------------------------

worst_display_df = (
    trouble_df.head(12)[
        [
            "Target",
            "horizon",
            ZONE_ID_COLUMN,
            ZONE_COLUMN,
            BOROUGH_COLUMN,
            "Champion",
            "evaluation_rows",
            "error_burden_pct",
            "error_concentration_ratio",
            "champion_mae",
            "mae_advantage",
            "bias",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            ZONE_ID_COLUMN: "Taxi Zone ID",
            ZONE_COLUMN: "Taxi Zone",
            BOROUGH_COLUMN: "Borough",
            "evaluation_rows": "Rows",
            "error_burden_pct": "Job error burden (%)",
            "error_concentration_ratio": "Error concentration (×)",
            "champion_mae": "Champion MAE",
            "mae_advantage": "MAE advantage",
            "bias": "Bias",
        }
    )
)

print(
    "Largest error-burden supported Taxi Zone surfaces"
)

display(
    worst_display_df.style.format(
        {
            "Job error burden (%)": "{:.3f}",
            "Error concentration (×)": "{:.3f}",
            "Champion MAE": "{:.3f}",
            "MAE advantage": "{:+.3f}",
            "Bias": "{:+.3f}",
        }
    )
)

Largest error-burden supported Taxi Zone surfaces


,Target,Horizon,Taxi Zone ID,Taxi Zone,Borough,Champion,Rows,Job error burden (%),Error concentration (×),Champion MAE,MAE advantage,Bias
0,Subway,5,100,Garment District,Manhattan,Tree,422,4.548,6.949,3161.636,+1307.575,+1110.617
1,Subway,1,100,Garment District,Manhattan,Tree,422,4.473,6.834,2784.710,+1684.501,-209.682
2,Subway,5,162,Midtown East,Manhattan,Tree,422,4.135,6.318,2874.385,+778.883,-374.468
3,Subway,2,100,Garment District,Manhattan,Transformer,422,4.093,6.254,3208.431,+1260.780,-513.245
4,Taxi trips,5,132,JFK Airport,Queens,Tree,430,3.752,9.640,139.656,+72.428,+25.065
5,Subway,2,162,Midtown East,Manhattan,Transformer,422,3.745,5.722,2935.365,+717.903,-328.504
6,Subway,1,162,Midtown East,Manhattan,Tree,422,3.639,5.560,2265.742,+1387.526,-641.077
7,Taxi trips,5,161,Midtown Center,Manhattan,Tree,430,3.329,8.556,123.946,+60.591,+24.061
8,Taxi trips,2,132,JFK Airport,Queens,Tree,430,3.299,8.478,127.961,+84.122,+27.260
9,Subway,5,186,Penn Station/Madison Sq West,Manhattan,Tree,422,3.289,5.026,2286.463,+1003.115,+219.945


**Findings.** The largest geographic contributions to total forecast error occur primarily in **high-activity Subway and Taxi-trip zones**.

Eight of the top 12 surfaces are Subway forecasts in Manhattan activity centers such as **Garment District, Midtown East, and Penn Station / Madison Sq West**. The remaining four are Taxi-trip surfaces led by **JFK Airport** and Midtown Center.

The largest burden comes from **Subway h=5 in Garment District**, which contributes **4.548% of that job's total absolute error** with an error concentration of **6.949×**. Despite that large contribution, the champion still improves on the weekly benchmark by approximately **1,307.575 riders of MAE**.

JFK Airport Taxi trips at h=5 shows the same distinction: the zone contributes **3.752% of job error** with a **9.640× error concentration**, while the Tree champion still improves on the benchmark by about **72.428 trips of MAE**.

High error burden therefore identifies **where absolute forecast error accumulates**, not necessarily where the champion performs poorly relative to the benchmark. Local forecast difficulty and systemwide error contribution are related but distinct properties.

**4.6.1.9.2E — Inspect the single largest geographic error-burden surface.**

A surface can contribute heavily to total MAE for two very different reasons: the model may be unusually unreliable there, or the underlying mobility series may simply be large and variable enough that even a comparatively strong forecast accumulates substantial absolute error.

The error-burden ranking above identifies the supported Target × Horizon × Taxi Zone surface that contributes the largest share of its forecasting job's absolute error. This diagnostic opens that surface and compares the observed mobility with the frozen champion, strongest pre-holdout runner-up, and Previous week · same Daypart benchmark.

The accompanying diagnostics report both **how much of the job's error budget the surface consumes** and whether the champion still improves on the benchmark. This distinguishes **systemic error importance** from **comparative forecast failure**.

Deeper explanation of why particular conditions create forecast error belongs in notebook 4.6.2.

In [82]:
# ---------------------------------------------------------------------
# 4.6.1.9.2E — Inspect the largest geographic error-burden surface
# ---------------------------------------------------------------------

highest_burden_surface = trouble_df.iloc[0].copy()

burden_metric = str(highest_burden_surface["metric"])
burden_horizon = int(highest_burden_surface["horizon"])
burden_zone_id = int(highest_burden_surface[ZONE_ID_COLUMN])
burden_zone_name = str(highest_burden_surface[ZONE_COLUMN])
burden_borough = str(highest_burden_surface[BOROUGH_COLUMN])

burden_series_df = load_holdout_inspection_rows(
    metric=burden_metric,
    horizon=burden_horizon,
    taxi_zone_id=burden_zone_id,
).copy()


# ---------------------------------------------------------------------
# Reconcile the plotted series to the zone-level burden surface
# ---------------------------------------------------------------------
# The chart is selected because of total error contribution, not because the
# champion necessarily performs poorly relative to the benchmark.

actual = burden_series_df["actual"].to_numpy(dtype=float)
champion_pred = burden_series_df[
    "champion_system_prediction"
].to_numpy(dtype=float)
runner_pred = burden_series_df[
    "runner_up_system_prediction"
].to_numpy(dtype=float)
benchmark_pred = burden_series_df[
    "benchmark_prediction"
].to_numpy(dtype=float)

champion_mae = float(np.mean(np.abs(actual - champion_pred)))
runner_mae = float(np.mean(np.abs(actual - runner_pred)))
benchmark_mae = float(np.mean(np.abs(actual - benchmark_pred)))
mae_advantage = benchmark_mae - champion_mae

if not np.isclose(
    champion_mae,
    float(highest_burden_surface["champion_mae"]),
    rtol=1e-6,
    atol=1e-6,
):
    raise RuntimeError(
        "Reconstructed champion MAE does not match the ranked zone surface."
    )

if not np.isclose(
    benchmark_mae,
    float(highest_burden_surface["benchmark_mae"]),
    rtol=1e-6,
    atol=1e-6,
):
    raise RuntimeError(
        "Reconstructed benchmark MAE does not match the ranked zone surface."
    )

if not np.isclose(
    mae_advantage,
    float(highest_burden_surface["mae_advantage"]),
    rtol=1e-6,
    atol=1e-6,
):
    raise RuntimeError(
        "Reconstructed MAE advantage does not match the ranked zone surface."
    )


# ---------------------------------------------------------------------
# Plot the high-burden surface in target-sequence order
# ---------------------------------------------------------------------
# Ordered observation sequence preserves the project's within-day mobility
# ordering while hover keeps the underlying calendar context visible.

x_values = burden_series_df["target_observation_sequence_id"]

customdata = np.column_stack(
    [
        burden_series_df["target_date"].dt.strftime("%Y-%m-%d"),
        burden_series_df["target_daypart"].astype(str),
        burden_series_df["target_temporal_bucket"].astype(str),
    ]
)

champion_family = FAMILY_LABELS[
    str(burden_series_df["champion_family"].iloc[0])
]
runner_family = FAMILY_LABELS[
    str(burden_series_df["runner_up_family"].iloc[0])
]

burden_fig = go.Figure()

series_specs = [
    ("actual", "Actual", BRAND_COLOR_SEQUENCE[0], 3.0, "solid"),
    (
        "champion_system_prediction",
        f"Champion · {champion_family}",
        BRAND_COLOR_SEQUENCE[1],
        2.5,
        "solid",
    ),
    (
        "runner_up_system_prediction",
        f"Runner-up · {runner_family}",
        BRAND_COLOR_SEQUENCE[2],
        2.0,
        "dash",
    ),
    (
        "benchmark_prediction",
        "Previous week · same Daypart",
        BRAND_COLOR_SEQUENCE[3],
        1.8,
        "dot",
    ),
]

for column, label, color, width, dash in series_specs:
    burden_fig.add_trace(
        go.Scatter(
            x=x_values,
            y=burden_series_df[column],
            mode="lines",
            name=label,
            line={"color": color, "width": width, "dash": dash},
            customdata=customdata,
            hovertemplate=(
                f"<b>{label}</b><br>"
                "Date: %{customdata[0]}<br>"
                "Daypart: %{customdata[1]}<br>"
                "Temporal bucket: %{customdata[2]}<br>"
                "Value: %{y:,.3f}"
                "<extra></extra>"
            ),
        )
    )

apply_branding(burden_fig)

burden_fig.update_layout(
    title=(
        "What the largest geographic error-burden surface looks like"
        f"<br><sup>{burden_zone_name} · {burden_borough} · "
        f"{TARGET_LABELS[burden_metric]} · h={burden_horizon}</sup>"
    ),
    xaxis_title="Ordered holdout target position",
    yaxis_title=TARGET_LABELS[burden_metric],
    hovermode="x unified",
    height=560,
    margin={"l": 70, "r": 35, "t": 105, "b": 105},
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.16,
        "yanchor": "top",
    },
)

burden_fig.add_annotation(
    x=0.99,
    y=0.98,
    xref="paper",
    yref="paper",
    xanchor="right",
    yanchor="top",
    align="left",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.88)",
    bordercolor="rgba(70,90,95,0.25)",
    borderwidth=1,
    borderpad=6,
    text=(
        "<b>Surface diagnostics</b><br>"
        f"Job error burden: {highest_burden_surface['error_burden_pct']:.3f}%<br>"
        f"Forecast share: {highest_burden_surface['forecast_share_pct']:.3f}%<br>"
        "Error concentration: "
        f"{highest_burden_surface['error_concentration_ratio']:.3f}×<br>"
        f"Champion MAE: {champion_mae:,.3f}<br>"
        f"Runner-up MAE: {runner_mae:,.3f}<br>"
        f"Benchmark MAE: {benchmark_mae:,.3f}<br>"
        f"MAE advantage: {mae_advantage:+,.3f}"
    ),
)

burden_fig.show()

**Findings.** The largest geographic error-burden surface is **Subway h=5 in Garment District**, where the Tree champion contributes **4.548% of the job's total absolute error** from only **0.654% of forecasts**. Its **6.949× error concentration** shows that this zone contributes far more error than its forecast volume alone would suggest.

The time series also shows why high error burden should not be interpreted as model failure. Subway ridership in Garment District is both **high-volume and strongly cyclical**, with repeated swings from roughly 5–10k riders to peaks above 60–70k. Small timing or amplitude misses around these large peaks accumulate substantial absolute error even when the model tracks the overall pattern well.

The Tree champion remains the strongest forecast on this surface. Its MAE is **3,161.636 riders**, compared with **3,774.667** for the Transformer runner-up and **4,469.211** for the weekly benchmark. That corresponds to a **+1,307.575-rider MAE advantage over the benchmark**.

This surface therefore represents **systemically important forecast error without comparative model failure**: it contributes heavily to total MAE because the underlying mobility series is large and variable, while the selected champion still materially improves on both the runner-up and the simple weekly benchmark.

### 4.6.1.9.3 Freeze the Downstream Handoff and Close the Tournament

The final blocks freeze the authoritative forecast-evaluation artifacts and summarize the completed tournament. The handoff preserves the selection-clean champion system together with unseen accuracy, runner-up robustness, geographic reliability, and difficult-surface evidence so 4.6.2 can explain the system without rebuilding or reopening model selection.

**4.6.1.9.3A — Freeze the completed forecast-evaluation handoff.**

The forecasting tournament now has four distinct layers of evidence: the validation-selected champion mapping, unseen-holdout performance, champion-versus-runner-up robustness, and geographic reliability diagnostics.

This final cell does not create another forecast or select another model. It records the authoritative 4.6.1 artifacts, verifies that the 15-job champion mapping is still selection-clean, and freezes the evidence that 4.6.2 will use to investigate forecast errors and feature drivers.

Chapter 4.6.2 should therefore explain the frozen forecasting system rather than reconstruct or reopen its tournament.

In [78]:
# ---------------------------------------------------------------------
# 4.6.1.9.3A — Freeze the forecast-evaluation handoff
# ---------------------------------------------------------------------

FORECAST_EVALUATION_HANDOFF_PATH = (
    FINAL_DIR
    / "forecast_evaluation_handoff_manifest.parquet"
)


# ---------------------------------------------------------------------
# 1) Define only the authoritative artifacts downstream analysis needs
# ---------------------------------------------------------------------

evaluation_artifacts = [
    {
        "artifact_id":
            "champion_registry",
        "path":
            CHAMPION_REGISTRY_PATH,
        "purpose":
            "Frozen validation-selected family for all 15 jobs",
    },
    {
        "artifact_id":
            "champion_holdout_predictions",
        "path":
            FINAL_HOLDOUT_CHAMPION_PREDICTION_PATH,
        "purpose":
            "Row-level unseen-holdout champion forecasts",
    },
    {
        "artifact_id":
            "champion_holdout_scorecard",
        "path":
            FINAL_HOLDOUT_SCORECARD_PATH,
        "purpose":
            "15-job unseen-holdout accuracy and benchmark skill",
    },
    {
        "artifact_id":
            "validation_holdout_generalization",
        "path":
            FINAL_HOLDOUT_GENERALIZATION_PATH,
        "purpose":
            "Validation-to-holdout transfer for frozen champions",
    },
    {
        "artifact_id":
            "pre_holdout_runner_up_registry",
        "path":
            RUNNER_UP_REGISTRY_PATH,
        "purpose":
            "One strongest pre-holdout alternative per champion job",
    },
    {
        "artifact_id":
            "champion_runner_up_holdout_rows",
        "path":
            RUNNER_UP_HOLDOUT_COMPARISON_PATH,
        "purpose":
            "Row-level champion / runner-up / benchmark comparison",
    },
    {
        "artifact_id":
            "champion_runner_up_holdout_scorecard",
        "path":
            RUNNER_UP_HOLDOUT_SCORECARD_PATH,
        "purpose":
            "15-job champion-versus-runner-up holdout point estimates",
    },
    {
        "artifact_id":
            "champion_runner_up_robustness",
        "path":
            RUNNER_UP_ROBUSTNESS_FINAL_PATH,
        "purpose":
            "Weekly-block uncertainty and practical-equivalence evidence",
    },
    {
        "artifact_id":
            "final_champion_board",
        "path":
            FINAL_CHAMPION_BOARD_PATH,
        "purpose":
            "Reader-facing 5 × 3 champion summary",
    },
    {
        "artifact_id":
            "zone_holdout_reliability",
        "path":
            FORECAST_ZONE_RELIABILITY_PATH,
        "purpose":
            "Taxi Zone MAE, benchmark comparison, and error-burden diagnostics",
    },
    {
        "artifact_id":
            "troublesome_zone_surfaces",
        "path":
            TROUBLESOME_SURFACE_PATH,
        "purpose":
            "Ranked support-qualified geographic error-burden surfaces",
    },
]


# ---------------------------------------------------------------------
# 2) Validate physical readability cheaply from Parquet metadata
# ---------------------------------------------------------------------

handoff_records = []

for artifact in evaluation_artifacts:
    path = Path(
        artifact["path"]
    )

    if not path.exists():
        raise FileNotFoundError(
            "Missing 4.6.1 handoff artifact: "
            f"{path}"
        )

    parquet_file = pq.ParquetFile(
        path
    )

    handoff_records.append({
        "artifact_id":
            artifact["artifact_id"],
        "path":
            str(path),
        "rows":
            int(
                parquet_file
                .metadata
                .num_rows
            ),
        "columns":
            int(
                len(
                    parquet_file
                    .schema_arrow
                    .names
                )
            ),
        "purpose":
            artifact["purpose"],
    })

forecast_evaluation_handoff_df = (
    pd.DataFrame(
        handoff_records
    )
)


# ---------------------------------------------------------------------
# 3) Re-prove that holdout evidence never changed model selection
# ---------------------------------------------------------------------

handoff_champion_df = (
    pd.read_parquet(
        CHAMPION_REGISTRY_PATH
    )
)

require_columns(
    handoff_champion_df,
    [
        "metric",
        "horizon",
        "champion_family",
        "champion_frozen",
        "holdout_used_for_selection",
    ],
    "forecast-evaluation champion handoff",
)

if len(handoff_champion_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Champion handoff registry does not "
        "contain exactly 15 jobs."
    )

if not handoff_champion_df[
    "champion_frozen"
].fillna(False).all():
    raise RuntimeError(
        "At least one downstream champion "
        "is not frozen."
    )

selection_clean_jobs = int(
    (
        ~handoff_champion_df[
            "holdout_used_for_selection"
        ].astype(bool)
    ).sum()
)

if selection_clean_jobs != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Holdout evidence appears in the "
        "champion-selection contract."
    )

expected_family_counts = {
    "tree": 8,
    "transformer": 4,
    "neural": 3,
}

if (
    handoff_champion_df[
        "champion_family"
    ]
    .value_counts()
    .to_dict()
    != expected_family_counts
):
    raise RuntimeError(
        "The final handoff no longer reproduces "
        "the frozen 8 Tree / 4 Transformer / "
        "3 Neural system."
    )


# ---------------------------------------------------------------------
# 4) Publish one compact directory of the completed evaluation evidence
# ---------------------------------------------------------------------

atomic_write_parquet(
    forecast_evaluation_handoff_df,
    FORECAST_EVALUATION_HANDOFF_PATH,
)

display(
    forecast_evaluation_handoff_df[
        [
            "artifact_id",
            "rows",
            "columns",
            "purpose",
        ]
    ]
    .rename(
        columns={
            "artifact_id":
                "Artifact",
            "rows":
                "Rows",
            "columns":
                "Columns",
            "purpose":
                "Downstream purpose",
        }
    )
)

print(
    "PASS — 4.6.1 forecast-evaluation "
    "handoff is frozen.\n"
    f"  Champion jobs:        "
    f"{EXPECTED_JOB_COUNT}\n"
    f"  Selection-clean jobs: "
    f"{selection_clean_jobs}\n"
    f"  Handoff artifacts:    "
    f"{len(forecast_evaluation_handoff_df)}\n"
    f"  Manifest:             "
    f"{FORECAST_EVALUATION_HANDOFF_PATH}",
    flush=True,
)

,Artifact,Rows,Columns,Downstream purpose
0,champion_registry,15,17,Frozen validation-selected family for all 15 jobs
1,champion_holdout_predictions,1427340,20,Row-level unseen-holdout champion forecasts
2,champion_holdout_scorecard,15,16,15-job unseen-holdout accuracy and benchmark s...
3,validation_holdout_generalization,15,25,Validation-to-holdout transfer for frozen cham...
4,pre_holdout_runner_up_registry,15,28,One strongest pre-holdout alternative per cham...
5,champion_runner_up_holdout_rows,1421436,20,Row-level champion / runner-up / benchmark com...
6,champion_runner_up_holdout_scorecard,15,23,15-job champion-versus-runner-up holdout point...
7,champion_runner_up_robustness,15,34,Weekly-block uncertainty and practical-equival...
8,final_champion_board,15,30,Reader-facing 5 × 3 champion summary
9,zone_holdout_reliability,3345,23,Target × Horizon × Taxi Zone reliability diagn...


PASS — 4.6.1 forecast-evaluation handoff is frozen.
  Champion jobs:        15
  Selection-clean jobs: 15
  Handoff artifacts:    11
  Manifest:             pipeline_data/4.6.1.final_tables/forecast_evaluation_handoff_manifest.parquet


**Findings.** The forecast-evaluation handoff preserves one complete and selection-clean **15-job champion system** together with its unseen-holdout, runner-up-robustness, geographic benchmark-comparison, and error-burden evidence.

All 15 champions remain frozen from ordinary validation, and **none uses final-holdout evidence for selection**. The downstream composition therefore remains **8 Tree, 4 Transformer, and 3 Neural** jobs.

The geographic handoff now keeps two distinct kinds of evidence: local reliability diagnostics such as benchmark skill and Relative MAE remain available for audit, while native-unit MAE, benchmark advantage, and error-burden measures identify which areas actually contribute materially to system error.

Notebook 4.6.2 can therefore investigate **where, when, and why the frozen forecasting system succeeds or fails** without reopening the model-family tournament or confusing locally extreme percentage errors with systemwide error contribution.

**4.6.1.9.3B — Close the notebook with the forecasting tournament map.**

A traditional single-elimination bracket would misrepresent this tournament. The four model families did not eliminate one another in a fixed sequence. Instead, every Target × Horizon formed its own four-family contest, the complete forecasting systems were compared on one shared validation population, and uncertainty determined whether the point leader had actually earned a clear practical win.

This final diagram separates the two phases that matter most:

- **Selection** uses ordinary validation to freeze the champion mapping.
- **Evaluation** begins only after that mapping is frozen and asks whether the selected system generalizes, remains competitive with its strongest alternative, and behaves reliably across time and geography.

The project-specific 8 Tree / 4 Transformer / 3 Neural result is therefore one example of the broader tournament process rather than the process itself.

In [79]:
# ---------------------------------------------------------------------
# 4.6.1.9.3B — Summarize who won the forecasting tournament and why
# ---------------------------------------------------------------------

TOURNAMENT_ROBUSTNESS_PATH = Path(
    "pipeline_data/4.6.1.final_tables/"
    "champion_runner_up_holdout_robustness.parquet"
)

TOURNAMENT_TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway",
}

TOURNAMENT_HORIZONS = [1, 2, 5]

# These seven jobs survived the primary uncertainty screen and therefore
# required the frozen absolute-bias tie-break. Keeping this roster explicit
# makes the final tournament summary auditable rather than inferred from prose.
NEAR_TIE_JOB_KEYS = {
    ("taxi_avg_trip_speed", 1),
    ("taxi_avg_trip_speed", 2),
    ("taxi_avg_trip_speed", 5),
    ("fhvhv_avg_trip_speed", 1),
    ("fhvhv_avg_trip_speed", 2),
    ("fhvhv_avg_trip_speed", 5),
    ("subway_ridership", 2),
}


# ---------------------------------------------------------------------
# 1) Recover the frozen 15-job champion result from its authoritative registry
# ---------------------------------------------------------------------

tournament_registry_df = pd.read_parquet(
    CHAMPION_REGISTRY_PATH,
    columns=["metric", "horizon", "champion_family"],
).copy()

tournament_registry_df["horizon"] = (
    tournament_registry_df["horizon"].astype(int)
)

if len(tournament_registry_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Tournament summary requires exactly 15 frozen champion jobs."
    )

if tournament_registry_df.duplicated(["metric", "horizon"]).any():
    raise RuntimeError(
        "Champion registry contains duplicate Target × Horizon jobs."
    )

observed_job_keys = set(
    zip(
        tournament_registry_df["metric"].astype(str),
        tournament_registry_df["horizon"].astype(int),
    )
)

expected_job_keys = {
    (metric, horizon)
    for metric in TOURNAMENT_TARGET_LABELS
    for horizon in TOURNAMENT_HORIZONS
}

if observed_job_keys != expected_job_keys:
    raise RuntimeError(
        "Champion registry does not reproduce the expected 5 × 3 job surface."
    )

tournament_registry_df["job_key"] = list(
    zip(
        tournament_registry_df["metric"].astype(str),
        tournament_registry_df["horizon"].astype(int),
    )
)

primary_winner_df = tournament_registry_df.loc[
    ~tournament_registry_df["job_key"].isin(NEAR_TIE_JOB_KEYS)
].copy()

bias_winner_df = tournament_registry_df.loc[
    tournament_registry_df["job_key"].isin(NEAR_TIE_JOB_KEYS)
].copy()

if len(primary_winner_df) != 8 or len(bias_winner_df) != 7:
    raise RuntimeError(
        "Frozen tournament no longer reproduces the expected "
        "8 clear-primary / 7 bias-resolved split."
    )

# All eight jobs fully settled by the primary screen were Tree wins.
if not primary_winner_df["champion_family"].eq("tree").all():
    raise RuntimeError(
        "Primary-winner roster changed from the frozen tournament result."
    )

bias_counts = (
    bias_winner_df["champion_family"]
    .value_counts()
    .to_dict()
)

if bias_counts != {"transformer": 4, "neural": 3}:
    raise RuntimeError(
        "Bias-resolved winner roster changed from the frozen tournament result."
    )

champion_counts = (
    tournament_registry_df["champion_family"]
    .value_counts()
    .reindex(FAMILY_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)

expected_champion_counts = {
    "classical": 0,
    "tree": 8,
    "neural": 3,
    "transformer": 4,
}

if champion_counts != expected_champion_counts:
    raise RuntimeError(
        "Frozen champion counts no longer match the 8/4/3 tournament result."
    )


def summarize_family_jobs(family):
    """
    Return a compact Target × Horizon ownership summary for one family.

    Grouping horizons within each target keeps the result readable while still
    showing exactly which forecasting jobs the family owns.
    """
    family_df = tournament_registry_df.loc[
        tournament_registry_df["champion_family"].eq(family)
    ]

    parts = []

    for metric, target_label in TOURNAMENT_TARGET_LABELS.items():
        horizons = sorted(
            family_df.loc[
                family_df["metric"].eq(metric),
                "horizon",
            ].astype(int).tolist()
        )

        if horizons:
            horizon_text = "/".join(
                f"h={horizon}" for horizon in horizons
            )
            parts.append(f"{target_label} {horizon_text}")

    return " · ".join(parts) if parts else "No champion jobs"


tree_jobs_text = summarize_family_jobs("tree")
transformer_jobs_text = summarize_family_jobs("transformer")
neural_jobs_text = summarize_family_jobs("neural")


# ---------------------------------------------------------------------
# 2) Recover the completed unseen runner-up robustness result
# ---------------------------------------------------------------------
# This is NOT used to change the frozen champions. It only tells the reader how
# strongly those validation-selected winners separated from second place later.

robustness_df = pd.read_parquet(
    TOURNAMENT_ROBUSTNESS_PATH,
    columns=[
        "metric",
        "horizon",
        "comparison_status",
    ],
).copy()

if len(robustness_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Tournament closeout requires exactly 15 holdout robustness jobs."
    )

robustness_df["status_key"] = (
    robustness_df["comparison_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

robustness_counts = (
    robustness_df["status_key"]
    .value_counts()
    .to_dict()
)

expected_statuses = {
    "champion_clear_win",
    "runner_up_clear_win",
    "practical_equivalence",
    "uncertain_difference",
}

unexpected_statuses = (
    set(robustness_counts) - expected_statuses
)

if unexpected_statuses:
    raise RuntimeError(
        "Unexpected holdout robustness classifications: "
        f"{sorted(unexpected_statuses)}"
    )

champion_clear_n = robustness_counts.get(
    "champion_clear_win", 0
)
runner_clear_n = robustness_counts.get(
    "runner_up_clear_win", 0
)
equivalent_n = robustness_counts.get(
    "practical_equivalence", 0
)
uncertain_n = robustness_counts.get(
    "uncertain_difference", 0
)

if (
    champion_clear_n
    + runner_clear_n
    + equivalent_n
    + uncertain_n
    != EXPECTED_JOB_COUNT
):
    raise RuntimeError(
        "Holdout robustness classifications do not sum to 15 jobs."
    )

runner_reversal_df = robustness_df.loc[
    robustness_df["status_key"].eq("runner_up_clear_win")
]

if len(runner_reversal_df) == 1:
    reversal_row = runner_reversal_df.iloc[0]
    reversal_label = (
        f"{TOURNAMENT_TARGET_LABELS[str(reversal_row['metric'])]} "
        f"h={int(reversal_row['horizon'])}"
    )
else:
    reversal_label = f"{len(runner_reversal_df)} jobs"


# ---------------------------------------------------------------------
# 3) Small drawing helpers
# ---------------------------------------------------------------------

def add_result_card(
    fig,
    y0,
    y1,
    title,
    body,
    *,
    accent=None,
    fill="white",
):
    """
    Add one full-width evidence card.

    The figure is intentionally vertical because Deepnote's notebook viewport
    is too narrow for a multi-column tournament bracket with explanatory text.
    """
    accent = accent or BRAND_COLORS["dark_teal"]

    fig.add_shape(
        type="rect",
        x0=0.06,
        x1=0.95,
        y0=y0,
        y1=y1,
        xref="x",
        yref="y",
        fillcolor=fill,
        line=dict(
            color="rgba(35, 74, 82, 0.22)",
            width=1.1,
        ),
        layer="below",
    )

    fig.add_shape(
        type="rect",
        x0=0.06,
        x1=0.075,
        y0=y0,
        y1=y1,
        xref="x",
        yref="y",
        fillcolor=accent,
        line=dict(width=0),
        layer="below",
    )

    fig.add_annotation(
        x=0.105,
        y=y1 - 0.025,
        xref="x",
        yref="y",
        text=f"<b>{title}</b>",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        align="left",
        font=dict(size=11),
    )

    fig.add_annotation(
        x=0.105,
        y=y1 - 0.048,
        xref="x",
        yref="y",
        text=body,
        showarrow=False,
        xanchor="left",
        yanchor="top",
        align="left",
        font=dict(size=9.5),
    )


def add_down_arrow(fig, y0, y1):
    """Connect result stages without introducing another Plotly trace."""
    fig.add_annotation(
        x=0.505,
        y=y1,
        ax=0.505,
        ay=y0,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        text="",
        showarrow=True,
        arrowhead=2,
        arrowsize=0.9,
        arrowwidth=1.3,
        arrowcolor="rgba(35, 74, 82, 0.45)",
    )


# ---------------------------------------------------------------------
# 4) Draw the tournament as an outcome story, not another methodology plan
# ---------------------------------------------------------------------

fig = go.Figure()

# Primary evidence decided more than half of the tournament immediately.
add_result_card(
    fig,
    0.765,
    0.895,
    "1 · Primary evidence settled 8 of the 15 jobs",
    (
        "Coverage-adjusted system MAE was tested with the paired weekly "
        "bootstrap and ±1% practical margin.<br>"
        "<b>Result: Tree won all 8 clear-primary jobs.</b>"
    ),
    accent=FAMILY_COLORS["tree"],
)

add_down_arrow(fig, 0.755, 0.725)

# The seven unresolved jobs were all settled by the first frozen tie-break.
add_result_card(
    fig,
    0.585,
    0.715,
    "2 · Absolute bias resolved all 7 near ties",
    (
        "<b>Neural won 3</b> — Taxi speed h=1 / h=2 / h=5.<br>"
        "<b>Transformer won 4</b> — FHVHV speed h=1 / h=2 / h=5 "
        "and Subway h=2."
    ),
    accent=BRAND_COLORS["seafoam"],
)

add_down_arrow(fig, 0.575, 0.545)

# Show the actual final ownership rather than merely saying champions froze.
add_result_card(
    fig,
    0.375,
    0.535,
    "3 · The frozen 15-job system had three winning families",
    (
        f"<b>Tree {champion_counts['tree']}</b> — {tree_jobs_text}<br>"
        f"<b>Transformer {champion_counts['transformer']}</b> — "
        f"{transformer_jobs_text}<br>"
        f"<b>Neural {champion_counts['neural']}</b> — {neural_jobs_text}<br>"
        "<b>Classical 0</b> — no final champion jobs."
    ),
    accent=BRAND_COLORS["dark_teal"],
    fill="rgba(237, 246, 249, 0.50)",
)


# ---------------------------------------------------------------------
# 5) Make the holdout boundary explicit
# ---------------------------------------------------------------------

fig.add_shape(
    type="line",
    x0=0.06,
    x1=0.95,
    y0=0.325,
    y1=0.325,
    xref="x",
    yref="y",
    line=dict(
        color="rgba(35, 74, 82, 0.36)",
        width=1.2,
        dash="dash",
    ),
)

fig.add_annotation(
    x=0.505,
    y=0.325,
    xref="x",
    yref="y",
    text=(
        "<b>CHAMPIONS FROZEN</b>"
        " · unseen holdout tests the result but cannot reselect it"
    ),
    showarrow=False,
    bgcolor="white",
    borderpad=3,
    font=dict(size=9.5),
    xanchor="center",
    yanchor="middle",
)

add_down_arrow(fig, 0.363, 0.292)


# ---------------------------------------------------------------------
# 6) Close with what actually happened on unseen data
# ---------------------------------------------------------------------

add_result_card(
    fig,
    0.105,
    0.285,
    "4 · Unseen holdout confirmed baseline value, but not every ranking",
    (
        "<b>All 15 frozen champions beat Previous week · same Daypart.</b> "
        "Benchmark skill improved from validation for 14 of 15 jobs.<br>"
        f"Against the strongest pre-holdout runner-up: "
        f"<b>{champion_clear_n} clear champion wins</b> · "
        f"{equivalent_n} practical equivalence · "
        f"{uncertain_n} uncertain differences · "
        f"<b>{runner_clear_n} clear runner-up win</b> "
        f"({reversal_label})."
    ),
    accent=BRAND_COLORS["terracotta"],
    fill="rgba(255, 221, 210, 0.24)",
)

fig.add_annotation(
    x=0.06,
    y=0.055,
    xref="x",
    yref="y",
    text=(
        "<b>What advances to 4.6.2:</b> the frozen champion system plus its "
        "reliability evidence — 4.6.2 explains <i>why</i> performance varies."
    ),
    showarrow=False,
    xanchor="left",
    yanchor="middle",
    align="left",
    font=dict(size=9.5),
)


# ---------------------------------------------------------------------
# 7) Brand and fit the synthesis to the notebook viewport
# ---------------------------------------------------------------------

apply_branding(fig)

fig.update_layout(
    title=dict(
        text=(
            "Who won the forecasting tournament — and why?"
            "<br><sup>"
            "Eight jobs were clear validation wins; seven needed the bias "
            "tie-break; holdout then tested the frozen result."
            "</sup>"
        ),
        x=0.06,
        xanchor="left",
        y=0.965,
        yanchor="top",
    ),
    height=480,
    margin=dict(l=16, r=16, t=48, b=8),
    showlegend=False,
    plot_bgcolor="white",
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    fixedrange=True,
)

fig.update_yaxes(
    visible=False,
    range=[0.02, 0.94],
    fixedrange=True,
)

fig.show()

**Findings.** The completed tournament produces a deliberately mixed forecasting system rather than one universal model-family winner.

Primary validation evidence settles **8 of the 15 jobs**, and **Tree wins all eight**. The remaining **7 near ties** are resolved by the frozen absolute-bias tie-break: **Neural wins all three Taxi-speed horizons**, while **Transformer wins all three FHVHV-speed horizons plus Subway h=2**. The final champion system is therefore **8 Tree / 4 Transformer / 3 Neural**, with Classical owning no final jobs.

The unseen holdout strongly supports the system's value relative to the operational benchmark: **all 15 frozen champions beat Previous week · same Daypart**. The runner-up audit is intentionally more demanding and shows **4 clear champion wins, 1 practical equivalence, 9 uncertain differences, and 1 clear runner-up win at Subway h=2**.

The tournament therefore closes with two distinct conclusions: the selected system generalizes successfully against the frozen benchmark, while superiority over the strongest advanced alternative is much more job-specific. That distinction is preserved for the reliability and failure analysis in 4.6.2 rather than being used to reselect models.

### Section Summary

Section 9 makes the frozen forecasting system inspectable from complementary perspectives: champion ownership, validation-to-holdout transfer, runner-up robustness, horizon behavior, individual forecast traces, geographic benchmark breadth, and geographic error burden. The revised spatial diagnostics make an important distinction explicit: **local proportional fragility is not the same as contribution to total MAE**. Sparse zones can produce extreme percentage errors while high-activity Subway and Taxi surfaces account for much more of the absolute-error budget. The section then freezes that complete evaluation evidence for downstream error analysis without reopening model selection.

## Close

This notebook selects, executes, and validates one coordinated forecasting system across **15 Target × Horizon jobs**.

Ordinary validation freezes a mixed **8 Tree / 4 Transformer / 3 Neural** champion system before the final holdout is opened. Primary evidence settles **8 jobs**, all won by Tree, while absolute bias resolves the remaining **7 near ties**, assigning the three Taxi-speed jobs to Neural and the three FHVHV-speed jobs plus Subway h=2 to Transformer.

The frozen system generalizes well to the unseen January–March 2026 holdout. Champion advanced-model coverage is **100%** across the **1,421,436 benchmark-supported evaluation rows**, and all **15 champions outperform Previous week · same Daypart**, with holdout benchmark skill ranging from approximately **19.2% to 37.7%**. Benchmark skill improves from validation to holdout for **14 of 15 jobs**, while Relative MAE increases only modestly; the six speed forecasts are especially stable.

The pre-specified runner-up audit gives a more cautious view of ranking stability. Its paired weekly bootstrap produces **4 clear champion wins, 1 practical equivalence, 9 uncertain differences, and 1 clear runner-up win**. Six uncertain comparisons can reverse direction across plausible weekly reweightings, while three retain the champion's directional advantage without reliably clearing the +1% practical threshold. The single clear reversal is **Subway h=2**, where Tree outperforms the frozen Transformer champion. Because the holdout is a robustness audit rather than a second selection tournament, the frozen champion mapping remains unchanged.

Geographic performance is broad but not universal. **100% of adequately supported Taxi Zones beat the weekly benchmark at every horizon for both Taxi speed and FHVHV speed**. FHVHV trips remain positive across **87.2%–93.4%** of supported zones, while Taxi trips range from **73.3% to 86.8%**. Subway remains broad at h=1 (**95.4%**) and h=5 (**96.1%**) but falls to **70.6% at h=2**, reinforcing that horizon's broader fragility.

The native-unit and error-burden diagnostics add an important qualification. The zones with the most extreme percentage-based reliability statistics are not necessarily the zones driving overall MAE. The largest absolute-error burdens instead concentrate in high-activity Subway surfaces in Midtown Manhattan and Taxi-trip surfaces such as JFK Airport. Garment District Subway h=5 alone contributes **4.548% of its job's absolute-error budget**, while still beating the weekly benchmark substantially. Geographic difficulty therefore has at least two distinct forms: **local proportional fragility** and **systemically consequential absolute error**.

The forecasting tournament is therefore complete. The final handoff preserves **11 authoritative artifacts** covering the selection-clean champion registry, unseen-holdout forecasts, validation-to-holdout transfer, runner-up robustness, geographic reliability, and difficult forecasting surfaces. Notebook **4.6.2 Analyze Forecast Errors and Feature Drivers** can now investigate where, when, and why this frozen forecasting system succeeds or fails without reopening model selection.

## Final File Inventory

The files below are the authoritative materialized outputs of notebook 4.6.1. Together they preserve the frozen champion mapping, unseen-holdout forecasts and scorecards, validation-to-holdout transfer evidence, runner-up robustness audit, and geographic reliability / error-burden surfaces required by downstream forecast-error analysis.

The inventory is built from the frozen forecast-evaluation handoff rather than by scanning the project directory. File sizes are reported only to make the handoff easier to audit operationally; the analytical contract is defined by the saved Parquet artifacts themselves.

In [80]:
# ---------------------------------------------------------------------
# Final File Inventory — authoritative 4.6.1 outputs
# ---------------------------------------------------------------------
# 9.3A already froze the exact downstream handoff. Reuse that manifest rather
# than searching the filesystem and accidentally including checkpoints,
# temporary transport files, or unrelated intermediate artifacts.

if len(forecast_evaluation_handoff_df) != 11:
    raise RuntimeError(
        "Final inventory expected exactly 11 frozen handoff artifacts."
    )

inventory_records = []

for row in forecast_evaluation_handoff_df.to_dict("records"):
    artifact_path = Path(row["path"])

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Frozen handoff artifact is missing: {artifact_path}"
        )

    # Row counts were already validated in 9.3A. The filesystem stat adds only
    # lightweight operational context and does not rescan the Parquet contents.
    inventory_records.append({
        "Artifact": row["artifact_id"],
        "File": artifact_path.name,
        "Rows": int(row["rows"]),
        "Size (MB)": artifact_path.stat().st_size / (1024 ** 2),
        "Purpose": row["purpose"],
    })


# The manifest itself is tiny, so read it normally rather than introducing a
# metadata-specific API merely to count eleven rows.
manifest_inventory_df = pd.read_parquet(
    FORECAST_EVALUATION_HANDOFF_PATH,
    columns=["artifact_id"],
)

inventory_records.append({
    "Artifact": "forecast_evaluation_handoff_manifest",
    "File": Path(FORECAST_EVALUATION_HANDOFF_PATH).name,
    "Rows": len(manifest_inventory_df),
    "Size (MB)": (
        Path(FORECAST_EVALUATION_HANDOFF_PATH).stat().st_size
        / (1024 ** 2)
    ),
    "Purpose": "Directory of the authoritative 4.6.1 evaluation handoff",
})



final_file_inventory_df = pd.DataFrame(inventory_records)

display(
    final_file_inventory_df.assign(
        **{"Size (MB)": final_file_inventory_df["Size (MB)"].round(3)}
    )
)

print(
    "PASS — final 4.6.1 file inventory is complete.\n"
    f"  Analytical artifacts: {len(forecast_evaluation_handoff_df)}\n"
    f"  Handoff manifest:     1\n"
    f"  Total listed files:   {len(final_file_inventory_df)}",
    flush=True,
)

,Artifact,File,Rows,Size (MB),Purpose
0,champion_registry,forecasting_champion_registry.parquet,15,0.014,Frozen validation-selected family for all 15 jobs
1,champion_holdout_predictions,champion_final_holdout_predictions.parquet,1427340,33.483,Row-level unseen-holdout champion forecasts
2,champion_holdout_scorecard,champion_final_holdout_scorecard.parquet,15,0.012,15-job unseen-holdout accuracy and benchmark s...
3,validation_holdout_generalization,champion_validation_holdout_generalization.par...,15,0.020,Validation-to-holdout transfer for frozen cham...
4,pre_holdout_runner_up_registry,frozen_pre_holdout_runner_up_registry.parquet,15,0.024,One strongest pre-holdout alternative per cham...
5,champion_runner_up_holdout_rows,champion_runner_up_holdout_rows.parquet,1421436,50.502,Row-level champion / runner-up / benchmark com...
6,champion_runner_up_holdout_scorecard,champion_runner_up_holdout_scorecard.parquet,15,0.019,15-job champion-versus-runner-up holdout point...
7,champion_runner_up_robustness,champion_runner_up_holdout_robustness.parquet,15,0.028,Weekly-block uncertainty and practical-equival...
8,final_champion_board,final_champion_board.parquet,15,0.023,Reader-facing 5 × 3 champion summary
9,zone_holdout_reliability,champion_holdout_zone_reliability.parquet,3345,0.291,Target × Horizon × Taxi Zone reliability diagn...


PASS — final 4.6.1 file inventory is complete.
  Analytical artifacts: 11
  Handoff manifest:     1
  Total listed files:   12


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>